In [2]:
#import statements 

from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import roc_auc_score
from __future__ import annotations
from typing import Sequence
import warnings
from __future__ import annotations
from typing import Sequence
import warnings




In [3]:
#Data Preprocessing and Cleaning

#School level predictors to merge onto student-level data 
SCHOOL_PREDICTORS = [
    "RATCMP1",
    "RATCMP2",
    "EDUSHORT",
]

# Raw bullying items used as dependent variables 
BULLYING_RAW_ITEMS = [
    "ST038Q04NA",  #verbal bullying 
    "ST038Q05NA",  #threat bullying 
    "ST038Q06NA",  #belongings taken/destroyed; physical bullying
    "ST038Q07NA",  #hit/pushed around; physical bullying
    "ST038Q08NA",  #relational bullying
]

PRIMARY_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]

def merge_country_data(
    stu_raw: pd.DataFrame,
    sch_raw: pd.DataFrame,
    country: str,
    school_predictors: Sequence[str] = SCHOOL_PREDICTORS,
) -> tuple[pd.DataFrame, dict]:
    """
    Filter student and school files to one PISA education system
    and merge selected school-level predictors onto students.

    Returns
    -------
    merged : pd.DataFrame
        Country-specific student-level dataset with school variables attached.

    merge_stats : dict
        Diagnostics describing the merge.
    """

    required_student = {"CNT", "CNTSCHID"}
    required_school = {"CNT", "CNTSCHID"}

    missing_stu = required_student - set(stu_raw.columns)
    missing_sch = required_school - set(sch_raw.columns)

    if missing_stu:
        raise ValueError(
            f"Student file is missing required merge columns: {sorted(missing_stu)}"
        )

    if missing_sch:
        raise ValueError(
            f"School file is missing required merge columns: {sorted(missing_sch)}"
        )

    #Filter both files to the same PISA education system
    stu = stu_raw.loc[stu_raw["CNT"] == country].copy()
    sch = sch_raw.loc[sch_raw["CNT"] == country].copy()

    if stu.empty:
        raise ValueError(f"{country}: no student records found.")

    if sch.empty:
        raise ValueError(f"{country}: no school records found.")

    #Determine which requested school predictors are available
    available_school_predictors = [
        p for p in school_predictors
        if p in sch.columns
    ]

    school_cols = [
        "CNT",
        "CNTSCHID",
        *available_school_predictors,
    ]

    sch = sch[school_cols].copy()

    #Each school should appear only once in the school dataset
    duplicate_schools = sch.duplicated(
        subset=["CNT", "CNTSCHID"],
        keep=False,
    )

    if duplicate_schools.any():
        examples = (
            sch.loc[
                duplicate_schools,
                ["CNT", "CNTSCHID"],
            ]
            .drop_duplicates()
            .head(10)
            .to_dict("records")
        )

        raise ValueError(
            f"{country}: duplicate school keys found. "
            f"Examples: {examples}"
        )

    n_students_before = len(stu)

    merged = stu.merge(
        sch,
        on=["CNT", "CNTSCHID"],
        how="left",
        validate="many_to_one",
        indicator="_school_merge",
    )

    #Merging school information must not change the number of students
    if len(merged) != n_students_before:
        raise RuntimeError(
            f"{country}: student row count changed during school merge "
            f"({n_students_before} -> {len(merged)})."
        )

    matched = merged["_school_merge"].eq("both")
    match_rate = float(matched.mean())

    if match_rate < 1.0:
        warnings.warn(
            f"{country}: only {match_rate:.2%} of students matched "
            "to a school record.",
            RuntimeWarning,
        )

    #Merge-quality indicator for auditing only
    merged["SCHOOL_MATCHED"] = matched.astype(int)

    merged = merged.drop(
        columns="_school_merge"
    )

    merge_stats = {
        "Country": country,
        "N_students": len(stu),
        "N_schools": sch["CNTSCHID"].nunique(),
        "N_students_school_matched": int(matched.sum()),
        "School_match_rate": match_rate,
        "School_predictors_available": available_school_predictors,
    }

    return merged, merge_stats

def recode_gender(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Recode PISA ST004D01T:

        1 = Female -> 0
        2 = Male   -> 1

    Missing values remain missing.
    """

    df = df.copy()

    if "ST004D01T" not in df.columns:
        return df

    observed = set(
        pd.to_numeric(
            df["ST004D01T"],
            errors="coerce",
        )
        .dropna()
        .unique()
    )

    unexpected = observed - {1, 2}

    if unexpected:
        raise ValueError(
            "Unexpected ST004D01T values found: "
            f"{sorted(unexpected)}"
        )

    df["ST004D01T"] = (
        df["ST004D01T"]
        .map({
            1: 0,
            2: 1,
        })
        .astype(float)
    )

    return df

def binarize_bullying_item(
    s: pd.Series,
    item_name: str,
) -> pd.Series:
    """
    Convert one PISA ST038 bullying-frequency item to:

        0 = Never or almost never
            OR a few times a year

        1 = A few times a month
            OR once a week or more

        NaN = Missing response

    Expected PISA response codes:

        1 = Never or almost never
        2 = A few times a year
        3 = A few times a month
        4 = Once a week or more
    """

    numeric = pd.to_numeric(
        s,
        errors="coerce",
    )

    nonmissing = numeric.dropna()

    invalid = nonmissing[
        ~nonmissing.isin([1, 2, 3, 4])
    ]

    if not invalid.empty:
        raise ValueError(
            f"{item_name}: unexpected response codes found: "
            f"{sorted(invalid.unique().tolist())}"
        )

    out = pd.Series(
        np.nan,
        index=s.index,
        dtype=float,
        name=item_name,
    )

    valid_mask = numeric.notna()

    out.loc[valid_mask] = (
        numeric.loc[valid_mask]
        .isin([3, 4])
        .astype(float)
    )

    return out

def combine_binary_or(
    *items: pd.Series,
) -> pd.Series:
    """
    Combine binary indicators using logical OR while
    preserving uncertainty caused by missing responses.

    Rules
    -----
    Any observed positive -> 1

    All component items observed and all negative -> 0

    Otherwise -> NaN

    Examples
    --------
    Q06 = 1,   Q07 = NaN -> 1
    Q06 = 0,   Q07 = 0   -> 0
    Q06 = 0,   Q07 = NaN -> NaN
    Q06 = NaN, Q07 = NaN -> NaN
    """

    frame = pd.concat(
        items,
        axis=1,
    )

    out = pd.Series(
        np.nan,
        index=frame.index,
        dtype=float,
    )

    any_positive = frame.eq(1).any(
        axis=1
    )

    all_observed = frame.notna().all(
        axis=1
    )

    all_negative = (
        all_observed
        & frame.eq(0).all(axis=1)
    )

    out.loc[any_positive] = 1.0

    out.loc[all_negative] = 0.0

    return out

def construct_bullying_outcomes(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construct the four study-specific binary bullying outcomes.

    BULLY_VERBAL:
        ST038Q04NA

    BULLY_THREAT:
        ST038Q05NA

    BULLY_PHYSICAL:
        ST038Q06NA OR ST038Q07NA

    BULLY_RELATIONAL:
        ST038Q08NA

    All outcomes use the threshold:
        at least a few times per month.
    """

    df = df.copy()

    required = set(
        BULLYING_RAW_ITEMS
    )

    missing = (
        required
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            "Required bullying questionnaire variables are missing: "
            f"{sorted(missing)}"
        )

    b04 = binarize_bullying_item(
        df["ST038Q04NA"],
        "ST038Q04NA",
    )

    b05 = binarize_bullying_item(
        df["ST038Q05NA"],
        "ST038Q05NA",
    )

    b06 = binarize_bullying_item(
        df["ST038Q06NA"],
        "ST038Q06NA",
    )

    b07 = binarize_bullying_item(
        df["ST038Q07NA"],
        "ST038Q07NA",
    )

    b08 = binarize_bullying_item(
        df["ST038Q08NA"],
        "ST038Q08NA",
    )

    # Verbal bullying:
    # made fun of by other students
    df["BULLY_VERBAL"] = b04

    # Threatening victimization:
    # kept separate because modality is ambiguous
    df["BULLY_THREAT"] = b05

    # Physical bullying:
    # belongings taken/destroyed OR hit/pushed
    df["BULLY_PHYSICAL"] = combine_binary_or(
        b06,
        b07,
    )

    # Relational bullying:
    # nasty rumours
    df["BULLY_RELATIONAL"] = b08

    return df

def predictor_availability_table(
    df: pd.DataFrame,
    predictors: Sequence[str],
    country: str,
) -> pd.DataFrame:
    """
    Create one audit row for every predictor in one country.

    This happens BEFORE modeling or imputation.
    """

    rows = []

    for feature in predictors:

        if feature not in df.columns:

            rows.append(
                {
                    "Country": country,
                    "Feature": feature,
                    "Present": False,
                    "N_total": len(df),
                    "N_nonmissing": 0,
                    "Missing_pct": np.nan,
                    "N_unique": 0,
                    "Usable": False,
                }
            )

            continue

        s = df[feature]

        n_nonmissing = int(
            s.notna().sum()
        )

        n_unique = int(
            s.dropna().nunique()
        )

        missing_pct = float(
            s.isna().mean() * 100
        )

        rows.append(
            {
                "Country": country,
                "Feature": feature,
                "Present": True,
                "N_total": len(df),
                "N_nonmissing": n_nonmissing,
                "Missing_pct": missing_pct,
                "N_unique": n_unique,
                "Usable": (
                    n_nonmissing > 0
                    and n_unique > 1
                ),
            }
        )

    return pd.DataFrame(rows)

def build_country_analysis_frame(
    stu_raw: pd.DataFrame,
    sch_raw: pd.DataFrame,
    country: str,
    predictors: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Build the canonical country-level analysis dataset.

    IMPORTANT:
    - no imputation
    - no scaling
    - no model fitting
    - no dropping students based on outcome missingness

    Those decisions belong to later stages.
    """

    df, merge_stats = merge_country_data(
        stu_raw=stu_raw,
        sch_raw=sch_raw,
        country=country,
    )

    df = recode_gender(
        df
    )

    df = construct_bullying_outcomes(
        df
    )

    # Raw bullying questions and dependent variables
    # must never accidentally enter X as predictors.
    forbidden_predictors = (
        set(BULLYING_RAW_ITEMS)
        | set(PRIMARY_OUTCOMES)
    )

    overlap = (
        set(predictors)
        & forbidden_predictors
    )

    if overlap:
        raise ValueError(
            "Outcome/bullying-item variables were accidentally included "
            f"as predictors: {sorted(overlap)}"
        )

    # Audit availability BEFORE silently removing predictors
    availability = predictor_availability_table(
        df=df,
        predictors=predictors,
        country=country,
    )

    available_predictors = [
        p
        for p in predictors
        if p in df.columns
    ]

    # Metadata required later for:
    # - identifying systems/students
    # - school-grouped cross-validation
    # - survey weighting
    metadata_cols = [
        c
        for c in [
            "CNT",
            "CNTSTUID",
            "CNTSCHID",
            "W_FSTUWT",
            "SCHOOL_MATCHED",
        ]
        if c in df.columns
    ]

    # Preserve all PISA student replicate weights
    replicate_weights = [
        c
        for c in df.columns
        if c.startswith("W_FSTURWT")
    ]

    replicate_weights = sorted(
        replicate_weights
    )

    # Keep raw bullying items for auditing/reproducibility.
    # These are NEVER predictors.
    raw_bullying_cols = [
        c
        for c in BULLYING_RAW_ITEMS
        if c in df.columns
    ]

    outcome_cols = [
        c
        for c in PRIMARY_OUTCOMES
        if c in df.columns
    ]

    keep_cols = list(
        dict.fromkeys(
            metadata_cols
            + replicate_weights
            + available_predictors
            + raw_bullying_cols
            + outcome_cols
        )
    )

    analysis_df = df[
        keep_cols
    ].copy()

    return (
        analysis_df,
        availability,
        merge_stats,
    )

In [4]:
# ============================================================
# 1. Paths
# ============================================================

STU_PATH = Path("../data/raw/Student Data.sav")
SCH_PATH = Path("../data/raw/School Data.sav")

PROCESSED_DIR = Path("../data/processed_final")
AUDIT_DIR = Path("../results/audit")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Predictor configuration
# ============================================================

# These are the substantive predictors apart from achievement
# plausible values.
BASE_PREDICTORS = [

    # ----------------------------
    # Individual-level predictors
    # ----------------------------
    "ST004D01T",   # Gender
    "AGE",         # Age
    "GRADE",       # Grade
    "BSMJ",        # Expected occupational status

    "JOYREAD",     # Joy of reading
    "SCREADCOMP",  # Reading self-concept: competence
    "SCREADDIFF",  # Reading self-concept: difficulty

    "COMPETE",     # Competitiveness
    "WORKMAST",    # Work mastery orientation
    "GFOFAIL",     # General fear of failure
    "EUDMO",       # Sense of meaning in life / eudaimonia
    "RESILIENCE",  # Resilience
    "MASTGOAL",    # Mastery goal orientation

    "ST185Q01HA",  # Life has meaning/purpose
    "ST184Q01HA",  # Growth mindset

    "SWBP",        # Subjective well-being

    "ST207Q01HA",
    "ST207Q02HA",
    "ST207Q03HA",
    "ST207Q04HA",
    "ST207Q05HA",  # Likes when someone stands up for bullied students


    # ----------------------------
    # Proximal-level predictors
    # ----------------------------
    "REPEAT",      # Grade repetition history
    "UNDREM",      # Understanding and remembering
    "METASUM",     # Summarizing
    "METASPAM",    # Assessing credibility


    # ----------------------------
    # Microsystem-level predictors
    # ----------------------------
    "EMOSUPS",     # Parental emotional support
    "DURECEC",     # Duration in early childhood education/care
    "BELONG",      # School belonging
    "PERCOMP",     # Perceived school competitiveness
    "PERCOOP",     # Perceived school cooperation
    "ATTLNACT",    # Attitudes toward learning activities
    "DISCLIMA",    # Disciplinary climate
    "TEACHSUP",    # Teacher support
    "DIRINS",      # Teacher-directed instruction
    "PERFEED",     # Perceived feedback
    "STIMREAD",    # Teacher stimulation of reading
    "ADAPTIVITY",  # Adaptation of instruction
    "TEACHINT",    # Perceived teacher interest
    "ST206Q01HA",  # Value of cooperation

    # Parent questionnaire item.
    # We preserve it for the availability audit for now.
    # It may NOT survive into the common final predictor universe.
    "PA006Q09TA",


    # ----------------------------
    # Macro / exosystem candidates
    # ----------------------------
    "ESCS",        # Socioeconomic status index
    "EDUSHORT",    # Educational resource shortage
    "RATCMP1",     # Computers per student
    "RATCMP2",     # Computers connected to Internet
]


# ============================================================
# 3. Preserve ALL plausible values for later decisions
# ============================================================

PV_READ = [
    f"PV{i}READ"
    for i in range(1, 11)
]

PV_MATH = [
    f"PV{i}MATH"
    for i in range(1, 11)
]

PV_SCIE = [
    f"PV{i}SCIE"
    for i in range(1, 11)
]

PLAUSIBLE_VALUES = (
    PV_READ
    + PV_MATH
    + PV_SCIE
)


# These are everything we want preserved in the country
# analysis files for now.
#
# This does NOT mean all 30 PVs will enter the same model.
PREDICTORS_TO_PRESERVE = (
    BASE_PREDICTORS
    + PLAUSIBLE_VALUES
)


# ============================================================
# 4. Load PISA files once
# ============================================================

print("Loading raw PISA student file...")

stu_raw, stu_meta = pyreadstat.read_sav(
    STU_PATH,
    apply_value_formats=False,
)

print(
    f"Student data loaded: "
    f"{stu_raw.shape[0]:,} rows × "
    f"{stu_raw.shape[1]:,} columns"
)


print("\nLoading raw PISA school file...")

sch_raw, sch_meta = pyreadstat.read_sav(
    SCH_PATH,
    apply_value_formats=False,
)

print(
    f"School data loaded: "
    f"{sch_raw.shape[0]:,} rows × "
    f"{sch_raw.shape[1]:,} columns"
)


# ============================================================
# 5. Identify participating PISA systems
# ============================================================

SYSTEM_CODES = sorted(
    stu_raw["CNT"]
    .dropna()
    .unique()
    .tolist()
)

print(
    f"\nSystems found in student file: "
    f"{len(SYSTEM_CODES)}"
)

print(SYSTEM_CODES)


# ============================================================
# 6. Weighted descriptive helper
# ============================================================

def weighted_mean(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    """
    Compute a weighted mean using observations with:
      - nonmissing values
      - nonmissing weights
      - finite weights
      - positive weights
    """

    values = pd.to_numeric(
        values,
        errors="coerce",
    )

    weights = pd.to_numeric(
        weights,
        errors="coerce",
    )

    valid = (
        values.notna()
        & weights.notna()
        & np.isfinite(weights)
        & (weights > 0)
    )

    if valid.sum() == 0:
        return np.nan

    return float(
        np.average(
            values.loc[valid],
            weights=weights.loc[valid],
        )
    )


# ============================================================
# 7. Outcome audit
# ============================================================

def outcome_audit_table(
    df: pd.DataFrame,
    country: str,
) -> pd.DataFrame:
    """
    Create descriptive diagnostics for the five study outcomes.

    Binary outcomes:
        N observed
        N missing
        N positive
        N negative
        unweighted prevalence
        weighted prevalence

    BEINGBULLIED:
        N observed
        N missing
        unweighted mean
        weighted mean
    """

    rows = []

    binary_outcomes = [
        "BULLY_VERBAL",
        "BULLY_THREAT",
        "BULLY_PHYSICAL",
        "BULLY_RELATIONAL",
    ]

    n_total = len(df)

    has_weight = (
        "W_FSTUWT"
        in df.columns
    )


    # ----------------------------
    # BEINGBULLIED
    # ----------------------------

    if "BEINGBULLIED" in df.columns:

        y = pd.to_numeric(
            df["BEINGBULLIED"],
            errors="coerce",
        )

        n_observed = int(
            y.notna().sum()
        )

        row = {
            "Country": country,
            "Outcome": "BEINGBULLIED",
            "Outcome_type": "continuous",
            "N_total": n_total,
            "N_observed": n_observed,
            "N_missing": int(
                y.isna().sum()
            ),
            "N_positive": np.nan,
            "N_negative": np.nan,
            "Unweighted_prevalence": np.nan,
            "Weighted_prevalence": np.nan,
            "Unweighted_mean": (
                float(y.mean())
                if n_observed > 0
                else np.nan
            ),
            "Weighted_mean": (
                weighted_mean(
                    y,
                    df["W_FSTUWT"],
                )
                if has_weight
                else np.nan
            ),
        }

        rows.append(row)


    # ----------------------------
    # Binary subtype outcomes
    # ----------------------------

    for outcome in binary_outcomes:

        if outcome not in df.columns:
            continue

        y = pd.to_numeric(
            df[outcome],
            errors="coerce",
        )

        observed = y.notna()

        n_observed = int(
            observed.sum()
        )

        n_positive = int(
            y.eq(1).sum()
        )

        n_negative = int(
            y.eq(0).sum()
        )

        unweighted_prevalence = (
            float(
                y.loc[observed].mean()
            )
            if n_observed > 0
            else np.nan
        )

        weighted_prevalence = (
            weighted_mean(
                y,
                df["W_FSTUWT"],
            )
            if has_weight
            else np.nan
        )

        row = {
            "Country": country,
            "Outcome": outcome,
            "Outcome_type": "binary",
            "N_total": n_total,
            "N_observed": n_observed,
            "N_missing": int(
                y.isna().sum()
            ),
            "N_positive": n_positive,
            "N_negative": n_negative,
            "Unweighted_prevalence": unweighted_prevalence,
            "Weighted_prevalence": weighted_prevalence,
            "Unweighted_mean": np.nan,
            "Weighted_mean": np.nan,
        }

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 8. Raw ST038 response audit
# ============================================================

def raw_bullying_item_audit(
    df: pd.DataFrame,
    country: str,
) -> pd.DataFrame:
    """
    Count responses 1-4 and missing values for each raw
    bullying questionnaire item.
    """

    rows = []

    for item in BULLYING_RAW_ITEMS:

        if item not in df.columns:
            continue

        s = pd.to_numeric(
            df[item],
            errors="coerce",
        )

        row = {
            "Country": country,
            "Item": item,
            "N_total": len(s),
            "N_response_1": int(
                s.eq(1).sum()
            ),
            "N_response_2": int(
                s.eq(2).sum()
            ),
            "N_response_3": int(
                s.eq(3).sum()
            ),
            "N_response_4": int(
                s.eq(4).sum()
            ),
            "N_missing": int(
                s.isna().sum()
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 9. Run preprocessing for every system
# ============================================================

analysis_frames = {}

availability_rows = []
merge_rows = []
outcome_rows = []
raw_item_rows = []
error_rows = []


for country in SYSTEM_CODES:

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"Processing {country}"
    )

    try:

        (
            analysis_df,
            availability_df,
            merge_stats,
        ) = build_country_analysis_frame(
            stu_raw=stu_raw,
            sch_raw=sch_raw,
            country=country,
            predictors=PREDICTORS_TO_PRESERVE,
        )


        # Keep in memory for this notebook session
        analysis_frames[country] = (
            analysis_df
        )


        # Save canonical processed country file locally
        processed_path = (
            PROCESSED_DIR
            / f"{country}_analysis.pkl"
        )

        analysis_df.to_pickle(
            processed_path
        )


        # ----------------------------
        # Predictor availability
        # ----------------------------

        availability_rows.append(
            availability_df
        )


        # ----------------------------
        # Merge diagnostics
        # ----------------------------

        merge_rows.append(
            merge_stats
        )


        # ----------------------------
        # Outcome diagnostics
        # ----------------------------

        outcome_rows.append(
            outcome_audit_table(
                analysis_df,
                country,
            )
        )


        # ----------------------------
        # Raw questionnaire diagnostics
        # ----------------------------

        raw_item_rows.append(
            raw_bullying_item_audit(
                analysis_df,
                country,
            )
        )


        print(
            f"{country}: "
            f"{analysis_df.shape[0]:,} students × "
            f"{analysis_df.shape[1]:,} retained columns"
        )


    except Exception as e:

        error_rows.append(
            {
                "Country": country,
                "Error_type": type(e).__name__,
                "Error": str(e),
            }
        )

        print(
            f"ERROR — {country}: "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# 10. Combine audit results
# ============================================================

merge_audit = pd.DataFrame(
    merge_rows
)

predictor_availability = (
    pd.concat(
        availability_rows,
        ignore_index=True,
    )
    if availability_rows
    else pd.DataFrame()
)

outcome_audit = (
    pd.concat(
        outcome_rows,
        ignore_index=True,
    )
    if outcome_rows
    else pd.DataFrame()
)

raw_item_audit = (
    pd.concat(
        raw_item_rows,
        ignore_index=True,
    )
    if raw_item_rows
    else pd.DataFrame()
)

error_log = pd.DataFrame(
    error_rows,
    columns=[
        "Country",
        "Error_type",
        "Error",
    ],
)


# ============================================================
# 11. Save audit tables
# ============================================================

merge_audit.to_csv(
    AUDIT_DIR / "merge_audit.csv",
    index=False,
)

predictor_availability.to_csv(
    AUDIT_DIR / "predictor_availability.csv",
    index=False,
)

outcome_audit.to_csv(
    AUDIT_DIR / "outcome_audit.csv",
    index=False,
)

raw_item_audit.to_csv(
    AUDIT_DIR / "raw_bullying_item_audit.csv",
    index=False,
)

error_log.to_csv(
    AUDIT_DIR / "preprocessing_errors.csv",
    index=False,
)


# ============================================================
# 12. Final run summary
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "PREPROCESSING COMPLETE"
)

print(
    "=" * 60
)

print(
    f"Systems discovered: "
    f"{len(SYSTEM_CODES)}"
)

print(
    f"Systems successfully processed: "
    f"{len(analysis_frames)}"
)

print(
    f"Systems with errors: "
    f"{len(error_log)}"
)

print(
    "\nSaved audit tables:"
)

print(
    " - results/audit/merge_audit.csv"
)

print(
    " - results/audit/predictor_availability.csv"
)

print(
    " - results/audit/outcome_audit.csv"
)

print(
    " - results/audit/raw_bullying_item_audit.csv"
)

print(
    " - results/audit/preprocessing_errors.csv"
)

print(
    "\nProcessed country datasets:"
)

print(
    " - data/processed_final/{COUNTRY}_analysis.pkl"
)

Loading raw PISA student file...
Student data loaded: 612,004 rows × 1,119 columns

Loading raw PISA school file...
School data loaded: 21,903 rows × 197 columns

Systems found in student file: 80
['ALB', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGR', 'BIH', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'COL', 'CRI', 'CZE', 'DEU', 'DNK', 'DOM', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GEO', 'GRC', 'HKG', 'HRV', 'HUN', 'IDN', 'IRL', 'ISL', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'KOR', 'KSV', 'LBN', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MEX', 'MKD', 'MLT', 'MNE', 'MYS', 'NLD', 'NOR', 'NZL', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'QAT', 'QAZ', 'QCI', 'QMR', 'QRT', 'ROU', 'RUS', 'SAU', 'SGP', 'SRB', 'SVK', 'SVN', 'SWE', 'TAP', 'THA', 'TUR', 'UKR', 'URY', 'USA', 'VNM']

Processing ALB
ALB: 6,359 students × 169 retained columns

Processing ARE
ARE: 19,277 students × 169 retained columns

Processing ARG
ARG: 11,975 students × 169 retained columns

Processing AUS
AUS: 14,273 students × 169 retained columns

Pr

In [5]:
# ============================================================
# SYSTEM + PREDICTOR ELIGIBILITY ANALYSIS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Load the audit files we just created
# ============================================================

AUDIT_DIR = Path("../results/audit")
ELIGIBILITY_DIR = Path("../results/eligibility")

ELIGIBILITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

merge_audit = pd.read_csv(
    AUDIT_DIR / "merge_audit.csv"
)

outcome_audit = pd.read_csv(
    AUDIT_DIR / "outcome_audit.csv"
)

predictor_availability = pd.read_csv(
    AUDIT_DIR / "predictor_availability.csv"
)


# ============================================================
# 2. Define the main PISA system universe
# ============================================================

RAW_SYSTEM_CODES = sorted(
    merge_audit["Country"]
    .dropna()
    .unique()
    .tolist()
)

# These are subnational PISA-adjudicated Russian regions.
# National Russia (RUS) is already represented separately,
# so these must not be counted as independent main systems.
EXCLUDED_SUBREGIONS = {
    "QMR": "Moscow Region (Russia)",
    "QRT": "Republic of Tatarstan (Russia)",
}

MAIN_SYSTEM_CODES = [
    c
    for c in RAW_SYSTEM_CODES
    if c not in EXCLUDED_SUBREGIONS
]


print("=" * 70)
print("SYSTEM UNIVERSE")
print("=" * 70)

print(
    "Raw CNT codes:",
    len(RAW_SYSTEM_CODES),
)

print(
    "Excluded adjudicated subregions:",
    list(EXCLUDED_SUBREGIONS.keys()),
)

print(
    "Main official systems represented in this dataset:",
    len(MAIN_SYSTEM_CODES),
)

print(
    "Cyprus (CYP) present in this file:",
    "CYP" in RAW_SYSTEM_CODES,
)


# ============================================================
# 3. Calculate outcome coverage for every system
# ============================================================

eligibility = outcome_audit[
    outcome_audit["Country"].isin(
        MAIN_SYSTEM_CODES
    )
].copy()

eligibility["Observed_pct"] = (
    eligibility["N_observed"]
    / eligibility["N_total"]
    * 100
)


# Has at least some usable outcome data
eligibility["Has_any_data"] = (
    eligibility["N_observed"] > 0
)


# Also calculate several possible minimum-coverage rules.
#
# IMPORTANT:
# We are NOT choosing one yet.
#
# These are only for seeing how sensitive our
# system sample would be to different rules.

for threshold in [50, 75, 90]:

    eligibility[
        f"Coverage_ge_{threshold}"
    ] = (
        eligibility["Observed_pct"]
        >= threshold
    )


eligibility.to_csv(
    ELIGIBILITY_DIR
    / "system_outcome_eligibility.csv",
    index=False,
)


# ============================================================
# 4. Compact counts for every outcome
# ============================================================

eligibility_counts = (
    eligibility
    .groupby(
        "Outcome",
        as_index=False,
    )
    .agg(
        N_systems_total=(
            "Country",
            "nunique",
        ),

        N_systems_any_data=(
            "Has_any_data",
            "sum",
        ),

        N_systems_ge_50pct=(
            "Coverage_ge_50",
            "sum",
        ),

        N_systems_ge_75pct=(
            "Coverage_ge_75",
            "sum",
        ),

        N_systems_ge_90pct=(
            "Coverage_ge_90",
            "sum",
        ),

        Min_observed_pct=(
            "Observed_pct",
            "min",
        ),

        Median_observed_pct=(
            "Observed_pct",
            "median",
        ),
    )
)


eligibility_counts.to_csv(
    ELIGIBILITY_DIR
    / "outcome_eligibility_counts.csv",
    index=False,
)


print("\n")
print("=" * 70)
print("OUTCOME ELIGIBILITY COUNTS")
print("=" * 70)

print(
    eligibility_counts.to_string(
        index=False
    )
)


# ============================================================
# 5. Identify low-coverage systems
# ============================================================

low_coverage = (
    eligibility[
        (
            eligibility["Has_any_data"]
        )
        &
        (
            eligibility["Observed_pct"]
            < 50
        )
    ]
    [
        [
            "Country",
            "Outcome",
            "N_total",
            "N_observed",
            "N_missing",
            "Observed_pct",
        ]
    ]
    .sort_values(
        [
            "Outcome",
            "Observed_pct",
        ]
    )
)


low_coverage.to_csv(
    ELIGIBILITY_DIR
    / "systems_below_50pct_outcome_coverage.csv",
    index=False,
)


print("\n")
print("=" * 70)
print("SYSTEMS WITH >0 BUT <50% OUTCOME COVERAGE")
print("=" * 70)

if low_coverage.empty:

    print(
        "None."
    )

else:

    print(
        low_coverage.to_string(
            index=False
        )
    )


# ============================================================
# 6. Build system sets for BEINGBULLIED and subtype analyses
# ============================================================

BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


def systems_for_outcome(
    outcome: str,
    minimum_coverage: float,
) -> set[str]:

    temp = eligibility[
        eligibility["Outcome"]
        == outcome
    ]

    if minimum_coverage == 0:

        temp = temp[
            temp["N_observed"] > 0
        ]

    else:

        temp = temp[
            temp["Observed_pct"]
            >= minimum_coverage
        ]

    return set(
        temp["Country"]
    )


def systems_for_all_subtypes(
    minimum_coverage: float,
) -> set[str]:

    sets = [
        systems_for_outcome(
            outcome,
            minimum_coverage,
        )
        for outcome
        in BINARY_OUTCOMES
    ]

    return set.intersection(
        *sets
    )


# ============================================================
# 7. Show sample-size sensitivity to outcome coverage rule
# ============================================================

system_set_rows = []


for threshold in [0, 50, 75, 90]:

    being_systems = systems_for_outcome(
        "BEINGBULLIED",
        threshold,
    )

    subtype_systems = systems_for_all_subtypes(
        threshold,
    )

    system_set_rows.append(
        {
            "Minimum_outcome_coverage_pct": threshold,
            "BEINGBULLIED_systems": len(
                being_systems
            ),
            "Subtype_common_systems": len(
                subtype_systems
            ),
        }
    )


system_set_summary = pd.DataFrame(
    system_set_rows
)


system_set_summary.to_csv(
    ELIGIBILITY_DIR
    / "system_set_sensitivity.csv",
    index=False,
)


print("\n")
print("=" * 70)
print("SYSTEM COUNT SENSITIVITY")
print("=" * 70)

print(
    system_set_summary.to_string(
        index=False
    )
)


# ============================================================
# 8. Predictor-summary helper
# ============================================================

def summarize_predictors(
    systems: set[str],
) -> pd.DataFrame:
    """
    Summarize predictor availability across a specified
    set of systems.

    This does NOT select predictors yet.
    """

    systems = sorted(
        systems
    )

    sub = predictor_availability[
        predictor_availability[
            "Country"
        ].isin(
            systems
        )
    ].copy()

    rows = []

    for feature, g in sub.groupby(
        "Feature"
    ):

        # Reindex ensures every expected system
        # is represented in the calculation.
        g = (
            g
            .set_index("Country")
            .reindex(systems)
        )

        missing_pct = pd.to_numeric(
            g["Missing_pct"],
            errors="coerce",
        )

        usable = (
            g["Usable"]
            .fillna(False)
            .astype(bool)
        )

        present = (
            g["Present"]
            .fillna(False)
            .astype(bool)
        )

        row = {
            "Feature": feature,

            "N_systems_expected": len(
                systems
            ),

            "N_systems_present": int(
                present.sum()
            ),

            "N_systems_usable": int(
                usable.sum()
            ),

            "N_systems_zero_observed": int(
                (
                    pd.to_numeric(
                        g["N_nonmissing"],
                        errors="coerce",
                    )
                    .fillna(0)
                    == 0
                ).sum()
            ),

            "Median_missing_pct": float(
                missing_pct.median()
            )
            if missing_pct.notna().any()
            else np.nan,

            "Max_missing_pct": float(
                missing_pct.max()
            )
            if missing_pct.notna().any()
            else np.nan,

            "Is_plausible_value": bool(
                feature.startswith("PV")
                and (
                    feature.endswith("READ")
                    or feature.endswith("MATH")
                    or feature.endswith("SCIE")
                )
            ),
        }


        # See how many systems satisfy several
        # candidate predictor-missingness limits.
        #
        # Again: these are NOT final decisions.

        for threshold in [
            20,
            30,
            40,
            50,
        ]:

            passes = (
                usable
                &
                missing_pct.le(
                    threshold
                )
            )

            row[
                f"N_systems_missing_le_{threshold}"
            ] = int(
                passes.sum()
            )

            row[
                f"All_systems_missing_le_{threshold}"
            ] = bool(
                passes.all()
            )


        rows.append(
            row
        )


    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "Is_plausible_value",
                "Feature",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 9. Predictor summaries using systems with ANY outcome data
# ============================================================

being_any_systems = systems_for_outcome(
    "BEINGBULLIED",
    0,
)

subtype_any_systems = systems_for_all_subtypes(
    0,
)


predictor_summary_being = summarize_predictors(
    being_any_systems
)

predictor_summary_subtypes = summarize_predictors(
    subtype_any_systems
)


predictor_summary_being.to_csv(
    ELIGIBILITY_DIR
    / "predictor_summary_BEINGBULLIED.csv",
    index=False,
)

predictor_summary_subtypes.to_csv(
    ELIGIBILITY_DIR
    / "predictor_summary_SUBTYPES.csv",
    index=False,
)


# ============================================================
# 10. Count candidate shared predictors under multiple rules
# ============================================================

threshold_rows = []


for family, summary_df in [
    (
        "BEINGBULLIED",
        predictor_summary_being,
    ),
    (
        "SUBTYPES",
        predictor_summary_subtypes,
    ),
]:

    # Handle achievement plausible values separately.
    non_pv = summary_df[
        ~summary_df[
            "Is_plausible_value"
        ]
    ].copy()

    for threshold in [
        20,
        30,
        40,
        50,
    ]:

        col = (
            f"All_systems_missing_le_{threshold}"
        )

        kept = (
            non_pv[
                non_pv[col]
            ]["Feature"]
            .tolist()
        )

        threshold_rows.append(
            {
                "Analysis_family": family,
                "Maximum_predictor_missing_pct": threshold,
                "N_shared_nonPV_predictors": len(
                    kept
                ),
                "Shared_predictors": "; ".join(
                    kept
                ),
            }
        )


predictor_threshold_sensitivity = (
    pd.DataFrame(
        threshold_rows
    )
)


predictor_threshold_sensitivity.to_csv(
    ELIGIBILITY_DIR
    / "predictor_threshold_sensitivity.csv",
    index=False,
)


print("\n")
print("=" * 70)
print("SHARED NON-PV PREDICTOR SENSITIVITY")
print("=" * 70)

print(
    predictor_threshold_sensitivity[
        [
            "Analysis_family",
            "Maximum_predictor_missing_pct",
            "N_shared_nonPV_predictors",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 11. Predictors that are completely unavailable somewhere
# ============================================================

problem_predictors_being = (
    predictor_summary_being[
        predictor_summary_being[
            "N_systems_zero_observed"
        ] > 0
    ]
    [
        [
            "Feature",
            "N_systems_expected",
            "N_systems_usable",
            "N_systems_zero_observed",
            "Median_missing_pct",
            "Max_missing_pct",
            "Is_plausible_value",
        ]
    ]
    .sort_values(
        [
            "N_systems_zero_observed",
            "Feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


problem_predictors_being.to_csv(
    ELIGIBILITY_DIR
    / "problem_predictors_BEINGBULLIED.csv",
    index=False,
)


problem_predictors_subtypes = (
    predictor_summary_subtypes[
        predictor_summary_subtypes[
            "N_systems_zero_observed"
        ] > 0
    ]
    [
        [
            "Feature",
            "N_systems_expected",
            "N_systems_usable",
            "N_systems_zero_observed",
            "Median_missing_pct",
            "Max_missing_pct",
            "Is_plausible_value",
        ]
    ]
    .sort_values(
        [
            "N_systems_zero_observed",
            "Feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


problem_predictors_subtypes.to_csv(
    ELIGIBILITY_DIR
    / "problem_predictors_SUBTYPES.csv",
    index=False,
)


# ============================================================
# 12. Final summary
# ============================================================

print("\n")
print("=" * 70)
print("ELIGIBILITY ANALYSIS COMPLETE")
print("=" * 70)

print(
    "Main represented systems:",
    len(MAIN_SYSTEM_CODES),
)

print(
    "BEINGBULLIED systems with any data:",
    len(being_any_systems),
)

print(
    "Systems with data for all four subtypes:",
    len(subtype_any_systems),
)

print(
    "\nOutputs saved to:"
)

print(
    "results/eligibility/"
)

SYSTEM UNIVERSE
Raw CNT codes: 80
Excluded adjudicated subregions: ['QMR', 'QRT']
Main official systems represented in this dataset: 78
Cyprus (CYP) present in this file: False


OUTCOME ELIGIBILITY COUNTS
         Outcome  N_systems_total  N_systems_any_data  N_systems_ge_50pct  N_systems_ge_75pct  N_systems_ge_90pct  Min_observed_pct  Median_observed_pct
    BEINGBULLIED               78                  74                  69                  53                  16               0.0            82.136513
  BULLY_PHYSICAL               78                  75                  70                  56                  18               0.0            83.662231
BULLY_RELATIONAL               78                  75                  70                  56                  18               0.0            83.770540
    BULLY_THREAT               78                  75                  70                  56                  18               0.0            83.803028
    BULLY_VERBAL             

In [6]:
# ============================================================
# PRINT THE IMPORTANT ELIGIBILITY RESULTS IN FULL
# ============================================================

print("\n" + "=" * 70)
print("1. SYSTEM COUNT SENSITIVITY")
print("=" * 70)
print(
    system_set_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("2. ALL SYSTEMS WITH <50% OUTCOME COVERAGE")
print("=" * 70)

if low_coverage.empty:
    print("None.")
else:
    print(
        low_coverage.to_string(
            index=False
        )
    )


print("\n" + "=" * 70)
print("3. SHARED NON-PV PREDICTOR SENSITIVITY")
print("=" * 70)

print(
    predictor_threshold_sensitivity[
        [
            "Analysis_family",
            "Maximum_predictor_missing_pct",
            "N_shared_nonPV_predictors",
            "Shared_predictors",
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 70)
print("4. COMPLETELY UNAVAILABLE PREDICTORS — BEINGBULLIED")
print("=" * 70)

if problem_predictors_being.empty:
    print("None.")
else:
    print(
        problem_predictors_being.to_string(
            index=False
        )
    )


print("\n" + "=" * 70)
print("5. COMPLETELY UNAVAILABLE PREDICTORS — SUBTYPES")
print("=" * 70)

if problem_predictors_subtypes.empty:
    print("None.")
else:
    print(
        problem_predictors_subtypes.to_string(
            index=False
        )
    )


1. SYSTEM COUNT SENSITIVITY
 Minimum_outcome_coverage_pct  BEINGBULLIED_systems  Subtype_common_systems
                            0                    74                      75
                           50                    69                      70
                           75                    53                      56
                           90                    16                      18

2. ALL SYSTEMS WITH <50% OUTCOME COVERAGE
Country          Outcome  N_total  N_observed  N_missing  Observed_pct
    DOM     BEINGBULLIED     5674        1397       4277     24.621079
    PAN     BEINGBULLIED     6270        1771       4499     28.245614
    MAR     BEINGBULLIED     6814        2378       4436     34.898738
    PER     BEINGBULLIED     6086        2161       3925     35.507723
    DEU     BEINGBULLIED     5451        2348       3103     43.074665
    DOM   BULLY_PHYSICAL     5674        1482       4192     26.119140
    PAN   BULLY_PHYSICAL     6270        1836      

In [7]:
# ============================================================
# AUDIT ORIGINAL SPSS MISSING CODES FOR BULLYING VARIABLES
# ============================================================

import numpy as np
import pandas as pd
import pyreadstat


# Systems we especially need to inspect
CHECK_SYSTEMS = [
    "DOM",
    "PAN",
    "MAR",
    "PER",
    "DEU",
    "KOR",
    "ISR",
    "LBN",
    "MKD",
]


CHECK_VARIABLES = [
    "CNT",
    "BEINGBULLIED",
    "ST038Q04NA",
    "ST038Q05NA",
    "ST038Q06NA",
    "ST038Q07NA",
    "ST038Q08NA",
]


# ------------------------------------------------------------
# 1. Reload ONLY these columns, but preserve SPSS user-missing
#    codes instead of automatically converting them to NaN.
# ------------------------------------------------------------

stu_missing_codes, missing_meta = pyreadstat.read_sav(
    STU_PATH,
    usecols=CHECK_VARIABLES,
    apply_value_formats=False,
    user_missing=True,
)


print(
    "Loaded:",
    stu_missing_codes.shape
)


# ------------------------------------------------------------
# 2. Show SPSS missing-code definitions
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SPSS MISSING-VALUE DEFINITIONS")
print("=" * 75)

missing_ranges = getattr(
    missing_meta,
    "missing_ranges",
    {},
)

for variable in CHECK_VARIABLES[1:]:

    print(
        f"\n{variable}"
    )

    print(
        "Missing definitions:",
        missing_ranges.get(
            variable,
            "None recorded",
        ),
    )


# ------------------------------------------------------------
# 3. Show value labels if present
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("VALUE LABELS")
print("=" * 75)

value_labels = getattr(
    missing_meta,
    "variable_value_labels",
    {},
)

for variable in CHECK_VARIABLES[1:]:

    print(
        f"\n{variable}:"
    )

    print(
        value_labels.get(
            variable,
            "No labels found",
        )
    )


# ------------------------------------------------------------
# 4. Examine raw response codes by system
# ------------------------------------------------------------

for country in CHECK_SYSTEMS:

    print("\n")
    print("=" * 75)
    print(country)
    print("=" * 75)

    sub = stu_missing_codes[
        stu_missing_codes["CNT"]
        == country
    ]

    print(
        "N students:",
        len(sub),
    )

    for variable in CHECK_VARIABLES[1:]:

        print(
            f"\n{variable}"
        )

        counts = (
            sub[variable]
            .value_counts(
                dropna=False
            )
            .sort_index()
        )

        print(
            counts.to_string()
        )

Loaded: (612004, 7)

SPSS MISSING-VALUE DEFINITIONS

BEINGBULLIED
Missing definitions: [{'lo': 95.0, 'hi': 99.0}]

ST038Q04NA
Missing definitions: [{'lo': 5.0, 'hi': 9.0}]

ST038Q05NA
Missing definitions: [{'lo': 5.0, 'hi': 9.0}]

ST038Q06NA
Missing definitions: [{'lo': 5.0, 'hi': 9.0}]

ST038Q07NA
Missing definitions: [{'lo': 5.0, 'hi': 9.0}]

ST038Q08NA
Missing definitions: [{'lo': 5.0, 'hi': 9.0}]

VALUE LABELS

BEINGBULLIED:
{95.0: 'Valid Skip', 97.0: 'Not Applicable', 98.0: 'Invalid', 99.0: 'No Response'}

ST038Q04NA:
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q05NA:
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q06NA:
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0

In [8]:
# ============================================================
# COMPACT MISSING-CODE AUDIT
# ============================================================

CHECK_SYSTEMS = [
    "DOM",
    "PAN",
    "MAR",
    "PER",
    "DEU",
    "KOR",
    "ISR",
    "LBN",
    "MKD",
]

CHECK_VARIABLES = [
    "BEINGBULLIED",
    "ST038Q04NA",
    "ST038Q05NA",
    "ST038Q06NA",
    "ST038Q07NA",
    "ST038Q08NA",
]


print("=" * 70)
print("VALUE LABELS")
print("=" * 70)

for variable in CHECK_VARIABLES:
    print(f"\n{variable}")
    print(
        missing_meta.variable_value_labels.get(
            variable,
            "No labels found"
        )
    )


print("\n" + "=" * 70)
print("MISSING-CODE COUNTS BY SYSTEM")
print("=" * 70)

for country in CHECK_SYSTEMS:

    print(f"\n--- {country} ---")

    sub = stu_missing_codes[
        stu_missing_codes["CNT"] == country
    ]

    for variable in CHECK_VARIABLES:

        counts = (
            sub[variable]
            .value_counts(dropna=False)
            .sort_index()
        )

        print(f"\n{variable}")
        print(counts.to_string())

VALUE LABELS

BEINGBULLIED
{95.0: 'Valid Skip', 97.0: 'Not Applicable', 98.0: 'Invalid', 99.0: 'No Response'}

ST038Q04NA
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q05NA
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q06NA
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q07NA
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0: 'A few times a month', 4.0: 'Once a week or more', 5.0: 'Valid Skip', 7.0: 'Not Applicable', 8.0: 'Invalid', 9.0: 'No Response'}

ST038Q08NA
{1.0: 'Never or almost never', 2.0: 'A few times a year', 3.0:

In [9]:
# ============================================================
# COMPACT MISSING-REASON SUMMARY
# ============================================================

CHECK_SYSTEMS = [
    "DOM",
    "PAN",
    "MAR",
    "PER",
    "DEU",
    "KOR",
    "ISR",
    "LBN",
    "MKD",
]

RAW_BULLY_ITEMS = [
    "ST038Q04NA",
    "ST038Q05NA",
    "ST038Q06NA",
    "ST038Q07NA",
    "ST038Q08NA",
]


# ============================================================
# 1. Raw bullying items
# ============================================================

raw_missing_rows = []

for country in CHECK_SYSTEMS:

    sub = stu_missing_codes[
        stu_missing_codes["CNT"] == country
    ]

    for item in RAW_BULLY_ITEMS:

        s = pd.to_numeric(
            sub[item],
            errors="coerce",
        )

        n_total = len(s)

        n_valid = int(
            s.isin([1, 2, 3, 4]).sum()
        )

        n_valid_skip = int(
            s.eq(5).sum()
        )

        n_not_applicable = int(
            s.eq(7).sum()
        )

        n_invalid = int(
            s.eq(8).sum()
        )

        n_no_response = int(
            s.eq(9).sum()
        )

        raw_missing_rows.append(
            {
                "Country": country,
                "Item": item,
                "N_total": n_total,

                "N_valid_1to4": n_valid,

                "Pct_valid_1to4": (
                    n_valid
                    / n_total
                    * 100
                ),

                "N_valid_skip_5": n_valid_skip,

                "Pct_valid_skip_5": (
                    n_valid_skip
                    / n_total
                    * 100
                ),

                "N_not_applicable_7": n_not_applicable,

                "Pct_not_applicable_7": (
                    n_not_applicable
                    / n_total
                    * 100
                ),

                "N_invalid_8": n_invalid,

                "Pct_invalid_8": (
                    n_invalid
                    / n_total
                    * 100
                ),

                "N_no_response_9": n_no_response,

                "Pct_no_response_9": (
                    n_no_response
                    / n_total
                    * 100
                ),
            }
        )


raw_missing_summary = pd.DataFrame(
    raw_missing_rows
)


# ============================================================
# 2. BEINGBULLIED
# ============================================================

being_missing_rows = []

for country in CHECK_SYSTEMS:

    sub = stu_missing_codes[
        stu_missing_codes["CNT"] == country
    ]

    s = pd.to_numeric(
        sub["BEINGBULLIED"],
        errors="coerce",
    )

    n_total = len(s)

    missing_codes = {
        95: "Valid Skip",
        97: "Not Applicable",
        98: "Invalid",
        99: "No Response",
    }

    missing_mask = s.isin(
        list(missing_codes.keys())
    )

    n_valid = int(
        (
            s.notna()
            & ~missing_mask
        ).sum()
    )

    being_missing_rows.append(
        {
            "Country": country,
            "N_total": n_total,

            "N_valid_score": n_valid,

            "Pct_valid_score": (
                n_valid
                / n_total
                * 100
            ),

            "N_valid_skip_95": int(
                s.eq(95).sum()
            ),

            "N_not_applicable_97": int(
                s.eq(97).sum()
            ),

            "N_invalid_98": int(
                s.eq(98).sum()
            ),

            "N_no_response_99": int(
                s.eq(99).sum()
            ),
        }
    )


being_missing_summary = pd.DataFrame(
    being_missing_rows
)


# ============================================================
# 3. Save
# ============================================================

raw_missing_summary.to_csv(
    ELIGIBILITY_DIR
    / "raw_bullying_missing_reasons.csv",
    index=False,
)

being_missing_summary.to_csv(
    ELIGIBILITY_DIR
    / "beingbullied_missing_reasons.csv",
    index=False,
)


# ============================================================
# 4. Print compact summaries
# ============================================================

print("\n" + "=" * 90)
print("RAW BULLYING ITEMS — MISSING REASONS")
print("=" * 90)

print(
    raw_missing_summary[
        [
            "Country",
            "Item",
            "Pct_valid_1to4",
            "Pct_valid_skip_5",
            "Pct_not_applicable_7",
            "Pct_invalid_8",
            "Pct_no_response_9",
        ]
    ]
    .round(2)
    .to_string(index=False)
)


print("\n" + "=" * 90)
print("BEINGBULLIED — MISSING REASONS")
print("=" * 90)

print(
    being_missing_summary
    .round(2)
    .to_string(index=False)
)


RAW BULLYING ITEMS — MISSING REASONS
Country       Item  Pct_valid_1to4  Pct_valid_skip_5  Pct_not_applicable_7  Pct_invalid_8  Pct_no_response_9
    DOM ST038Q04NA           26.31               0.0                   0.0            0.0              38.16
    DOM ST038Q05NA           26.10               0.0                   0.0            0.0              38.37
    DOM ST038Q06NA           26.89               0.0                   0.0            0.0              37.57
    DOM ST038Q07NA           25.96               0.0                   0.0            0.0              38.51
    DOM ST038Q08NA           26.17               0.0                   0.0            0.0              38.30
    PAN ST038Q04NA           29.86               0.0                   0.0            0.0              18.84
    PAN ST038Q05NA           29.25               0.0                   0.0            0.0              19.44
    PAN ST038Q06NA           29.46               0.0                   0.0            0.0 

In [10]:
# ============================================================
# FINAL MISSING-REASON AUDIT INCLUDING SYSTEM-MISSING NaN
# ============================================================

CHECK_SYSTEMS = [
    "DOM",
    "PAN",
    "MAR",
    "PER",
    "DEU",
    "KOR",
    "ISR",
    "LBN",
    "MKD",
]

RAW_BULLY_ITEMS = [
    "ST038Q04NA",
    "ST038Q05NA",
    "ST038Q06NA",
    "ST038Q07NA",
    "ST038Q08NA",
]


# ============================================================
# 1. Raw bullying items
# ============================================================

rows = []

for country in CHECK_SYSTEMS:

    sub = stu_missing_codes[
        stu_missing_codes["CNT"] == country
    ]

    for item in RAW_BULLY_ITEMS:

        s = pd.to_numeric(
            sub[item],
            errors="coerce",
        )

        n_total = len(s)

        counts = {
            "valid": int(
                s.isin([1, 2, 3, 4]).sum()
            ),
            "valid_skip": int(
                s.eq(5).sum()
            ),
            "not_applicable": int(
                s.eq(7).sum()
            ),
            "invalid": int(
                s.eq(8).sum()
            ),
            "no_response": int(
                s.eq(9).sum()
            ),
            "system_missing": int(
                s.isna().sum()
            ),
        }

        rows.append(
            {
                "Country": country,
                "Item": item,
                "N_total": n_total,

                "Pct_valid": (
                    counts["valid"]
                    / n_total
                    * 100
                ),

                "Pct_valid_skip": (
                    counts["valid_skip"]
                    / n_total
                    * 100
                ),

                "Pct_not_applicable": (
                    counts["not_applicable"]
                    / n_total
                    * 100
                ),

                "Pct_invalid": (
                    counts["invalid"]
                    / n_total
                    * 100
                ),

                "Pct_no_response": (
                    counts["no_response"]
                    / n_total
                    * 100
                ),

                "Pct_system_missing": (
                    counts["system_missing"]
                    / n_total
                    * 100
                ),
            }
        )


raw_missing_complete = pd.DataFrame(
    rows
)


# ============================================================
# 2. BEINGBULLIED
# ============================================================

rows = []

for country in CHECK_SYSTEMS:

    sub = stu_missing_codes[
        stu_missing_codes["CNT"] == country
    ]

    s = pd.to_numeric(
        sub["BEINGBULLIED"],
        errors="coerce",
    )

    n_total = len(s)

    missing_codes = [
        95,
        97,
        98,
        99,
    ]

    valid_score = (
        s.notna()
        & ~s.isin(
            missing_codes
        )
    )

    rows.append(
        {
            "Country": country,
            "N_total": n_total,

            "Pct_valid_score": (
                valid_score.sum()
                / n_total
                * 100
            ),

            "Pct_valid_skip_95": (
                s.eq(95).sum()
                / n_total
                * 100
            ),

            "Pct_not_applicable_97": (
                s.eq(97).sum()
                / n_total
                * 100
            ),

            "Pct_invalid_98": (
                s.eq(98).sum()
                / n_total
                * 100
            ),

            "Pct_no_response_99": (
                s.eq(99).sum()
                / n_total
                * 100
            ),

            "Pct_system_missing": (
                s.isna().sum()
                / n_total
                * 100
            ),
        }
    )


being_missing_complete = pd.DataFrame(
    rows
)


# ============================================================
# 3. Check that percentages sum to 100
# ============================================================

raw_missing_complete[
    "Pct_total_check"
] = (
    raw_missing_complete[
        [
            "Pct_valid",
            "Pct_valid_skip",
            "Pct_not_applicable",
            "Pct_invalid",
            "Pct_no_response",
            "Pct_system_missing",
        ]
    ]
    .sum(axis=1)
)


being_missing_complete[
    "Pct_total_check"
] = (
    being_missing_complete[
        [
            "Pct_valid_score",
            "Pct_valid_skip_95",
            "Pct_not_applicable_97",
            "Pct_invalid_98",
            "Pct_no_response_99",
            "Pct_system_missing",
        ]
    ]
    .sum(axis=1)
)


# ============================================================
# 4. Print
# ============================================================

print("\n" + "=" * 100)
print("RAW ST038 ITEMS — COMPLETE MISSINGNESS BREAKDOWN")
print("=" * 100)

print(
    raw_missing_complete
    .round(2)
    .to_string(
        index=False
    )
)


print("\n" + "=" * 100)
print("BEINGBULLIED — COMPLETE MISSINGNESS BREAKDOWN")
print("=" * 100)

print(
    being_missing_complete
    .round(2)
    .to_string(
        index=False
    )
)


RAW ST038 ITEMS — COMPLETE MISSINGNESS BREAKDOWN
Country       Item  N_total  Pct_valid  Pct_valid_skip  Pct_not_applicable  Pct_invalid  Pct_no_response  Pct_system_missing  Pct_total_check
    DOM ST038Q04NA     5674      26.31             0.0                 0.0          0.0            38.16               35.53            100.0
    DOM ST038Q05NA     5674      26.10             0.0                 0.0          0.0            38.37               35.53            100.0
    DOM ST038Q06NA     5674      26.89             0.0                 0.0          0.0            37.57               35.53            100.0
    DOM ST038Q07NA     5674      25.96             0.0                 0.0          0.0            38.51               35.53            100.0
    DOM ST038Q08NA     5674      26.17             0.0                 0.0          0.0            38.30               35.53            100.0
    PAN ST038Q04NA     6270      29.86             0.0                 0.0          0.0           

In [11]:
# ============================================================
# FREEZE SYSTEM SETS + AUDIT THE FINAL PREDICTOR UNIVERSE
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Freeze outcome-specific primary system sets
# ============================================================

PRIMARY_BEING_SYSTEMS = sorted(
    systems_for_outcome(
        "BEINGBULLIED",
        0,
    )
)

PRIMARY_SUBTYPE_SYSTEMS = sorted(
    systems_for_all_subtypes(
        0,
    )
)


# Coverage-restricted sensitivity sets
SENSITIVITY_BEING_50_SYSTEMS = sorted(
    systems_for_outcome(
        "BEINGBULLIED",
        50,
    )
)

SENSITIVITY_SUBTYPE_50_SYSTEMS = sorted(
    systems_for_all_subtypes(
        50,
    )
)


print("=" * 75)
print("FROZEN SYSTEM SETS")
print("=" * 75)

print(
    "Primary BEINGBULLIED systems:",
    len(PRIMARY_BEING_SYSTEMS),
)

print(
    "Primary subtype systems:",
    len(PRIMARY_SUBTYPE_SYSTEMS),
)

print(
    "50% sensitivity BEINGBULLIED systems:",
    len(SENSITIVITY_BEING_50_SYSTEMS),
)

print(
    "50% sensitivity subtype systems:",
    len(SENSITIVITY_SUBTYPE_50_SYSTEMS),
)


# ============================================================
# 2. Helper: identify plausible values
# ============================================================

def is_plausible_value(
    feature: str,
) -> bool:

    return (
        feature.startswith("PV")
        and (
            feature.endswith("READ")
            or feature.endswith("MATH")
            or feature.endswith("SCIE")
        )
    )


# ============================================================
# 3. Build detailed predictor comparison table
# ============================================================

def predictor_universe_audit(
    systems,
    analysis_label,
):

    systems = sorted(
        systems
    )

    sub = predictor_availability[
        predictor_availability[
            "Country"
        ].isin(
            systems
        )
    ].copy()

    rows = []

    for feature in sorted(
        sub["Feature"]
        .dropna()
        .unique()
    ):

        # Plausible values will be handled separately later
        if is_plausible_value(
            feature
        ):
            continue

        g = sub[
            sub["Feature"]
            == feature
        ].copy()

        # Make sure all expected systems are represented
        g = (
            g
            .set_index(
                "Country"
            )
            .reindex(
                systems
            )
            .reset_index()
        )

        g["Missing_pct"] = pd.to_numeric(
            g["Missing_pct"],
            errors="coerce",
        )

        g["N_nonmissing"] = pd.to_numeric(
            g["N_nonmissing"],
            errors="coerce",
        )

        g["Usable"] = (
            g["Usable"]
            .fillna(False)
            .astype(bool)
        )


        # ----------------------------
        # Worst system
        # ----------------------------

        if g["Missing_pct"].notna().any():

            worst_idx = (
                g["Missing_pct"]
                .idxmax()
            )

            worst_country = g.loc[
                worst_idx,
                "Country",
            ]

            max_missing = float(
                g.loc[
                    worst_idx,
                    "Missing_pct",
                ]
            )

        else:

            worst_country = None
            max_missing = np.nan


        # ----------------------------
        # Counts
        # ----------------------------

        n_systems = len(
            systems
        )

        n_usable = int(
            g["Usable"].sum()
        )

        n_zero_observed = int(
            (
                g["N_nonmissing"]
                .fillna(0)
                == 0
            ).sum()
        )


        # ----------------------------
        # Threshold sensitivity
        # ----------------------------

        row = {

            "Analysis": analysis_label,

            "Feature": feature,

            "N_systems": n_systems,

            "N_systems_usable": n_usable,

            "N_systems_zero_observed": (
                n_zero_observed
            ),

            "Median_missing_pct": float(
                g["Missing_pct"]
                .median()
            ),

            "Max_missing_pct": (
                max_missing
            ),

            "Worst_system": (
                worst_country
            ),

            "Available_everywhere": (
                n_usable
                == n_systems
            ),
        }


        for threshold in [
            20,
            30,
            40,
            50,
            60,
            70,
        ]:

            passes = (
                g["Usable"]
                &
                g["Missing_pct"]
                .le(
                    threshold
                )
            )

            row[
                f"All_systems_le_{threshold}"
            ] = bool(
                passes.all()
            )


        rows.append(
            row
        )


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "Available_everywhere",
                "Max_missing_pct",
                "Feature",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 4. Build primary predictor audits
# ============================================================

predictor_audit_being = (
    predictor_universe_audit(
        PRIMARY_BEING_SYSTEMS,
        "BEINGBULLIED",
    )
)


predictor_audit_subtypes = (
    predictor_universe_audit(
        PRIMARY_SUBTYPE_SYSTEMS,
        "SUBTYPES",
    )
)


# ============================================================
# 5. Save complete tables
# ============================================================

PREDICTOR_AUDIT_DIR = Path(
    "../results/predictor_universe"
)

PREDICTOR_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


predictor_audit_being.to_csv(
    PREDICTOR_AUDIT_DIR
    / "predictor_universe_BEINGBULLIED.csv",
    index=False,
)


predictor_audit_subtypes.to_csv(
    PREDICTOR_AUDIT_DIR
    / "predictor_universe_SUBTYPES.csv",
    index=False,
)


# ============================================================
# 6. Print concise table
# ============================================================

DISPLAY_COLS = [
    "Feature",
    "N_systems",
    "N_systems_usable",
    "N_systems_zero_observed",
    "Median_missing_pct",
    "Max_missing_pct",
    "Worst_system",
    "Available_everywhere",
]


print("\n")
print("=" * 100)
print("BEINGBULLIED — NON-PV PREDICTOR UNIVERSE")
print("=" * 100)

print(
    predictor_audit_being[
        DISPLAY_COLS
    ]
    .round(2)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("SUBTYPES — NON-PV PREDICTOR UNIVERSE")
print("=" * 100)

print(
    predictor_audit_subtypes[
        DISPLAY_COLS
    ]
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Count predictors surviving different maximum-missing rules
# ============================================================

summary_rows = []


for label, audit in [

    (
        "BEINGBULLIED",
        predictor_audit_being,
    ),

    (
        "SUBTYPES",
        predictor_audit_subtypes,
    ),

]:

    for threshold in [
        20,
        30,
        40,
        50,
        60,
        70,
    ]:

        column = (
            f"All_systems_le_{threshold}"
        )

        kept = (
            audit.loc[
                audit[column],
                "Feature",
            ]
            .tolist()
        )

        summary_rows.append(
            {
                "Analysis": label,

                "Max_missing_pct": (
                    threshold
                ),

                "N_predictors": (
                    len(kept)
                ),

                "Predictors": (
                    "; ".join(
                        kept
                    )
                ),
            }
        )


predictor_rule_summary = pd.DataFrame(
    summary_rows
)


predictor_rule_summary.to_csv(
    PREDICTOR_AUDIT_DIR
    / "predictor_missingness_rule_sensitivity.csv",
    index=False,
)


print("\n")
print("=" * 100)
print("PREDICTOR UNIVERSE SIZE UNDER DIFFERENT MISSINGNESS RULES")
print("=" * 100)

print(
    predictor_rule_summary[
        [
            "Analysis",
            "Max_missing_pct",
            "N_predictors",
        ]
    ]
    .to_string(
        index=False
    )
)

FROZEN SYSTEM SETS
Primary BEINGBULLIED systems: 74
Primary subtype systems: 75
50% sensitivity BEINGBULLIED systems: 69
50% sensitivity subtype systems: 70


BEINGBULLIED — NON-PV PREDICTOR UNIVERSE
   Feature  N_systems  N_systems_usable  N_systems_zero_observed  Median_missing_pct  Max_missing_pct Worst_system  Available_everywhere
       AGE         74                74                        0                0.00             0.00          ALB                  True
 ST004D01T         74                74                        0                0.00             0.01          CAN                  True
  DISCLIMA         74                74                        0                2.46            12.92          DEU                  True
      ESCS         74                74                        0                1.59            14.58          DEU                  True
   JOYREAD         74                74                        0                2.96            16.34          DOM 

In [12]:
# ============================================================
# PRINT ONLY THE FINAL PREDICTOR-UNIVERSE DECISION INFORMATION
# ============================================================

print("=" * 90)
print("PREDICTOR UNIVERSE SIZE")
print("=" * 90)

print(
    predictor_rule_summary[
        [
            "Analysis",
            "Max_missing_pct",
            "N_predictors",
        ]
    ].to_string(index=False)
)


# ============================================================
# Show exact predictor sets at the most relevant thresholds
# ============================================================

for analysis_name, audit in [
    ("BEINGBULLIED", predictor_audit_being),
    ("SUBTYPES", predictor_audit_subtypes),
]:

    print("\n")
    print("=" * 90)
    print(analysis_name)
    print("=" * 90)

    for threshold in [40, 50, 60, 70]:

        col = f"All_systems_le_{threshold}"

        kept = (
            audit.loc[
                audit[col],
                "Feature",
            ]
            .sort_values()
            .tolist()
        )

        print(
            f"\nMaximum missingness = {threshold}%"
        )

        print(
            f"N predictors = {len(kept)}"
        )

        print(kept)


# ============================================================
# Show predictors that are available everywhere but have
# high missingness in at least one system
# ============================================================

for analysis_name, audit in [
    ("BEINGBULLIED", predictor_audit_being),
    ("SUBTYPES", predictor_audit_subtypes),
]:

    print("\n")
    print("=" * 90)
    print(
        f"{analysis_name} — AVAILABLE EVERYWHERE "
        "BUT MAX MISSINGNESS > 40%"
    )
    print("=" * 90)

    problem = audit[
        (
            audit["Available_everywhere"]
        )
        &
        (
            audit["Max_missing_pct"] > 40
        )
    ][
        [
            "Feature",
            "Median_missing_pct",
            "Max_missing_pct",
            "Worst_system",
        ]
    ].sort_values(
        "Max_missing_pct"
    )

    if problem.empty:
        print("None.")
    else:
        print(
            problem
            .round(2)
            .to_string(index=False)
        )


# ============================================================
# Show predictors that are completely absent in one or
# more systems
# ============================================================

for analysis_name, audit in [
    ("BEINGBULLIED", predictor_audit_being),
    ("SUBTYPES", predictor_audit_subtypes),
]:

    print("\n")
    print("=" * 90)
    print(
        f"{analysis_name} — NOT AVAILABLE IN EVERY SYSTEM"
    )
    print("=" * 90)

    absent = audit[
        ~audit[
            "Available_everywhere"
        ]
    ][
        [
            "Feature",
            "N_systems",
            "N_systems_usable",
            "N_systems_zero_observed",
            "Median_missing_pct",
            "Max_missing_pct",
            "Worst_system",
        ]
    ].sort_values(
        [
            "N_systems_usable",
            "Feature",
        ],
        ascending=[
            False,
            True,
        ]
    )

    if absent.empty:
        print("None.")
    else:
        print(
            absent
            .round(2)
            .to_string(index=False)
        )

PREDICTOR UNIVERSE SIZE
    Analysis  Max_missing_pct  N_predictors
BEINGBULLIED               20             7
BEINGBULLIED               30             7
BEINGBULLIED               40            12
BEINGBULLIED               50            14
BEINGBULLIED               60            14
BEINGBULLIED               70            17
    SUBTYPES               20             7
    SUBTYPES               30             7
    SUBTYPES               40            12
    SUBTYPES               50            14
    SUBTYPES               60            14
    SUBTYPES               70            17


BEINGBULLIED

Maximum missingness = 40%
N predictors = 12
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']

Maximum missingness = 50%
N predictors = 14
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'BELONG', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'MASTGOAL', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']

Maximu

In [13]:
# ============================================================
# CONFIRM FINAL 50% COMMON PREDICTOR SET
# ============================================================

being_50 = set(
    predictor_audit_being.loc[
        predictor_audit_being["All_systems_le_50"],
        "Feature"
    ]
)

subtype_50 = set(
    predictor_audit_subtypes.loc[
        predictor_audit_subtypes["All_systems_le_50"],
        "Feature"
    ]
)

common_50 = (
    being_50
    & subtype_50
)


print("=" * 80)
print("BEINGBULLIED — 50% SET")
print("=" * 80)

print(
    sorted(being_50)
)

print(
    "\nN =",
    len(being_50)
)


print("\n" + "=" * 80)
print("SUBTYPES — 50% SET")
print("=" * 80)

print(
    sorted(subtype_50)
)

print(
    "\nN =",
    len(subtype_50)
)


print("\n" + "=" * 80)
print("COMMON SET ACROSS ALL FIVE OUTCOMES")
print("=" * 80)

print(
    sorted(common_50)
)

print(
    "\nN =",
    len(common_50)
)


print("\nOnly in BEINGBULLIED:")
print(
    sorted(
        being_50
        - subtype_50
    )
)

print("\nOnly in SUBTYPES:")
print(
    sorted(
        subtype_50
        - being_50
    )
)

BEINGBULLIED — 50% SET
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'BELONG', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'MASTGOAL', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']

N = 14

SUBTYPES — 50% SET
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'BELONG', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'MASTGOAL', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']

N = 14

COMMON SET ACROSS ALL FIVE OUTCOMES
['ADAPTIVITY', 'AGE', 'ATTLNACT', 'BELONG', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'MASTGOAL', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']

N = 14

Only in BEINGBULLIED:
[]

Only in SUBTYPES:
[]


In [14]:
# ============================================================
# ACHIEVEMENT PLAUSIBLE-VALUE AVAILABILITY AUDIT
# ============================================================

PRIMARY_NONPV_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


# ============================================================
# 1. Define the 10 plausible values for each domain
# ============================================================

PV_READ = [
    f"PV{i}READ"
    for i in range(1, 11)
]

PV_MATH = [
    f"PV{i}MATH"
    for i in range(1, 11)
]

PV_SCIE = [
    f"PV{i}SCIE"
    for i in range(1, 11)
]

ALL_PVS = (
    PV_READ
    + PV_MATH
    + PV_SCIE
)


# ============================================================
# 2. Summarize PV availability for a system set
# ============================================================

def audit_pv_availability(
    systems,
    analysis_label,
):

    systems = sorted(
        systems
    )

    sub = predictor_availability[
        (
            predictor_availability["Country"]
            .isin(systems)
        )
        &
        (
            predictor_availability["Feature"]
            .isin(ALL_PVS)
        )
    ].copy()

    rows = []

    for country in systems:

        g = sub[
            sub["Country"] == country
        ]

        feature_lookup = (
            g.set_index("Feature")
        )


        # ----------------------------------------
        # Check each achievement domain
        # ----------------------------------------

        domain_results = {}

        for domain, pv_list in [
            ("READ", PV_READ),
            ("MATH", PV_MATH),
            ("SCIE", PV_SCIE),
        ]:

            present = [
                pv
                for pv in pv_list
                if pv in feature_lookup.index
            ]

            usable = [
                pv
                for pv in pv_list
                if (
                    pv in feature_lookup.index
                    and bool(
                        feature_lookup.loc[
                            pv,
                            "Usable",
                        ]
                    )
                )
            ]

            zero_observed = [
                pv
                for pv in pv_list
                if (
                    pv not in feature_lookup.index
                    or (
                        pd.to_numeric(
                            feature_lookup.loc[
                                pv,
                                "N_nonmissing",
                            ],
                            errors="coerce",
                        )
                        == 0
                    )
                )
            ]


            domain_results[
                f"N_{domain}_present"
            ] = len(
                present
            )

            domain_results[
                f"N_{domain}_usable"
            ] = len(
                usable
            )

            domain_results[
                f"N_{domain}_zero_observed"
            ] = len(
                zero_observed
            )


        # ----------------------------------------
        # Require all 10 PVs in all 3 domains
        # ----------------------------------------

        all_30_usable = (
            domain_results[
                "N_READ_usable"
            ] == 10
            and
            domain_results[
                "N_MATH_usable"
            ] == 10
            and
            domain_results[
                "N_SCIE_usable"
            ] == 10
        )


        rows.append(
            {
                "Analysis": analysis_label,
                "Country": country,
                **domain_results,
                "All_30_PVs_usable": (
                    all_30_usable
                ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 3. Audit both frozen primary system sets
# ============================================================

pv_audit_being = audit_pv_availability(
    PRIMARY_BEING_SYSTEMS,
    "BEINGBULLIED",
)

pv_audit_subtypes = audit_pv_availability(
    PRIMARY_SUBTYPE_SYSTEMS,
    "SUBTYPES",
)


# ============================================================
# 4. Identify PV-eligible systems
# ============================================================

PV_ELIGIBLE_BEING_SYSTEMS = sorted(
    pv_audit_being.loc[
        pv_audit_being[
            "All_30_PVs_usable"
        ],
        "Country",
    ].tolist()
)


PV_ELIGIBLE_SUBTYPE_SYSTEMS = sorted(
    pv_audit_subtypes.loc[
        pv_audit_subtypes[
            "All_30_PVs_usable"
        ],
        "Country",
    ].tolist()
)


# ============================================================
# 5. Print failures
# ============================================================

print("=" * 85)
print("BEINGBULLIED — PLAUSIBLE VALUE AVAILABILITY")
print("=" * 85)

print(
    "Primary systems:",
    len(PRIMARY_BEING_SYSTEMS),
)

print(
    "Systems with all 30 PV columns usable:",
    len(PV_ELIGIBLE_BEING_SYSTEMS),
)

failed_being = pv_audit_being[
    ~pv_audit_being[
        "All_30_PVs_usable"
    ]
]

print("\nSystems failing PV requirement:")

if failed_being.empty:
    print("None.")
else:
    print(
        failed_being.to_string(
            index=False
        )
    )


print("\n" + "=" * 85)
print("SUBTYPES — PLAUSIBLE VALUE AVAILABILITY")
print("=" * 85)

print(
    "Primary systems:",
    len(PRIMARY_SUBTYPE_SYSTEMS),
)

print(
    "Systems with all 30 PV columns usable:",
    len(PV_ELIGIBLE_SUBTYPE_SYSTEMS),
)

failed_subtypes = pv_audit_subtypes[
    ~pv_audit_subtypes[
        "All_30_PVs_usable"
    ]
]

print("\nSystems failing PV requirement:")

if failed_subtypes.empty:
    print("None.")
else:
    print(
        failed_subtypes.to_string(
            index=False
        )
    )


# ============================================================
# 6. Save
# ============================================================

PV_AUDIT_DIR = Path(
    "../results/plausible_values"
)

PV_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


pv_audit_being.to_csv(
    PV_AUDIT_DIR
    / "pv_availability_BEINGBULLIED.csv",
    index=False,
)

pv_audit_subtypes.to_csv(
    PV_AUDIT_DIR
    / "pv_availability_SUBTYPES.csv",
    index=False,
)

BEINGBULLIED — PLAUSIBLE VALUE AVAILABILITY
Primary systems: 74
Systems with all 30 PV columns usable: 73

Systems failing PV requirement:
    Analysis Country  N_READ_present  N_READ_usable  N_READ_zero_observed  N_MATH_present  N_MATH_usable  N_MATH_zero_observed  N_SCIE_present  N_SCIE_usable  N_SCIE_zero_observed  All_30_PVs_usable
BEINGBULLIED     VNM              10              0                    10              10              0                    10              10              0                    10              False

SUBTYPES — PLAUSIBLE VALUE AVAILABILITY
Primary systems: 75
Systems with all 30 PV columns usable: 74

Systems failing PV requirement:
Analysis Country  N_READ_present  N_READ_usable  N_READ_zero_observed  N_MATH_present  N_MATH_usable  N_MATH_zero_observed  N_SCIE_present  N_SCIE_usable  N_SCIE_zero_observed  All_30_PVs_usable
SUBTYPES     VNM              10              0                    10              10              0                    10          

In [15]:
# ============================================================
# FINAL MODEL-READINESS AUDIT
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)


# ============================================================
# 1. Frozen primary configuration
# ============================================================

PRIMARY_NONPV_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


REPLICATE_WEIGHT_COLS = [
    f"W_FSTURWT{i}"
    for i in range(1, 81)
]


N_CV_SPLITS = 5

CV_RANDOM_STATE = 42


# ============================================================
# 2. Helper: obtain a country's canonical dataset
# ============================================================

def get_country_frame(
    country: str,
) -> pd.DataFrame:
    """
    Return the canonical processed dataset for one system.

    Uses the in-memory analysis_frames dictionary if available.
    """

    if country not in analysis_frames:
        raise KeyError(
            f"{country}: canonical analysis frame "
            "not found in analysis_frames."
        )

    return analysis_frames[
        country
    ].copy()


# ============================================================
# 3. Helper: basic data requirements
# ============================================================

def basic_readiness_checks(
    df: pd.DataFrame,
) -> dict:
    """
    Check identifiers, survey weights and frozen predictors.
    """

    missing_predictors = [
        p
        for p in PRIMARY_NONPV_PREDICTORS
        if p not in df.columns
    ]

    replicate_weights_present = [
        w
        for w in REPLICATE_WEIGHT_COLS
        if w in df.columns
    ]

    nonnumeric_predictors = [
        p
        for p in PRIMARY_NONPV_PREDICTORS
        if (
            p in df.columns
            and not pd.api.types.is_numeric_dtype(
                df[p]
            )
        )
    ]

    if "CNTSCHID" in df.columns:

        n_school_missing = int(
            df["CNTSCHID"]
            .isna()
            .sum()
        )

        n_schools = int(
            df["CNTSCHID"]
            .nunique()
        )

    else:

        n_school_missing = len(
            df
        )

        n_schools = 0


    if "W_FSTUWT" in df.columns:

        weights = pd.to_numeric(
            df["W_FSTUWT"],
            errors="coerce",
        )

        n_weight_missing = int(
            weights.isna().sum()
        )

        n_weight_nonpositive = int(
            (
                weights.notna()
                & (weights <= 0)
            ).sum()
        )

    else:

        n_weight_missing = len(
            df
        )

        n_weight_nonpositive = np.nan


    return {

        "N_students": len(
            df
        ),

        "N_schools": n_schools,

        "N_missing_school_ID": (
            n_school_missing
        ),

        "Has_W_FSTUWT": (
            "W_FSTUWT"
            in df.columns
        ),

        "N_missing_W_FSTUWT": (
            n_weight_missing
        ),

        "N_nonpositive_W_FSTUWT": (
            n_weight_nonpositive
        ),

        "N_replicate_weights": len(
            replicate_weights_present
        ),

        "All_80_replicate_weights": (
            len(
                replicate_weights_present
            )
            == 80
        ),

        "N_missing_frozen_predictors": len(
            missing_predictors
        ),

        "Missing_frozen_predictors": (
            "; ".join(
                missing_predictors
            )
        ),

        "N_nonnumeric_predictors": len(
            nonnumeric_predictors
        ),

        "Nonnumeric_predictors": (
            "; ".join(
                nonnumeric_predictors
            )
        ),
    }


# ============================================================
# 4. Helper: regression CV feasibility
# ============================================================

def check_regression_cv(
    df: pd.DataFrame,
    outcome: str = "BEINGBULLIED",
    n_splits: int = N_CV_SPLITS,
) -> dict:
    """
    Check whether school-grouped CV can be created for the
    continuous BEINGBULLIED outcome.
    """

    required = [
        outcome,
        "CNTSCHID",
    ]

    if any(
        c not in df.columns
        for c in required
    ):
        return {
            "CV_feasible": False,
            "CV_reason": (
                "Missing outcome or CNTSCHID"
            ),
            "N_outcome_observed": 0,
            "N_outcome_schools": 0,
        }


    temp = df.loc[
        (
            df[outcome].notna()
            & df["CNTSCHID"].notna()
        )
    ].copy()


    n_observed = len(
        temp
    )

    n_schools = int(
        temp["CNTSCHID"]
        .nunique()
    )


    if n_observed == 0:

        return {
            "CV_feasible": False,
            "CV_reason": (
                "No observed outcome values"
            ),
            "N_outcome_observed": 0,
            "N_outcome_schools": 0,
        }


    if n_schools < n_splits:

        return {
            "CV_feasible": False,
            "CV_reason": (
                f"Only {n_schools} schools "
                f"for {n_splits}-fold CV"
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_outcome_schools": (
                n_schools
            ),
        }


    # Actually instantiate the folds to make sure
    # sklearn can create them without an exception.

    splitter = GroupKFold(
        n_splits=n_splits
    )


    try:

        folds = list(
            splitter.split(
                X=temp[
                    PRIMARY_NONPV_PREDICTORS
                ],
                y=temp[
                    outcome
                ],
                groups=temp[
                    "CNTSCHID"
                ],
            )
        )

    except Exception as e:

        return {
            "CV_feasible": False,
            "CV_reason": (
                f"{type(e).__name__}: {e}"
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_outcome_schools": (
                n_schools
            ),
        }


    return {
        "CV_feasible": (
            len(folds)
            == n_splits
        ),
        "CV_reason": "OK",
        "N_outcome_observed": (
            n_observed
        ),
        "N_outcome_schools": (
            n_schools
        ),
    }


# ============================================================
# 5. Helper: classification CV feasibility
# ============================================================

def check_classification_cv(
    df: pd.DataFrame,
    outcome: str,
    n_splits: int = N_CV_SPLITS,
) -> dict:
    """
    Check whether stratified school-grouped CV produces
    usable folds for a binary bullying outcome.

    Every train and validation fold must contain both classes.
    """

    required = [
        outcome,
        "CNTSCHID",
    ]

    if any(
        c not in df.columns
        for c in required
    ):

        return {
            "CV_feasible": False,
            "CV_reason": (
                "Missing outcome or CNTSCHID"
            ),
            "N_outcome_observed": 0,
            "N_positive": 0,
            "N_negative": 0,
            "N_outcome_schools": 0,
            "N_positive_schools": 0,
        }


    temp = df.loc[
        (
            df[outcome].notna()
            & df["CNTSCHID"].notna()
        )
    ].copy()


    y = (
        temp[outcome]
        .astype(int)
    )


    n_observed = len(
        temp
    )

    n_positive = int(
        y.eq(1).sum()
    )

    n_negative = int(
        y.eq(0).sum()
    )

    n_schools = int(
        temp["CNTSCHID"]
        .nunique()
    )


    positive_schools = int(
        temp.loc[
            y.eq(1),
            "CNTSCHID",
        ]
        .nunique()
    )


    if y.nunique() < 2:

        return {
            "CV_feasible": False,
            "CV_reason": (
                "Outcome has only one class"
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_positive": (
                n_positive
            ),
            "N_negative": (
                n_negative
            ),
            "N_outcome_schools": (
                n_schools
            ),
            "N_positive_schools": (
                positive_schools
            ),
        }


    if n_schools < n_splits:

        return {
            "CV_feasible": False,
            "CV_reason": (
                f"Only {n_schools} schools "
                f"for {n_splits}-fold CV"
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_positive": (
                n_positive
            ),
            "N_negative": (
                n_negative
            ),
            "N_outcome_schools": (
                n_schools
            ),
            "N_positive_schools": (
                positive_schools
            ),
        }


    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=CV_RANDOM_STATE,
    )


    try:

        folds = list(
            splitter.split(
                X=temp[
                    PRIMARY_NONPV_PREDICTORS
                ],
                y=y,
                groups=temp[
                    "CNTSCHID"
                ],
            )
        )

    except Exception as e:

        return {
            "CV_feasible": False,
            "CV_reason": (
                f"{type(e).__name__}: {e}"
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_positive": (
                n_positive
            ),
            "N_negative": (
                n_negative
            ),
            "N_outcome_schools": (
                n_schools
            ),
            "N_positive_schools": (
                positive_schools
            ),
        }


    # Make sure both classes occur in every
    # training and validation fold.

    fold_problems = []


    for fold_number, (
        train_idx,
        test_idx,
    ) in enumerate(
        folds,
        start=1,
    ):

        y_train = y.iloc[
            train_idx
        ]

        y_test = y.iloc[
            test_idx
        ]


        if y_train.nunique() < 2:

            fold_problems.append(
                f"fold {fold_number}: "
                "training set has one class"
            )


        if y_test.nunique() < 2:

            fold_problems.append(
                f"fold {fold_number}: "
                "validation set has one class"
            )


    if fold_problems:

        return {
            "CV_feasible": False,
            "CV_reason": "; ".join(
                fold_problems
            ),
            "N_outcome_observed": (
                n_observed
            ),
            "N_positive": (
                n_positive
            ),
            "N_negative": (
                n_negative
            ),
            "N_outcome_schools": (
                n_schools
            ),
            "N_positive_schools": (
                positive_schools
            ),
        }


    return {
        "CV_feasible": True,
        "CV_reason": "OK",
        "N_outcome_observed": (
            n_observed
        ),
        "N_positive": (
            n_positive
        ),
        "N_negative": (
            n_negative
        ),
        "N_outcome_schools": (
            n_schools
        ),
        "N_positive_schools": (
            positive_schools
        ),
    }


# ============================================================
# 6. Run basic checks once per primary system
# ============================================================

ALL_PRIMARY_SYSTEMS = sorted(
    set(
        PRIMARY_BEING_SYSTEMS
    )
    |
    set(
        PRIMARY_SUBTYPE_SYSTEMS
    )
)


basic_rows = []


for country in ALL_PRIMARY_SYSTEMS:

    df = get_country_frame(
        country
    )

    checks = basic_readiness_checks(
        df
    )

    basic_rows.append(
        {
            "Country": country,
            **checks,
        }
    )


basic_readiness = pd.DataFrame(
    basic_rows
)


# ============================================================
# 7. Run BEINGBULLIED CV checks
# ============================================================

being_rows = []


for country in PRIMARY_BEING_SYSTEMS:

    df = get_country_frame(
        country
    )

    checks = check_regression_cv(
        df=df,
        outcome="BEINGBULLIED",
    )

    being_rows.append(
        {
            "Country": country,
            "Outcome": "BEINGBULLIED",
            **checks,
        }
    )


being_cv_readiness = pd.DataFrame(
    being_rows
)


# ============================================================
# 8. Run subtype CV checks
# ============================================================

subtype_rows = []


for country in PRIMARY_SUBTYPE_SYSTEMS:

    df = get_country_frame(
        country
    )

    for outcome in BINARY_OUTCOMES:

        checks = (
            check_classification_cv(
                df=df,
                outcome=outcome,
            )
        )

        subtype_rows.append(
            {
                "Country": country,
                "Outcome": outcome,
                **checks,
            }
        )


subtype_cv_readiness = pd.DataFrame(
    subtype_rows
)


# ============================================================
# 9. Save detailed audit files
# ============================================================

READINESS_DIR = Path(
    "../results/model_readiness"
)

READINESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


basic_readiness.to_csv(
    READINESS_DIR
    / "basic_model_readiness.csv",
    index=False,
)


being_cv_readiness.to_csv(
    READINESS_DIR
    / "beingbullied_cv_readiness.csv",
    index=False,
)


subtype_cv_readiness.to_csv(
    READINESS_DIR
    / "subtype_cv_readiness.csv",
    index=False,
)


# ============================================================
# 10. Print compact results
# ============================================================

print("=" * 90)
print("BASIC MODEL READINESS")
print("=" * 90)

print(
    "Systems checked:",
    len(
        basic_readiness
    )
)

print(
    "Systems missing W_FSTUWT:",
    int(
        (
            ~basic_readiness[
                "Has_W_FSTUWT"
            ]
        ).sum()
    )
)

print(
    "Systems without all 80 replicate weights:",
    int(
        (
            ~basic_readiness[
                "All_80_replicate_weights"
            ]
        ).sum()
    )
)

print(
    "Systems missing frozen predictors:",
    int(
        (
            basic_readiness[
                "N_missing_frozen_predictors"
            ] > 0
        ).sum()
    )
)

print(
    "Systems with nonnumeric frozen predictors:",
    int(
        (
            basic_readiness[
                "N_nonnumeric_predictors"
            ] > 0
        ).sum()
    )
)


print("\n" + "=" * 90)
print("BEINGBULLIED — 5-FOLD SCHOOL-GROUPED CV")
print("=" * 90)

print(
    "Systems checked:",
    len(
        being_cv_readiness
    )
)

print(
    "Systems passing:",
    int(
        being_cv_readiness[
            "CV_feasible"
        ].sum()
    )
)

print(
    "Systems failing:",
    int(
        (
            ~being_cv_readiness[
                "CV_feasible"
            ]
        ).sum()
    )
)


failed_being_cv = (
    being_cv_readiness[
        ~being_cv_readiness[
            "CV_feasible"
        ]
    ]
)


if not failed_being_cv.empty:

    print(
        "\nFailures:"
    )

    print(
        failed_being_cv.to_string(
            index=False
        )
    )


print("\n" + "=" * 90)
print("SUBTYPES — 5-FOLD STRATIFIED SCHOOL-GROUPED CV")
print("=" * 90)

print(
    "Country-outcome combinations checked:",
    len(
        subtype_cv_readiness
    )
)

print(
    "Combinations passing:",
    int(
        subtype_cv_readiness[
            "CV_feasible"
        ].sum()
    )
)

print(
    "Combinations failing:",
    int(
        (
            ~subtype_cv_readiness[
                "CV_feasible"
            ]
        ).sum()
    )
)


failed_subtype_cv = (
    subtype_cv_readiness[
        ~subtype_cv_readiness[
            "CV_feasible"
        ]
    ]
)


if not failed_subtype_cv.empty:

    print(
        "\nFailures:"
    )

    print(
        failed_subtype_cv.to_string(
            index=False
        )
    )


print("\n" + "=" * 90)
print("RAREST BINARY OUTCOMES")
print("=" * 90)

print(
    subtype_cv_readiness[
        [
            "Country",
            "Outcome",
            "N_outcome_observed",
            "N_positive",
            "N_negative",
            "N_outcome_schools",
            "N_positive_schools",
            "CV_feasible",
        ]
    ]
    .sort_values(
        "N_positive"
    )
    .head(15)
    .to_string(
        index=False
    )
)

BASIC MODEL READINESS
Systems checked: 75
Systems missing W_FSTUWT: 0
Systems without all 80 replicate weights: 0
Systems missing frozen predictors: 0
Systems with nonnumeric frozen predictors: 0

BEINGBULLIED — 5-FOLD SCHOOL-GROUPED CV
Systems checked: 74
Systems passing: 74
Systems failing: 0

SUBTYPES — 5-FOLD STRATIFIED SCHOOL-GROUPED CV
Country-outcome combinations checked: 300
Combinations passing: 300
Combinations failing: 0

RAREST BINARY OUTCOMES
Country          Outcome  N_outcome_observed  N_positive  N_negative  N_outcome_schools  N_positive_schools  CV_feasible
    NLD     BULLY_THREAT                3617          59        3558                151                  39         True
    KOR     BULLY_THREAT                6583          75        6508                188                  58         True
    KOR   BULLY_PHYSICAL                6570         106        6464                188                  66         True
    ISL     BULLY_THREAT                2449         117

In [21]:
# ============================================================
# PRIMARY XGBOOST MODELING FRAMEWORK
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)

from xgboost import (
    XGBClassifier,
    XGBRegressor,
)


# ============================================================
# 1. Frozen primary predictors
# ============================================================

PRIMARY_NONPV_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_CV_SPLITS = 5

CV_RANDOM_STATE = 42


# ============================================================
# 2. Fixed reference XGBoost configuration
# ============================================================

#
# IMPORTANT:
# These are called REFERENCE parameters, not "best" parameters.
#
# We are using one common model specification across systems
# so differences in results are not caused by each country
# receiving a completely different model configuration.
#

COMMON_XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 5,
    "reg_lambda": 4.0,
    "reg_alpha": 0.6,
    "gamma": 0.0,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}


# ============================================================
# 3. Create model objects
# ============================================================

def make_xgb_regressor():
    """
    Return the reference XGBoost regression model.
    """

    return XGBRegressor(
        **COMMON_XGB_PARAMS,
        objective="reg:squarederror",
        eval_metric="rmse",
    )


def make_xgb_classifier():
    """
    Return the reference XGBoost binary classifier.
    """

    return XGBClassifier(
        **COMMON_XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
    )


# ============================================================
# 4. Survey-weight helper
# ============================================================

def normalize_training_weights(
    weights: pd.Series,
) -> np.ndarray:
    """
    Normalize survey weights to mean 1 within each training fold.

    This preserves relative student weighting while preventing
    the absolute scale of PISA weights from changing the scale
    of XGBoost's weighted loss and regularization.
    """

    w = pd.to_numeric(
        weights,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    if (
        np.isnan(w).any()
        or np.isinf(w).any()
        or (w <= 0).any()
    ):
        raise ValueError(
            "Training weights contain missing, infinite, "
            "or non-positive values."
        )

    return (
        w / w.mean()
    )


# ============================================================
# 5. Weighted metric helpers
# ============================================================

def weighted_rmse(
    y_true,
    y_pred,
    weights,
) -> float:
    """
    Weighted root mean squared error.
    """

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    return float(
        np.sqrt(
            np.average(
                (y_true - y_pred) ** 2,
                weights=weights,
            )
        )
    )


def regression_metrics(
    y_true,
    y_pred,
    weights,
) -> dict:
    """
    Weighted regression metrics.
    """

    return {
        "RMSE": weighted_rmse(
            y_true,
            y_pred,
            weights,
        ),

        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred,
                sample_weight=weights,
            )
        ),

        "R2": float(
            r2_score(
                y_true,
                y_pred,
                sample_weight=weights,
            )
        ),
    }


def classification_metrics(
    y_true,
    y_prob,
    weights,
) -> dict:
    """
    Weighted binary classification metrics.
    """

    return {
        "ROC_AUC": float(
            roc_auc_score(
                y_true,
                y_prob,
                sample_weight=weights,
            )
        ),

        "PR_AUC": float(
            average_precision_score(
                y_true,
                y_prob,
                sample_weight=weights,
            )
        ),
    }


# ============================================================
# 6. Regression baseline
# ============================================================

def weighted_training_mean(
    y,
    weights,
) -> float:
    """
    Weighted mean of the training outcome.
    """

    return float(
        np.average(
            np.asarray(
                y,
                dtype=float,
            ),
            weights=np.asarray(
                weights,
                dtype=float,
            ),
        )
    )


# ============================================================
# 7. Prepare one outcome dataset
# ============================================================

def prepare_model_data(
    df: pd.DataFrame,
    outcome: str,
    predictors=PRIMARY_NONPV_PREDICTORS,
) -> pd.DataFrame:
    """
    Select students eligible for one model.

    Rows must have:
      - observed outcome
      - school ID
      - valid positive final student weight

    Predictor NaNs are intentionally retained because
    XGBoost can handle missing predictor values natively.
    """

    required_cols = (
        list(
            predictors
        )
        + [
            outcome,
            "CNTSCHID",
            "W_FSTUWT",
        ]
    )

    missing_cols = [
        c
        for c in required_cols
        if c not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required model columns: "
            f"{missing_cols}"
        )


    temp = df[
        required_cols
    ].copy()


    temp["W_FSTUWT"] = pd.to_numeric(
        temp["W_FSTUWT"],
        errors="coerce",
    )


    valid = (
        temp[outcome].notna()
        & temp["CNTSCHID"].notna()
        & temp["W_FSTUWT"].notna()
        & np.isfinite(
            temp["W_FSTUWT"]
        )
        & (
            temp["W_FSTUWT"] > 0
        )
    )


    temp = temp.loc[
        valid
    ].copy()


    return temp


# ============================================================
# 8. Run one country's BEINGBULLIED model
# ============================================================

def run_country_regression_cv(
    df: pd.DataFrame,
    country: str,
    outcome: str = "BEINGBULLIED",
    predictors=PRIMARY_NONPV_PREDICTORS,
    n_splits: int = N_CV_SPLITS,
):
    """
    Run school-grouped out-of-fold XGBoost regression.

    Returns
    -------
    overall_metrics
    fold_metrics
    oof_predictions
    """

    data = prepare_model_data(
        df=df,
        outcome=outcome,
        predictors=predictors,
    )


    X = data[
        list(
            predictors
        )
    ]

    y = pd.to_numeric(
        data[outcome],
        errors="coerce",
    )

    groups = data[
        "CNTSCHID"
    ]

    weights = data[
        "W_FSTUWT"
    ]


    splitter = GroupKFold(
        n_splits=n_splits
    )


    oof_pred = np.full(
        len(data),
        np.nan,
        dtype=float,
    )

    oof_baseline = np.full(
        len(data),
        np.nan,
        dtype=float,
    )

    fold_number = np.full(
        len(data),
        -1,
        dtype=int,
    )


    fold_rows = []


    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        start=1,
    ):


        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]


        y_train = y.iloc[
            train_idx
        ]

        y_test = y.iloc[
            test_idx
        ]


        w_train_raw = weights.iloc[
            train_idx
        ]

        w_test = weights.iloc[
            test_idx
        ].to_numpy(
            dtype=float
        )


        # Normalize only the training weights for model fitting
        w_train_fit = normalize_training_weights(
            w_train_raw
        )


        model = make_xgb_regressor()


        model.fit(
            X_train,
            y_train,
            sample_weight=w_train_fit,
        )


        pred = model.predict(
            X_test
        )


        # Baseline prediction:
        # weighted training-fold mean for every validation student

        baseline_value = weighted_training_mean(
            y_train,
            w_train_raw,
        )

        baseline_pred = np.full(
            len(test_idx),
            baseline_value,
            dtype=float,
        )


        oof_pred[
            test_idx
        ] = pred

        oof_baseline[
            test_idx
        ] = baseline_pred

        fold_number[
            test_idx
        ] = fold


        model_metrics = regression_metrics(
            y_test,
            pred,
            w_test,
        )

        baseline_metrics = regression_metrics(
            y_test,
            baseline_pred,
            w_test,
        )


        fold_rows.append(
            {
                "Country": country,
                "Outcome": outcome,
                "Fold": fold,

                "N_train": len(
                    train_idx
                ),

                "N_test": len(
                    test_idx
                ),

                "N_train_schools": int(
                    groups.iloc[
                        train_idx
                    ].nunique()
                ),

                "N_test_schools": int(
                    groups.iloc[
                        test_idx
                    ].nunique()
                ),

                **{
                    f"Model_{k}": v
                    for k, v
                    in model_metrics.items()
                },

                **{
                    f"Baseline_{k}": v
                    for k, v
                    in baseline_metrics.items()
                },
            }
        )


    if np.isnan(
        oof_pred
    ).any():
        raise RuntimeError(
            f"{country}: some students never received "
            "an OOF regression prediction."
        )


    overall_model = regression_metrics(
        y,
        oof_pred,
        weights,
    )

    overall_baseline = regression_metrics(
        y,
        oof_baseline,
        weights,
    )


    overall_metrics = pd.DataFrame(
        [
            {
                "Country": country,
                "Outcome": outcome,
                "Model_type": "regression",

                "N": len(
                    data
                ),

                "N_schools": int(
                    groups.nunique()
                ),

                **{
                    f"Model_{k}": v
                    for k, v
                    in overall_model.items()
                },

                **{
                    f"Baseline_{k}": v
                    for k, v
                    in overall_baseline.items()
                },
            }
        ]
    )


    fold_metrics = pd.DataFrame(
        fold_rows
    )


    oof_predictions = data[
        [
            "CNTSCHID",
            "W_FSTUWT",
        ]
    ].copy()


    oof_predictions[
        "Country"
    ] = country

    oof_predictions[
        "Outcome"
    ] = outcome

    oof_predictions[
        "Observed"
    ] = y.to_numpy()

    oof_predictions[
        "Predicted"
    ] = oof_pred

    oof_predictions[
        "Baseline_predicted"
    ] = oof_baseline

    oof_predictions[
        "Fold"
    ] = fold_number


    return (
        overall_metrics,
        fold_metrics,
        oof_predictions,
    )


# ============================================================
# 9. Run one country's binary subtype model
# ============================================================

def run_country_classification_cv(
    df: pd.DataFrame,
    country: str,
    outcome: str,
    predictors=PRIMARY_NONPV_PREDICTORS,
    n_splits: int = N_CV_SPLITS,
):
    """
    Run stratified school-grouped out-of-fold XGBoost
    classification.

    Returns
    -------
    overall_metrics
    fold_metrics
    oof_predictions
    """

    data = prepare_model_data(
        df=df,
        outcome=outcome,
        predictors=predictors,
    )


    X = data[
        list(
            predictors
        )
    ]

    y = (
        pd.to_numeric(
            data[outcome],
            errors="coerce",
        )
        .astype(int)
    )

    groups = data[
        "CNTSCHID"
    ]

    weights = data[
        "W_FSTUWT"
    ]


    if y.nunique() != 2:
        raise ValueError(
            f"{country} — {outcome}: "
            "binary outcome does not contain both classes."
        )


    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=CV_RANDOM_STATE,
    )


    oof_prob = np.full(
        len(data),
        np.nan,
        dtype=float,
    )

    oof_baseline = np.full(
        len(data),
        np.nan,
        dtype=float,
    )

    fold_number = np.full(
        len(data),
        -1,
        dtype=int,
    )


    fold_rows = []


    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        start=1,
    ):


        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]


        y_train = y.iloc[
            train_idx
        ]

        y_test = y.iloc[
            test_idx
        ]


        w_train_raw = weights.iloc[
            train_idx
        ]

        w_test = weights.iloc[
            test_idx
        ].to_numpy(
            dtype=float
        )


        w_train_fit = normalize_training_weights(
            w_train_raw
        )


        model = make_xgb_classifier()


        model.fit(
            X_train,
            y_train,
            sample_weight=w_train_fit,
        )


        prob = model.predict_proba(
            X_test
        )[:, 1]


        # Baseline:
        # weighted prevalence in the training fold

        baseline_probability = float(
            np.average(
                y_train,
                weights=w_train_raw,
            )
        )

        baseline_prob = np.full(
            len(test_idx),
            baseline_probability,
            dtype=float,
        )


        oof_prob[
            test_idx
        ] = prob

        oof_baseline[
            test_idx
        ] = baseline_prob

        fold_number[
            test_idx
        ] = fold


        model_metrics = classification_metrics(
            y_test,
            prob,
            w_test,
        )

        baseline_metrics = classification_metrics(
            y_test,
            baseline_prob,
            w_test,
        )


        fold_rows.append(
            {
                "Country": country,
                "Outcome": outcome,
                "Fold": fold,

                "N_train": len(
                    train_idx
                ),

                "N_test": len(
                    test_idx
                ),

                "N_train_positive": int(
                    y_train.sum()
                ),

                "N_test_positive": int(
                    y_test.sum()
                ),

                "N_train_schools": int(
                    groups.iloc[
                        train_idx
                    ].nunique()
                ),

                "N_test_schools": int(
                    groups.iloc[
                        test_idx
                    ].nunique()
                ),

                **{
                    f"Model_{k}": v
                    for k, v
                    in model_metrics.items()
                },

                **{
                    f"Baseline_{k}": v
                    for k, v
                    in baseline_metrics.items()
                },
            }
        )


    if np.isnan(
        oof_prob
    ).any():
        raise RuntimeError(
            f"{country} — {outcome}: "
            "some students never received an OOF prediction."
        )


    overall_model = classification_metrics(
        y,
        oof_prob,
        weights,
    )

    weighted_prevalence = float(
        np.average(
            y,
            weights=weights,
        )
    )

    overall_baseline = {
        "ROC_AUC": 0.5,
        "PR_AUC": weighted_prevalence,
    }

    overall_metrics = pd.DataFrame(
        [
            {
                "Country": country,
                "Outcome": outcome,
                "Model_type": "classification",

                "N": len(
                    data
                ),

                "N_positive": int(
                    y.sum()
                ),

                "N_negative": int(
                    y.eq(0).sum()
                ),

                "N_schools": int(
                    groups.nunique()
                ),

                "Weighted_prevalence": (
                    weighted_prevalence
                ),

                **{
                    f"Model_{k}": v
                    for k, v
                    in overall_model.items()
                },

                **{
                    f"Baseline_{k}": v
                    for k, v
                    in overall_baseline.items()
                },
            }
        ]
    )


    fold_metrics = pd.DataFrame(
        fold_rows
    )


    oof_predictions = data[
        [
            "CNTSCHID",
            "W_FSTUWT",
        ]
    ].copy()


    oof_predictions[
        "Country"
    ] = country

    oof_predictions[
        "Outcome"
    ] = outcome

    oof_predictions[
        "Observed"
    ] = y.to_numpy()

    oof_predictions[
        "Predicted_probability"
    ] = oof_prob

    oof_predictions[
        "Baseline_probability"
    ] = oof_baseline

    oof_predictions[
        "Fold"
    ] = fold_number


    return (
        overall_metrics,
        fold_metrics,
        oof_predictions,
    )

In [22]:
# ============================================================
# MODEL SMOKE TEST — USA
# ============================================================

TEST_COUNTRY = "USA"

test_df = analysis_frames[
    TEST_COUNTRY
]


# ------------------------------------------------------------
# 1. BEINGBULLIED regression
# ------------------------------------------------------------

(
    usa_being_metrics,
    usa_being_folds,
    usa_being_oof,
) = run_country_regression_cv(
    df=test_df,
    country=TEST_COUNTRY,
    outcome="BEINGBULLIED",
)


print("=" * 90)
print("USA — BEINGBULLIED")
print("=" * 90)

print(
    usa_being_metrics
    .round(4)
    .to_string(
        index=False
    )
)

print("\nFold metrics:")

print(
    usa_being_folds
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 2. All four subtype classifiers
# ------------------------------------------------------------

for outcome in BINARY_OUTCOMES:

    (
        metrics,
        folds,
        oof,
    ) = run_country_classification_cv(
        df=test_df,
        country=TEST_COUNTRY,
        outcome=outcome,
    )


    print("\n")
    print("=" * 90)
    print(
        f"USA — {outcome}"
    )
    print("=" * 90)

    print(
        metrics
        .round(4)
        .to_string(
            index=False
        )
    )

    print("\nFold metrics:")

    print(
        folds
        .round(4)
        .to_string(
            index=False
        )
    )

USA — BEINGBULLIED
Country      Outcome Model_type    N  N_schools  Model_RMSE  Model_MAE  Model_R2  Baseline_RMSE  Baseline_MAE  Baseline_R2
    USA BEINGBULLIED regression 4524        162      0.9573     0.7895    0.1612         1.0458        0.8958      -0.0012

Fold metrics:
Country      Outcome  Fold  N_train  N_test  N_train_schools  N_test_schools  Model_RMSE  Model_MAE  Model_R2  Baseline_RMSE  Baseline_MAE  Baseline_R2
    USA BEINGBULLIED     1     3617     907              129              33      0.9609     0.7999    0.0767         1.0006        0.8804      -0.0012
    USA BEINGBULLIED     2     3620     904              130              32      0.9120     0.7526    0.1858         1.0128        0.8770      -0.0042
    USA BEINGBULLIED     3     3620     904              130              32      0.9874     0.8175    0.1645         1.0845        0.9219      -0.0079
    USA BEINGBULLIED     4     3618     906              129              33      0.9820     0.7998    0.1568   

In [23]:
test_metrics, test_folds, test_oof = run_country_classification_cv(
    df=analysis_frames["USA"],
    country="USA",
    outcome="BULLY_VERBAL",
)

print(
    test_metrics
    .round(4)
    .to_string(index=False)
)

Country      Outcome     Model_type    N  N_positive  N_negative  N_schools  Weighted_prevalence  Model_ROC_AUC  Model_PR_AUC  Baseline_ROC_AUC  Baseline_PR_AUC
    USA BULLY_VERBAL classification 4556         780        3776        162               0.1715         0.6932        0.3097               0.5           0.1715


In [24]:
# ============================================================
# RUN PRIMARY MODELS ACROSS ALL SYSTEMS
# ============================================================

from pathlib import Path
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Output directories
# ============================================================

MODEL_DIR = Path(
    "../results/primary_models"
)

OOF_DIR = (
    MODEL_DIR
    / "oof_predictions"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OOF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Storage
# ============================================================

overall_rows = []
fold_rows = []
error_rows = []


start_time = time.time()


# ============================================================
# 3. BEINGBULLIED models
# ============================================================

print("=" * 90)
print("RUNNING BEINGBULLIED MODELS")
print("=" * 90)


for i, country in enumerate(
    PRIMARY_BEING_SYSTEMS,
    start=1,
):

    print(
        f"[{i}/{len(PRIMARY_BEING_SYSTEMS)}] "
        f"{country}",
        end=" ... ",
    )

    try:

        df = analysis_frames[
            country
        ]

        (
            overall,
            folds,
            oof,
        ) = run_country_regression_cv(
            df=df,
            country=country,
            outcome="BEINGBULLIED",
        )


        overall_rows.append(
            overall
        )

        fold_rows.append(
            folds
        )


        oof.to_pickle(
            OOF_DIR
            / f"{country}_BEINGBULLIED.pkl"
        )


        print(
            "OK | "
            f"R2={overall.iloc[0]['Model_R2']:.3f}"
        )


    except Exception as e:

        error_rows.append(
            {
                "Country": country,
                "Outcome": "BEINGBULLIED",
                "Error_type": (
                    type(e).__name__
                ),
                "Error": str(e),
            }
        )

        print(
            f"ERROR: "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# 4. Binary subtype models
# ============================================================

print("\n")
print("=" * 90)
print("RUNNING BULLYING-SUBTYPE MODELS")
print("=" * 90)


total_subtype_models = (
    len(PRIMARY_SUBTYPE_SYSTEMS)
    * len(BINARY_OUTCOMES)
)

counter = 0


for country in PRIMARY_SUBTYPE_SYSTEMS:

    df = analysis_frames[
        country
    ]

    for outcome in BINARY_OUTCOMES:

        counter += 1

        print(
            f"[{counter}/{total_subtype_models}] "
            f"{country} — {outcome}",
            end=" ... ",
        )

        try:

            (
                overall,
                folds,
                oof,
            ) = run_country_classification_cv(
                df=df,
                country=country,
                outcome=outcome,
            )


            overall_rows.append(
                overall
            )

            fold_rows.append(
                folds
            )


            oof.to_pickle(
                OOF_DIR
                / f"{country}_{outcome}.pkl"
            )


            print(
                "OK | "
                f"ROC={overall.iloc[0]['Model_ROC_AUC']:.3f} | "
                f"PR={overall.iloc[0]['Model_PR_AUC']:.3f}"
            )


        except Exception as e:

            error_rows.append(
                {
                    "Country": country,
                    "Outcome": outcome,
                    "Error_type": (
                        type(e).__name__
                    ),
                    "Error": str(e),
                }
            )

            print(
                f"ERROR: "
                f"{type(e).__name__}: {e}"
            )


# ============================================================
# 5. Combine results
# ============================================================

overall_performance = (
    pd.concat(
        overall_rows,
        ignore_index=True,
        sort=False,
    )
    if overall_rows
    else pd.DataFrame()
)


fold_performance = (
    pd.concat(
        fold_rows,
        ignore_index=True,
        sort=False,
    )
    if fold_rows
    else pd.DataFrame()
)


model_errors = pd.DataFrame(
    error_rows,
    columns=[
        "Country",
        "Outcome",
        "Error_type",
        "Error",
    ],
)


# ============================================================
# 6. Save
# ============================================================

overall_performance.to_csv(
    MODEL_DIR
    / "overall_performance.csv",
    index=False,
)


fold_performance.to_csv(
    MODEL_DIR
    / "fold_performance.csv",
    index=False,
)


model_errors.to_csv(
    MODEL_DIR
    / "model_errors.csv",
    index=False,
)


# ============================================================
# 7. Runtime
# ============================================================

elapsed_seconds = (
    time.time()
    - start_time
)

elapsed_minutes = (
    elapsed_seconds
    / 60
)


# ============================================================
# 8. Summary
# ============================================================

print("\n")
print("=" * 90)
print("PRIMARY MODEL RUN COMPLETE")
print("=" * 90)

print(
    "Expected models:",
    (
        len(PRIMARY_BEING_SYSTEMS)
        +
        total_subtype_models
    )
)

print(
    "Successful models:",
    len(
        overall_performance
    )
)

print(
    "Errors:",
    len(
        model_errors
    )
)

print(
    f"Runtime: "
    f"{elapsed_minutes:.2f} minutes"
)


# ============================================================
# 9. BEINGBULLIED performance summary
# ============================================================

being_results = (
    overall_performance[
        overall_performance[
            "Outcome"
        ]
        == "BEINGBULLIED"
    ]
)


print("\n")
print("=" * 90)
print("BEINGBULLIED PERFORMANCE")
print("=" * 90)

print(
    being_results[
        [
            "Model_RMSE",
            "Model_MAE",
            "Model_R2",
            "Baseline_RMSE",
            "Baseline_MAE",
            "Baseline_R2",
        ]
    ]
    .describe()
    .round(3)
    .to_string()
)


print(
    "\nSystems with Model_R2 <= 0:"
)

poor_regression = (
    being_results[
        being_results[
            "Model_R2"
        ]
        <= 0
    ]
    [
        [
            "Country",
            "N",
            "N_schools",
            "Model_RMSE",
            "Model_R2",
            "Baseline_RMSE",
        ]
    ]
    .sort_values(
        "Model_R2"
    )
)


if poor_regression.empty:

    print(
        "None."
    )

else:

    print(
        poor_regression
        .round(3)
        .to_string(
            index=False
        )
    )


# ============================================================
# 10. Classification performance summary
# ============================================================

classification_results = (
    overall_performance[
        overall_performance[
            "Model_type"
        ]
        == "classification"
    ]
)


print("\n")
print("=" * 90)
print("CLASSIFICATION PERFORMANCE BY OUTCOME")
print("=" * 90)


classification_summary = (
    classification_results
    .groupby(
        "Outcome"
    )
    .agg(
        N_models=(
            "Country",
            "count",
        ),

        Mean_ROC_AUC=(
            "Model_ROC_AUC",
            "mean",
        ),

        Median_ROC_AUC=(
            "Model_ROC_AUC",
            "median",
        ),

        Min_ROC_AUC=(
            "Model_ROC_AUC",
            "min",
        ),

        Max_ROC_AUC=(
            "Model_ROC_AUC",
            "max",
        ),

        Mean_PR_AUC=(
            "Model_PR_AUC",
            "mean",
        ),

        Median_PR_AUC=(
            "Model_PR_AUC",
            "median",
        ),

        Mean_prevalence=(
            "Weighted_prevalence",
            "mean",
        ),
    )
    .reset_index()
)


print(
    classification_summary
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Identify weak classifiers
# ============================================================

weak_classifiers = (
    classification_results[
        classification_results[
            "Model_ROC_AUC"
        ]
        <= 0.55
    ]
    [
        [
            "Country",
            "Outcome",
            "N",
            "N_positive",
            "Weighted_prevalence",
            "Model_ROC_AUC",
            "Model_PR_AUC",
            "Baseline_PR_AUC",
        ]
    ]
    .sort_values(
        [
            "Model_ROC_AUC",
            "Country",
        ]
    )
)


print("\n")
print("=" * 90)
print("CLASSIFIERS WITH ROC-AUC <= 0.55")
print("=" * 90)


if weak_classifiers.empty:

    print(
        "None."
    )

else:

    print(
        weak_classifiers
        .round(3)
        .to_string(
            index=False
        )
    )

RUNNING BEINGBULLIED MODELS
[1/74] ALB ... OK | R2=0.103
[2/74] ARE ... OK | R2=0.210
[3/74] ARG ... OK | R2=0.133
[4/74] AUS ... OK | R2=0.207
[5/74] AUT ... OK | R2=0.127
[6/74] BEL ... OK | R2=0.150
[7/74] BGR ... OK | R2=0.102
[8/74] BIH ... OK | R2=0.097
[9/74] BLR ... OK | R2=0.134
[10/74] BRA ... OK | R2=0.086
[11/74] BRN ... OK | R2=0.142
[12/74] CAN ... OK | R2=0.148
[13/74] CHE ... OK | R2=0.093
[14/74] CHL ... OK | R2=0.096
[15/74] COL ... OK | R2=0.076
[16/74] CRI ... OK | R2=0.116
[17/74] CZE ... OK | R2=0.110
[18/74] DEU ... OK | R2=0.094
[19/74] DNK ... OK | R2=0.133
[20/74] DOM ... OK | R2=0.060
[21/74] ESP ... OK | R2=0.145
[22/74] EST ... OK | R2=0.126
[23/74] FIN ... OK | R2=0.179
[24/74] FRA ... OK | R2=0.108
[25/74] GBR ... OK | R2=0.225
[26/74] GEO ... OK | R2=0.149
[27/74] GRC ... OK | R2=0.082
[28/74] HKG ... OK | R2=0.121
[29/74] HRV ... OK | R2=0.124
[30/74] HUN ... OK | R2=0.173
[31/74] IDN ... OK | R2=0.070
[32/74] IRL ... OK | R2=0.173
[33/74] ISL ... OK | 

In [25]:
# ============================================================
# OUT-OF-FOLD TREESHAP ATTRIBUTION FRAMEWORK
# + USA SMOKE TEST
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)


# ============================================================
# 1. Frozen attribution configuration
# ============================================================

ATTRIBUTION_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


ATTRIBUTION_BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


ATTRIBUTION_OUTCOMES = [
    "BEINGBULLIED",
    *ATTRIBUTION_BINARY_OUTCOMES,
]


N_ATTRIBUTION_FOLDS = 5

ATTRIBUTION_RANDOM_STATE = 42


# ============================================================
# 2. Exact TreeSHAP extraction
# ============================================================

def get_exact_tree_shap(
    model,
    X_test: pd.DataFrame,
):
    """
    Calculate exact TreeSHAP contributions for held-out rows.

    Returns
    -------
    shap_values : np.ndarray
        Shape = (n_students, n_predictors)

    bias_values : np.ndarray
        SHAP expected-value / bias term for each student.

    max_additivity_error : float
        Maximum difference between:
            sum(SHAP contributions + bias)
        and:
            XGBoost raw prediction margin.
    """

    booster = model.get_booster()

    feature_names = list(
        X_test.columns
    )


    dtest = xgb.DMatrix(
        X_test,
        feature_names=feature_names,
    )


    contributions = booster.predict(
        dtest,
        pred_contribs=True,
        approx_contribs=False,
        validate_features=True,
    )


    contributions = np.asarray(
        contributions,
        dtype=float,
    )


    expected_columns = (
        len(feature_names)
        + 1
    )


    if (
        contributions.ndim != 2
        or contributions.shape[1]
        != expected_columns
    ):
        raise RuntimeError(
            "Unexpected SHAP contribution shape: "
            f"{contributions.shape}; expected "
            f"(n_rows, {expected_columns})."
        )


    # Every column except the final column
    # corresponds to one predictor.
    shap_values = (
        contributions[
            :,
            :-1
        ]
    )


    # Final column is XGBoost's SHAP bias term.
    bias_values = (
        contributions[
            :,
            -1
        ]
    )


    # --------------------------------------------------------
    # Additivity QA check
    # --------------------------------------------------------
    #
    # SHAP contributions + bias should sum to the
    # raw XGBoost margin.
    #
    # Regression:
    #   raw margin = model prediction scale
    #
    # Binary logistic classification:
    #   raw margin = log-odds
    #

    raw_margin = booster.predict(
        dtest,
        output_margin=True,
        validate_features=True,
    )


    raw_margin = np.asarray(
        raw_margin,
        dtype=float,
    ).reshape(-1)


    contribution_sum = (
        contributions.sum(
            axis=1
        )
    )


    max_additivity_error = float(
        np.max(
            np.abs(
                contribution_sum
                - raw_margin
            )
        )
    )


    # Allow tiny floating-point differences,
    # but fail if SHAP clearly does not reconstruct
    # the model margin.
    if max_additivity_error > 1e-3:
        raise RuntimeError(
            "TreeSHAP additivity check failed. "
            f"Maximum error = "
            f"{max_additivity_error:.8f}"
        )


    return (
        shap_values,
        bias_values,
        max_additivity_error,
    )


# ============================================================
# 3. Weighted SHAP aggregation helper
# ============================================================

def weighted_shap_summary(
    shap_values: np.ndarray,
    weights,
    feature_names,
) -> pd.DataFrame:
    """
    Compute survey-weighted SHAP importance.

    Main importance:
        weighted mean absolute SHAP

    Also stores weighted signed mean SHAP for QA,
    but signed mean SHAP is NOT interpreted as a
    causal/protective/harmful effect.
    """

    shap_values = np.asarray(
        shap_values,
        dtype=float,
    )


    weights = np.asarray(
        weights,
        dtype=float,
    )


    if shap_values.shape[0] != len(
        weights
    ):
        raise ValueError(
            "SHAP rows and weight rows do not match."
        )


    if shap_values.shape[1] != len(
        feature_names
    ):
        raise ValueError(
            "SHAP columns and feature-name count "
            "do not match."
        )


    if (
        np.isnan(weights).any()
        or np.isinf(weights).any()
        or (weights <= 0).any()
    ):
        raise ValueError(
            "Invalid survey weights encountered "
            "during SHAP aggregation."
        )


    if (
        np.isnan(shap_values).any()
        or np.isinf(shap_values).any()
    ):
        raise ValueError(
            "Non-finite SHAP values encountered."
        )


    mean_abs_shap = np.average(
        np.abs(
            shap_values
        ),
        axis=0,
        weights=weights,
    )


    mean_signed_shap = np.average(
        shap_values,
        axis=0,
        weights=weights,
    )


    total_abs_shap = float(
        mean_abs_shap.sum()
    )


    if total_abs_shap <= 0:
        raise RuntimeError(
            "Total absolute SHAP importance "
            "is zero."
        )


    importance_share = (
        mean_abs_shap
        / total_abs_shap
    )


    summary = pd.DataFrame(
        {
            "Feature": list(
                feature_names
            ),

            "MeanAbsSHAP": (
                mean_abs_shap
            ),

            "ImportanceShare": (
                importance_share
            ),

            "MeanSignedSHAP": (
                mean_signed_shap
            ),
        }
    )


    # Deterministic ordering:
    # descending importance, then alphabetically
    # if two features tie exactly.
    summary = (
        summary
        .sort_values(
            [
                "MeanAbsSHAP",
                "Feature",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    summary["Rank"] = np.arange(
        1,
        len(summary) + 1,
    )


    return summary


# ============================================================
# 4. Main country/outcome OOF SHAP function
# ============================================================

def run_country_oof_shap(
    df: pd.DataFrame,
    country: str,
    outcome: str,
    predictors=ATTRIBUTION_PREDICTORS,
    n_splits: int = N_ATTRIBUTION_FOLDS,
):
    """
    Calculate held-out TreeSHAP attribution for one
    country/outcome model.

    Procedure
    ---------
    1. Use exactly the same eligible students as modeling.
    2. Recreate the same school-grouped CV scheme.
    3. Fit XGBoost on training schools only.
    4. Calculate SHAP on held-out schools only.
    5. Pool all held-out SHAP values.
    6. Weight attribution using W_FSTUWT.
    7. Rank all predictors.
    8. Quantify fold-to-fold rank stability.

    Returns
    -------
    overall_ranking
    fold_importance
    qa_table
    row_level_shap
    """

    predictors = list(
        predictors
    )


    # --------------------------------------------------------
    # Prepare exactly the same analysis sample
    # --------------------------------------------------------

    data = prepare_model_data(
        df=df,
        outcome=outcome,
        predictors=predictors,
    )


    X = data[
        predictors
    ]


    groups = data[
        "CNTSCHID"
    ]


    weights = pd.to_numeric(
        data["W_FSTUWT"],
        errors="coerce",
    )


    # --------------------------------------------------------
    # Determine model type + recreate SAME CV procedure
    # --------------------------------------------------------

    if outcome == "BEINGBULLIED":

        model_type = "regression"

        shap_space = (
            "BEINGBULLIED_index_units"
        )

        y = pd.to_numeric(
            data[outcome],
            errors="coerce",
        )


        splitter = GroupKFold(
            n_splits=n_splits
        )


        splits = splitter.split(
            X,
            y,
            groups,
        )


    else:

        model_type = "classification"

        shap_space = "log_odds"

        y = (
            pd.to_numeric(
                data[outcome],
                errors="coerce",
            )
            .astype(int)
        )


        if y.nunique() != 2:
            raise ValueError(
                f"{country} — {outcome}: "
                "outcome does not contain both classes."
            )


        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=(
                ATTRIBUTION_RANDOM_STATE
            ),
        )


        splits = splitter.split(
            X,
            y,
            groups,
        )


    # --------------------------------------------------------
    # Storage for every student's held-out SHAP values
    # --------------------------------------------------------

    n_rows = len(
        data
    )

    n_features = len(
        predictors
    )


    oof_shap = np.full(
        (
            n_rows,
            n_features,
        ),
        np.nan,
        dtype=float,
    )


    oof_bias = np.full(
        n_rows,
        np.nan,
        dtype=float,
    )


    fold_assignment = np.full(
        n_rows,
        -1,
        dtype=int,
    )


    fold_importance_rows = []

    qa_rows = []


    # ========================================================
    # 5. Fit each fold and attribute ONLY held-out students
    # ========================================================

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splits,
        start=1,
    ):


        X_train = X.iloc[
            train_idx
        ]


        X_test = X.iloc[
            test_idx
        ]


        y_train = y.iloc[
            train_idx
        ]


        w_train_raw = weights.iloc[
            train_idx
        ]


        w_test = (
            weights.iloc[
                test_idx
            ]
            .to_numpy(
                dtype=float
            )
        )


        # Same weight-normalization procedure
        # as primary performance models.
        w_train_fit = (
            normalize_training_weights(
                w_train_raw
            )
        )


        # ----------------------------------------------------
        # Same fixed XGBoost model specification
        # ----------------------------------------------------

        if model_type == "regression":

            model = (
                make_xgb_regressor()
            )

        else:

            model = (
                make_xgb_classifier()
            )


        model.fit(
            X_train,
            y_train,
            sample_weight=w_train_fit,
        )


        # ----------------------------------------------------
        # Exact TreeSHAP on held-out schools
        # ----------------------------------------------------

        (
            shap_test,
            bias_test,
            max_additivity_error,
        ) = get_exact_tree_shap(
            model=model,
            X_test=X_test,
        )


        # Store held-out contributions
        oof_shap[
            test_idx,
            :
        ] = shap_test


        oof_bias[
            test_idx
        ] = bias_test


        fold_assignment[
            test_idx
        ] = fold


        # ----------------------------------------------------
        # Fold-specific importance
        # ----------------------------------------------------

        fold_summary = (
            weighted_shap_summary(
                shap_values=shap_test,
                weights=w_test,
                feature_names=predictors,
            )
        )


        fold_summary.insert(
            0,
            "Fold",
            fold,
        )


        fold_summary.insert(
            0,
            "Outcome",
            outcome,
        )


        fold_summary.insert(
            0,
            "Country",
            country,
        )


        fold_importance_rows.append(
            fold_summary
        )


        # ----------------------------------------------------
        # Fold QA
        # ----------------------------------------------------

        qa_rows.append(
            {
                "Country": country,
                "Outcome": outcome,
                "Model_type": model_type,
                "Fold": fold,

                "N_train": len(
                    train_idx
                ),

                "N_test": len(
                    test_idx
                ),

                "N_train_schools": int(
                    groups.iloc[
                        train_idx
                    ].nunique()
                ),

                "N_test_schools": int(
                    groups.iloc[
                        test_idx
                    ].nunique()
                ),

                "Max_SHAP_additivity_error": (
                    max_additivity_error
                ),
            }
        )


    # ========================================================
    # 6. Verify every student received exactly one OOF SHAP row
    # ========================================================

    if np.isnan(
        oof_shap
    ).any():

        raise RuntimeError(
            f"{country} — {outcome}: "
            "some held-out SHAP values are missing."
        )


    if np.isnan(
        oof_bias
    ).any():

        raise RuntimeError(
            f"{country} — {outcome}: "
            "some SHAP bias values are missing."
        )


    if (
        fold_assignment
        < 1
    ).any():

        raise RuntimeError(
            f"{country} — {outcome}: "
            "some students have no fold assignment."
        )


    # ========================================================
    # 7. Pool ALL held-out students for final importance
    # ========================================================

    overall_ranking = (
        weighted_shap_summary(
            shap_values=oof_shap,
            weights=weights.to_numpy(
                dtype=float
            ),
            feature_names=predictors,
        )
    )


    # ========================================================
    # 8. Fold-level stability
    # ========================================================

    fold_importance = pd.concat(
        fold_importance_rows,
        ignore_index=True,
    )


    stability_rows = []


    for feature in predictors:

        g = fold_importance[
            fold_importance[
                "Feature"
            ]
            == feature
        ].copy()


        fold_ranks = (
            g["Rank"]
            .to_numpy(
                dtype=float
            )
        )


        fold_shares = (
            g["ImportanceShare"]
            .to_numpy(
                dtype=float
            )
        )


        q25 = float(
            np.quantile(
                fold_ranks,
                0.25,
            )
        )


        q75 = float(
            np.quantile(
                fold_ranks,
                0.75,
            )
        )


        stability_rows.append(
            {
                "Feature": feature,

                "MeanFoldRank": float(
                    np.mean(
                        fold_ranks
                    )
                ),

                "MedianFoldRank": float(
                    np.median(
                        fold_ranks
                    )
                ),

                "FoldRankSD": float(
                    np.std(
                        fold_ranks,
                        ddof=0,
                    )
                ),

                "FoldRankIQR": (
                    q75
                    - q25
                ),

                "BestFoldRank": int(
                    np.min(
                        fold_ranks
                    )
                ),

                "WorstFoldRank": int(
                    np.max(
                        fold_ranks
                    )
                ),

                "MeanFoldImportanceShare": float(
                    np.mean(
                        fold_shares
                    )
                ),

                "SDFoldImportanceShare": float(
                    np.std(
                        fold_shares,
                        ddof=0,
                    )
                ),

                # We track multiple top-k values
                # because 10/14 predictors is broad.
                "Top5FoldCount": int(
                    (
                        fold_ranks <= 5
                    ).sum()
                ),

                "Top5FoldRate": float(
                    (
                        fold_ranks <= 5
                    ).mean()
                ),

                "Top7FoldCount": int(
                    (
                        fold_ranks <= 7
                    ).sum()
                ),

                "Top7FoldRate": float(
                    (
                        fold_ranks <= 7
                    ).mean()
                ),

                "Top10FoldCount": int(
                    (
                        fold_ranks <= 10
                    ).sum()
                ),

                "Top10FoldRate": float(
                    (
                        fold_ranks <= 10
                    ).mean()
                ),
            }
        )


    stability = pd.DataFrame(
        stability_rows
    )


    overall_ranking = (
        overall_ranking
        .merge(
            stability,
            on="Feature",
            how="left",
            validate="one_to_one",
        )
    )


    overall_ranking.insert(
        0,
        "SHAP_space",
        shap_space,
    )


    overall_ranking.insert(
        0,
        "Model_type",
        model_type,
    )


    overall_ranking.insert(
        0,
        "Outcome",
        outcome,
    )


    overall_ranking.insert(
        0,
        "Country",
        country,
    )


    overall_ranking[
        "N_students"
    ] = len(
        data
    )


    overall_ranking[
        "N_schools"
    ] = int(
        groups.nunique()
    )


    # ========================================================
    # 9. Row-level held-out SHAP output
    # ========================================================

    row_level_shap = pd.DataFrame(
        oof_shap,
        index=data.index,
        columns=[
            f"SHAP__{feature}"
            for feature
            in predictors
        ],
    )


    row_level_shap.insert(
        0,
        "Fold",
        fold_assignment,
    )


    row_level_shap.insert(
        0,
        "SHAP_Bias",
        oof_bias,
    )


    row_level_shap.insert(
        0,
        "W_FSTUWT",
        weights.to_numpy(
            dtype=float
        ),
    )


    row_level_shap.insert(
        0,
        "CNTSCHID",
        groups.to_numpy(),
    )


    row_level_shap.insert(
        0,
        "Observed",
        y.to_numpy(),
    )


    row_level_shap.insert(
        0,
        "Outcome",
        outcome,
    )


    row_level_shap.insert(
        0,
        "Country",
        country,
    )


    # Preserve student ID when available.
    if (
        "CNTSTUID"
        in df.columns
    ):

        student_ids = (
            df.loc[
                data.index,
                "CNTSTUID",
            ]
            .to_numpy()
        )

        row_level_shap.insert(
            1,
            "CNTSTUID",
            student_ids,
        )


    # ========================================================
    # 10. QA table
    # ========================================================

    qa_table = pd.DataFrame(
        qa_rows
    )


    return (
        overall_ranking,
        fold_importance,
        qa_table,
        row_level_shap,
    )


# ============================================================
# 11. USA ATTRIBUTION SMOKE TEST
# ============================================================

TEST_COUNTRY = "USA"

usa_attribution_results = {}

test_df = analysis_frames[
    TEST_COUNTRY
]


for outcome in ATTRIBUTION_OUTCOMES:

    print("\n")
    print("=" * 110)
    print(
        f"USA — {outcome}"
    )
    print("=" * 110)


    (
        ranking,
        fold_importance,
        qa,
        row_shap,
    ) = run_country_oof_shap(
        df=test_df,
        country=TEST_COUNTRY,
        outcome=outcome,
    )


    usa_attribution_results[
        outcome
    ] = {
        "ranking": ranking,
        "fold_importance": (
            fold_importance
        ),
        "qa": qa,
        "row_shap": row_shap,
    }


    # --------------------------------------------------------
    # Print the FULL 14-predictor ranking
    # --------------------------------------------------------

    print(
        "\nHeld-out SHAP ranking:"
    )


    print(
        ranking[
            [
                "Rank",
                "Feature",
                "MeanAbsSHAP",
                "ImportanceShare",
                "MedianFoldRank",
                "FoldRankIQR",
                "BestFoldRank",
                "WorstFoldRank",
                "Top5FoldRate",
                "Top7FoldRate",
                "Top10FoldRate",
            ]
        ]
        .round(4)
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # QA
    # --------------------------------------------------------

    print(
        "\nSHAP additivity QA:"
    )


    print(
        qa[
            [
                "Fold",
                "N_train",
                "N_test",
                "N_train_schools",
                "N_test_schools",
                "Max_SHAP_additivity_error",
            ]
        ]
        .to_string(
            index=False
        )
    )


    print(
        "\nImportance shares sum to:",
        round(
            ranking[
                "ImportanceShare"
            ].sum(),
            8,
        )
    )



USA — BEINGBULLIED

Held-out SHAP ranking:
 Rank    Feature  MeanAbsSHAP  ImportanceShare  MedianFoldRank  FoldRankIQR  BestFoldRank  WorstFoldRank  Top5FoldRate  Top7FoldRate  Top10FoldRate
    1     BELONG       0.3421           0.3923             1.0          0.0             1              1           1.0           1.0            1.0
    2   DISCLIMA       0.1177           0.1349             2.0          0.0             2              2           1.0           1.0            1.0
    3       ESCS       0.0647           0.0742             3.0          0.0             3              4           1.0           1.0            1.0
    4    JOYREAD       0.0459           0.0527             5.0          1.0             4              5           1.0           1.0            1.0
    5   MASTGOAL       0.0416           0.0477             6.0          1.0             5             10           0.4           0.8            1.0
    6  ST004D01T       0.0412           0.0472             6.0     

In [26]:
for outcome in ATTRIBUTION_OUTCOMES:

    result = usa_attribution_results[
        outcome
    ]

    ranking = result[
        "ranking"
    ]

    qa = result[
        "qa"
    ]

    print("\n")
    print("=" * 90)
    print(outcome)
    print("=" * 90)

    print(
        ranking[
            [
                "Rank",
                "Feature",
                "ImportanceShare",
                "MedianFoldRank",
                "BestFoldRank",
                "WorstFoldRank",
                "Top5FoldRate",
            ]
        ]
        .round(4)
        .to_string(
            index=False
        )
    )

    print(
        "\nMax additivity error:",
        qa[
            "Max_SHAP_additivity_error"
        ].max()
    )

    print(
        "Importance sum:",
        ranking[
            "ImportanceShare"
        ].sum()
    )



BEINGBULLIED
 Rank    Feature  ImportanceShare  MedianFoldRank  BestFoldRank  WorstFoldRank  Top5FoldRate
    1     BELONG           0.3923             1.0             1              1           1.0
    2   DISCLIMA           0.1349             2.0             2              2           1.0
    3       ESCS           0.0742             3.0             3              4           1.0
    4    JOYREAD           0.0527             5.0             4              5           1.0
    5   MASTGOAL           0.0477             6.0             5             10           0.4
    6  ST004D01T           0.0472             6.0             3             11           0.4
    7 ADAPTIVITY           0.0449             6.0             4              8           0.2
    8   STIMREAD           0.0375             8.0             8             10           0.0
    9   EDUSHORT           0.0349             9.0             7             12           0.0
   10     UNDREM           0.0329             9.0      

In [27]:
# ============================================================
# RUN OOF SHAP ATTRIBUTION ACROSS ALL SYSTEMS
# ============================================================

from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Output directory
# ============================================================

ATTR_DIR = Path(
    "../results/primary_attribution"
)

ATTR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Storage
# ============================================================

all_rankings = []
all_fold_importance = []
all_qa = []
attribution_errors = []


start_time = time.time()

model_counter = 0

expected_models = (
    len(PRIMARY_BEING_SYSTEMS)
    +
    len(PRIMARY_SUBTYPE_SYSTEMS)
    * len(ATTRIBUTION_BINARY_OUTCOMES)
)


# ============================================================
# 3. Helper: add Top-1 fold stability
# ============================================================

def add_top1_stability(
    ranking: pd.DataFrame,
    fold_importance: pd.DataFrame,
) -> pd.DataFrame:

    top1 = (
        fold_importance
        .groupby(
            "Feature",
            as_index=False,
        )
        .agg(
            Top1FoldCount=(
                "Rank",
                lambda s: int(
                    (s == 1).sum()
                ),
            ),
            Top1FoldRate=(
                "Rank",
                lambda s: float(
                    (s == 1).mean()
                ),
            ),
        )
    )

    ranking = (
        ranking
        .merge(
            top1,
            on="Feature",
            how="left",
            validate="one_to_one",
        )
    )

    return ranking


# ============================================================
# 4. Helper: checkpoint current results
# ============================================================

def save_attribution_checkpoint():

    if all_rankings:

        rankings_df = pd.concat(
            all_rankings,
            ignore_index=True,
        )

        rankings_df.to_csv(
            ATTR_DIR
            / "oof_shap_rankings.csv",
            index=False,
        )


    if all_fold_importance:

        folds_df = pd.concat(
            all_fold_importance,
            ignore_index=True,
        )

        folds_df.to_csv(
            ATTR_DIR
            / "fold_shap_importance.csv",
            index=False,
        )


    if all_qa:

        qa_df = pd.concat(
            all_qa,
            ignore_index=True,
        )

        qa_df.to_csv(
            ATTR_DIR
            / "shap_qa.csv",
            index=False,
        )


    errors_df = pd.DataFrame(
        attribution_errors,
        columns=[
            "Country",
            "Outcome",
            "Error_type",
            "Error",
        ],
    )

    errors_df.to_csv(
        ATTR_DIR
        / "attribution_errors.csv",
        index=False,
    )


# ============================================================
# 5. Helper: run one model
# ============================================================

def process_attribution_model(
    country,
    outcome,
):

    global model_counter

    model_counter += 1


    print(
        f"[{model_counter}/{expected_models}] "
        f"{country} — {outcome}",
        end=" ... ",
    )


    try:

        df = analysis_frames[
            country
        ]


        (
            ranking,
            fold_importance,
            qa,
            row_shap,
        ) = run_country_oof_shap(
            df=df,
            country=country,
            outcome=outcome,
        )


        # ----------------------------------------------------
        # Add Top-1 fold stability
        # ----------------------------------------------------

        ranking = add_top1_stability(
            ranking=ranking,
            fold_importance=fold_importance,
        )


        # ----------------------------------------------------
        # Store compact outputs
        # ----------------------------------------------------

        all_rankings.append(
            ranking
        )

        all_fold_importance.append(
            fold_importance
        )

        all_qa.append(
            qa
        )


        top_row = (
            ranking
            .sort_values(
                "Rank"
            )
            .iloc[0]
        )


        print(
            "OK | "
            f"#1={top_row['Feature']} | "
            f"Share={top_row['ImportanceShare']:.3f} | "
            f"Top1FoldRate={top_row['Top1FoldRate']:.1f}"
        )


        # ----------------------------------------------------
        # We do NOT retain row-level SHAP for all 374 models
        # in memory right now.
        # ----------------------------------------------------

        del row_shap


    except Exception as e:

        attribution_errors.append(
            {
                "Country": country,
                "Outcome": outcome,
                "Error_type": (
                    type(e).__name__
                ),
                "Error": str(e),
            }
        )


        print(
            f"ERROR: "
            f"{type(e).__name__}: {e}"
        )


    # --------------------------------------------------------
    # Save every 25 models so progress is not lost
    # --------------------------------------------------------

    if (
        model_counter % 25
        == 0
    ):

        save_attribution_checkpoint()

        gc.collect()


# ============================================================
# 6. BEINGBULLIED attribution
# ============================================================

print("=" * 100)
print("RUNNING BEINGBULLIED OOF SHAP ATTRIBUTION")
print("=" * 100)


for country in PRIMARY_BEING_SYSTEMS:

    process_attribution_model(
        country=country,
        outcome="BEINGBULLIED",
    )


# ============================================================
# 7. Bullying subtype attribution
# ============================================================

print("\n")
print("=" * 100)
print("RUNNING SUBTYPE OOF SHAP ATTRIBUTION")
print("=" * 100)


for country in PRIMARY_SUBTYPE_SYSTEMS:

    for outcome in ATTRIBUTION_BINARY_OUTCOMES:

        process_attribution_model(
            country=country,
            outcome=outcome,
        )


# ============================================================
# 8. Final save
# ============================================================

save_attribution_checkpoint()


rankings_df = pd.concat(
    all_rankings,
    ignore_index=True,
)


folds_df = pd.concat(
    all_fold_importance,
    ignore_index=True,
)


qa_df = pd.concat(
    all_qa,
    ignore_index=True,
)


errors_df = pd.DataFrame(
    attribution_errors,
    columns=[
        "Country",
        "Outcome",
        "Error_type",
        "Error",
    ],
)


# ============================================================
# 9. Basic structural validation
# ============================================================

successful_models = (
    rankings_df[
        [
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


expected_ranking_rows = (
    successful_models
    * len(
        ATTRIBUTION_PREDICTORS
    )
)


actual_ranking_rows = len(
    rankings_df
)


# ------------------------------------------------------------
# Check every model has exactly 14 predictors
# ------------------------------------------------------------

features_per_model = (
    rankings_df
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    ["Feature"]
    .nunique()
)


bad_feature_counts = (
    features_per_model[
        features_per_model
        != len(
            ATTRIBUTION_PREDICTORS
        )
    ]
)


# ============================================================
# 10. Importance-share QA
# ============================================================

importance_sums = (
    rankings_df
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    ["ImportanceShare"]
    .sum()
)


min_importance_sum = float(
    importance_sums.min()
)

max_importance_sum = float(
    importance_sums.max()
)


# ============================================================
# 11. Additivity QA
# ============================================================

max_additivity_error = float(
    qa_df[
        "Max_SHAP_additivity_error"
    ].max()
)


# ============================================================
# 12. Rank-1 predictor frequencies by outcome
# ============================================================

rank1 = (
    rankings_df[
        rankings_df[
            "Rank"
        ]
        == 1
    ]
    .copy()
)


rank1_frequency = (
    rank1
    .groupby(
        [
            "Outcome",
            "Feature",
        ]
    )
    .size()
    .reset_index(
        name="N_systems"
    )
)


systems_per_outcome = (
    rank1
    .groupby(
        "Outcome"
    )
    ["Country"]
    .nunique()
    .rename(
        "N_total_systems"
    )
    .reset_index()
)


rank1_frequency = (
    rank1_frequency
    .merge(
        systems_per_outcome,
        on="Outcome",
        how="left",
    )
)


rank1_frequency[
    "Percent_systems"
] = (
    100
    * rank1_frequency[
        "N_systems"
    ]
    / rank1_frequency[
        "N_total_systems"
    ]
)


rank1_frequency = (
    rank1_frequency
    .sort_values(
        [
            "Outcome",
            "N_systems",
            "Feature",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
)


rank1_frequency.to_csv(
    ATTR_DIR
    / "rank1_predictor_frequency.csv",
    index=False,
)


# ============================================================
# 13. Final-rank #1 stability
# ============================================================

rank1_stability = (
    rank1[
        [
            "Country",
            "Outcome",
            "Feature",
            "ImportanceShare",
            "Top1FoldCount",
            "Top1FoldRate",
        ]
    ]
    .copy()
)


rank1_stability.to_csv(
    ATTR_DIR
    / "rank1_stability.csv",
    index=False,
)


# ============================================================
# 14. Stability summary
# ============================================================

n_rank1_all5 = int(
    (
        rank1[
            "Top1FoldRate"
        ]
        == 1.0
    ).sum()
)


n_rank1_atleast4 = int(
    (
        rank1[
            "Top1FoldRate"
        ]
        >= 0.8
    ).sum()
)


n_rank1_atleast3 = int(
    (
        rank1[
            "Top1FoldRate"
        ]
        >= 0.6
    ).sum()
)


# ============================================================
# 15. Runtime
# ============================================================

elapsed_minutes = (
    time.time()
    - start_time
) / 60


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 100)
print("OOF SHAP ATTRIBUTION COMPLETE")
print("=" * 100)

print(
    "Expected models:",
    expected_models
)

print(
    "Successful models:",
    successful_models
)

print(
    "Errors:",
    len(
        errors_df
    )
)

print(
    "Expected ranking rows:",
    expected_ranking_rows
)

print(
    "Actual ranking rows:",
    actual_ranking_rows
)

print(
    "Models with incorrect feature count:",
    len(
        bad_feature_counts
    )
)

print(
    f"Runtime: "
    f"{elapsed_minutes:.2f} minutes"
)


print("\n")
print("=" * 100)
print("SHAP QA")
print("=" * 100)

print(
    "Minimum importance-share sum:",
    min_importance_sum
)

print(
    "Maximum importance-share sum:",
    max_importance_sum
)

print(
    "Maximum SHAP additivity error:",
    max_additivity_error
)


print("\n")
print("=" * 100)
print("FINAL #1 PREDICTOR STABILITY")
print("=" * 100)

print(
    "Final #1 predictor was #1 in all 5 folds:",
    f"{n_rank1_all5}/{successful_models}",
    f"({100 * n_rank1_all5 / successful_models:.1f}%)"
)

print(
    "Final #1 predictor was #1 in >=4/5 folds:",
    f"{n_rank1_atleast4}/{successful_models}",
    f"({100 * n_rank1_atleast4 / successful_models:.1f}%)"
)

print(
    "Final #1 predictor was #1 in >=3/5 folds:",
    f"{n_rank1_atleast3}/{successful_models}",
    f"({100 * n_rank1_atleast3 / successful_models:.1f}%)"
)


print("\n")
print("=" * 100)
print("#1 PREDICTOR FREQUENCIES BY OUTCOME")
print("=" * 100)


for outcome in ATTRIBUTION_OUTCOMES:

    print("\n")
    print(
        "-" * 80
    )

    print(
        outcome
    )

    print(
        "-" * 80
    )


    temp = (
        rank1_frequency[
            rank1_frequency[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "Feature",
                "N_systems",
                "N_total_systems",
                "Percent_systems",
            ]
        ]
    )


    print(
        temp
        .round(1)
        .to_string(
            index=False
        )
    )

RUNNING BEINGBULLIED OOF SHAP ATTRIBUTION
[1/374] ALB — BEINGBULLIED ... OK | #1=BELONG | Share=0.228 | Top1FoldRate=1.0
[2/374] ARE — BEINGBULLIED ... OK | #1=BELONG | Share=0.318 | Top1FoldRate=1.0
[3/374] ARG — BEINGBULLIED ... OK | #1=BELONG | Share=0.338 | Top1FoldRate=1.0
[4/374] AUS — BEINGBULLIED ... OK | #1=BELONG | Share=0.391 | Top1FoldRate=1.0
[5/374] AUT — BEINGBULLIED ... OK | #1=BELONG | Share=0.316 | Top1FoldRate=1.0
[6/374] BEL — BEINGBULLIED ... OK | #1=BELONG | Share=0.390 | Top1FoldRate=1.0
[7/374] BGR — BEINGBULLIED ... OK | #1=BELONG | Share=0.278 | Top1FoldRate=1.0
[8/374] BIH — BEINGBULLIED ... OK | #1=BELONG | Share=0.294 | Top1FoldRate=1.0
[9/374] BLR — BEINGBULLIED ... OK | #1=BELONG | Share=0.267 | Top1FoldRate=1.0
[10/374] BRA — BEINGBULLIED ... OK | #1=BELONG | Share=0.290 | Top1FoldRate=1.0
[11/374] BRN — BEINGBULLIED ... OK | #1=BELONG | Share=0.195 | Top1FoldRate=1.0
[12/374] CAN — BEINGBULLIED ... OK | #1=BELONG | Share=0.375 | Top1FoldRate=1.0
[13/374

In [28]:
# ============================================================
# XGBOOST HYPERPARAMETER ROBUSTNESS ANALYSIS
#
# Purpose:
# Test whether the primary OOF-SHAP predictor rankings remain
# similar under reasonable alternative XGBoost specifications.
#
# IMPORTANT:
# - Same students
# - Same outcomes
# - Same 14 predictors
# - Same survey weights
# - Same 5 school-grouped folds
# - Only XGBoost hyperparameters change
# ============================================================

from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Directories
# ============================================================

ROBUST_DIR = Path(
    "../results/hyperparameter_robustness"
)

ROBUST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Make sure reference rankings exist
# ============================================================

if "rankings_df" not in globals():

    rankings_df = pd.read_csv(
        "../results/primary_attribution/"
        "oof_shap_rankings.csv"
    )


# ============================================================
# 3. Preserve the REFERENCE XGBoost parameters
# ============================================================

REFERENCE_XGB_PARAMS = (
    COMMON_XGB_PARAMS.copy()
)


print(
    "REFERENCE PARAMETERS:"
)

for key, value in (
    REFERENCE_XGB_PARAMS.items()
):
    print(
        f"  {key}: {value}"
    )


# ============================================================
# 4. Alternative model specifications
# ============================================================
#
# These are NOT being treated as "better" models.
# They intentionally perturb model complexity and
# regularization around our reference specification.
# ============================================================

ROBUSTNESS_CONFIGS = {

    # --------------------------------------------------------
    # Simpler / shallower trees
    # --------------------------------------------------------
    "shallower": {
        "max_depth": 3,
        "min_child_weight": 5,
        "reg_lambda": 4.0,
        "reg_alpha": 0.6,
        "gamma": 0.0,
        "n_estimators": 300,
        "learning_rate": 0.05,
    },


    # --------------------------------------------------------
    # More flexible / deeper trees
    # --------------------------------------------------------
    "deeper": {
        "max_depth": 5,
        "min_child_weight": 5,
        "reg_lambda": 4.0,
        "reg_alpha": 0.6,
        "gamma": 0.0,
        "n_estimators": 300,
        "learning_rate": 0.05,
    },


    # --------------------------------------------------------
    # Stronger regularization
    # --------------------------------------------------------
    "more_regularized": {
        "max_depth": 4,
        "min_child_weight": 10,
        "reg_lambda": 8.0,
        "reg_alpha": 1.0,
        "gamma": 0.1,
        "n_estimators": 300,
        "learning_rate": 0.05,
    },
}


# ============================================================
# 5. Helper: logistic transform
# ============================================================

def stable_sigmoid(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    result = np.empty_like(
        x
    )

    positive = (
        x >= 0
    )

    negative = ~positive


    result[
        positive
    ] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -x[
                    positive
                ]
            )
        )
    )


    exp_x = np.exp(
        x[
            negative
        ]
    )

    result[
        negative
    ] = (
        exp_x
        /
        (
            1.0
            +
            exp_x
        )
    )


    return result


# ============================================================
# 6. Storage
# ============================================================

robust_rankings = []

robust_performance = []

robust_errors = []


expected_models_per_config = (
    len(PRIMARY_BEING_SYSTEMS)
    +
    len(PRIMARY_SUBTYPE_SYSTEMS)
    * len(ATTRIBUTION_BINARY_OUTCOMES)
)


expected_total_runs = (
    expected_models_per_config
    * len(
        ROBUSTNESS_CONFIGS
    )
)


global_counter = 0

start_time = time.time()


# ============================================================
# 7. Helper: save checkpoint
# ============================================================

def save_robustness_checkpoint():

    if robust_rankings:

        pd.concat(
            robust_rankings,
            ignore_index=True,
        ).to_csv(
            ROBUST_DIR
            / "alternative_shap_rankings.csv",
            index=False,
        )


    if robust_performance:

        pd.DataFrame(
            robust_performance
        ).to_csv(
            ROBUST_DIR
            / "alternative_performance.csv",
            index=False,
        )


    pd.DataFrame(
        robust_errors,
        columns=[
            "Config",
            "Country",
            "Outcome",
            "Error_type",
            "Error",
        ],
    ).to_csv(
        ROBUST_DIR
        / "robustness_errors.csv",
        index=False,
    )


# ============================================================
# 8. Run one alternative model
# ============================================================

def run_robustness_model(
    config_name,
    country,
    outcome,
):

    global global_counter

    global_counter += 1


    print(
        f"[{global_counter}/{expected_total_runs}] "
        f"{config_name} | "
        f"{country} — {outcome}",
        end=" ... ",
    )


    try:

        (
            ranking,
            fold_importance,
            qa,
            row_shap,
        ) = run_country_oof_shap(
            df=analysis_frames[
                country
            ],
            country=country,
            outcome=outcome,
        )


        # ----------------------------------------------------
        # Save ranking
        # ----------------------------------------------------

        ranking = (
            ranking.copy()
        )

        ranking.insert(
            0,
            "Config",
            config_name,
        )


        robust_rankings.append(
            ranking
        )


        # ----------------------------------------------------
        # Recover OOF prediction directly from SHAP
        #
        # SHAP values + bias reconstruct raw model output.
        # ----------------------------------------------------

        shap_columns = [
            f"SHAP__{feature}"
            for feature
            in ATTRIBUTION_PREDICTORS
        ]


        raw_margin = (
            row_shap[
                "SHAP_Bias"
            ].to_numpy(
                dtype=float
            )
            +
            row_shap[
                shap_columns
            ]
            .sum(
                axis=1
            )
            .to_numpy(
                dtype=float
            )
        )


        observed = (
            row_shap[
                "Observed"
            ]
            .to_numpy(
                dtype=float
            )
        )


        weights = (
            row_shap[
                "W_FSTUWT"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # Performance
        # ----------------------------------------------------

        if outcome == "BEINGBULLIED":

            metrics = (
                regression_metrics(
                    observed,
                    raw_margin,
                    weights,
                )
            )


            performance_row = {
                "Config": config_name,
                "Country": country,
                "Outcome": outcome,
                "Model_type": (
                    "regression"
                ),

                "RMSE": metrics[
                    "RMSE"
                ],

                "MAE": metrics[
                    "MAE"
                ],

                "R2": metrics[
                    "R2"
                ],

                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,
            }


        else:

            probabilities = (
                stable_sigmoid(
                    raw_margin
                )
            )


            metrics = (
                classification_metrics(
                    observed,
                    probabilities,
                    weights,
                )
            )


            performance_row = {
                "Config": config_name,
                "Country": country,
                "Outcome": outcome,
                "Model_type": (
                    "classification"
                ),

                "RMSE": np.nan,
                "MAE": np.nan,
                "R2": np.nan,

                "ROC_AUC": metrics[
                    "ROC_AUC"
                ],

                "PR_AUC": metrics[
                    "PR_AUC"
                ],
            }


        robust_performance.append(
            performance_row
        )


        # ----------------------------------------------------
        # Compact progress output
        # ----------------------------------------------------

        top = (
            ranking
            .sort_values(
                "Rank"
            )
            .iloc[0]
        )


        print(
            "OK | "
            f"#1={top['Feature']} | "
            f"share="
            f"{top['ImportanceShare']:.3f}"
        )


        del row_shap
        del fold_importance
        del qa


    except Exception as e:

        robust_errors.append(
            {
                "Config": (
                    config_name
                ),

                "Country": country,

                "Outcome": outcome,

                "Error_type": (
                    type(e).__name__
                ),

                "Error": str(e),
            }
        )


        print(
            f"ERROR: "
            f"{type(e).__name__}: {e}"
        )


    if (
        global_counter % 25
        == 0
    ):

        save_robustness_checkpoint()

        gc.collect()


# ============================================================
# 9. Run all alternative configurations
# ============================================================

try:

    for (
        config_name,
        config_overrides,
    ) in ROBUSTNESS_CONFIGS.items():


        print("\n")
        print(
            "=" * 110
        )

        print(
            f"RUNNING CONFIG: "
            f"{config_name}"
        )

        print(
            "=" * 110
        )


        # ----------------------------------------------------
        # Change ONLY the requested hyperparameters.
        # Everything else stays identical to reference.
        # ----------------------------------------------------

        COMMON_XGB_PARAMS = {
            **REFERENCE_XGB_PARAMS,
            **config_overrides,
        }


        print(
            "\nParameters changed:"
        )

        for (
            key,
            value,
        ) in config_overrides.items():

            print(
                f"  {key}: {value}"
            )


        print("\n")


        # ----------------------------------------------------
        # Continuous outcome
        # ----------------------------------------------------

        for country in (
            PRIMARY_BEING_SYSTEMS
        ):

            run_robustness_model(
                config_name=(
                    config_name
                ),

                country=country,

                outcome=(
                    "BEINGBULLIED"
                ),
            )


        # ----------------------------------------------------
        # Binary outcomes
        # ----------------------------------------------------

        for country in (
            PRIMARY_SUBTYPE_SYSTEMS
        ):

            for outcome in (
                ATTRIBUTION_BINARY_OUTCOMES
            ):

                run_robustness_model(
                    config_name=(
                        config_name
                    ),

                    country=country,

                    outcome=outcome,
                )


        # Save at end of each config
        save_robustness_checkpoint()


finally:

    # --------------------------------------------------------
    # VERY IMPORTANT:
    # restore original reference parameters even if the
    # robustness run is interrupted.
    # --------------------------------------------------------

    COMMON_XGB_PARAMS = (
        REFERENCE_XGB_PARAMS.copy()
    )


# ============================================================
# 10. Combine alternative results
# ============================================================

alternative_rankings = (
    pd.concat(
        robust_rankings,
        ignore_index=True,
    )
)


alternative_performance = (
    pd.DataFrame(
        robust_performance
    )
)


errors_df = pd.DataFrame(
    robust_errors,
    columns=[
        "Config",
        "Country",
        "Outcome",
        "Error_type",
        "Error",
    ],
)


# ============================================================
# 11. Prepare reference rankings
# ============================================================

reference = (
    rankings_df[
        [
            "Country",
            "Outcome",
            "Feature",
            "Rank",
            "ImportanceShare",
        ]
    ]
    .copy()
)


reference = (
    reference.rename(
        columns={
            "Rank": (
                "ReferenceRank"
            ),

            "ImportanceShare": (
                "ReferenceShare"
            ),
        }
    )
)


# ============================================================
# 12. Compare each alternative with reference
# ============================================================

comparison_rows = []


for config_name in (
    ROBUSTNESS_CONFIGS.keys()
):

    alt = (
        alternative_rankings[
            alternative_rankings[
                "Config"
            ]
            == config_name
        ]
        [
            [
                "Country",
                "Outcome",
                "Feature",
                "Rank",
                "ImportanceShare",
            ]
        ]
        .copy()
    )


    alt = alt.rename(
        columns={
            "Rank": (
                "AlternativeRank"
            ),

            "ImportanceShare": (
                "AlternativeShare"
            ),
        }
    )


    merged = (
        reference
        .merge(
            alt,
            on=[
                "Country",
                "Outcome",
                "Feature",
            ],
            how="inner",
            validate="one_to_one",
        )
    )


    for (
        country,
        outcome,
    ), group in merged.groupby(
        [
            "Country",
            "Outcome",
        ]
    ):


        if len(group) != len(
            ATTRIBUTION_PREDICTORS
        ):
            raise RuntimeError(
                f"{config_name} | "
                f"{country} | {outcome}: "
                "incorrect number of features "
                "during robustness comparison."
            )


        # ----------------------------------------------------
        # Spearman rank correlation
        #
        # Since ReferenceRank and AlternativeRank are already
        # ranks, Pearson correlation of these rank vectors =
        # Spearman rank correlation.
        # ----------------------------------------------------

        reference_ranks = (
            group[
                "ReferenceRank"
            ]
            .to_numpy(
                dtype=float
            )
        )


        alternative_ranks = (
            group[
                "AlternativeRank"
            ]
            .to_numpy(
                dtype=float
            )
        )


        rank_correlation = float(
            np.corrcoef(
                reference_ranks,
                alternative_ranks,
            )[0, 1]
        )


        # ----------------------------------------------------
        # Cosine similarity of FULL 14-feature SHAP vectors
        # ----------------------------------------------------

        reference_vector = (
            group[
                "ReferenceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        alternative_vector = (
            group[
                "AlternativeShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        cosine_similarity = float(
            np.dot(
                reference_vector,
                alternative_vector,
            )
            /
            (
                np.linalg.norm(
                    reference_vector
                )
                *
                np.linalg.norm(
                    alternative_vector
                )
            )
        )


        # ----------------------------------------------------
        # Top-1 agreement
        # ----------------------------------------------------

        reference_top1 = (
            group.loc[
                group[
                    "ReferenceRank"
                ].idxmin(),
                "Feature",
            ]
        )


        alternative_top1 = (
            group.loc[
                group[
                    "AlternativeRank"
                ].idxmin(),
                "Feature",
            ]
        )


        top1_same = (
            reference_top1
            == alternative_top1
        )


        # ----------------------------------------------------
        # Top-5 Jaccard similarity
        # ----------------------------------------------------

        reference_top5 = set(
            group.loc[
                group[
                    "ReferenceRank"
                ]
                <= 5,
                "Feature",
            ]
        )


        alternative_top5 = set(
            group.loc[
                group[
                    "AlternativeRank"
                ]
                <= 5,
                "Feature",
            ]
        )


        top5_jaccard = (
            len(
                reference_top5
                & alternative_top5
            )
            /
            len(
                reference_top5
                | alternative_top5
            )
        )


        # ----------------------------------------------------
        # Top-7 Jaccard
        # ----------------------------------------------------

        reference_top7 = set(
            group.loc[
                group[
                    "ReferenceRank"
                ]
                <= 7,
                "Feature",
            ]
        )


        alternative_top7 = set(
            group.loc[
                group[
                    "AlternativeRank"
                ]
                <= 7,
                "Feature",
            ]
        )


        top7_jaccard = (
            len(
                reference_top7
                & alternative_top7
            )
            /
            len(
                reference_top7
                | alternative_top7
            )
        )


        # ----------------------------------------------------
        # Average absolute change in normalized SHAP share
        # ----------------------------------------------------

        mean_abs_share_difference = float(
            np.mean(
                np.abs(
                    reference_vector
                    - alternative_vector
                )
            )
        )


        comparison_rows.append(
            {
                "Config": (
                    config_name
                ),

                "Country": country,

                "Outcome": outcome,

                "RankCorrelation": (
                    rank_correlation
                ),

                "CosineSimilarity": (
                    cosine_similarity
                ),

                "Top1Same": (
                    int(
                        top1_same
                    )
                ),

                "Top5Jaccard": (
                    top5_jaccard
                ),

                "Top7Jaccard": (
                    top7_jaccard
                ),

                "MeanAbsShareDifference": (
                    mean_abs_share_difference
                ),
            }
        )


comparisons = pd.DataFrame(
    comparison_rows
)


comparisons.to_csv(
    ROBUST_DIR
    / "reference_vs_alternatives.csv",
    index=False,
)


# ============================================================
# 13. Robustness summary by configuration
# ============================================================

overall_robustness_summary = (
    comparisons
    .groupby(
        "Config"
    )
    .agg(

        N_models=(
            "Country",
            "count",
        ),

        Mean_rank_correlation=(
            "RankCorrelation",
            "mean",
        ),

        Median_rank_correlation=(
            "RankCorrelation",
            "median",
        ),

        Min_rank_correlation=(
            "RankCorrelation",
            "min",
        ),

        Mean_cosine_similarity=(
            "CosineSimilarity",
            "mean",
        ),

        Median_cosine_similarity=(
            "CosineSimilarity",
            "median",
        ),

        Min_cosine_similarity=(
            "CosineSimilarity",
            "min",
        ),

        Top1_agreement_rate=(
            "Top1Same",
            "mean",
        ),

        Mean_top5_jaccard=(
            "Top5Jaccard",
            "mean",
        ),

        Mean_top7_jaccard=(
            "Top7Jaccard",
            "mean",
        ),

        Mean_abs_share_difference=(
            "MeanAbsShareDifference",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 14. Robustness summary by outcome
# ============================================================

outcome_robustness_summary = (
    comparisons
    .groupby(
        [
            "Config",
            "Outcome",
        ]
    )
    .agg(

        N_models=(
            "Country",
            "count",
        ),

        Mean_rank_correlation=(
            "RankCorrelation",
            "mean",
        ),

        Median_rank_correlation=(
            "RankCorrelation",
            "median",
        ),

        Mean_cosine_similarity=(
            "CosineSimilarity",
            "mean",
        ),

        Top1_agreement_rate=(
            "Top1Same",
            "mean",
        ),

        Mean_top5_jaccard=(
            "Top5Jaccard",
            "mean",
        ),

        Mean_top7_jaccard=(
            "Top7Jaccard",
            "mean",
        ),
    )
    .reset_index()
)


overall_robustness_summary.to_csv(
    ROBUST_DIR
    / "overall_robustness_summary.csv",
    index=False,
)


outcome_robustness_summary.to_csv(
    ROBUST_DIR
    / "outcome_robustness_summary.csv",
    index=False,
)


# ============================================================
# 15. Alternative performance summary
# ============================================================

regression_perf = (
    alternative_performance[
        alternative_performance[
            "Outcome"
        ]
        == "BEINGBULLIED"
    ]
)


classification_perf = (
    alternative_performance[
        alternative_performance[
            "Outcome"
        ]
        != "BEINGBULLIED"
    ]
)


regression_summary = (
    regression_perf
    .groupby(
        "Config"
    )
    .agg(
        Mean_R2=(
            "R2",
            "mean",
        ),

        Median_R2=(
            "R2",
            "median",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),
    )
    .reset_index()
)


classification_summary = (
    classification_perf
    .groupby(
        [
            "Config",
            "Outcome",
        ]
    )
    .agg(
        Mean_ROC_AUC=(
            "ROC_AUC",
            "mean",
        ),

        Median_ROC_AUC=(
            "ROC_AUC",
            "median",
        ),

        Min_ROC_AUC=(
            "ROC_AUC",
            "min",
        ),

        Mean_PR_AUC=(
            "PR_AUC",
            "mean",
        ),
    )
    .reset_index()
)


regression_summary.to_csv(
    ROBUST_DIR
    / "alternative_regression_performance.csv",
    index=False,
)


classification_summary.to_csv(
    ROBUST_DIR
    / "alternative_classification_performance.csv",
    index=False,
)


# ============================================================
# 16. Final validation
# ============================================================

successful_runs = (
    alternative_rankings[
        [
            "Config",
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


elapsed_minutes = (
    time.time()
    - start_time
) / 60


print("\n")
print("=" * 110)
print("HYPERPARAMETER ROBUSTNESS COMPLETE")
print("=" * 110)

print(
    "Expected alternative models:",
    expected_total_runs
)

print(
    "Successful alternative models:",
    successful_runs
)

print(
    "Errors:",
    len(
        errors_df
    )
)

print(
    f"Runtime: "
    f"{elapsed_minutes:.2f} minutes"
)


print("\n")
print("=" * 110)
print("OVERALL RANKING ROBUSTNESS")
print("=" * 110)

print(
    overall_robustness_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("ROBUSTNESS BY OUTCOME")
print("=" * 110)

print(
    outcome_robustness_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("ALTERNATIVE REGRESSION PERFORMANCE")
print("=" * 110)

print(
    regression_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("ALTERNATIVE CLASSIFICATION PERFORMANCE")
print("=" * 110)

print(
    classification_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\nReference XGBoost parameters restored:")
print(
    COMMON_XGB_PARAMS
    == REFERENCE_XGB_PARAMS
)

REFERENCE PARAMETERS:
  n_estimators: 300
  max_depth: 4
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_weight: 5
  reg_lambda: 4.0
  reg_alpha: 0.6
  gamma: 0.0
  random_state: 42
  n_jobs: -1
  tree_method: hist


RUNNING CONFIG: shallower

Parameters changed:
  max_depth: 3
  min_child_weight: 5
  reg_lambda: 4.0
  reg_alpha: 0.6
  gamma: 0.0
  n_estimators: 300
  learning_rate: 0.05


[1/1122] shallower | ALB — BEINGBULLIED ... OK | #1=BELONG | share=0.242
[2/1122] shallower | ARE — BEINGBULLIED ... OK | #1=BELONG | share=0.331
[3/1122] shallower | ARG — BEINGBULLIED ... OK | #1=BELONG | share=0.360
[4/1122] shallower | AUS — BEINGBULLIED ... OK | #1=BELONG | share=0.413
[5/1122] shallower | AUT — BEINGBULLIED ... OK | #1=BELONG | share=0.338
[6/1122] shallower | BEL — BEINGBULLIED ... OK | #1=BELONG | share=0.424
[7/1122] shallower | BGR — BEINGBULLIED ... OK | #1=BELONG | share=0.291
[8/1122] shallower | BIH — BEINGBULLIED ... OK | #1=BELONG | share=0.

In [29]:
# ============================================================
# GLOBAL PREDICTOR ARCHITECTURE
#
# Uses the frozen PRIMARY OOF-SHAP rankings.
# No model refitting.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Output directory
# ============================================================

PATTERN_DIR = Path(
    "../results/pattern_analysis"
)

PATTERN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Load frozen primary rankings if needed
# ============================================================

if "rankings_df" not in globals():

    rankings_df = pd.read_csv(
        "../results/primary_attribution/"
        "oof_shap_rankings.csv"
    )


primary_rankings = (
    rankings_df.copy()
)


# ============================================================
# 3. Basic structural validation
# ============================================================

EXPECTED_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


expected_system_counts = {
    "BEINGBULLIED": 74,
    "BULLY_VERBAL": 75,
    "BULLY_THREAT": 75,
    "BULLY_PHYSICAL": 75,
    "BULLY_RELATIONAL": 75,
}


observed_system_counts = (
    primary_rankings
    .groupby(
        "Outcome"
    )
    ["Country"]
    .nunique()
)


print("=" * 100)
print("SYSTEM COUNTS")
print("=" * 100)

print(
    observed_system_counts
    .to_string()
)


for outcome, expected_n in (
    expected_system_counts.items()
):

    actual_n = int(
        observed_system_counts[
            outcome
        ]
    )

    if actual_n != expected_n:

        raise RuntimeError(
            f"{outcome}: expected "
            f"{expected_n} systems, "
            f"found {actual_n}."
        )


# ============================================================
# 4. Predictor summary within each outcome
# ============================================================

summary_rows = []


for outcome in EXPECTED_OUTCOMES:

    temp = (
        primary_rankings[
            primary_rankings[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    n_systems = (
        temp[
            "Country"
        ]
        .nunique()
    )


    for feature in (
        ATTRIBUTION_PREDICTORS
    ):

        g = (
            temp[
                temp[
                    "Feature"
                ]
                == feature
            ]
            .copy()
        )


        if len(g) != n_systems:

            raise RuntimeError(
                f"{outcome} — {feature}: "
                "feature missing from one or more systems."
            )


        ranks = (
            g[
                "Rank"
            ]
            .to_numpy(
                dtype=float
            )
        )


        shares = (
            g[
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        summary_rows.append(
            {
                "Outcome": outcome,
                "Feature": feature,
                "N_systems": n_systems,

                # --------------------------------------------
                # Rank recurrence
                # --------------------------------------------

                "Rank1_count": int(
                    (
                        ranks == 1
                    ).sum()
                ),

                "Rank1_pct": float(
                    100
                    * (
                        ranks == 1
                    ).mean()
                ),

                "Top3_count": int(
                    (
                        ranks <= 3
                    ).sum()
                ),

                "Top3_pct": float(
                    100
                    * (
                        ranks <= 3
                    ).mean()
                ),

                "Top5_count": int(
                    (
                        ranks <= 5
                    ).sum()
                ),

                "Top5_pct": float(
                    100
                    * (
                        ranks <= 5
                    ).mean()
                ),

                "Top7_count": int(
                    (
                        ranks <= 7
                    ).sum()
                ),

                "Top7_pct": float(
                    100
                    * (
                        ranks <= 7
                    ).mean()
                ),

                "Top10_count": int(
                    (
                        ranks <= 10
                    ).sum()
                ),

                "Top10_pct": float(
                    100
                    * (
                        ranks <= 10
                    ).mean()
                ),


                # --------------------------------------------
                # Rank location
                # --------------------------------------------

                "Mean_rank": float(
                    np.mean(
                        ranks
                    )
                ),

                "Median_rank": float(
                    np.median(
                        ranks
                    )
                ),

                "Rank_SD": float(
                    np.std(
                        ranks,
                        ddof=0,
                    )
                ),

                "Best_rank": int(
                    np.min(
                        ranks
                    )
                ),

                "Worst_rank": int(
                    np.max(
                        ranks
                    )
                ),


                # --------------------------------------------
                # Attribution magnitude
                # --------------------------------------------

                "Mean_importance_share": float(
                    np.mean(
                        shares
                    )
                ),

                "Median_importance_share": float(
                    np.median(
                        shares
                    )
                ),

                "SD_importance_share": float(
                    np.std(
                        shares,
                        ddof=0,
                    )
                ),

                "Min_importance_share": float(
                    np.min(
                        shares
                    )
                ),

                "Max_importance_share": float(
                    np.max(
                        shares
                    )
                ),
            }
        )


predictor_summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# 5. Rank predictors within each outcome by average SHAP share
# ============================================================

predictor_summary[
    "GlobalImportanceRank"
] = (
    predictor_summary
    .groupby(
        "Outcome"
    )
    [
        "Mean_importance_share"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


predictor_summary = (
    predictor_summary
    .sort_values(
        [
            "Outcome",
            "GlobalImportanceRank",
        ]
    )
    .reset_index(
        drop=True
    )
)


predictor_summary.to_csv(
    PATTERN_DIR
    / "predictor_summary_by_outcome.csv",
    index=False,
)


# ============================================================
# 6. Mean attribution profile for each outcome
# ============================================================

mean_profile = (
    predictor_summary[
        [
            "Outcome",
            "Feature",
            "Mean_importance_share",
        ]
    ]
    .pivot(
        index="Outcome",
        columns="Feature",
        values="Mean_importance_share",
    )
)


# Force consistent predictor column order
mean_profile = (
    mean_profile[
        ATTRIBUTION_PREDICTORS
    ]
)


mean_profile.to_csv(
    PATTERN_DIR
    / "mean_outcome_importance_profiles.csv"
)


# ============================================================
# 7. Similarity BETWEEN the five outcome-level profiles
#
# Full 14-feature cosine similarity.
# ============================================================

outcome_cosine_rows = []


for i, outcome_a in enumerate(
    EXPECTED_OUTCOMES
):

    for outcome_b in (
        EXPECTED_OUTCOMES[
            i + 1:
        ]
    ):

        a = (
            mean_profile.loc[
                outcome_a
            ]
            .to_numpy(
                dtype=float
            )
        )

        b = (
            mean_profile.loc[
                outcome_b
            ]
            .to_numpy(
                dtype=float
            )
        )


        cosine = float(
            np.dot(
                a,
                b,
            )
            /
            (
                np.linalg.norm(
                    a
                )
                *
                np.linalg.norm(
                    b
                )
            )
        )


        outcome_cosine_rows.append(
            {
                "Outcome_A": outcome_a,
                "Outcome_B": outcome_b,
                "Cosine_similarity": cosine,
            }
        )


outcome_profile_similarity = (
    pd.DataFrame(
        outcome_cosine_rows
    )
)


outcome_profile_similarity.to_csv(
    PATTERN_DIR
    / "outcome_profile_cosine_similarity.csv",
    index=False,
)


# ============================================================
# 8. Within-system subtype similarity
#
# For every system we compare its four subtype SHAP vectors.
# This asks:
#
# "Within the SAME education system, do different bullying
#  subtypes rely on similar predictor architectures?"
# ============================================================

SUBTYPE_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


subtype_similarity_rows = []


for country in (
    PRIMARY_SUBTYPE_SYSTEMS
):

    country_data = (
        primary_rankings[
            (
                primary_rankings[
                    "Country"
                ]
                == country
            )
            &
            (
                primary_rankings[
                    "Outcome"
                ]
                .isin(
                    SUBTYPE_OUTCOMES
                )
            )
        ]
    )


    vectors = {}


    for outcome in (
        SUBTYPE_OUTCOMES
    ):

        temp = (
            country_data[
                country_data[
                    "Outcome"
                ]
                == outcome
            ]
            .set_index(
                "Feature"
            )
            .loc[
                ATTRIBUTION_PREDICTORS
            ]
        )


        vectors[
            outcome
        ] = (
            temp[
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


    for i, outcome_a in enumerate(
        SUBTYPE_OUTCOMES
    ):

        for outcome_b in (
            SUBTYPE_OUTCOMES[
                i + 1:
            ]
        ):

            a = (
                vectors[
                    outcome_a
                ]
            )

            b = (
                vectors[
                    outcome_b
                ]
            )


            cosine = float(
                np.dot(
                    a,
                    b,
                )
                /
                (
                    np.linalg.norm(
                        a
                    )
                    *
                    np.linalg.norm(
                        b
                    )
                )
            )


            subtype_similarity_rows.append(
                {
                    "Country": country,
                    "Outcome_A": outcome_a,
                    "Outcome_B": outcome_b,
                    "Cosine_similarity": cosine,
                }
            )


within_system_subtype_similarity = (
    pd.DataFrame(
        subtype_similarity_rows
    )
)


within_system_subtype_similarity.to_csv(
    PATTERN_DIR
    / "within_system_subtype_similarity.csv",
    index=False,
)


# ============================================================
# 9. Summarize subtype-pair similarities
# ============================================================

subtype_pair_summary = (
    within_system_subtype_similarity
    .groupby(
        [
            "Outcome_A",
            "Outcome_B",
        ]
    )
    .agg(
        N_systems=(
            "Country",
            "count",
        ),

        Mean_cosine=(
            "Cosine_similarity",
            "mean",
        ),

        Median_cosine=(
            "Cosine_similarity",
            "median",
        ),

        SD_cosine=(
            "Cosine_similarity",
            "std",
        ),

        Min_cosine=(
            "Cosine_similarity",
            "min",
        ),

        Max_cosine=(
            "Cosine_similarity",
            "max",
        ),
    )
    .reset_index()
)


subtype_pair_summary.to_csv(
    PATTERN_DIR
    / "subtype_pair_similarity_summary.csv",
    index=False,
)


# ============================================================
# 10. Print global ranking for each outcome
# ============================================================

print("\n")
print("=" * 100)
print("GLOBAL PREDICTOR ARCHITECTURE")
print("=" * 100)


for outcome in EXPECTED_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        predictor_summary[
            predictor_summary[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "GlobalImportanceRank",
                "Feature",
                "Mean_importance_share",
                "Median_rank",
                "Rank1_count",
                "Rank1_pct",
                "Top3_pct",
                "Top5_pct",
                "Top7_pct",
                "Top10_pct",
            ]
        ]
    )


    print(
        temp
        .round(3)
        .to_string(
            index=False
        )
    )


# ============================================================
# 11. Print outcome-level profile similarity
# ============================================================

print("\n")
print("=" * 100)
print("SIMILARITY BETWEEN MEAN OUTCOME PROFILES")
print("=" * 100)


print(
    outcome_profile_similarity
    .sort_values(
        "Cosine_similarity",
        ascending=False,
    )
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 12. Print within-system subtype similarity
# ============================================================

print("\n")
print("=" * 100)
print("WITHIN-SYSTEM SUBTYPE SIMILARITY")
print("=" * 100)


print(
    subtype_pair_summary
    .sort_values(
        "Mean_cosine",
        ascending=False,
    )
    .round(4)
    .to_string(
        index=False
    )
)

SYSTEM COUNTS
Outcome
BEINGBULLIED        74
BULLY_PHYSICAL      75
BULLY_RELATIONAL    75
BULLY_THREAT        75
BULLY_VERBAL        75


GLOBAL PREDICTOR ARCHITECTURE


------------------------------------------------------------------------------------------
BEINGBULLIED
------------------------------------------------------------------------------------------
 GlobalImportanceRank    Feature  Mean_importance_share  Median_rank  Rank1_count  Rank1_pct  Top3_pct  Top5_pct  Top7_pct  Top10_pct
                    1     BELONG                  0.295          1.0           73     98.649   100.000   100.000   100.000    100.000
                    2   DISCLIMA                  0.123          2.0            1      1.351    94.595   100.000   100.000    100.000
                    3  ST004D01T                  0.112          3.0            0      0.000    83.784    89.189    95.946     97.297
                    4    JOYREAD                  0.054          6.0            0      0.000     6

In [30]:
# ============================================================
# CROSS-SYSTEM PREDICTOR PROFILE SIMILARITY
#
# No model fitting.
# Uses frozen primary OOF-SHAP rankings.
#
# Measures:
#   1. Full 14-feature SHAP cosine similarity
#   2. Full 14-feature rank correlation
#   3. Top-5 Jaccard
#   4. Top-7 Jaccard
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd


# ============================================================
# 1. Output directory
# ============================================================

SIM_DIR = Path(
    "../results/system_similarity"
)

SIM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Load frozen rankings if needed
# ============================================================

if "rankings_df" not in globals():

    rankings_df = pd.read_csv(
        "../results/primary_attribution/"
        "oof_shap_rankings.csv"
    )


similarity_data = (
    rankings_df.copy()
)


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


FEATURE_ORDER = list(
    ATTRIBUTION_PREDICTORS
)


# ============================================================
# 3. Helper functions
# ============================================================

def cosine_similarity(
    a,
    b,
):

    a = np.asarray(
        a,
        dtype=float,
    )

    b = np.asarray(
        b,
        dtype=float,
    )


    denominator = (
        np.linalg.norm(a)
        *
        np.linalg.norm(b)
    )


    if denominator == 0:

        return np.nan


    return float(
        np.dot(
            a,
            b,
        )
        /
        denominator
    )


def rank_correlation(
    ranks_a,
    ranks_b,
):

    ranks_a = np.asarray(
        ranks_a,
        dtype=float,
    )

    ranks_b = np.asarray(
        ranks_b,
        dtype=float,
    )


    # Because these are already rank vectors,
    # Pearson correlation of ranks = Spearman rho.
    return float(
        np.corrcoef(
            ranks_a,
            ranks_b,
        )[0, 1]
    )


def jaccard_similarity(
    set_a,
    set_b,
):

    union = (
        set_a
        |
        set_b
    )


    if len(union) == 0:

        return np.nan


    return float(
        len(
            set_a
            &
            set_b
        )
        /
        len(
            union
        )
    )


# ============================================================
# 4. Pairwise system similarity
# ============================================================

pairwise_rows = []


for outcome in OUTCOMES:

    outcome_data = (
        similarity_data[
            similarity_data[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    systems = sorted(
        outcome_data[
            "Country"
        ]
        .unique()
    )


    print(
        f"{outcome}: "
        f"{len(systems)} systems"
    )


    # --------------------------------------------------------
    # Prepare one vector/ranking/set per system
    # --------------------------------------------------------

    system_profiles = {}


    for country in systems:

        temp = (
            outcome_data[
                outcome_data[
                    "Country"
                ]
                == country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                FEATURE_ORDER
            ]
        )


        if len(temp) != len(
            FEATURE_ORDER
        ):

            raise RuntimeError(
                f"{country} — {outcome}: "
                "incorrect feature count."
            )


        importance_vector = (
            temp[
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        rank_vector = (
            temp[
                "Rank"
            ]
            .to_numpy(
                dtype=float
            )
        )


        top5 = set(
            temp[
                temp[
                    "Rank"
                ]
                <= 5
            ]
            .index
        )


        top7 = set(
            temp[
                temp[
                    "Rank"
                ]
                <= 7
            ]
            .index
        )


        system_profiles[
            country
        ] = {
            "importance": (
                importance_vector
            ),

            "ranks": (
                rank_vector
            ),

            "top5": (
                top5
            ),

            "top7": (
                top7
            ),
        }


    # --------------------------------------------------------
    # Every unique pair of systems
    # --------------------------------------------------------

    for (
        country_a,
        country_b,
    ) in combinations(
        systems,
        2,
    ):

        a = (
            system_profiles[
                country_a
            ]
        )

        b = (
            system_profiles[
                country_b
            ]
        )


        full_cosine = (
            cosine_similarity(
                a[
                    "importance"
                ],
                b[
                    "importance"
                ],
            )
        )


        full_rank_corr = (
            rank_correlation(
                a[
                    "ranks"
                ],
                b[
                    "ranks"
                ],
            )
        )


        top5_jaccard = (
            jaccard_similarity(
                a[
                    "top5"
                ],
                b[
                    "top5"
                ],
            )
        )


        top7_jaccard = (
            jaccard_similarity(
                a[
                    "top7"
                ],
                b[
                    "top7"
                ],
            )
        )


        pairwise_rows.append(
            {
                "Outcome": outcome,

                "System_A": (
                    country_a
                ),

                "System_B": (
                    country_b
                ),

                "CosineSimilarity": (
                    full_cosine
                ),

                "RankCorrelation": (
                    full_rank_corr
                ),

                "Top5Jaccard": (
                    top5_jaccard
                ),

                "Top7Jaccard": (
                    top7_jaccard
                ),
            }
        )


pairwise_similarity = (
    pd.DataFrame(
        pairwise_rows
    )
)


pairwise_similarity.to_csv(
    SIM_DIR
    / "pairwise_system_similarity.csv",
    index=False,
)


# ============================================================
# 5. Distribution summary by outcome
# ============================================================

similarity_summary = (
    pairwise_similarity
    .groupby(
        "Outcome"
    )
    .agg(

        N_pairs=(
            "System_A",
            "count",
        ),

        Mean_cosine=(
            "CosineSimilarity",
            "mean",
        ),

        Median_cosine=(
            "CosineSimilarity",
            "median",
        ),

        SD_cosine=(
            "CosineSimilarity",
            "std",
        ),

        Min_cosine=(
            "CosineSimilarity",
            "min",
        ),

        Q05_cosine=(
            "CosineSimilarity",
            lambda x: float(
                x.quantile(
                    0.05
                )
            ),
        ),

        Q25_cosine=(
            "CosineSimilarity",
            lambda x: float(
                x.quantile(
                    0.25
                )
            ),
        ),

        Q75_cosine=(
            "CosineSimilarity",
            lambda x: float(
                x.quantile(
                    0.75
                )
            ),
        ),

        Q95_cosine=(
            "CosineSimilarity",
            lambda x: float(
                x.quantile(
                    0.95
                )
            ),
        ),

        Max_cosine=(
            "CosineSimilarity",
            "max",
        ),

        Mean_rank_corr=(
            "RankCorrelation",
            "mean",
        ),

        Median_rank_corr=(
            "RankCorrelation",
            "median",
        ),

        Min_rank_corr=(
            "RankCorrelation",
            "min",
        ),

        Mean_top5_jaccard=(
            "Top5Jaccard",
            "mean",
        ),

        Median_top5_jaccard=(
            "Top5Jaccard",
            "median",
        ),

        Mean_top7_jaccard=(
            "Top7Jaccard",
            "mean",
        ),

        Median_top7_jaccard=(
            "Top7Jaccard",
            "median",
        ),
    )
    .reset_index()
)


similarity_summary.to_csv(
    SIM_DIR
    / "similarity_summary_by_outcome.csv",
    index=False,
)


# ============================================================
# 6. System-level average similarity
#
# For each system:
# "How similar is this system to all other systems?"
# ============================================================

system_rows = []


for outcome in OUTCOMES:

    temp = (
        pairwise_similarity[
            pairwise_similarity[
                "Outcome"
            ]
            == outcome
        ]
    )


    systems = sorted(
        set(
            temp[
                "System_A"
            ]
        )
        |
        set(
            temp[
                "System_B"
            ]
        )
    )


    for country in systems:

        country_pairs = (
            temp[
                (
                    temp[
                        "System_A"
                    ]
                    == country
                )
                |
                (
                    temp[
                        "System_B"
                    ]
                    == country
                )
            ]
        )


        system_rows.append(
            {
                "Outcome": outcome,
                "Country": country,

                "MeanCosineToOthers": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].mean()
                ),

                "MedianCosineToOthers": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].median()
                ),

                "MinCosineToOthers": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].min()
                ),

                "MeanRankCorrelationToOthers": float(
                    country_pairs[
                        "RankCorrelation"
                    ].mean()
                ),

                "MeanTop5JaccardToOthers": float(
                    country_pairs[
                        "Top5Jaccard"
                    ].mean()
                ),

                "MeanTop7JaccardToOthers": float(
                    country_pairs[
                        "Top7Jaccard"
                    ].mean()
                ),
            }
        )


system_similarity_summary = (
    pd.DataFrame(
        system_rows
    )
)


system_similarity_summary.to_csv(
    SIM_DIR
    / "system_average_similarity.csv",
    index=False,
)


# ============================================================
# 7. Most similar and least similar pairs
# ============================================================

extreme_rows = []


for outcome in OUTCOMES:

    temp = (
        pairwise_similarity[
            pairwise_similarity[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineSimilarity"
        )
    )


    # 10 least similar
    bottom = (
        temp.head(
            10
        )
        .copy()
    )

    bottom[
        "Type"
    ] = "Least_similar"


    # 10 most similar
    top = (
        temp.tail(
            10
        )
        .sort_values(
            "CosineSimilarity",
            ascending=False,
        )
        .copy()
    )

    top[
        "Type"
    ] = "Most_similar"


    extreme_rows.append(
        bottom
    )

    extreme_rows.append(
        top
    )


extreme_pairs = (
    pd.concat(
        extreme_rows,
        ignore_index=True,
    )
)


extreme_pairs.to_csv(
    SIM_DIR
    / "extreme_similarity_pairs.csv",
    index=False,
)


# ============================================================
# 8. Lowest-average-similarity systems
#
# Potentially distinctive predictor profiles.
# ============================================================

distinctive_systems = (
    system_similarity_summary
    .sort_values(
        [
            "Outcome",
            "MeanCosineToOthers",
        ]
    )
)


distinctive_systems.to_csv(
    SIM_DIR
    / "systems_ranked_by_distinctiveness.csv",
    index=False,
)


# ============================================================
# 9. QA
# ============================================================

expected_pair_counts = {
    "BEINGBULLIED": (
        74 * 73 // 2
    ),

    "BULLY_VERBAL": (
        75 * 74 // 2
    ),

    "BULLY_THREAT": (
        75 * 74 // 2
    ),

    "BULLY_PHYSICAL": (
        75 * 74 // 2
    ),

    "BULLY_RELATIONAL": (
        75 * 74 // 2
    ),
}


print("\n")
print("=" * 105)
print("PAIR COUNT QA")
print("=" * 105)


for outcome in OUTCOMES:

    observed = int(
        (
            pairwise_similarity[
                "Outcome"
            ]
            == outcome
        ).sum()
    )

    expected = (
        expected_pair_counts[
            outcome
        ]
    )

    print(
        f"{outcome}: "
        f"{observed} / {expected}"
    )

    if observed != expected:

        raise RuntimeError(
            f"Incorrect pair count "
            f"for {outcome}."
        )


# ============================================================
# 10. Print distribution summaries
# ============================================================

print("\n")
print("=" * 105)
print("CROSS-SYSTEM SIMILARITY DISTRIBUTIONS")
print("=" * 105)


print(
    similarity_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Print 5 most distinctive systems per outcome
# ============================================================

print("\n")
print("=" * 105)
print("MOST DISTINCTIVE SYSTEM PROFILES")
print("=" * 105)


for outcome in OUTCOMES:

    print("\n")
    print(
        "-" * 85
    )

    print(
        outcome
    )

    print(
        "-" * 85
    )


    temp = (
        system_similarity_summary[
            system_similarity_summary[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "MeanCosineToOthers"
        )
        .head(
            5
        )
        [
            [
                "Country",
                "MeanCosineToOthers",
                "MedianCosineToOthers",
                "MinCosineToOthers",
                "MeanRankCorrelationToOthers",
                "MeanTop5JaccardToOthers",
                "MeanTop7JaccardToOthers",
            ]
        ]
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 12. Print least similar pairs
# ============================================================

print("\n")
print("=" * 105)
print("LEAST SIMILAR SYSTEM PAIRS")
print("=" * 105)


for outcome in OUTCOMES:

    print("\n")
    print(
        "-" * 85
    )

    print(
        outcome
    )

    print(
        "-" * 85
    )


    temp = (
        pairwise_similarity[
            pairwise_similarity[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineSimilarity"
        )
        .head(
            5
        )
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )

BEINGBULLIED: 74 systems
BULLY_VERBAL: 75 systems
BULLY_THREAT: 75 systems
BULLY_PHYSICAL: 75 systems
BULLY_RELATIONAL: 75 systems


PAIR COUNT QA
BEINGBULLIED: 2701 / 2701
BULLY_VERBAL: 2775 / 2775
BULLY_THREAT: 2775 / 2775
BULLY_PHYSICAL: 2775 / 2775
BULLY_RELATIONAL: 2775 / 2775


CROSS-SYSTEM SIMILARITY DISTRIBUTIONS
         Outcome  N_pairs  Mean_cosine  Median_cosine  SD_cosine  Min_cosine  Q05_cosine  Q25_cosine  Q75_cosine  Q95_cosine  Max_cosine  Mean_rank_corr  Median_rank_corr  Min_rank_corr  Mean_top5_jaccard  Median_top5_jaccard  Mean_top7_jaccard  Median_top7_jaccard
    BEINGBULLIED     2701       0.9546         0.9637     0.0364      0.6985      0.8905      0.9404      0.9797      0.9916      0.9983          0.6807            0.6967         0.1473             0.5375               0.4286             0.5957               0.5556
  BULLY_PHYSICAL     2775       0.9437         0.9500     0.0320      0.7499      0.8817      0.9271      0.9667      0.9832      0.9958         

In [31]:
# ============================================================
# DATA-QUALITY CHECK FOR CROSS-SYSTEM DISTINCTIVENESS
#
# Questions:
# 1. Are unusual SHAP profiles associated with low outcome
#    coverage, predictor missingness, sample size, or schools?
#
# 2. Do the cross-system similarity conclusions remain when
#    restricting to systems with >= 50% outcome coverage?
#
# No model fitting.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

DQ_DIR = Path(
    "../results/system_similarity"
)

DQ_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DQ_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


DQ_PREDICTORS = list(
    ATTRIBUTION_PREDICTORS
)


COVERAGE_SENSITIVITY_THRESHOLD = 50.0


# ============================================================
# 2. Reload similarity results if needed
# ============================================================

if "pairwise_similarity" not in globals():

    pairwise_similarity = pd.read_csv(
        DQ_DIR
        / "pairwise_system_similarity.csv"
    )


if "system_similarity_summary" not in globals():

    system_similarity_summary = pd.read_csv(
        DQ_DIR
        / "system_average_similarity.csv"
    )


# ============================================================
# 3. Build outcome + predictor data-quality diagnostics
# ============================================================

diagnostic_rows = []

predictor_missingness_rows = []


for outcome in DQ_OUTCOMES:

    systems = sorted(
        system_similarity_summary.loc[
            system_similarity_summary[
                "Outcome"
            ]
            == outcome,
            "Country",
        ]
        .unique()
    )


    for country in systems:

        df = analysis_frames[
            country
        ]


        # ----------------------------------------------------
        # Validate required columns
        # ----------------------------------------------------

        required = [
            outcome,
            "W_FSTUWT",
            "CNTSCHID",
            *DQ_PREDICTORS,
        ]


        missing_columns = [
            col
            for col in required
            if col not in df.columns
        ]


        if missing_columns:

            raise RuntimeError(
                f"{country} — {outcome}: "
                f"missing columns "
                f"{missing_columns}"
            )


        # ----------------------------------------------------
        # Outcome / weight / school eligibility
        # ----------------------------------------------------

        y = pd.to_numeric(
            df[outcome],
            errors="coerce",
        )


        weights = pd.to_numeric(
            df["W_FSTUWT"],
            errors="coerce",
        )


        valid_outcome = (
            y.notna()
        )


        valid_weight = (
            weights.notna()
            &
            np.isfinite(
                weights
            )
            &
            (
                weights > 0
            )
        )


        valid_school = (
            df[
                "CNTSCHID"
            ]
            .notna()
        )


        # This matches the eligibility logic used
        # before XGBoost fitting.
        model_mask = (
            valid_outcome
            &
            valid_weight
            &
            valid_school
        )


        n_total = int(
            len(
                df
            )
        )


        n_outcome = int(
            valid_outcome.sum()
        )


        n_model = int(
            model_mask.sum()
        )


        n_schools_model = int(
            df.loc[
                model_mask,
                "CNTSCHID",
            ]
            .nunique()
        )


        outcome_coverage_pct = float(
            100
            * n_outcome
            / n_total
        )


        model_row_coverage_pct = float(
            100
            * n_model
            / n_total
        )


        # ----------------------------------------------------
        # Predictor missingness among actual modeling rows
        # ----------------------------------------------------

        predictor_missing_pcts = []


        for feature in DQ_PREDICTORS:

            values = pd.to_numeric(
                df.loc[
                    model_mask,
                    feature,
                ],
                errors="coerce",
            )


            missing_pct = float(
                100
                * values.isna().mean()
            )


            predictor_missing_pcts.append(
                missing_pct
            )


            predictor_missingness_rows.append(
                {
                    "Country": country,
                    "Outcome": outcome,
                    "Feature": feature,
                    "Missing_pct": (
                        missing_pct
                    ),
                }
            )


        predictor_missing_pcts = (
            np.asarray(
                predictor_missing_pcts,
                dtype=float,
            )
        )


        diagnostic_rows.append(
            {
                "Country": country,
                "Outcome": outcome,

                "N_total": (
                    n_total
                ),

                "N_outcome_observed": (
                    n_outcome
                ),

                "N_model": (
                    n_model
                ),

                "N_schools_model": (
                    n_schools_model
                ),

                "Outcome_coverage_pct": (
                    outcome_coverage_pct
                ),

                "Outcome_missing_pct": (
                    100
                    - outcome_coverage_pct
                ),

                "Model_row_coverage_pct": (
                    model_row_coverage_pct
                ),

                "Mean_predictor_missing_pct": float(
                    np.mean(
                        predictor_missing_pcts
                    )
                ),

                "Median_predictor_missing_pct": float(
                    np.median(
                        predictor_missing_pcts
                    )
                ),

                "Max_predictor_missing_pct": float(
                    np.max(
                        predictor_missing_pcts
                    )
                ),

                "N_predictors_missing_25pct": int(
                    (
                        predictor_missing_pcts
                        >= 25
                    ).sum()
                ),

                "N_predictors_missing_40pct": int(
                    (
                        predictor_missing_pcts
                        >= 40
                    ).sum()
                ),

                "Log10_N_model": float(
                    np.log10(
                        n_model
                    )
                ),
            }
        )


data_quality = pd.DataFrame(
    diagnostic_rows
)


predictor_missingness_long = pd.DataFrame(
    predictor_missingness_rows
)


data_quality.to_csv(
    DQ_DIR
    / "system_outcome_data_quality.csv",
    index=False,
)


predictor_missingness_long.to_csv(
    DQ_DIR
    / "system_outcome_predictor_missingness.csv",
    index=False,
)


# ============================================================
# 4. Merge data quality with system distinctiveness
# ============================================================

distinctiveness_data = (
    system_similarity_summary
    .merge(
        data_quality,
        on=[
            "Country",
            "Outcome",
        ],
        how="left",
        validate="one_to_one",
    )
)


distinctiveness_data[
    "Distinctiveness"
] = (
    1
    -
    distinctiveness_data[
        "MeanCosineToOthers"
    ]
)


distinctiveness_data.to_csv(
    DQ_DIR
    / "distinctiveness_with_data_quality.csv",
    index=False,
)


# ============================================================
# 5. Correlate distinctiveness with data-quality measures
#
# Spearman is used because we care about monotonic
# association and do not want to assume linearity.
# ============================================================

quality_metrics = [
    "Outcome_missing_pct",
    "Mean_predictor_missing_pct",
    "Max_predictor_missing_pct",
    "Log10_N_model",
    "N_schools_model",
]


correlation_rows = []


for outcome in DQ_OUTCOMES:

    temp = (
        distinctiveness_data[
            distinctiveness_data[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    for metric in quality_metrics:

        rho = (
            temp[
                "Distinctiveness"
            ]
            .corr(
                temp[
                    metric
                ],
                method="spearman",
            )
        )


        correlation_rows.append(
            {
                "Outcome": outcome,
                "Quality_metric": metric,
                "Spearman_rho": float(
                    rho
                ),
                "N_systems": int(
                    len(
                        temp
                    )
                ),
            }
        )


quality_correlations = pd.DataFrame(
    correlation_rows
)


quality_correlations.to_csv(
    DQ_DIR
    / "distinctiveness_data_quality_correlations.csv",
    index=False,
)


# ============================================================
# 6. 50% outcome-coverage sensitivity set
# ============================================================

coverage50_systems = {}


for outcome in DQ_OUTCOMES:

    eligible = (
        data_quality.loc[
            (
                data_quality[
                    "Outcome"
                ]
                == outcome
            )
            &
            (
                data_quality[
                    "Outcome_coverage_pct"
                ]
                >= COVERAGE_SENSITIVITY_THRESHOLD
            ),
            "Country",
        ]
        .tolist()
    )


    coverage50_systems[
        outcome
    ] = set(
        eligible
    )


# ============================================================
# 7. Recalculate pairwise summaries within >=50% systems
# ============================================================

coverage50_summary_rows = []

coverage50_system_rows = []


for outcome in DQ_OUTCOMES:

    eligible = (
        coverage50_systems[
            outcome
        ]
    )


    temp = (
        pairwise_similarity[
            (
                pairwise_similarity[
                    "Outcome"
                ]
                == outcome
            )
            &
            (
                pairwise_similarity[
                    "System_A"
                ]
                .isin(
                    eligible
                )
            )
            &
            (
                pairwise_similarity[
                    "System_B"
                ]
                .isin(
                    eligible
                )
            )
        ]
        .copy()
    )


    n_systems = len(
        eligible
    )


    expected_pairs = (
        n_systems
        * (
            n_systems - 1
        )
        // 2
    )


    if len(temp) != expected_pairs:

        raise RuntimeError(
            f"{outcome}: expected "
            f"{expected_pairs} coverage-50 pairs, "
            f"found {len(temp)}."
        )


    coverage50_summary_rows.append(
        {
            "Outcome": outcome,
            "N_systems": n_systems,
            "N_pairs": len(
                temp
            ),

            "Mean_cosine": float(
                temp[
                    "CosineSimilarity"
                ].mean()
            ),

            "Median_cosine": float(
                temp[
                    "CosineSimilarity"
                ].median()
            ),

            "SD_cosine": float(
                temp[
                    "CosineSimilarity"
                ].std()
            ),

            "Min_cosine": float(
                temp[
                    "CosineSimilarity"
                ].min()
            ),

            "Q05_cosine": float(
                temp[
                    "CosineSimilarity"
                ].quantile(
                    0.05
                )
            ),

            "Mean_rank_corr": float(
                temp[
                    "RankCorrelation"
                ].mean()
            ),

            "Mean_top5_jaccard": float(
                temp[
                    "Top5Jaccard"
                ].mean()
            ),

            "Mean_top7_jaccard": float(
                temp[
                    "Top7Jaccard"
                ].mean()
            ),
        }
    )


    # --------------------------------------------------------
    # Recalculate each eligible system's average cosine to
    # other eligible systems only.
    # --------------------------------------------------------

    for country in sorted(
        eligible
    ):

        country_pairs = (
            temp[
                (
                    temp[
                        "System_A"
                    ]
                    == country
                )
                |
                (
                    temp[
                        "System_B"
                    ]
                    == country
                )
            ]
        )


        coverage50_system_rows.append(
            {
                "Outcome": outcome,
                "Country": country,

                "MeanCosineToOthers_50": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].mean()
                ),

                "MedianCosineToOthers_50": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].median()
                ),

                "MinCosineToOthers_50": float(
                    country_pairs[
                        "CosineSimilarity"
                    ].min()
                ),
            }
        )


coverage50_summary = pd.DataFrame(
    coverage50_summary_rows
)


coverage50_system_similarity = pd.DataFrame(
    coverage50_system_rows
)


coverage50_summary.to_csv(
    DQ_DIR
    / "coverage50_similarity_summary.csv",
    index=False,
)


coverage50_system_similarity.to_csv(
    DQ_DIR
    / "coverage50_system_similarity.csv",
    index=False,
)


# ============================================================
# 8. Identify systems excluded by 50% sensitivity
# ============================================================

excluded_rows = []


for outcome in DQ_OUTCOMES:

    all_systems = set(
        data_quality.loc[
            data_quality[
                "Outcome"
            ]
            == outcome,
            "Country",
        ]
    )


    excluded = sorted(
        all_systems
        -
        coverage50_systems[
            outcome
        ]
    )


    for country in excluded:

        dq_row = (
            data_quality[
                (
                    data_quality[
                        "Country"
                    ]
                    == country
                )
                &
                (
                    data_quality[
                        "Outcome"
                    ]
                    == outcome
                )
            ]
            .iloc[0]
        )


        sim_row = (
            system_similarity_summary[
                (
                    system_similarity_summary[
                        "Country"
                    ]
                    == country
                )
                &
                (
                    system_similarity_summary[
                        "Outcome"
                    ]
                    == outcome
                )
            ]
            .iloc[0]
        )


        excluded_rows.append(
            {
                "Outcome": outcome,
                "Country": country,

                "Outcome_coverage_pct": (
                    dq_row[
                        "Outcome_coverage_pct"
                    ]
                ),

                "Mean_predictor_missing_pct": (
                    dq_row[
                        "Mean_predictor_missing_pct"
                    ]
                ),

                "Original_MeanCosineToOthers": (
                    sim_row[
                        "MeanCosineToOthers"
                    ]
                ),
            }
        )


excluded_coverage50 = pd.DataFrame(
    excluded_rows
)


excluded_coverage50.to_csv(
    DQ_DIR
    / "coverage50_excluded_systems.csv",
    index=False,
)


# ============================================================
# 9. PRINT: data-quality correlations
# ============================================================

print("\n")
print("=" * 110)
print("DISTINCTIVENESS vs DATA QUALITY")
print("=" * 110)


print(
    quality_correlations
    .pivot(
        index="Outcome",
        columns="Quality_metric",
        values="Spearman_rho",
    )
    .round(
        3
    )
    .to_string()
)


# ============================================================
# 10. PRINT: distinctive systems + their data quality
# ============================================================

print("\n")
print("=" * 110)
print("MOST DISTINCTIVE SYSTEMS WITH DATA QUALITY")
print("=" * 110)


for outcome in DQ_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )
    print(
        outcome
    )
    print(
        "-" * 90
    )


    temp = (
        distinctiveness_data[
            distinctiveness_data[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "MeanCosineToOthers"
        )
        .head(
            8
        )
        [
            [
                "Country",
                "MeanCosineToOthers",
                "Outcome_coverage_pct",
                "N_model",
                "N_schools_model",
                "Mean_predictor_missing_pct",
                "Max_predictor_missing_pct",
            ]
        ]
    )


    print(
        temp
        .round(
            2
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 11. PRINT: >=50% coverage sensitivity
# ============================================================

print("\n")
print("=" * 110)
print("50% OUTCOME-COVERAGE SENSITIVITY")
print("=" * 110)


print(
    coverage50_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 12. PRINT: most distinctive systems after restriction
# ============================================================

print("\n")
print("=" * 110)
print("MOST DISTINCTIVE SYSTEMS — >=50% COVERAGE ONLY")
print("=" * 110)


for outcome in DQ_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )
    print(
        outcome
    )
    print(
        "-" * 90
    )


    temp = (
        coverage50_system_similarity[
            coverage50_system_similarity[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "MeanCosineToOthers_50"
        )
        .head(
            5
        )
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 13. PRINT: excluded systems
# ============================================================

print("\n")
print("=" * 110)
print("SYSTEMS EXCLUDED BY >=50% COVERAGE SENSITIVITY")
print("=" * 110)


if excluded_coverage50.empty:

    print(
        "None."
    )

else:

    print(
        excluded_coverage50
        .sort_values(
            [
                "Outcome",
                "Outcome_coverage_pct",
            ]
        )
        .round(
            2
        )
        .to_string(
            index=False
        )
    )



DISTINCTIVENESS vs DATA QUALITY
Quality_metric    Log10_N_model  Max_predictor_missing_pct  Mean_predictor_missing_pct  N_schools_model  Outcome_missing_pct
Outcome                                                                                                                     
BEINGBULLIED              0.056                      0.154                       0.187           -0.136               -0.206
BULLY_PHYSICAL           -0.165                      0.068                       0.078           -0.167                0.030
BULLY_RELATIONAL          0.109                      0.085                       0.104           -0.006               -0.110
BULLY_THREAT             -0.125                      0.068                       0.065           -0.022                0.099
BULLY_VERBAL              0.032                      0.078                       0.085           -0.172               -0.131


MOST DISTINCTIVE SYSTEMS WITH DATA QUALITY


--------------------------------------------

In [32]:
# ============================================================
# CROSS-SYSTEM SIMILARITY AFTER REMOVING BELONG
#
# Purpose:
# Determine whether high international similarity is driven
# mainly by the universal BELONG predictor, or whether the
# remaining 13-predictor architecture is also conserved.
#
# No model fitting.
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

BACKBONE_DIR = Path(
    "../results/system_similarity"
)

BACKBONE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BACKBONE_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


REMOVED_FEATURE = "BELONG"


REMAINING_FEATURES = [
    feature
    for feature in ATTRIBUTION_PREDICTORS
    if feature != REMOVED_FEATURE
]


print(
    "Removed feature:",
    REMOVED_FEATURE
)

print(
    "Remaining predictors:",
    len(REMAINING_FEATURES)
)


# ============================================================
# 2. Load frozen primary rankings if needed
# ============================================================

if "rankings_df" not in globals():

    rankings_df = pd.read_csv(
        "../results/primary_attribution/"
        "oof_shap_rankings.csv"
    )


# ============================================================
# 3. Construct 13-feature renormalized profiles
# ============================================================

no_belong = (
    rankings_df[
        rankings_df[
            "Feature"
        ]
        != REMOVED_FEATURE
    ]
    .copy()
)


# For every country/outcome:
# renormalize remaining importance shares to sum to 1.

remaining_sum = (
    no_belong
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    ["ImportanceShare"]
    .transform(
        "sum"
    )
)


if (
    remaining_sum
    <= 0
).any():

    raise RuntimeError(
        "Non-positive remaining SHAP total "
        "after removing BELONG."
    )


no_belong[
    "ImportanceShare_NoBelong"
] = (
    no_belong[
        "ImportanceShare"
    ]
    /
    remaining_sum
)


# ============================================================
# 4. QA: every model should have 13 predictors
# ============================================================

feature_counts = (
    no_belong
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    ["Feature"]
    .nunique()
)


bad_counts = (
    feature_counts[
        feature_counts
        != len(
            REMAINING_FEATURES
        )
    ]
)


if len(
    bad_counts
) > 0:

    raise RuntimeError(
        "Some models do not contain exactly "
        "13 predictors after BELONG removal."
    )


share_sums = (
    no_belong
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "ImportanceShare_NoBelong"
    ]
    .sum()
)


print(
    "Renormalized share range:",
    share_sums.min(),
    "to",
    share_sums.max(),
)


# ============================================================
# 5. Re-rank remaining predictors
# ============================================================

no_belong[
    "Rank_NoBelong"
] = (
    no_belong
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "ImportanceShare_NoBelong"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


no_belong.to_csv(
    BACKBONE_DIR
    / "rankings_without_belong.csv",
    index=False,
)


# ============================================================
# 6. Pairwise similarity without BELONG
# ============================================================

no_belong_pair_rows = []


for outcome in BACKBONE_OUTCOMES:

    outcome_data = (
        no_belong[
            no_belong[
                "Outcome"
            ]
            == outcome
        ]
    )


    systems = sorted(
        outcome_data[
            "Country"
        ]
        .unique()
    )


    profiles = {}


    for country in systems:

        temp = (
            outcome_data[
                outcome_data[
                    "Country"
                ]
                == country
            ]
            .set_index(
                "Feature"
            )
            .loc[
                REMAINING_FEATURES
            ]
        )


        share_vector = (
            temp[
                "ImportanceShare_NoBelong"
            ]
            .to_numpy(
                dtype=float
            )
        )


        rank_vector = (
            temp[
                "Rank_NoBelong"
            ]
            .to_numpy(
                dtype=float
            )
        )


        top5 = set(
            temp.loc[
                temp[
                    "Rank_NoBelong"
                ]
                <= 5
            ]
            .index
        )


        top7 = set(
            temp.loc[
                temp[
                    "Rank_NoBelong"
                ]
                <= 7
            ]
            .index
        )


        profiles[
            country
        ] = {
            "shares": share_vector,
            "ranks": rank_vector,
            "top5": top5,
            "top7": top7,
        }


    for (
        system_a,
        system_b,
    ) in combinations(
        systems,
        2,
    ):

        a = profiles[
            system_a
        ]

        b = profiles[
            system_b
        ]


        # ----------------------------------------------------
        # Cosine
        # ----------------------------------------------------

        cosine = float(
            np.dot(
                a[
                    "shares"
                ],
                b[
                    "shares"
                ],
            )
            /
            (
                np.linalg.norm(
                    a[
                        "shares"
                    ]
                )
                *
                np.linalg.norm(
                    b[
                        "shares"
                    ]
                )
            )
        )


        # ----------------------------------------------------
        # Spearman rank correlation
        # ----------------------------------------------------

        rank_corr = float(
            np.corrcoef(
                a[
                    "ranks"
                ],
                b[
                    "ranks"
                ],
            )[0, 1]
        )


        # ----------------------------------------------------
        # Jaccard
        # ----------------------------------------------------

        top5_jaccard = float(
            len(
                a[
                    "top5"
                ]
                &
                b[
                    "top5"
                ]
            )
            /
            len(
                a[
                    "top5"
                ]
                |
                b[
                    "top5"
                ]
            )
        )


        top7_jaccard = float(
            len(
                a[
                    "top7"
                ]
                &
                b[
                    "top7"
                ]
            )
            /
            len(
                a[
                    "top7"
                ]
                |
                b[
                    "top7"
                ]
            )
        )


        no_belong_pair_rows.append(
            {
                "Outcome": outcome,
                "System_A": system_a,
                "System_B": system_b,

                "Cosine_NoBelong": (
                    cosine
                ),

                "RankCorrelation_NoBelong": (
                    rank_corr
                ),

                "Top5Jaccard_NoBelong": (
                    top5_jaccard
                ),

                "Top7Jaccard_NoBelong": (
                    top7_jaccard
                ),
            }
        )


pairwise_no_belong = pd.DataFrame(
    no_belong_pair_rows
)


pairwise_no_belong.to_csv(
    BACKBONE_DIR
    / "pairwise_similarity_without_belong.csv",
    index=False,
)


# ============================================================
# 7. Summary without BELONG
# ============================================================

no_belong_summary = (
    pairwise_no_belong
    .groupby(
        "Outcome"
    )
    .agg(

        N_pairs=(
            "System_A",
            "count",
        ),

        Mean_cosine_no_belong=(
            "Cosine_NoBelong",
            "mean",
        ),

        Median_cosine_no_belong=(
            "Cosine_NoBelong",
            "median",
        ),

        SD_cosine_no_belong=(
            "Cosine_NoBelong",
            "std",
        ),

        Min_cosine_no_belong=(
            "Cosine_NoBelong",
            "min",
        ),

        Q05_cosine_no_belong=(
            "Cosine_NoBelong",
            lambda x: float(
                x.quantile(
                    0.05
                )
            ),
        ),

        Q95_cosine_no_belong=(
            "Cosine_NoBelong",
            lambda x: float(
                x.quantile(
                    0.95
                )
            ),
        ),

        Mean_rank_corr_no_belong=(
            "RankCorrelation_NoBelong",
            "mean",
        ),

        Median_rank_corr_no_belong=(
            "RankCorrelation_NoBelong",
            "median",
        ),

        Mean_top5_jaccard_no_belong=(
            "Top5Jaccard_NoBelong",
            "mean",
        ),

        Mean_top7_jaccard_no_belong=(
            "Top7Jaccard_NoBelong",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 8. Compare against original similarity
# ============================================================

if "similarity_summary" not in globals():

    similarity_summary = pd.read_csv(
        BACKBONE_DIR
        / "similarity_summary_by_outcome.csv"
    )


comparison = (
    similarity_summary[
        [
            "Outcome",
            "Mean_cosine",
            "Median_cosine",
            "Mean_rank_corr",
            "Mean_top5_jaccard",
            "Mean_top7_jaccard",
        ]
    ]
    .merge(
        no_belong_summary,
        on="Outcome",
        how="inner",
        validate="one_to_one",
    )
)


comparison[
    "Delta_mean_cosine"
] = (
    comparison[
        "Mean_cosine_no_belong"
    ]
    -
    comparison[
        "Mean_cosine"
    ]
)


comparison[
    "Delta_rank_corr"
] = (
    comparison[
        "Mean_rank_corr_no_belong"
    ]
    -
    comparison[
        "Mean_rank_corr"
    ]
)


comparison.to_csv(
    BACKBONE_DIR
    / "original_vs_no_belong_similarity.csv",
    index=False,
)


# ============================================================
# 9. System distinctiveness without BELONG
# ============================================================

system_rows = []


for outcome in BACKBONE_OUTCOMES:

    temp = (
        pairwise_no_belong[
            pairwise_no_belong[
                "Outcome"
            ]
            == outcome
        ]
    )


    systems = sorted(
        set(
            temp[
                "System_A"
            ]
        )
        |
        set(
            temp[
                "System_B"
            ]
        )
    )


    for country in systems:

        country_pairs = (
            temp[
                (
                    temp[
                        "System_A"
                    ]
                    == country
                )
                |
                (
                    temp[
                        "System_B"
                    ]
                    == country
                )
            ]
        )


        system_rows.append(
            {
                "Outcome": outcome,
                "Country": country,

                "MeanCosineToOthers_NoBelong": float(
                    country_pairs[
                        "Cosine_NoBelong"
                    ]
                    .mean()
                ),

                "MedianCosineToOthers_NoBelong": float(
                    country_pairs[
                        "Cosine_NoBelong"
                    ]
                    .median()
                ),

                "MinCosineToOthers_NoBelong": float(
                    country_pairs[
                        "Cosine_NoBelong"
                    ]
                    .min()
                ),

                "MeanRankCorrToOthers_NoBelong": float(
                    country_pairs[
                        "RankCorrelation_NoBelong"
                    ]
                    .mean()
                ),
            }
        )


system_no_belong = pd.DataFrame(
    system_rows
)


system_no_belong.to_csv(
    BACKBONE_DIR
    / "system_distinctiveness_without_belong.csv",
    index=False,
)


# ============================================================
# 10. Mean remaining-predictor architecture by outcome
# ============================================================

remaining_architecture = (
    no_belong
    .groupby(
        [
            "Outcome",
            "Feature",
        ],
        as_index=False,
    )
    .agg(

        Mean_share_no_belong=(
            "ImportanceShare_NoBelong",
            "mean",
        ),

        Median_share_no_belong=(
            "ImportanceShare_NoBelong",
            "median",
        ),

        Mean_rank_no_belong=(
            "Rank_NoBelong",
            "mean",
        ),

        Median_rank_no_belong=(
            "Rank_NoBelong",
            "median",
        ),
    )
)


remaining_architecture[
    "ArchitectureRank"
] = (
    remaining_architecture
    .groupby(
        "Outcome"
    )
    [
        "Mean_share_no_belong"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


remaining_architecture = (
    remaining_architecture
    .sort_values(
        [
            "Outcome",
            "ArchitectureRank",
        ]
    )
)


remaining_architecture.to_csv(
    BACKBONE_DIR
    / "mean_architecture_without_belong.csv",
    index=False,
)


# ============================================================
# 11. PRINT: original vs no-BELONG similarity
# ============================================================

print("\n")
print("=" * 110)
print("ORIGINAL vs BELONG-REMOVED CROSS-SYSTEM SIMILARITY")
print("=" * 110)


print(
    comparison[
        [
            "Outcome",

            "Mean_cosine",
            "Mean_cosine_no_belong",
            "Delta_mean_cosine",

            "Mean_rank_corr",
            "Mean_rank_corr_no_belong",
            "Delta_rank_corr",

            "Mean_top5_jaccard_no_belong",
            "Mean_top7_jaccard_no_belong",
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 12. PRINT: no-BELONG distribution
# ============================================================

print("\n")
print("=" * 110)
print("BELONG-REMOVED SIMILARITY DISTRIBUTIONS")
print("=" * 110)


print(
    no_belong_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 13. PRINT: most distinctive systems without BELONG
# ============================================================

print("\n")
print("=" * 110)
print("MOST DISTINCTIVE SYSTEMS AFTER REMOVING BELONG")
print("=" * 110)


for outcome in BACKBONE_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        system_no_belong[
            system_no_belong[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "MeanCosineToOthers_NoBelong"
        )
        .head(
            5
        )
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 14. PRINT: remaining predictor architecture
# ============================================================

print("\n")
print("=" * 110)
print("PREDICTOR ARCHITECTURE AFTER REMOVING BELONG")
print("=" * 110)


for outcome in BACKBONE_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        remaining_architecture[
            remaining_architecture[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "ArchitectureRank",
                "Feature",
                "Mean_share_no_belong",
                "Median_rank_no_belong",
            ]
        ]
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )

Removed feature: BELONG
Remaining predictors: 13
Renormalized share range: 0.9999999999999999 to 1.0


ORIGINAL vs BELONG-REMOVED CROSS-SYSTEM SIMILARITY
         Outcome  Mean_cosine  Mean_cosine_no_belong  Delta_mean_cosine  Mean_rank_corr  Mean_rank_corr_no_belong  Delta_rank_corr  Mean_top5_jaccard_no_belong  Mean_top7_jaccard_no_belong
    BEINGBULLIED       0.9546                 0.9286            -0.0260          0.6807                    0.6011          -0.0796                       0.4855                       0.5730
  BULLY_PHYSICAL       0.9437                 0.9275            -0.0162          0.5763                    0.4743          -0.1020                       0.4397                       0.5093
BULLY_RELATIONAL       0.9473                 0.9173            -0.0300          0.4636                    0.3297          -0.1339                       0.3546                       0.4834
    BULLY_THREAT       0.9357                 0.9164            -0.0193          0.4886   

In [33]:
# ============================================================
# SYSTEM DEVIATIONS FROM THE INTERNATIONAL CONSENSUS PROFILE
#
# Purpose:
# For each outcome:
#
# 1. Construct the mean international SHAP profile.
# 2. Compare every system against that profile.
# 3. Quantify how far each system deviates.
# 4. Identify WHICH predictors drive that deviation.
#
# Uses frozen OOF-SHAP attribution.
# No XGBoost refitting.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

DEV_DIR = Path(
    "../results/system_deviation"
)

DEV_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DEV_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


DEV_FEATURES = list(
    ATTRIBUTION_PREDICTORS
)


# ============================================================
# 2. Load frozen rankings if needed
# ============================================================

if "rankings_df" not in globals():

    rankings_df = pd.read_csv(
        "../results/primary_attribution/"
        "oof_shap_rankings.csv"
    )


dev_data = rankings_df.copy()


# ============================================================
# 3. Build international consensus profile
#
# For each outcome + feature:
# average normalized OOF-SHAP importance across systems.
# ============================================================

consensus = (
    dev_data
    .groupby(
        [
            "Outcome",
            "Feature",
        ],
        as_index=False,
    )
    .agg(

        ConsensusMeanShare=(
            "ImportanceShare",
            "mean",
        ),

        ConsensusMedianShare=(
            "ImportanceShare",
            "median",
        ),

        ConsensusSDShare=(
            "ImportanceShare",
            "std",
        ),
    )
)


# ============================================================
# 4. Normalize consensus exactly to sum to 1
# ============================================================

consensus[
    "ConsensusMeanShare"
] = (
    consensus[
        "ConsensusMeanShare"
    ]
    /
    consensus
    .groupby(
        "Outcome"
    )
    [
        "ConsensusMeanShare"
    ]
    .transform(
        "sum"
    )
)


consensus.to_csv(
    DEV_DIR
    / "international_consensus_profiles.csv",
    index=False,
)


# ============================================================
# 5. Merge each system with consensus
# ============================================================

deviation_long = (
    dev_data[
        [
            "Country",
            "Outcome",
            "Feature",
            "Rank",
            "ImportanceShare",
        ]
    ]
    .merge(
        consensus[
            [
                "Outcome",
                "Feature",
                "ConsensusMeanShare",
                "ConsensusSDShare",
            ]
        ],
        on=[
            "Outcome",
            "Feature",
        ],
        how="left",
        validate="many_to_one",
    )
)


# Raw signed deviation
#
# Positive:
# feature receives MORE attribution in this system
# than the international outcome average.
#
# Negative:
# feature receives LESS attribution.

deviation_long[
    "ShareDeviation"
] = (
    deviation_long[
        "ImportanceShare"
    ]
    -
    deviation_long[
        "ConsensusMeanShare"
    ]
)


deviation_long[
    "AbsoluteShareDeviation"
] = (
    deviation_long[
        "ShareDeviation"
    ]
    .abs()
)


# ============================================================
# 6. Standardized deviation
#
# Useful because some features naturally vary more
# internationally than others.
#
# This is descriptive — not a statistical z-test.
# ============================================================

deviation_long[
    "StandardizedDeviation"
] = (
    deviation_long[
        "ShareDeviation"
    ]
    /
    deviation_long[
        "ConsensusSDShare"
    ]
    .replace(
        0,
        np.nan,
    )
)


# ============================================================
# 7. Rank each system's deviation drivers
# ============================================================

deviation_long[
    "DeviationDriverRank"
] = (
    deviation_long
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "AbsoluteShareDeviation"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


deviation_long.to_csv(
    DEV_DIR
    / "system_feature_deviations.csv",
    index=False,
)


# ============================================================
# 8. System-level distance from consensus
#
# Compute several complementary distances.
# ============================================================

system_deviation_rows = []


for outcome in DEV_OUTCOMES:

    outcome_consensus = (
        consensus[
            consensus[
                "Outcome"
            ]
            == outcome
        ]
        .set_index(
            "Feature"
        )
        .loc[
            DEV_FEATURES,
            "ConsensusMeanShare",
        ]
        .to_numpy(
            dtype=float
        )
    )


    systems = sorted(
        dev_data.loc[
            dev_data[
                "Outcome"
            ]
            == outcome,
            "Country",
        ]
        .unique()
    )


    for country in systems:

        temp = (
            dev_data[
                (
                    dev_data[
                        "Country"
                    ]
                    == country
                )
                &
                (
                    dev_data[
                        "Outcome"
                    ]
                    == outcome
                )
            ]
            .set_index(
                "Feature"
            )
            .loc[
                DEV_FEATURES
            ]
        )


        system_vector = (
            temp[
                "ImportanceShare"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # Cosine similarity to consensus
        # ----------------------------------------------------

        cosine_to_consensus = float(
            np.dot(
                system_vector,
                outcome_consensus,
            )
            /
            (
                np.linalg.norm(
                    system_vector
                )
                *
                np.linalg.norm(
                    outcome_consensus
                )
            )
        )


        # ----------------------------------------------------
        # L1 distance
        #
        # Sum of absolute share differences.
        # ----------------------------------------------------

        l1_distance = float(
            np.abs(
                system_vector
                -
                outcome_consensus
            )
            .sum()
        )


        # ----------------------------------------------------
        # Euclidean distance
        # ----------------------------------------------------

        l2_distance = float(
            np.sqrt(
                np.square(
                    system_vector
                    -
                    outcome_consensus
                )
                .sum()
            )
        )


        system_deviation_rows.append(
            {
                "Country": country,
                "Outcome": outcome,

                "CosineToConsensus": (
                    cosine_to_consensus
                ),

                "CosineDistanceToConsensus": (
                    1
                    -
                    cosine_to_consensus
                ),

                "L1DistanceToConsensus": (
                    l1_distance
                ),

                "L2DistanceToConsensus": (
                    l2_distance
                ),
            }
        )


system_deviation = pd.DataFrame(
    system_deviation_rows
)


system_deviation.to_csv(
    DEV_DIR
    / "system_distances_from_consensus.csv",
    index=False,
)


# ============================================================
# 9. Extract top deviation drivers
# ============================================================

top_drivers = (
    deviation_long[
        deviation_long[
            "DeviationDriverRank"
        ]
        <= 5
    ]
    .copy()
)


top_drivers.to_csv(
    DEV_DIR
    / "top5_deviation_drivers.csv",
    index=False,
)


# ============================================================
# 10. Most over-emphasized predictor per system
# ============================================================

positive_deviation = (
    deviation_long[
        deviation_long[
            "ShareDeviation"
        ]
        > 0
    ]
    .copy()
)


most_overemphasized = (
    positive_deviation
    .sort_values(
        [
            "Country",
            "Outcome",
            "ShareDeviation",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .groupby(
        [
            "Country",
            "Outcome",
        ],
        as_index=False,
    )
    .first()
)


most_overemphasized.to_csv(
    DEV_DIR
    / "most_overemphasized_predictor.csv",
    index=False,
)


# ============================================================
# 11. Which predictors most often drive system deviations?
#
# Count predictor appearing among top-3 absolute deviations.
# ============================================================

top3_driver_frequency = (
    deviation_long[
        deviation_long[
            "DeviationDriverRank"
        ]
        <= 3
    ]
    .groupby(
        [
            "Outcome",
            "Feature",
        ]
    )
    .size()
    .reset_index(
        name="Top3DeviationCount"
    )
)


systems_per_outcome = (
    dev_data
    .groupby(
        "Outcome"
    )
    ["Country"]
    .nunique()
    .rename(
        "N_systems"
    )
    .reset_index()
)


top3_driver_frequency = (
    top3_driver_frequency
    .merge(
        systems_per_outcome,
        on="Outcome",
        how="left",
    )
)


top3_driver_frequency[
    "Top3DeviationPct"
] = (
    100
    *
    top3_driver_frequency[
        "Top3DeviationCount"
    ]
    /
    top3_driver_frequency[
        "N_systems"
    ]
)


top3_driver_frequency = (
    top3_driver_frequency
    .sort_values(
        [
            "Outcome",
            "Top3DeviationCount",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


top3_driver_frequency.to_csv(
    DEV_DIR
    / "deviation_driver_frequency.csv",
    index=False,
)


# ============================================================
# 12. QA
# ============================================================

consensus_sums = (
    consensus
    .groupby(
        "Outcome"
    )
    [
        "ConsensusMeanShare"
    ]
    .sum()
)


print("\n")
print("=" * 110)
print("CONSENSUS PROFILE QA")
print("=" * 110)


print(
    consensus_sums
    .round(
        10
    )
    .to_string()
)


# ============================================================
# 13. Print international consensus architecture
# ============================================================

print("\n")
print("=" * 110)
print("INTERNATIONAL CONSENSUS PREDICTOR PROFILES")
print("=" * 110)


for outcome in DEV_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )

    print(
        outcome
    )

    print(
        "-" * 90
    )


    temp = (
        consensus[
            consensus[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "ConsensusMeanShare",
            ascending=False,
        )
        [
            [
                "Feature",
                "ConsensusMeanShare",
                "ConsensusSDShare",
            ]
        ]
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 14. Print most distinctive systems by consensus distance
# ============================================================

print("\n")
print("=" * 110)
print("SYSTEMS FARTHEST FROM INTERNATIONAL CONSENSUS")
print("=" * 110)


for outcome in DEV_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )

    print(
        outcome
    )

    print(
        "-" * 90
    )


    temp = (
        system_deviation[
            system_deviation[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineToConsensus"
        )
        .head(
            8
        )
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 15. Print WHY the 5 most distinctive systems differ
# ============================================================

print("\n")
print("=" * 110)
print("TOP DEVIATION DRIVERS FOR DISTINCTIVE SYSTEMS")
print("=" * 110)


for outcome in DEV_OUTCOMES:

    distinctive = (
        system_deviation[
            system_deviation[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineToConsensus"
        )
        .head(
            5
        )
        [
            "Country"
        ]
        .tolist()
    )


    print("\n")
    print(
        "=" * 90
    )

    print(
        outcome
    )

    print(
        "=" * 90
    )


    for country in distinctive:

        print(
            f"\n{country}"
        )


        temp = (
            deviation_long[
                (
                    deviation_long[
                        "Outcome"
                    ]
                    == outcome
                )
                &
                (
                    deviation_long[
                        "Country"
                    ]
                    == country
                )
            ]
            .sort_values(
                "AbsoluteShareDeviation",
                ascending=False,
            )
            .head(
                5
            )
            [
                [
                    "Feature",
                    "ImportanceShare",
                    "ConsensusMeanShare",
                    "ShareDeviation",
                    "StandardizedDeviation",
                ]
            ]
        )


        print(
            temp
            .round(
                4
            )
            .to_string(
                index=False
            )
        )


# ============================================================
# 16. Print common deviation-driving predictors
# ============================================================

print("\n")
print("=" * 110)
print("MOST COMMON DRIVERS OF CROSS-SYSTEM DEVIATION")
print("=" * 110)


for outcome in DEV_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )

    print(
        outcome
    )

    print(
        "-" * 90
    )


    temp = (
        top3_driver_frequency[
            top3_driver_frequency[
                "Outcome"
            ]
            == outcome
        ]
        .head(
            10
        )
    )


    print(
        temp
        .round(
            2
        )
        .to_string(
            index=False
        )
    )



CONSENSUS PROFILE QA
Outcome
BEINGBULLIED        1.0
BULLY_PHYSICAL      1.0
BULLY_RELATIONAL    1.0
BULLY_THREAT        1.0
BULLY_VERBAL        1.0


INTERNATIONAL CONSENSUS PREDICTOR PROFILES


------------------------------------------------------------------------------------------
BEINGBULLIED
------------------------------------------------------------------------------------------
   Feature  ConsensusMeanShare  ConsensusSDShare
    BELONG              0.2952            0.0549
  DISCLIMA              0.1234            0.0243
 ST004D01T              0.1121            0.0448
   JOYREAD              0.0540            0.0155
      ESCS              0.0538            0.0128
  MASTGOAL              0.0474            0.0133
  ATTLNACT              0.0472            0.0178
  STIMREAD              0.0456            0.0091
  EDUSHORT              0.0445            0.0140
ADAPTIVITY              0.0416            0.0082
   METASUM              0.0390            0.0131
    UNDREM         

In [34]:
# ============================================================
# FOLD-LEVEL PERFORMANCE STABILITY QC
#
# No model fitting.
# Uses the saved 5-fold performance results.
#
# Purpose:
# Check whether good overall performance is supported
# consistently across held-out school folds.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Load fold performance
# ============================================================

MODEL_DIR = Path(
    "../results/primary_models"
)

fold_perf = pd.read_csv(
    MODEL_DIR
    / "fold_performance.csv"
)


print(
    "Shape:",
    fold_perf.shape
)

print(
    "\nColumns:"
)

print(
    fold_perf.columns.tolist()
)


# ============================================================
# 2. Separate regression and classification
# ============================================================

reg = (
    fold_perf[
        fold_perf[
            "Outcome"
        ]
        == "BEINGBULLIED"
    ]
    .copy()
)


clf = (
    fold_perf[
        fold_perf[
            "Outcome"
        ]
        != "BEINGBULLIED"
    ]
    .copy()
)


# ============================================================
# 3. REGRESSION QC
# ============================================================

reg[
    "RMSE_improvement"
] = (
    1
    -
    (
        reg[
            "Model_RMSE"
        ]
        /
        reg[
            "Baseline_RMSE"
        ]
    )
)


reg[
    "Beats_baseline_RMSE"
] = (
    reg[
        "Model_RMSE"
    ]
    <
    reg[
        "Baseline_RMSE"
    ]
)


reg_system_summary = (
    reg
    .groupby(
        "Country"
    )
    .agg(

        N_folds=(
            "Fold",
            "count",
        ),

        Mean_R2=(
            "Model_R2",
            "mean",
        ),

        Median_R2=(
            "Model_R2",
            "median",
        ),

        Min_R2=(
            "Model_R2",
            "min",
        ),

        Max_R2=(
            "Model_R2",
            "max",
        ),

        SD_R2=(
            "Model_R2",
            "std",
        ),

        N_negative_R2=(
            "Model_R2",
            lambda x: int(
                (
                    x < 0
                ).sum()
            ),
        ),

        Mean_RMSE_improvement=(
            "RMSE_improvement",
            "mean",
        ),

        N_folds_beating_RMSE_baseline=(
            "Beats_baseline_RMSE",
            "sum",
        ),
    )
    .reset_index()
)


# ============================================================
# 4. CLASSIFICATION QC
# ============================================================

clf[
    "ROC_above_chance"
] = (
    clf[
        "Model_ROC_AUC"
    ]
    >
    0.5
)


clf[
    "ROC_above_055"
] = (
    clf[
        "Model_ROC_AUC"
    ]
    >
    0.55
)


clf[
    "PR_beats_baseline"
] = (
    clf[
        "Model_PR_AUC"
    ]
    >
    clf[
        "Baseline_PR_AUC"
    ]
)


clf[
    "PR_improvement"
] = (
    clf[
        "Model_PR_AUC"
    ]
    -
    clf[
        "Baseline_PR_AUC"
    ]
)


clf_system_summary = (
    clf
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    .agg(

        N_folds=(
            "Fold",
            "count",
        ),

        Mean_ROC_AUC=(
            "Model_ROC_AUC",
            "mean",
        ),

        Median_ROC_AUC=(
            "Model_ROC_AUC",
            "median",
        ),

        Min_ROC_AUC=(
            "Model_ROC_AUC",
            "min",
        ),

        Max_ROC_AUC=(
            "Model_ROC_AUC",
            "max",
        ),

        SD_ROC_AUC=(
            "Model_ROC_AUC",
            "std",
        ),

        N_folds_ROC_above_chance=(
            "ROC_above_chance",
            "sum",
        ),

        N_folds_ROC_above_055=(
            "ROC_above_055",
            "sum",
        ),

        Mean_PR_AUC=(
            "Model_PR_AUC",
            "mean",
        ),

        Min_PR_AUC=(
            "Model_PR_AUC",
            "min",
        ),

        Mean_PR_improvement=(
            "PR_improvement",
            "mean",
        ),

        N_folds_PR_beating_baseline=(
            "PR_beats_baseline",
            "sum",
        ),
    )
    .reset_index()
)


# ============================================================
# 5. Overall regression QC summary
# ============================================================

reg_qc = pd.DataFrame(
    {
        "Metric": [
            "Systems",
            "All 5 folds R2 > 0",
            "At least 4/5 folds R2 > 0",
            "All 5 folds beat RMSE baseline",
            "Median system Min_R2",
            "Worst fold R2 overall",
            "Median system SD_R2",
        ],

        "Value": [
            len(
                reg_system_summary
            ),

            int(
                (
                    reg_system_summary[
                        "N_negative_R2"
                    ]
                    == 0
                ).sum()
            ),

            int(
                (
                    reg_system_summary[
                        "N_negative_R2"
                    ]
                    <= 1
                ).sum()
            ),

            int(
                (
                    reg_system_summary[
                        "N_folds_beating_RMSE_baseline"
                    ]
                    == 5
                ).sum()
            ),

            float(
                reg_system_summary[
                    "Min_R2"
                ]
                .median()
            ),

            float(
                reg[
                    "Model_R2"
                ]
                .min()
            ),

            float(
                reg_system_summary[
                    "SD_R2"
                ]
                .median()
            ),
        ],
    }
)


# ============================================================
# 6. Classification QC by outcome
# ============================================================

classification_qc_rows = []


for outcome in sorted(
    clf[
        "Outcome"
    ]
    .unique()
):

    temp = (
        clf_system_summary[
            clf_system_summary[
                "Outcome"
            ]
            == outcome
        ]
    )


    classification_qc_rows.append(
        {
            "Outcome": outcome,

            "N_models": len(
                temp
            ),

            "All5_ROC_above_0.50": int(
                (
                    temp[
                        "N_folds_ROC_above_chance"
                    ]
                    == 5
                ).sum()
            ),

            "AtLeast4_ROC_above_0.50": int(
                (
                    temp[
                        "N_folds_ROC_above_chance"
                    ]
                    >= 4
                ).sum()
            ),

            "All5_ROC_above_0.55": int(
                (
                    temp[
                        "N_folds_ROC_above_055"
                    ]
                    == 5
                ).sum()
            ),

            "All5_PR_beats_baseline": int(
                (
                    temp[
                        "N_folds_PR_beating_baseline"
                    ]
                    == 5
                ).sum()
            ),

            "Median_min_fold_ROC": float(
                temp[
                    "Min_ROC_AUC"
                ]
                .median()
            ),

            "Worst_fold_ROC": float(
                temp[
                    "Min_ROC_AUC"
                ]
                .min()
            ),

            "Median_ROC_SD": float(
                temp[
                    "SD_ROC_AUC"
                ]
                .median()
            ),

            "Median_PR_improvement": float(
                temp[
                    "Mean_PR_improvement"
                ]
                .median()
            ),
        }
    )


classification_qc = pd.DataFrame(
    classification_qc_rows
)


# ============================================================
# 7. Identify potentially unstable regression systems
# ============================================================

reg_unstable = (
    reg_system_summary[
        (
            reg_system_summary[
                "N_negative_R2"
            ]
            > 0
        )
        |
        (
            reg_system_summary[
                "N_folds_beating_RMSE_baseline"
            ]
            < 5
        )
    ]
    .sort_values(
        "Min_R2"
    )
)


# ============================================================
# 8. Identify potentially unstable classifiers
# ============================================================

clf_unstable = (
    clf_system_summary[
        (
            clf_system_summary[
                "N_folds_ROC_above_chance"
            ]
            < 5
        )
        |
        (
            clf_system_summary[
                "N_folds_PR_beating_baseline"
            ]
            < 5
        )
    ]
    .sort_values(
        [
            "Outcome",
            "Min_ROC_AUC",
        ]
    )
)


# ============================================================
# 9. Save
# ============================================================

QC_DIR = Path(
    "../results/model_qc"
)

QC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


reg_system_summary.to_csv(
    QC_DIR
    / "regression_fold_stability.csv",
    index=False,
)


clf_system_summary.to_csv(
    QC_DIR
    / "classification_fold_stability.csv",
    index=False,
)


reg_unstable.to_csv(
    QC_DIR
    / "regression_unstable_models.csv",
    index=False,
)


clf_unstable.to_csv(
    QC_DIR
    / "classification_unstable_models.csv",
    index=False,
)


# ============================================================
# 10. Print
# ============================================================

print("\n")
print("=" * 105)
print("REGRESSION FOLD STABILITY")
print("=" * 105)

print(
    reg_qc
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 105)
print("CLASSIFICATION FOLD STABILITY")
print("=" * 105)

print(
    classification_qc
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 105)
print("REGRESSION MODELS WITH ANY FOLD WARNING")
print("=" * 105)

if reg_unstable.empty:

    print(
        "None."
    )

else:

    print(
        reg_unstable
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 105)
print("CLASSIFICATION MODELS WITH ANY FOLD WARNING")
print("=" * 105)

if clf_unstable.empty:

    print(
        "None."
    )

else:

    print(
        clf_unstable
        .round(
            4
        )
        .to_string(
            index=False
        )
    )

Shape: (1870, 19)

Columns:
['Country', 'Outcome', 'Fold', 'N_train', 'N_test', 'N_train_schools', 'N_test_schools', 'Model_RMSE', 'Model_MAE', 'Model_R2', 'Baseline_RMSE', 'Baseline_MAE', 'Baseline_R2', 'N_train_positive', 'N_test_positive', 'Model_ROC_AUC', 'Model_PR_AUC', 'Baseline_ROC_AUC', 'Baseline_PR_AUC']


REGRESSION FOLD STABILITY
                        Metric   Value
                       Systems 74.0000
            All 5 folds R2 > 0 72.0000
     At least 4/5 folds R2 > 0 73.0000
All 5 folds beat RMSE baseline 73.0000
          Median system Min_R2  0.0790
         Worst fold R2 overall -0.0294
           Median system SD_R2  0.0298


CLASSIFICATION FOLD STABILITY
         Outcome  N_models  All5_ROC_above_0.50  AtLeast4_ROC_above_0.50  All5_ROC_above_0.55  All5_PR_beats_baseline  Median_min_fold_ROC  Worst_fold_ROC  Median_ROC_SD  Median_PR_improvement
  BULLY_PHYSICAL        75                   75                       75                   75                      75   

In [35]:
# ============================================================
# ACHIEVEMENT-AUGMENTED PLAUSIBLE-VALUE SMOKE TEST
#
# Tests:
#   USA — BEINGBULLIED
#   USA — BULLY_PHYSICAL
#
# Across PV1 ... PV10.
#
# No averaging of student plausible values.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen non-achievement predictors
# ============================================================

PRIMARY_NONPV_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


# ============================================================
# 2. PV helpers
# ============================================================

def get_pv_predictors(pv_number):

    return [
        f"PV{pv_number}READ",
        f"PV{pv_number}MATH",
        f"PV{pv_number}SCIE",
    ]


ALL_PV_COLUMNS = [
    column
    for pv in range(1, 11)
    for column in get_pv_predictors(pv)
]


# ============================================================
# 3. Check system PV usability
# ============================================================

def has_usable_all_pvs(df):

    for column in ALL_PV_COLUMNS:

        if column not in df.columns:
            return False

        values = pd.to_numeric(
            df[column],
            errors="coerce",
        )

        if values.notna().sum() == 0:
            return False

        if values.nunique(
            dropna=True
        ) <= 1:
            return False

    return True


PV_BEING_SYSTEMS = [
    country
    for country in PRIMARY_BEING_SYSTEMS
    if has_usable_all_pvs(
        analysis_frames[country]
    )
]


PV_SUBTYPE_SYSTEMS = [
    country
    for country in PRIMARY_SUBTYPE_SYSTEMS
    if has_usable_all_pvs(
        analysis_frames[country]
    )
]


print(
    "Achievement-eligible BEINGBULLIED systems:",
    len(PV_BEING_SYSTEMS),
)

print(
    "Achievement-eligible subtype systems:",
    len(PV_SUBTYPE_SYSTEMS),
)


print(
    "\nBEING systems excluded:"
)

print(
    sorted(
        set(PRIMARY_BEING_SYSTEMS)
        -
        set(PV_BEING_SYSTEMS)
    )
)


print(
    "\nSubtype systems excluded:"
)

print(
    sorted(
        set(PRIMARY_SUBTYPE_SYSTEMS)
        -
        set(PV_SUBTYPE_SYSTEMS)
    )
)


# Expected from our earlier audit
assert len(PV_BEING_SYSTEMS) == 73
assert len(PV_SUBTYPE_SYSTEMS) == 74


# ============================================================
# 4. Stable sigmoid for classifier margins
# ============================================================

def pv_sigmoid(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    output = np.empty_like(x)

    positive = (
        x >= 0
    )

    negative = ~positive


    output[positive] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -x[positive]
            )
        )
    )


    exp_x = np.exp(
        x[negative]
    )

    output[negative] = (
        exp_x
        /
        (
            1.0
            +
            exp_x
        )
    )


    return output


# ============================================================
# 5. Storage
# ============================================================

PV_SMOKE_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_PHYSICAL",
]


pv_smoke_performance = []

pv_smoke_rankings = []


# ============================================================
# 6. Preserve original 14-feature list
#
# We modify the list IN PLACE because run_country_oof_shap()
# may hold a reference to the original list object.
# ============================================================

original_attribution_predictors = (
    ATTRIBUTION_PREDICTORS.copy()
)


# ============================================================
# 7. Run USA over PV1 ... PV10
# ============================================================

try:

    for pv in range(1, 11):

        pv_features = (
            get_pv_predictors(pv)
        )


        current_predictors = (
            PRIMARY_NONPV_PREDICTORS
            +
            pv_features
        )


        # ----------------------------------------------------
        # Replace predictor list in place
        # ----------------------------------------------------

        ATTRIBUTION_PREDICTORS[:] = (
            current_predictors
        )


        print("\n")
        print(
            "=" * 90
        )

        print(
            f"PV RUN {pv}"
        )

        print(
            "=" * 90
        )


        for outcome in PV_SMOKE_OUTCOMES:

            (
                ranking,
                fold_importance,
                qa,
                row_shap,
            ) = run_country_oof_shap(
                df=analysis_frames[
                    "USA"
                ],
                country="USA",
                outcome=outcome,
            )


            # ------------------------------------------------
            # Structural QA
            # ------------------------------------------------

            if len(ranking) != 17:

                raise RuntimeError(
                    f"PV{pv} — {outcome}: "
                    f"expected 17 predictors, "
                    f"found {len(ranking)}."
                )


            importance_sum = float(
                ranking[
                    "ImportanceShare"
                ].sum()
            )


            max_additivity_error = float(
                qa[
                    "Max_SHAP_additivity_error"
                ].max()
            )


            # ------------------------------------------------
            # Recover OOF prediction from exact SHAP values
            # ------------------------------------------------

            shap_columns = [
                f"SHAP__{feature}"
                for feature
                in current_predictors
            ]


            raw_prediction = (
                row_shap[
                    "SHAP_Bias"
                ].to_numpy(
                    dtype=float
                )
                +
                row_shap[
                    shap_columns
                ]
                .sum(
                    axis=1
                )
                .to_numpy(
                    dtype=float
                )
            )


            observed = (
                row_shap[
                    "Observed"
                ]
                .to_numpy(
                    dtype=float
                )
            )


            weights = (
                row_shap[
                    "W_FSTUWT"
                ]
                .to_numpy(
                    dtype=float
                )
            )


            # ------------------------------------------------
            # Performance
            # ------------------------------------------------

            if outcome == "BEINGBULLIED":

                metrics = regression_metrics(
                    observed,
                    raw_prediction,
                    weights,
                )


                performance_row = {
                    "PV": pv,
                    "Outcome": outcome,
                    "R2": metrics[
                        "R2"
                    ],
                    "RMSE": metrics[
                        "RMSE"
                    ],
                    "ROC_AUC": np.nan,
                    "PR_AUC": np.nan,
                }


            else:

                probability = pv_sigmoid(
                    raw_prediction
                )


                metrics = classification_metrics(
                    observed,
                    probability,
                    weights,
                )


                performance_row = {
                    "PV": pv,
                    "Outcome": outcome,
                    "R2": np.nan,
                    "RMSE": np.nan,
                    "ROC_AUC": metrics[
                        "ROC_AUC"
                    ],
                    "PR_AUC": metrics[
                        "PR_AUC"
                    ],
                }


            performance_row[
                "ImportanceSum"
            ] = importance_sum


            performance_row[
                "MaxAdditivityError"
            ] = max_additivity_error


            top_row = (
                ranking
                .sort_values(
                    "Rank"
                )
                .iloc[0]
            )


            performance_row[
                "TopPredictor"
            ] = top_row[
                "Feature"
            ]


            performance_row[
                "TopShare"
            ] = top_row[
                "ImportanceShare"
            ]


            pv_smoke_performance.append(
                performance_row
            )


            # ------------------------------------------------
            # Convert PV-specific achievement names into
            # conceptual achievement predictors
            # ------------------------------------------------

            conceptual_map = {
                f"PV{pv}READ":
                    "READ_ACHIEVEMENT",

                f"PV{pv}MATH":
                    "MATH_ACHIEVEMENT",

                f"PV{pv}SCIE":
                    "SCIENCE_ACHIEVEMENT",
            }


            ranking_copy = (
                ranking.copy()
            )


            ranking_copy[
                "PV"
            ] = pv


            ranking_copy[
                "ConceptualFeature"
            ] = (
                ranking_copy[
                    "Feature"
                ]
                .replace(
                    conceptual_map
                )
            )


            pv_smoke_rankings.append(
                ranking_copy
            )


            print(
                f"{outcome}: "
                f"#1={top_row['Feature']} | "
                f"share={top_row['ImportanceShare']:.3f} | "
                f"importance sum={importance_sum:.6f} | "
                f"max error={max_additivity_error:.2e}"
            )


finally:

    # --------------------------------------------------------
    # Restore frozen primary 14-feature list
    # --------------------------------------------------------

    ATTRIBUTION_PREDICTORS[:] = (
        original_attribution_predictors
    )


# ============================================================
# 8. Combine results
# ============================================================

pv_smoke_performance = pd.DataFrame(
    pv_smoke_performance
)


pv_smoke_rankings = pd.concat(
    pv_smoke_rankings,
    ignore_index=True,
)


# ============================================================
# 9. Aggregate conceptual predictor importance across 10 PVs
#
# IMPORTANT:
# Variation across PVs here is PV sensitivity,
# NOT a sampling standard error.
# ============================================================

pv_conceptual_summary = (
    pv_smoke_rankings
    .groupby(
        [
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanImportanceShare=(
            "ImportanceShare",
            "mean",
        ),

        SDImportanceAcrossPVs=(
            "ImportanceShare",
            "std",
        ),

        MeanRankAcrossPVs=(
            "Rank",
            "mean",
        ),

        MedianRankAcrossPVs=(
            "Rank",
            "median",
        ),

        BestRankAcrossPVs=(
            "Rank",
            "min",
        ),

        WorstRankAcrossPVs=(
            "Rank",
            "max",
        ),

        Top5PVRate=(
            "Rank",
            lambda x: float(
                (
                    x <= 5
                ).mean()
            ),
        ),
    )
    .reset_index()
)


pv_conceptual_summary[
    "ConceptualRank"
] = (
    pv_conceptual_summary
    .groupby(
        "Outcome"
    )
    [
        "MeanImportanceShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


pv_conceptual_summary = (
    pv_conceptual_summary
    .sort_values(
        [
            "Outcome",
            "ConceptualRank",
        ]
    )
)


# ============================================================
# 10. Performance summary across PVs
# ============================================================

performance_summary = (
    pv_smoke_performance
    .groupby(
        "Outcome"
    )
    .agg(

        Mean_R2=(
            "R2",
            "mean",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Max_R2=(
            "R2",
            "max",
        ),

        Mean_ROC_AUC=(
            "ROC_AUC",
            "mean",
        ),

        Min_ROC_AUC=(
            "ROC_AUC",
            "min",
        ),

        Max_ROC_AUC=(
            "ROC_AUC",
            "max",
        ),

        Mean_PR_AUC=(
            "PR_AUC",
            "mean",
        ),

        Max_additivity_error=(
            "MaxAdditivityError",
            "max",
        ),
    )
    .reset_index()
)


# ============================================================
# 11. Print results
# ============================================================

print("\n")
print("=" * 100)
print("USA ACHIEVEMENT-PV PERFORMANCE")
print("=" * 100)


print(
    performance_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("USA CONCEPTUAL PREDICTOR ARCHITECTURE ACROSS 10 PVs")
print("=" * 100)


for outcome in PV_SMOKE_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )

    print(
        outcome
    )

    print(
        "-" * 90
    )


    temp = (
        pv_conceptual_summary[
            pv_conceptual_summary[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "ConceptualRank",
                "ConceptualFeature",
                "MeanImportanceShare",
                "SDImportanceAcrossPVs",
                "MedianRankAcrossPVs",
                "BestRankAcrossPVs",
                "WorstRankAcrossPVs",
                "Top5PVRate",
            ]
        ]
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


print(
    "\nPrimary predictor list restored:",
    ATTRIBUTION_PREDICTORS
    ==
    original_attribution_predictors
)

Achievement-eligible BEINGBULLIED systems: 73
Achievement-eligible subtype systems: 74

BEING systems excluded:
['VNM']

Subtype systems excluded:
['VNM']


PV RUN 1
BEINGBULLIED: #1=BELONG | share=0.349 | importance sum=1.000000 | max error=2.23e-06
BULLY_PHYSICAL: #1=BELONG | share=0.205 | importance sum=1.000000 | max error=6.48e-06


PV RUN 2
BEINGBULLIED: #1=BELONG | share=0.352 | importance sum=1.000000 | max error=2.29e-06
BULLY_PHYSICAL: #1=BELONG | share=0.205 | importance sum=1.000000 | max error=6.57e-06


PV RUN 3
BEINGBULLIED: #1=BELONG | share=0.336 | importance sum=1.000000 | max error=2.10e-06
BULLY_PHYSICAL: #1=BELONG | share=0.209 | importance sum=1.000000 | max error=6.15e-06


PV RUN 4
BEINGBULLIED: #1=BELONG | share=0.346 | importance sum=1.000000 | max error=1.48e-06
BULLY_PHYSICAL: #1=BELONG | share=0.219 | importance sum=1.000000 | max error=7.36e-06


PV RUN 5
BEINGBULLIED: #1=BELONG | share=0.342 | importance sum=1.000000 | max error=1.90e-06
BULLY_PHYSICAL: #

In [36]:
# ============================================================
# FULL ACHIEVEMENT-AUGMENTED PV ANALYSIS
#
# 10 matched plausible-value runs:
#
#   PV1READ + PV1MATH + PV1SCIE
#   ...
#   PV10READ + PV10MATH + PV10SCIE
#
# Primary summaries:
#   1. Performance averaged across 10 PVs
#   2. SHAP importance averaged across 10 PVs
#   3. Total ACHIEVEMENT FAMILY importance
#   4. PV-to-PV stability
#   5. Performance change vs frozen 14-predictor model
#
# This is SUPPLEMENTARY to the frozen primary analysis.
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

PV_DIR = Path(
    "../results/achievement_pv"
)

PV_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NONPV_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


PV_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


PV_BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


ACHIEVEMENT_CONCEPTS = [
    "READ_ACHIEVEMENT",
    "MATH_ACHIEVEMENT",
    "SCIENCE_ACHIEVEMENT",
]


# ============================================================
# 2. PV helpers
# ============================================================

def get_pv_predictors(pv):

    return [
        f"PV{pv}READ",
        f"PV{pv}MATH",
        f"PV{pv}SCIE",
    ]


ALL_PV_COLUMNS = [
    col
    for pv in range(1, 11)
    for col in get_pv_predictors(pv)
]


def has_usable_all_pvs(df):

    for col in ALL_PV_COLUMNS:

        if col not in df.columns:
            return False

        values = pd.to_numeric(
            df[col],
            errors="coerce",
        )

        if values.notna().sum() == 0:
            return False

        if values.nunique(
            dropna=True
        ) <= 1:
            return False

    return True


# ============================================================
# 3. Determine eligible systems
# ============================================================

PV_BEING_SYSTEMS = [
    country
    for country in PRIMARY_BEING_SYSTEMS
    if has_usable_all_pvs(
        analysis_frames[country]
    )
]


PV_SUBTYPE_SYSTEMS = [
    country
    for country in PRIMARY_SUBTYPE_SYSTEMS
    if has_usable_all_pvs(
        analysis_frames[country]
    )
]


print(
    "PV-eligible BEINGBULLIED systems:",
    len(PV_BEING_SYSTEMS)
)

print(
    "PV-eligible subtype systems:",
    len(PV_SUBTYPE_SYSTEMS)
)


print(
    "Excluded BEING systems:",
    sorted(
        set(PRIMARY_BEING_SYSTEMS)
        -
        set(PV_BEING_SYSTEMS)
    )
)


print(
    "Excluded subtype systems:",
    sorted(
        set(PRIMARY_SUBTYPE_SYSTEMS)
        -
        set(PV_SUBTYPE_SYSTEMS)
    )
)


assert len(PV_BEING_SYSTEMS) == 73
assert len(PV_SUBTYPE_SYSTEMS) == 74


# ============================================================
# 4. Stable sigmoid
# ============================================================

def pv_sigmoid(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    out = np.empty_like(x)

    pos = (
        x >= 0
    )

    neg = ~pos


    out[pos] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -x[pos]
            )
        )
    )


    exp_x = np.exp(
        x[neg]
    )

    out[neg] = (
        exp_x
        /
        (
            1.0
            +
            exp_x
        )
    )


    return out


# ============================================================
# 5. Output paths
# ============================================================

RANK_PATH = (
    PV_DIR
    / "pv_model_rankings.csv"
)

PERF_PATH = (
    PV_DIR
    / "pv_model_performance.csv"
)

QA_PATH = (
    PV_DIR
    / "pv_model_qa.csv"
)

ERROR_PATH = (
    PV_DIR
    / "pv_model_errors.csv"
)


# ============================================================
# 6. Resume support
# ============================================================

if PERF_PATH.exists():

    existing_perf = pd.read_csv(
        PERF_PATH
    )

else:

    existing_perf = pd.DataFrame()


if RANK_PATH.exists():

    existing_rankings = pd.read_csv(
        RANK_PATH
    )

else:

    existing_rankings = pd.DataFrame()


if QA_PATH.exists():

    existing_qa = pd.read_csv(
        QA_PATH
    )

else:

    existing_qa = pd.DataFrame()


if ERROR_PATH.exists():

    existing_errors = pd.read_csv(
        ERROR_PATH
    )

else:

    existing_errors = pd.DataFrame()


completed = set()


if not existing_perf.empty:

    completed = set(
        zip(
            existing_perf[
                "PV"
            ].astype(int),

            existing_perf[
                "Country"
            ],

            existing_perf[
                "Outcome"
            ],
        )
    )


ranking_batches = (
    [existing_rankings]
    if not existing_rankings.empty
    else []
)


performance_rows = (
    existing_perf
    .to_dict(
        "records"
    )
    if not existing_perf.empty
    else []
)


qa_rows = (
    existing_qa
    .to_dict(
        "records"
    )
    if not existing_qa.empty
    else []
)


error_rows = (
    existing_errors
    .to_dict(
        "records"
    )
    if not existing_errors.empty
    else []
)


# ============================================================
# 7. Expected number of models
# ============================================================

models_per_pv = (
    len(PV_BEING_SYSTEMS)
    +
    (
        len(PV_SUBTYPE_SYSTEMS)
        *
        len(PV_BINARY_OUTCOMES)
    )
)


EXPECTED_MODELS = (
    10
    *
    models_per_pv
)


print(
    "\nExpected total PV models:",
    EXPECTED_MODELS
)

print(
    "Already completed:",
    len(completed)
)

print(
    "Remaining:",
    EXPECTED_MODELS
    -
    len(completed)
)


# ============================================================
# 8. Checkpoint function
# ============================================================

def save_pv_checkpoint():

    if ranking_batches:

        pd.concat(
            ranking_batches,
            ignore_index=True,
        ).to_csv(
            RANK_PATH,
            index=False,
        )


    pd.DataFrame(
        performance_rows
    ).to_csv(
        PERF_PATH,
        index=False,
    )


    pd.DataFrame(
        qa_rows
    ).to_csv(
        QA_PATH,
        index=False,
    )


    pd.DataFrame(
        error_rows
    ).to_csv(
        ERROR_PATH,
        index=False,
    )


# ============================================================
# 9. Preserve frozen primary predictor list
# ============================================================

ORIGINAL_ATTRIBUTION_PREDICTORS = (
    ATTRIBUTION_PREDICTORS.copy()
)


# ============================================================
# 10. Run one model
# ============================================================

new_models_this_session = 0

start_time = time.time()


def run_one_pv_model(
    pv,
    country,
    outcome,
):

    global new_models_this_session


    key = (
        pv,
        country,
        outcome,
    )


    if key in completed:
        return


    pv_features = (
        get_pv_predictors(
            pv
        )
    )


    current_predictors = (
        NONPV_PREDICTORS
        +
        pv_features
    )


    # --------------------------------------------------------
    # run_country_oof_shap() uses this predictor list.
    # --------------------------------------------------------

    ATTRIBUTION_PREDICTORS[:] = (
        current_predictors
    )


    model_number = (
        len(completed)
        +
        1
    )


    print(
        f"[{model_number}/{EXPECTED_MODELS}] "
        f"PV{pv} | "
        f"{country} — {outcome}",
        end=" ... ",
    )


    try:

        (
            ranking,
            fold_importance,
            qa,
            row_shap,
        ) = run_country_oof_shap(
            df=analysis_frames[
                country
            ],
            country=country,
            outcome=outcome,
        )


        # ----------------------------------------------------
        # Structural QA
        # ----------------------------------------------------

        if len(ranking) != 17:

            raise RuntimeError(
                f"Expected 17 predictors, "
                f"found {len(ranking)}."
            )


        importance_sum = float(
            ranking[
                "ImportanceShare"
            ]
            .sum()
        )


        max_additivity_error = float(
            qa[
                "Max_SHAP_additivity_error"
            ]
            .max()
        )


        # ----------------------------------------------------
        # Conceptual feature names
        # ----------------------------------------------------

        conceptual_map = {
            f"PV{pv}READ":
                "READ_ACHIEVEMENT",

            f"PV{pv}MATH":
                "MATH_ACHIEVEMENT",

            f"PV{pv}SCIE":
                "SCIENCE_ACHIEVEMENT",
        }


        ranking = (
            ranking.copy()
        )


        ranking.insert(
            0,
            "PV",
            pv,
        )


        ranking[
            "ConceptualFeature"
        ] = (
            ranking[
                "Feature"
            ]
            .replace(
                conceptual_map
            )
        )


        ranking[
            "IsAchievement"
        ] = (
            ranking[
                "ConceptualFeature"
            ]
            .isin(
                ACHIEVEMENT_CONCEPTS
            )
        )


        achievement_family_share = float(
            ranking.loc[
                ranking[
                    "IsAchievement"
                ],
                "ImportanceShare",
            ]
            .sum()
        )


        ranking[
            "AchievementFamilyShare"
        ] = (
            achievement_family_share
        )


        ranking_batches.append(
            ranking
        )


        # ----------------------------------------------------
        # Recover exact OOF predictions from SHAP
        # ----------------------------------------------------

        shap_columns = [
            f"SHAP__{feature}"
            for feature
            in current_predictors
        ]


        raw_prediction = (
            row_shap[
                "SHAP_Bias"
            ]
            .to_numpy(
                dtype=float
            )
            +
            row_shap[
                shap_columns
            ]
            .sum(
                axis=1
            )
            .to_numpy(
                dtype=float
            )
        )


        observed = (
            row_shap[
                "Observed"
            ]
            .to_numpy(
                dtype=float
            )
        )


        weights = (
            row_shap[
                "W_FSTUWT"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # Performance
        # ----------------------------------------------------

        if outcome == "BEINGBULLIED":

            metrics = (
                regression_metrics(
                    observed,
                    raw_prediction,
                    weights,
                )
            )


            perf_row = {
                "PV": pv,
                "Country": country,
                "Outcome": outcome,
                "N": len(
                    observed
                ),

                "R2": metrics[
                    "R2"
                ],

                "RMSE": metrics[
                    "RMSE"
                ],

                "MAE": metrics[
                    "MAE"
                ],

                "ROC_AUC": np.nan,
                "PR_AUC": np.nan,

                "AchievementFamilyShare":
                    achievement_family_share,
            }


        else:

            probability = (
                pv_sigmoid(
                    raw_prediction
                )
            )


            metrics = (
                classification_metrics(
                    observed,
                    probability,
                    weights,
                )
            )


            perf_row = {
                "PV": pv,
                "Country": country,
                "Outcome": outcome,
                "N": len(
                    observed
                ),

                "R2": np.nan,
                "RMSE": np.nan,
                "MAE": np.nan,

                "ROC_AUC": metrics[
                    "ROC_AUC"
                ],

                "PR_AUC": metrics[
                    "PR_AUC"
                ],

                "AchievementFamilyShare":
                    achievement_family_share,
            }


        performance_rows.append(
            perf_row
        )


        qa_rows.append(
            {
                "PV": pv,
                "Country": country,
                "Outcome": outcome,

                "ImportanceSum":
                    importance_sum,

                "MaxAdditivityError":
                    max_additivity_error,
            }
        )


        completed.add(
            key
        )


        new_models_this_session += 1


        top = (
            ranking
            .sort_values(
                "Rank"
            )
            .iloc[0]
        )


        print(
            "OK | "
            f"#1={top['ConceptualFeature']} | "
            f"achievement="
            f"{achievement_family_share:.3f}"
        )


        del fold_importance
        del qa
        del row_shap


        # ----------------------------------------------------
        # Checkpoint every 50 newly completed models
        # ----------------------------------------------------

        if (
            new_models_this_session
            % 50
            == 0
        ):

            save_pv_checkpoint()

            gc.collect()


    except Exception as e:

        error_rows.append(
            {
                "PV": pv,
                "Country": country,
                "Outcome": outcome,

                "Error_type":
                    type(e).__name__,

                "Error":
                    str(e),
            }
        )


        print(
            f"ERROR | "
            f"{type(e).__name__}: "
            f"{e}"
        )


# ============================================================
# 11. Run all 10 plausible values
# ============================================================

try:

    for pv in range(
        1,
        11,
    ):

        print("\n")
        print(
            "=" * 110
        )

        print(
            f"STARTING PV{pv}"
        )

        print(
            "=" * 110
        )


        # ----------------------------------------------------
        # BEINGBULLIED
        # ----------------------------------------------------

        for country in (
            PV_BEING_SYSTEMS
        ):

            run_one_pv_model(
                pv=pv,
                country=country,
                outcome="BEINGBULLIED",
            )


        # ----------------------------------------------------
        # Four subtype outcomes
        # ----------------------------------------------------

        for country in (
            PV_SUBTYPE_SYSTEMS
        ):

            for outcome in (
                PV_BINARY_OUTCOMES
            ):

                run_one_pv_model(
                    pv=pv,
                    country=country,
                    outcome=outcome,
                )


        save_pv_checkpoint()


finally:

    # Always save progress
    save_pv_checkpoint()


    # Restore frozen primary 14 predictors
    ATTRIBUTION_PREDICTORS[:] = (
        ORIGINAL_ATTRIBUTION_PREDICTORS
    )


# ============================================================
# 12. Reload final saved files
# ============================================================

pv_rankings = pd.read_csv(
    RANK_PATH
)


pv_performance = pd.read_csv(
    PERF_PATH
)


pv_qa = pd.read_csv(
    QA_PATH
)


pv_errors = pd.read_csv(
    ERROR_PATH
)


# ============================================================
# 13. Basic completion QA
# ============================================================

successful_models = (
    pv_performance[
        [
            "PV",
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


elapsed_minutes = (
    time.time()
    -
    start_time
) / 60


print("\n")
print(
    "=" * 110
)

print(
    "FULL PV ANALYSIS COMPLETE"
)

print(
    "=" * 110
)


print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Successful models:",
    successful_models
)

print(
    "Errors:",
    len(
        pv_errors
    )
)

print(
    f"Session runtime: "
    f"{elapsed_minutes:.2f} minutes"
)


print(
    "Primary predictor list restored:",
    ATTRIBUTION_PREDICTORS
    ==
    ORIGINAL_ATTRIBUTION_PREDICTORS
)


# ============================================================
# 14. QA summaries
# ============================================================

print(
    "\nMaximum SHAP additivity error:",
    pv_qa[
        "MaxAdditivityError"
    ]
    .max()
)

print(
    "Importance sum range:",
    pv_qa[
        "ImportanceSum"
    ]
    .min(),
    "to",
    pv_qa[
        "ImportanceSum"
    ]
    .max(),
)


# ============================================================
# 15. Aggregate each conceptual predictor across 10 PVs
#
# First aggregate within each system/outcome.
# ============================================================

system_feature_pv_summary = (
    pv_rankings
    .groupby(
        [
            "Country",
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanImportanceShare=(
            "ImportanceShare",
            "mean",
        ),

        SDImportanceAcrossPVs=(
            "ImportanceShare",
            "std",
        ),

        MeanRankAcrossPVs=(
            "Rank",
            "mean",
        ),

        MedianRankAcrossPVs=(
            "Rank",
            "median",
        ),

        BestRankAcrossPVs=(
            "Rank",
            "min",
        ),

        WorstRankAcrossPVs=(
            "Rank",
            "max",
        ),

        Top5PVRate=(
            "Rank",
            lambda x: float(
                (
                    x <= 5
                )
                .mean()
            ),
        ),
    )
    .reset_index()
)


system_feature_pv_summary[
    "ConceptualRank"
] = (
    system_feature_pv_summary
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "MeanImportanceShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


system_feature_pv_summary.to_csv(
    PV_DIR
    / "system_conceptual_predictor_summary.csv",
    index=False,
)


# ============================================================
# 16. Achievement-family share per system
# ============================================================

system_achievement = (
    pv_performance
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    .agg(

        MeanAchievementShare=(
            "AchievementFamilyShare",
            "mean",
        ),

        SDAchievementAcrossPVs=(
            "AchievementFamilyShare",
            "std",
        ),

        MinAchievementShare=(
            "AchievementFamilyShare",
            "min",
        ),

        MaxAchievementShare=(
            "AchievementFamilyShare",
            "max",
        ),
    )
    .reset_index()
)


system_achievement.to_csv(
    PV_DIR
    / "system_achievement_family_share.csv",
    index=False,
)


achievement_summary = (
    system_achievement
    .groupby(
        "Outcome"
    )
    .agg(

        N_systems=(
            "Country",
            "count",
        ),

        Mean_system_achievement_share=(
            "MeanAchievementShare",
            "mean",
        ),

        Median_system_achievement_share=(
            "MeanAchievementShare",
            "median",
        ),

        Min_system_achievement_share=(
            "MeanAchievementShare",
            "min",
        ),

        Max_system_achievement_share=(
            "MeanAchievementShare",
            "max",
        ),

        Mean_PV_SD=(
            "SDAchievementAcrossPVs",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 17. Aggregate performance across the 10 PVs per system
# ============================================================

system_pv_performance = (
    pv_performance
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    .agg(

        PV_Mean_R2=(
            "R2",
            "mean",
        ),

        PV_SD_R2=(
            "R2",
            "std",
        ),

        PV_Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        PV_Mean_ROC_AUC=(
            "ROC_AUC",
            "mean",
        ),

        PV_SD_ROC_AUC=(
            "ROC_AUC",
            "std",
        ),

        PV_Mean_PR_AUC=(
            "PR_AUC",
            "mean",
        ),

        PV_SD_PR_AUC=(
            "PR_AUC",
            "std",
        ),
    )
    .reset_index()
)


# ============================================================
# 18. Compare performance with frozen primary model
# ============================================================

primary_performance = pd.read_csv(
    "../results/primary_models/"
    "overall_performance.csv"
)


required_primary_columns = [
    "Country",
    "Outcome",
    "Model_R2",
    "Model_ROC_AUC",
    "Model_PR_AUC",
]


missing_primary_cols = [
    col
    for col in required_primary_columns
    if col not in primary_performance.columns
]


if missing_primary_cols:

    raise RuntimeError(
        "overall_performance.csv is missing: "
        f"{missing_primary_cols}\n"
        f"Available columns: "
        f"{primary_performance.columns.tolist()}"
    )


performance_comparison = (
    system_pv_performance
    .merge(
        primary_performance[
            required_primary_columns
        ],
        on=[
            "Country",
            "Outcome",
        ],
        how="left",
        validate="one_to_one",
    )
)


performance_comparison[
    "Delta_R2"
] = (
    performance_comparison[
        "PV_Mean_R2"
    ]
    -
    performance_comparison[
        "Model_R2"
    ]
)


performance_comparison[
    "Delta_ROC_AUC"
] = (
    performance_comparison[
        "PV_Mean_ROC_AUC"
    ]
    -
    performance_comparison[
        "Model_ROC_AUC"
    ]
)


performance_comparison[
    "Delta_PR_AUC"
] = (
    performance_comparison[
        "PV_Mean_PR_AUC"
    ]
    -
    performance_comparison[
        "Model_PR_AUC"
    ]
)


performance_comparison.to_csv(
    PV_DIR
    / "performance_vs_primary.csv",
    index=False,
)


# ============================================================
# 19. Cross-system performance-change summary
# ============================================================

performance_change_rows = []


for outcome in PV_OUTCOMES:

    temp = (
        performance_comparison[
            performance_comparison[
                "Outcome"
            ]
            == outcome
        ]
    )


    if outcome == "BEINGBULLIED":

        delta = (
            temp[
                "Delta_R2"
            ]
            .dropna()
        )


        performance_change_rows.append(
            {
                "Outcome": outcome,

                "N_systems":
                    len(delta),

                "Metric":
                    "R2",

                "Mean_delta":
                    delta.mean(),

                "Median_delta":
                    delta.median(),

                "Min_delta":
                    delta.min(),

                "Max_delta":
                    delta.max(),

                "Pct_systems_improved":
                    100
                    *
                    (
                        delta > 0
                    )
                    .mean(),
            }
        )


    else:

        delta_roc = (
            temp[
                "Delta_ROC_AUC"
            ]
            .dropna()
        )

        delta_pr = (
            temp[
                "Delta_PR_AUC"
            ]
            .dropna()
        )


        performance_change_rows.append(
            {
                "Outcome": outcome,

                "N_systems":
                    len(delta_roc),

                "Metric":
                    "ROC_AUC",

                "Mean_delta":
                    delta_roc.mean(),

                "Median_delta":
                    delta_roc.median(),

                "Min_delta":
                    delta_roc.min(),

                "Max_delta":
                    delta_roc.max(),

                "Pct_systems_improved":
                    100
                    *
                    (
                        delta_roc > 0
                    )
                    .mean(),
            }
        )


        performance_change_rows.append(
            {
                "Outcome": outcome,

                "N_systems":
                    len(delta_pr),

                "Metric":
                    "PR_AUC",

                "Mean_delta":
                    delta_pr.mean(),

                "Median_delta":
                    delta_pr.median(),

                "Min_delta":
                    delta_pr.min(),

                "Max_delta":
                    delta_pr.max(),

                "Pct_systems_improved":
                    100
                    *
                    (
                        delta_pr > 0
                    )
                    .mean(),
            }
        )


performance_change_summary = pd.DataFrame(
    performance_change_rows
)


performance_change_summary.to_csv(
    PV_DIR
    / "performance_change_summary.csv",
    index=False,
)


# ============================================================
# 20. International conceptual predictor architecture
#
# Equal weight to each education system.
# ============================================================

international_architecture = (
    system_feature_pv_summary
    .groupby(
        [
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanSystemShare=(
            "MeanImportanceShare",
            "mean",
        ),

        MedianSystemShare=(
            "MeanImportanceShare",
            "median",
        ),

        MedianSystemRank=(
            "ConceptualRank",
            "median",
        ),

        Top5SystemRate=(
            "ConceptualRank",
            lambda x: float(
                (
                    x <= 5
                )
                .mean()
            ),
        ),

        MeanPVInstability=(
            "SDImportanceAcrossPVs",
            "mean",
        ),
    )
    .reset_index()
)


international_architecture[
    "InternationalRank"
] = (
    international_architecture
    .groupby(
        "Outcome"
    )
    [
        "MeanSystemShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


international_architecture = (
    international_architecture
    .sort_values(
        [
            "Outcome",
            "InternationalRank",
        ]
    )
)


international_architecture.to_csv(
    PV_DIR
    / "international_conceptual_architecture.csv",
    index=False,
)


# ============================================================
# 21. PRINT RESULTS
# ============================================================

print("\n")
print(
    "=" * 110
)

print(
    "ACHIEVEMENT FAMILY SHARE BY OUTCOME"
)

print(
    "=" * 110
)


print(
    achievement_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print(
    "=" * 110
)

print(
    "PERFORMANCE CHANGE VS PRIMARY 14-PREDICTOR MODEL"
)

print(
    "=" * 110
)


print(
    performance_change_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print(
    "=" * 110
)

print(
    "INTERNATIONAL ACHIEVEMENT-AUGMENTED ARCHITECTURE"
)

print(
    "=" * 110
)


for outcome in PV_OUTCOMES:

    print("\n")
    print(
        "-" * 90
    )

    print(
        outcome
    )

    print(
        "-" * 90
    )


    temp = (
        international_architecture[
            international_architecture[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "InternationalRank",
                "ConceptualFeature",
                "MeanSystemShare",
                "MedianSystemRank",
                "Top5SystemRate",
                "MeanPVInstability",
            ]
        ]
    )


    print(
        temp
        .round(
            4
        )
        .to_string(
            index=False
        )
    )
    

PV-eligible BEINGBULLIED systems: 73
PV-eligible subtype systems: 74
Excluded BEING systems: ['VNM']
Excluded subtype systems: ['VNM']

Expected total PV models: 3690
Already completed: 0
Remaining: 3690


STARTING PV1
[1/3690] PV1 | ALB — BEINGBULLIED ... OK | #1=BELONG | achievement=0.192
[2/3690] PV1 | ARE — BEINGBULLIED ... OK | #1=BELONG | achievement=0.150
[3/3690] PV1 | ARG — BEINGBULLIED ... OK | #1=BELONG | achievement=0.169
[4/3690] PV1 | AUS — BEINGBULLIED ... OK | #1=BELONG | achievement=0.097
[5/3690] PV1 | AUT — BEINGBULLIED ... OK | #1=BELONG | achievement=0.162
[6/3690] PV1 | BEL — BEINGBULLIED ... OK | #1=BELONG | achievement=0.142
[7/3690] PV1 | BGR — BEINGBULLIED ... OK | #1=BELONG | achievement=0.141
[8/3690] PV1 | BIH — BEINGBULLIED ... OK | #1=BELONG | achievement=0.194
[9/3690] PV1 | BLR — BEINGBULLIED ... OK | #1=BELONG | achievement=0.194
[10/3690] PV1 | BRA — BEINGBULLIED ... OK | #1=BELONG | achievement=0.137
[11/3690] PV1 | BRN — BEINGBULLIED ... OK | #1=REA

EmptyDataError: No columns to parse from file

In [37]:
# ============================================================
# RECOVER COMPLETED PV ANALYSIS AFTER EMPTY ERROR-LOG CRASH
#
# IMPORTANT:
# This does NOT fit any models.
# It only reloads the files already saved during the 5-hour run
# and completes the aggregation / reporting stage.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError


# ============================================================
# 1. Paths
# ============================================================

PV_DIR = Path(
    "../results/achievement_pv"
)

RANK_PATH = (
    PV_DIR
    / "pv_model_rankings.csv"
)

PERF_PATH = (
    PV_DIR
    / "pv_model_performance.csv"
)

QA_PATH = (
    PV_DIR
    / "pv_model_qa.csv"
)

ERROR_PATH = (
    PV_DIR
    / "pv_model_errors.csv"
)


# ============================================================
# 2. Reload completed saved results
# ============================================================

pv_rankings = pd.read_csv(
    RANK_PATH
)

pv_performance = pd.read_csv(
    PERF_PATH
)

pv_qa = pd.read_csv(
    QA_PATH
)


# Empty error file means there were no recorded errors.
try:

    pv_errors = pd.read_csv(
        ERROR_PATH
    )

except EmptyDataError:

    pv_errors = pd.DataFrame(
        columns=[
            "PV",
            "Country",
            "Outcome",
            "Error_type",
            "Error",
        ]
    )


# ============================================================
# 3. Constants
# ============================================================

PV_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


ACHIEVEMENT_CONCEPTS = [
    "READ_ACHIEVEMENT",
    "MATH_ACHIEVEMENT",
    "SCIENCE_ACHIEVEMENT",
]


EXPECTED_MODELS = 3690


# ============================================================
# 4. Completion QA
# ============================================================

successful_models = (
    pv_performance[
        [
            "PV",
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


ranking_models = (
    pv_rankings[
        [
            "PV",
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


qa_models = (
    pv_qa[
        [
            "PV",
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


expected_ranking_rows = (
    EXPECTED_MODELS
    * 17
)


duplicate_perf_models = (
    pv_performance
    .duplicated(
        [
            "PV",
            "Country",
            "Outcome",
        ]
    )
    .sum()
)


print("=" * 110)
print("RECOVERED PV ANALYSIS — COMPLETION QA")
print("=" * 110)

print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Performance models:",
    successful_models
)

print(
    "Ranking models:",
    ranking_models
)

print(
    "QA models:",
    qa_models
)

print(
    "Recorded errors:",
    len(pv_errors)
)

print(
    "Ranking rows:",
    len(pv_rankings),
    "/",
    expected_ranking_rows,
)

print(
    "Duplicate performance model rows:",
    duplicate_perf_models
)


if successful_models != EXPECTED_MODELS:

    raise RuntimeError(
        f"Expected {EXPECTED_MODELS} models, "
        f"but performance file contains "
        f"{successful_models} unique models."
    )


if ranking_models != EXPECTED_MODELS:

    raise RuntimeError(
        "Ranking file does not contain all "
        "expected models."
    )


if qa_models != EXPECTED_MODELS:

    raise RuntimeError(
        "QA file does not contain all "
        "expected models."
    )


if len(pv_rankings) != expected_ranking_rows:

    raise RuntimeError(
        f"Expected {expected_ranking_rows} ranking rows, "
        f"found {len(pv_rankings)}."
    )


if duplicate_perf_models != 0:

    raise RuntimeError(
        "Duplicate PV model rows were found."
    )


# ============================================================
# 5. SHAP QA
# ============================================================

print("\n")
print("=" * 110)
print("SHAP QA")
print("=" * 110)

print(
    "Importance sum range:",
    pv_qa[
        "ImportanceSum"
    ].min(),
    "to",
    pv_qa[
        "ImportanceSum"
    ].max(),
)

print(
    "Maximum additivity error:",
    pv_qa[
        "MaxAdditivityError"
    ].max(),
)


# ============================================================
# 6. Aggregate predictors across 10 PVs within each system
# ============================================================

system_feature_pv_summary = (
    pv_rankings
    .groupby(
        [
            "Country",
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanImportanceShare=(
            "ImportanceShare",
            "mean",
        ),

        SDImportanceAcrossPVs=(
            "ImportanceShare",
            "std",
        ),

        MeanRankAcrossPVs=(
            "Rank",
            "mean",
        ),

        MedianRankAcrossPVs=(
            "Rank",
            "median",
        ),

        BestRankAcrossPVs=(
            "Rank",
            "min",
        ),

        WorstRankAcrossPVs=(
            "Rank",
            "max",
        ),

        Top5PVRate=(
            "Rank",
            lambda x: float(
                (
                    x <= 5
                ).mean()
            ),
        ),
    )
    .reset_index()
)


system_feature_pv_summary[
    "ConceptualRank"
] = (
    system_feature_pv_summary
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "MeanImportanceShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


system_feature_pv_summary.to_csv(
    PV_DIR
    / "system_conceptual_predictor_summary.csv",
    index=False,
)


# ============================================================
# 7. Achievement-family share
# ============================================================

system_achievement = (
    pv_performance
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    .agg(

        MeanAchievementShare=(
            "AchievementFamilyShare",
            "mean",
        ),

        SDAchievementAcrossPVs=(
            "AchievementFamilyShare",
            "std",
        ),

        MinAchievementShare=(
            "AchievementFamilyShare",
            "min",
        ),

        MaxAchievementShare=(
            "AchievementFamilyShare",
            "max",
        ),
    )
    .reset_index()
)


system_achievement.to_csv(
    PV_DIR
    / "system_achievement_family_share.csv",
    index=False,
)


achievement_summary = (
    system_achievement
    .groupby(
        "Outcome"
    )
    .agg(

        N_systems=(
            "Country",
            "count",
        ),

        Mean_system_achievement_share=(
            "MeanAchievementShare",
            "mean",
        ),

        Median_system_achievement_share=(
            "MeanAchievementShare",
            "median",
        ),

        Min_system_achievement_share=(
            "MeanAchievementShare",
            "min",
        ),

        Max_system_achievement_share=(
            "MeanAchievementShare",
            "max",
        ),

        Mean_PV_SD=(
            "SDAchievementAcrossPVs",
            "mean",
        ),
    )
    .reset_index()
)


achievement_summary.to_csv(
    PV_DIR
    / "achievement_family_summary.csv",
    index=False,
)


# ============================================================
# 8. Aggregate performance across 10 PVs per system
# ============================================================

system_pv_performance = (
    pv_performance
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    .agg(

        PV_Mean_R2=(
            "R2",
            "mean",
        ),

        PV_SD_R2=(
            "R2",
            "std",
        ),

        PV_Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        PV_Mean_ROC_AUC=(
            "ROC_AUC",
            "mean",
        ),

        PV_SD_ROC_AUC=(
            "ROC_AUC",
            "std",
        ),

        PV_Mean_PR_AUC=(
            "PR_AUC",
            "mean",
        ),

        PV_SD_PR_AUC=(
            "PR_AUC",
            "std",
        ),
    )
    .reset_index()
)


# ============================================================
# 9. Load frozen primary model performance
# ============================================================

primary_performance = pd.read_csv(
    "../results/primary_models/"
    "overall_performance.csv"
)


required_primary_columns = [
    "Country",
    "Outcome",
    "Model_R2",
    "Model_ROC_AUC",
    "Model_PR_AUC",
]


missing_primary_cols = [
    col
    for col in required_primary_columns
    if col not in primary_performance.columns
]


if missing_primary_cols:

    raise RuntimeError(
        "Primary performance file missing columns: "
        f"{missing_primary_cols}"
    )


# ============================================================
# 10. Performance change vs primary model
# ============================================================

performance_comparison = (
    system_pv_performance
    .merge(
        primary_performance[
            required_primary_columns
        ],
        on=[
            "Country",
            "Outcome",
        ],
        how="left",
        validate="one_to_one",
    )
)


performance_comparison[
    "Delta_R2"
] = (
    performance_comparison[
        "PV_Mean_R2"
    ]
    -
    performance_comparison[
        "Model_R2"
    ]
)


performance_comparison[
    "Delta_ROC_AUC"
] = (
    performance_comparison[
        "PV_Mean_ROC_AUC"
    ]
    -
    performance_comparison[
        "Model_ROC_AUC"
    ]
)


performance_comparison[
    "Delta_PR_AUC"
] = (
    performance_comparison[
        "PV_Mean_PR_AUC"
    ]
    -
    performance_comparison[
        "Model_PR_AUC"
    ]
)


performance_comparison.to_csv(
    PV_DIR
    / "performance_vs_primary.csv",
    index=False,
)


# ============================================================
# 11. Cross-system performance-change summary
# ============================================================

performance_change_rows = []


for outcome in PV_OUTCOMES:

    temp = (
        performance_comparison[
            performance_comparison[
                "Outcome"
            ]
            == outcome
        ]
    )


    if outcome == "BEINGBULLIED":

        delta = (
            temp[
                "Delta_R2"
            ]
            .dropna()
        )


        performance_change_rows.append(
            {
                "Outcome": outcome,
                "N_systems": len(delta),
                "Metric": "R2",
                "Mean_delta": delta.mean(),
                "Median_delta": delta.median(),
                "Min_delta": delta.min(),
                "Max_delta": delta.max(),

                "Pct_systems_improved": (
                    100
                    *
                    (
                        delta > 0
                    ).mean()
                ),
            }
        )


    else:

        for metric_name, delta_column in [
            (
                "ROC_AUC",
                "Delta_ROC_AUC",
            ),
            (
                "PR_AUC",
                "Delta_PR_AUC",
            ),
        ]:

            delta = (
                temp[
                    delta_column
                ]
                .dropna()
            )


            performance_change_rows.append(
                {
                    "Outcome": outcome,
                    "N_systems": len(delta),
                    "Metric": metric_name,
                    "Mean_delta": delta.mean(),
                    "Median_delta": delta.median(),
                    "Min_delta": delta.min(),
                    "Max_delta": delta.max(),

                    "Pct_systems_improved": (
                        100
                        *
                        (
                            delta > 0
                        ).mean()
                    ),
                }
            )


performance_change_summary = pd.DataFrame(
    performance_change_rows
)


performance_change_summary.to_csv(
    PV_DIR
    / "performance_change_summary.csv",
    index=False,
)


# ============================================================
# 12. International conceptual architecture
# ============================================================

international_architecture = (
    system_feature_pv_summary
    .groupby(
        [
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanSystemShare=(
            "MeanImportanceShare",
            "mean",
        ),

        MedianSystemShare=(
            "MeanImportanceShare",
            "median",
        ),

        MedianSystemRank=(
            "ConceptualRank",
            "median",
        ),

        Top5SystemRate=(
            "ConceptualRank",
            lambda x: float(
                (
                    x <= 5
                ).mean()
            ),
        ),

        MeanPVInstability=(
            "SDImportanceAcrossPVs",
            "mean",
        ),
    )
    .reset_index()
)


international_architecture[
    "InternationalRank"
] = (
    international_architecture
    .groupby(
        "Outcome"
    )
    [
        "MeanSystemShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


international_architecture = (
    international_architecture
    .sort_values(
        [
            "Outcome",
            "InternationalRank",
        ]
    )
)


international_architecture.to_csv(
    PV_DIR
    / "international_conceptual_architecture.csv",
    index=False,
)


# ============================================================
# 13. Final output
# ============================================================

print("\n")
print("=" * 110)
print("FULL PV ANALYSIS COMPLETE")
print("=" * 110)

print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Successful models:",
    successful_models
)

print(
    "Errors:",
    len(pv_errors)
)


print("\n")
print("=" * 110)
print("ACHIEVEMENT FAMILY SHARE BY OUTCOME")
print("=" * 110)

print(
    achievement_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("PERFORMANCE CHANGE VS PRIMARY 14-PREDICTOR MODEL")
print("=" * 110)

print(
    performance_change_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("INTERNATIONAL ACHIEVEMENT-AUGMENTED ARCHITECTURE")
print("=" * 110)


for outcome in PV_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        international_architecture[
            international_architecture[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "InternationalRank",
                "ConceptualFeature",
                "MeanSystemShare",
                "MedianSystemRank",
                "Top5SystemRate",
                "MeanPVInstability",
            ]
        ]
    )


    print(
        temp
        .round(4)
        .to_string(
            index=False
        )
    )

RECOVERED PV ANALYSIS — COMPLETION QA
Expected models: 3690
Performance models: 3690
Ranking models: 3690
QA models: 3690
Recorded errors: 0
Ranking rows: 62730 / 62730
Duplicate performance model rows: 0


SHAP QA
Importance sum range: 0.9999999999999996 to 1.0000000000000002
Maximum additivity error: 1.1458992958068848e-05


FULL PV ANALYSIS COMPLETE
Expected models: 3690
Successful models: 3690
Errors: 0


ACHIEVEMENT FAMILY SHARE BY OUTCOME
         Outcome  N_systems  Mean_system_achievement_share  Median_system_achievement_share  Min_system_achievement_share  Max_system_achievement_share  Mean_PV_SD
    BEINGBULLIED         73                         0.1808                           0.1752                        0.0980                        0.3582      0.0190
  BULLY_PHYSICAL         74                         0.2351                           0.2331                        0.1247                        0.4012      0.0200
BULLY_RELATIONAL         74                         0.2249 

In [38]:
# ============================================================
# DOES ACHIEVEMENT CHANGE THE ORIGINAL 14-PREDICTOR ARCHITECTURE?
#
# No model fitting.
#
# For every system/outcome:
#
# 1. Take the achievement-augmented PV models.
# 2. Average each predictor's SHAP share across PV1...PV10.
# 3. Remove READ/MATH/SCIENCE achievement.
# 4. Renormalize the original 14 predictors to sum to 1.
# 5. Compare against the frozen primary 14-predictor model.
#
# Metrics:
#   - Cosine similarity
#   - Rank correlation
#   - Top-1 agreement
#   - Top-5 Jaccard
#   - Top-7 Jaccard
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

ARCH_DIR = Path(
    "../results/achievement_pv"
)

PRIMARY_RANK_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


NONPV_FEATURES = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


ACHIEVEMENT_FEATURES = [
    "READ_ACHIEVEMENT",
    "MATH_ACHIEVEMENT",
    "SCIENCE_ACHIEVEMENT",
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


# ============================================================
# 2. Load saved results
# ============================================================

primary = pd.read_csv(
    PRIMARY_RANK_PATH
)


augmented = pd.read_csv(
    ARCH_DIR
    / "system_conceptual_predictor_summary.csv"
)


# ============================================================
# 3. Restrict augmented models to original 14 predictors
# ============================================================

aug_nonpv = (
    augmented[
        augmented[
            "ConceptualFeature"
        ]
        .isin(
            NONPV_FEATURES
        )
    ]
    .copy()
)


# ============================================================
# 4. Renormalize original 14 predictors within augmented model
# ============================================================

nonpv_total = (
    aug_nonpv
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "MeanImportanceShare"
    ]
    .transform(
        "sum"
    )
)


aug_nonpv[
    "RenormalizedNonPVShare"
] = (
    aug_nonpv[
        "MeanImportanceShare"
    ]
    /
    nonpv_total
)


# Re-rank the 14 predictors after achievement removal

aug_nonpv[
    "RenormalizedNonPVRank"
] = (
    aug_nonpv
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "RenormalizedNonPVShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


# ============================================================
# 5. QA
# ============================================================

counts = (
    aug_nonpv
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "ConceptualFeature"
    ]
    .nunique()
)


bad_counts = (
    counts[
        counts != 14
    ]
)


if len(bad_counts) > 0:

    raise RuntimeError(
        "Some augmented system/outcome models "
        "do not contain all 14 original predictors."
    )


share_sums = (
    aug_nonpv
    .groupby(
        [
            "Country",
            "Outcome",
        ]
    )
    [
        "RenormalizedNonPVShare"
    ]
    .sum()
)


print(
    "Renormalized non-achievement share range:",
    share_sums.min(),
    "to",
    share_sums.max(),
)


# ============================================================
# 6. Pair each augmented model with its primary model
#
# VNM naturally disappears because it has no PV analysis.
# ============================================================

comparison_rows = []


available_models = (
    aug_nonpv[
        [
            "Country",
            "Outcome",
        ]
    ]
    .drop_duplicates()
)


for _, model_row in available_models.iterrows():

    country = (
        model_row[
            "Country"
        ]
    )

    outcome = (
        model_row[
            "Outcome"
        ]
    )


    # --------------------------------------------------------
    # Primary 14-feature profile
    # --------------------------------------------------------

    p = (
        primary[
            (
                primary[
                    "Country"
                ]
                == country
            )
            &
            (
                primary[
                    "Outcome"
                ]
                == outcome
            )
        ]
        .set_index(
            "Feature"
        )
        .loc[
            NONPV_FEATURES
        ]
    )


    # --------------------------------------------------------
    # Augmented model after achievement removal
    # --------------------------------------------------------

    a = (
        aug_nonpv[
            (
                aug_nonpv[
                    "Country"
                ]
                == country
            )
            &
            (
                aug_nonpv[
                    "Outcome"
                ]
                == outcome
            )
        ]
        .set_index(
            "ConceptualFeature"
        )
        .loc[
            NONPV_FEATURES
        ]
    )


    primary_share = (
        p[
            "ImportanceShare"
        ]
        .to_numpy(
            dtype=float
        )
    )


    augmented_share = (
        a[
            "RenormalizedNonPVShare"
        ]
        .to_numpy(
            dtype=float
        )
    )


    primary_rank = (
        p[
            "Rank"
        ]
        .to_numpy(
            dtype=float
        )
    )


    augmented_rank = (
        a[
            "RenormalizedNonPVRank"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Cosine
    # --------------------------------------------------------

    cosine = float(
        np.dot(
            primary_share,
            augmented_share,
        )
        /
        (
            np.linalg.norm(
                primary_share
            )
            *
            np.linalg.norm(
                augmented_share
            )
        )
    )


    # --------------------------------------------------------
    # Rank correlation
    # --------------------------------------------------------

    rank_corr = float(
        np.corrcoef(
            primary_rank,
            augmented_rank,
        )[0, 1]
    )


    # --------------------------------------------------------
    # Top-k sets
    # --------------------------------------------------------

    primary_top1 = (
        p[
            "Rank"
        ]
        .idxmin()
    )


    augmented_top1 = (
        a[
            "RenormalizedNonPVRank"
        ]
        .idxmin()
    )


    primary_top5 = set(
        p.loc[
            p[
                "Rank"
            ]
            <= 5
        ]
        .index
    )


    augmented_top5 = set(
        a.loc[
            a[
                "RenormalizedNonPVRank"
            ]
            <= 5
        ]
        .index
    )


    primary_top7 = set(
        p.loc[
            p[
                "Rank"
            ]
            <= 7
        ]
        .index
    )


    augmented_top7 = set(
        a.loc[
            a[
                "RenormalizedNonPVRank"
            ]
            <= 7
        ]
        .index
    )


    top5_jaccard = float(
        len(
            primary_top5
            &
            augmented_top5
        )
        /
        len(
            primary_top5
            |
            augmented_top5
        )
    )


    top7_jaccard = float(
        len(
            primary_top7
            &
            augmented_top7
        )
        /
        len(
            primary_top7
            |
            augmented_top7
        )
    )


    # --------------------------------------------------------
    # Share displacement
    # --------------------------------------------------------

    mean_abs_share_change = float(
        np.abs(
            primary_share
            -
            augmented_share
        )
        .mean()
    )


    comparison_rows.append(
        {
            "Country": country,
            "Outcome": outcome,

            "CosineSimilarity":
                cosine,

            "RankCorrelation":
                rank_corr,

            "Top1Same":
                int(
                    primary_top1
                    ==
                    augmented_top1
                ),

            "PrimaryTop1":
                primary_top1,

            "AugmentedTop1":
                augmented_top1,

            "Top5Jaccard":
                top5_jaccard,

            "Top7Jaccard":
                top7_jaccard,

            "MeanAbsoluteShareChange":
                mean_abs_share_change,
        }
    )


architecture_comparison = pd.DataFrame(
    comparison_rows
)


architecture_comparison.to_csv(
    ARCH_DIR
    / "primary_vs_augmented_nonpv_architecture.csv",
    index=False,
)


# ============================================================
# 7. Summary by outcome
# ============================================================

architecture_summary = (
    architecture_comparison
    .groupby(
        "Outcome"
    )
    .agg(

        N_systems=(
            "Country",
            "count",
        ),

        Mean_cosine=(
            "CosineSimilarity",
            "mean",
        ),

        Median_cosine=(
            "CosineSimilarity",
            "median",
        ),

        Min_cosine=(
            "CosineSimilarity",
            "min",
        ),

        Mean_rank_corr=(
            "RankCorrelation",
            "mean",
        ),

        Median_rank_corr=(
            "RankCorrelation",
            "median",
        ),

        Min_rank_corr=(
            "RankCorrelation",
            "min",
        ),

        Top1_agreement_rate=(
            "Top1Same",
            "mean",
        ),

        Mean_top5_jaccard=(
            "Top5Jaccard",
            "mean",
        ),

        Mean_top7_jaccard=(
            "Top7Jaccard",
            "mean",
        ),

        Mean_abs_share_change=(
            "MeanAbsoluteShareChange",
            "mean",
        ),
    )
    .reset_index()
)


architecture_summary.to_csv(
    ARCH_DIR
    / "primary_vs_augmented_architecture_summary.csv",
    index=False,
)


# ============================================================
# 8. International mean residual architecture
# ============================================================

residual_international = (
    aug_nonpv
    .groupby(
        [
            "Outcome",
            "ConceptualFeature",
        ]
    )
    .agg(

        MeanRenormalizedShare=(
            "RenormalizedNonPVShare",
            "mean",
        ),

        MedianRenormalizedRank=(
            "RenormalizedNonPVRank",
            "median",
        ),

        Top5SystemRate=(
            "RenormalizedNonPVRank",
            lambda x: float(
                (
                    x <= 5
                ).mean()
            ),
        ),
    )
    .reset_index()
)


residual_international[
    "InternationalRank"
] = (
    residual_international
    .groupby(
        "Outcome"
    )
    [
        "MeanRenormalizedShare"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


residual_international = (
    residual_international
    .sort_values(
        [
            "Outcome",
            "InternationalRank",
        ]
    )
)


residual_international.to_csv(
    ARCH_DIR
    / "augmented_residual_nonpv_architecture.csv",
    index=False,
)


# ============================================================
# 9. Systems with largest architecture changes
# ============================================================

largest_changes = (
    architecture_comparison
    .sort_values(
        [
            "Outcome",
            "CosineSimilarity",
        ]
    )
)


largest_changes.to_csv(
    ARCH_DIR
    / "systems_with_largest_architecture_changes.csv",
    index=False,
)


# ============================================================
# 10. Print summary
# ============================================================

print("\n")
print("=" * 110)
print("PRIMARY vs ACHIEVEMENT-AUGMENTED NON-PV ARCHITECTURE")
print("=" * 110)


print(
    architecture_summary
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Print systems with largest changes
# ============================================================

print("\n")
print("=" * 110)
print("SYSTEMS WITH LARGEST NON-PV ARCHITECTURE CHANGES")
print("=" * 110)


for outcome in OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        architecture_comparison[
            architecture_comparison[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineSimilarity"
        )
        .head(
            5
        )
    )


    print(
        temp
        .round(4)
        .to_string(
            index=False
        )
    )


# ============================================================
# 12. Print residual architecture
# ============================================================

print("\n")
print("=" * 110)
print("INTERNATIONAL NON-PV ARCHITECTURE AFTER ADDING ACHIEVEMENT")
print("=" * 110)


for outcome in OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        residual_international[
            residual_international[
                "Outcome"
            ]
            == outcome
        ]
        [
            [
                "InternationalRank",
                "ConceptualFeature",
                "MeanRenormalizedShare",
                "MedianRenormalizedRank",
                "Top5SystemRate",
            ]
        ]
    )


    print(
        temp
        .round(4)
        .to_string(
            index=False
        )
    )

Renormalized non-achievement share range: 0.9999999999999999 to 1.0000000000000002


PRIMARY vs ACHIEVEMENT-AUGMENTED NON-PV ARCHITECTURE
         Outcome  N_systems  Mean_cosine  Median_cosine  Min_cosine  Mean_rank_corr  Median_rank_corr  Min_rank_corr  Top1_agreement_rate  Mean_top5_jaccard  Mean_top7_jaccard  Mean_abs_share_change
    BEINGBULLIED         73       0.9974         0.9984      0.9708          0.9482            0.9648         0.6747               1.0000             0.8069             0.8596                 0.0055
  BULLY_PHYSICAL         74       0.9933         0.9947      0.9735          0.8813            0.8945         0.5560               0.9324             0.7310             0.7740                 0.0075
BULLY_RELATIONAL         74       0.9943         0.9951      0.9729          0.8789            0.9055         0.6440               0.9865             0.7230             0.7580                 0.0073
    BULLY_THREAT         74       0.9918         0.9934      0.954

In [39]:
# ============================================================
# PISA REPLICATE-WEIGHT UNCERTAINTY FOR OUTCOME ESTIMATES
#
# Uses:
#   W_FSTUWT
#   W_FSTURWT1 ... W_FSTURWT80
#
# Fay BRR:
#
# Var(theta) =
#   1 / [80 * (1 - 0.5)^2]
#   * sum_r (theta_r - theta)^2
#
# = 1/20 * sum_r(...)
#
# BEINGBULLIED:
#   survey-weighted mean
#
# Binary subtype outcomes:
#   survey-weighted prevalence
#
# NO MODEL FITTING.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

UNCERTAINTY_DIR = Path(
    "../results/design_uncertainty"
)

UNCERTAINTY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


FINAL_WEIGHT = (
    "W_FSTUWT"
)


REPLICATE_WEIGHTS = [
    f"W_FSTURWT{i}"
    for i in range(
        1,
        81,
    )
]


N_REPLICATES = 80

FAY_K = 0.5


# ============================================================
# 2. Fay variance multiplier
# ============================================================

FAY_VARIANCE_MULTIPLIER = (
    1
    /
    (
        N_REPLICATES
        *
        (
            1
            -
            FAY_K
        ) ** 2
    )
)


print(
    "Fay variance multiplier:",
    FAY_VARIANCE_MULTIPLIER
)

# Should equal 0.05 = 1 / 20

assert np.isclose(
    FAY_VARIANCE_MULTIPLIER,
    1 / 20,
)


# ============================================================
# 3. Weighted mean helper
# ============================================================

def weighted_mean(
    values,
    weights,
):

    values = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    weights = pd.to_numeric(
        weights,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    mask = (
        np.isfinite(
            values
        )
        &
        np.isfinite(
            weights
        )
        &
        (
            weights > 0
        )
    )


    if mask.sum() == 0:

        return np.nan


    return float(
        np.average(
            values[
                mask
            ],
            weights=weights[
                mask
            ],
        )
    )


# ============================================================
# 4. System sets
# ============================================================

OUTCOME_SYSTEMS = {
    "BEINGBULLIED":
        list(
            PRIMARY_BEING_SYSTEMS
        ),

    "BULLY_VERBAL":
        list(
            PRIMARY_SUBTYPE_SYSTEMS
        ),

    "BULLY_THREAT":
        list(
            PRIMARY_SUBTYPE_SYSTEMS
        ),

    "BULLY_PHYSICAL":
        list(
            PRIMARY_SUBTYPE_SYSTEMS
        ),

    "BULLY_RELATIONAL":
        list(
            PRIMARY_SUBTYPE_SYSTEMS
        ),
}


# ============================================================
# 5. Validate replicate weights
# ============================================================

all_systems = sorted(
    set(
        PRIMARY_BEING_SYSTEMS
    )
    |
    set(
        PRIMARY_SUBTYPE_SYSTEMS
    )
)


weight_qa_rows = []


for country in all_systems:

    df = (
        analysis_frames[
            country
        ]
    )


    missing_weights = [
        column
        for column in (
            [FINAL_WEIGHT]
            +
            REPLICATE_WEIGHTS
        )
        if column not in df.columns
    ]


    weight_qa_rows.append(
        {
            "Country":
                country,

            "MissingWeightColumns":
                len(
                    missing_weights
                ),

            "MissingColumns":
                ",".join(
                    missing_weights
                ),
        }
    )


weight_qa = pd.DataFrame(
    weight_qa_rows
)


if (
    weight_qa[
        "MissingWeightColumns"
    ]
    > 0
).any():

    print(
        weight_qa[
            weight_qa[
                "MissingWeightColumns"
            ]
            > 0
        ]
    )

    raise RuntimeError(
        "One or more systems are missing "
        "required replicate weights."
    )


print(
    "Systems with all 80 replicate weights:",
    len(
        weight_qa
    )
)


# ============================================================
# 6. Calculate full-sample and replicate estimates
# ============================================================

estimate_rows = []

replicate_rows = []


for outcome in OUTCOMES:

    systems = (
        OUTCOME_SYSTEMS[
            outcome
        ]
    )


    for country in systems:

        df = (
            analysis_frames[
                country
            ]
        )


        y = pd.to_numeric(
            df[
                outcome
            ],
            errors="coerce",
        )


        # -----------------------------------------------
        # Main estimate using final student weight
        # -----------------------------------------------

        theta = weighted_mean(
            y,
            df[
                FINAL_WEIGHT
            ],
        )


        if not np.isfinite(
            theta
        ):

            raise RuntimeError(
                f"{country} — {outcome}: "
                "main weighted estimate is invalid."
            )


        # -----------------------------------------------
        # Replicate estimates
        # -----------------------------------------------

        replicate_estimates = []


        for r, weight_column in enumerate(
            REPLICATE_WEIGHTS,
            start=1,
        ):

            theta_r = weighted_mean(
                y,
                df[
                    weight_column
                ],
            )


            replicate_estimates.append(
                theta_r
            )


            replicate_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "Replicate":
                        r,

                    "Estimate":
                        theta_r,
                }
            )


        replicate_estimates = np.asarray(
            replicate_estimates,
            dtype=float,
        )


        if not np.all(
            np.isfinite(
                replicate_estimates
            )
        ):

            raise RuntimeError(
                f"{country} — {outcome}: "
                "one or more replicate estimates "
                "are invalid."
            )


        # -----------------------------------------------
        # Fay BRR variance
        # -----------------------------------------------

        variance = float(
            FAY_VARIANCE_MULTIPLIER
            *
            np.square(
                replicate_estimates
                -
                theta
            )
            .sum()
        )


        se = float(
            np.sqrt(
                variance
            )
        )


        ci_lower_raw = float(
            theta
            -
            1.96
            *
            se
        )


        ci_upper_raw = float(
            theta
            +
            1.96
            *
            se
        )


        # -----------------------------------------------
        # For binary prevalence only:
        # clip displayed CI to valid probability range.
        #
        # Raw CI is still preserved separately.
        # -----------------------------------------------

        if outcome in BINARY_OUTCOMES:

            ci_lower = float(
                max(
                    0,
                    ci_lower_raw,
                )
            )

            ci_upper = float(
                min(
                    1,
                    ci_upper_raw,
                )
            )

            estimate_type = (
                "Weighted prevalence"
            )


        else:

            ci_lower = (
                ci_lower_raw
            )

            ci_upper = (
                ci_upper_raw
            )

            estimate_type = (
                "Weighted mean"
            )


        valid_y = (
            y.notna()
        )


        estimate_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    outcome,

                "EstimateType":
                    estimate_type,

                "N_unweighted":
                    int(
                        valid_y.sum()
                    ),

                "N_schools":
                    int(
                        df.loc[
                            valid_y,
                            "CNTSCHID",
                        ]
                        .nunique()
                    ),

                "Estimate":
                    theta,

                "SE":
                    se,

                "Variance":
                    variance,

                "CI95_Lower":
                    ci_lower,

                "CI95_Upper":
                    ci_upper,

                "CI95_Lower_Raw":
                    ci_lower_raw,

                "CI95_Upper_Raw":
                    ci_upper_raw,

                "ReplicateSD":
                    float(
                        replicate_estimates.std(
                            ddof=1
                        )
                    ),
            }
        )


# ============================================================
# 7. Save
# ============================================================

outcome_uncertainty = pd.DataFrame(
    estimate_rows
)


replicate_estimates_long = pd.DataFrame(
    replicate_rows
)


outcome_uncertainty.to_csv(
    UNCERTAINTY_DIR
    / "outcome_replicate_weight_uncertainty.csv",
    index=False,
)


replicate_estimates_long.to_csv(
    UNCERTAINTY_DIR
    / "outcome_replicate_estimates_long.csv",
    index=False,
)


weight_qa.to_csv(
    UNCERTAINTY_DIR
    / "replicate_weight_qa.csv",
    index=False,
)


# ============================================================
# 8. Summary across education systems
# ============================================================

uncertainty_summary = (
    outcome_uncertainty
    .groupby(
        "Outcome"
    )
    .agg(

        N_systems=(
            "Country",
            "count",
        ),

        Median_estimate=(
            "Estimate",
            "median",
        ),

        Median_SE=(
            "SE",
            "median",
        ),

        Mean_SE=(
            "SE",
            "mean",
        ),

        Min_SE=(
            "SE",
            "min",
        ),

        Max_SE=(
            "SE",
            "max",
        ),

        Median_CI_width=(
            "CI95_Upper",
            lambda x: np.nan,
        ),
    )
    .reset_index()
)


# Calculate CI width separately

outcome_uncertainty[
    "CI95_Width"
] = (
    outcome_uncertainty[
        "CI95_Upper"
    ]
    -
    outcome_uncertainty[
        "CI95_Lower"
    ]
)


ci_width_summary = (
    outcome_uncertainty
    .groupby(
        "Outcome"
    )
    [
        "CI95_Width"
    ]
    .median()
    .rename(
        "Median_CI_width"
    )
    .reset_index()
)


uncertainty_summary = (
    uncertainty_summary
    .drop(
        columns=[
            "Median_CI_width"
        ]
    )
    .merge(
        ci_width_summary,
        on="Outcome",
        how="left",
    )
)


uncertainty_summary.to_csv(
    UNCERTAINTY_DIR
    / "outcome_uncertainty_summary.csv",
    index=False,
)


# ============================================================
# 9. QA
# ============================================================

expected_estimates = (
    len(
        PRIMARY_BEING_SYSTEMS
    )
    +
    (
        4
        *
        len(
            PRIMARY_SUBTYPE_SYSTEMS
        )
    )
)


expected_replicate_rows = (
    expected_estimates
    *
    80
)


print("\n")
print("=" * 110)
print("REPLICATE-WEIGHT QA")
print("=" * 110)


print(
    "Expected system-outcome estimates:",
    expected_estimates
)

print(
    "Observed system-outcome estimates:",
    len(
        outcome_uncertainty
    )
)

print(
    "Expected replicate estimates:",
    expected_replicate_rows
)

print(
    "Observed replicate estimates:",
    len(
        replicate_estimates_long
    )
)


assert (
    len(
        outcome_uncertainty
    )
    ==
    expected_estimates
)


assert (
    len(
        replicate_estimates_long
    )
    ==
    expected_replicate_rows
)


# ============================================================
# 10. Print cross-system uncertainty summary
# ============================================================

print("\n")
print("=" * 110)
print("REPLICATE-WEIGHT OUTCOME UNCERTAINTY SUMMARY")
print("=" * 110)


print(
    uncertainty_summary
    .round(
        5
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Least precise estimates
# ============================================================

print("\n")
print("=" * 110)
print("LARGEST STANDARD ERRORS BY OUTCOME")
print("=" * 110)


for outcome in OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        outcome_uncertainty[
            outcome_uncertainty[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "SE",
            ascending=False,
        )
        .head(
            5
        )
        [
            [
                "Country",
                "Estimate",
                "SE",
                "CI95_Lower",
                "CI95_Upper",
                "N_unweighted",
                "N_schools",
            ]
        ]
    )


    print(
        temp
        .round(
            5
        )
        .to_string(
            index=False
        )
    )

Fay variance multiplier: 0.05
Systems with all 80 replicate weights: 75


REPLICATE-WEIGHT QA
Expected system-outcome estimates: 374
Observed system-outcome estimates: 374
Expected replicate estimates: 29920
Observed replicate estimates: 29920


REPLICATE-WEIGHT OUTCOME UNCERTAINTY SUMMARY
         Outcome  N_systems  Median_estimate  Median_SE  Mean_SE  Min_SE  Max_SE  Median_CI_width
    BEINGBULLIED         74          0.02814    0.01743  0.01834 0.00985 0.04621          0.06832
  BULLY_PHYSICAL         75          0.11614    0.00551  0.00608 0.00205 0.01684          0.02160
BULLY_RELATIONAL         75          0.12615    0.00524  0.00559 0.00241 0.01562          0.02054
    BULLY_THREAT         75          0.07487    0.00463  0.00484 0.00127 0.01479          0.01816
    BULLY_VERBAL         75          0.13685    0.00547  0.00586 0.00290 0.01397          0.02144


LARGEST STANDARD ERRORS BY OUTCOME


----------------------------------------------------------------------------------

In [40]:
# ============================================================
# 40% PREDICTOR-MISSINGNESS SENSITIVITY
#
# Purpose:
# Test whether the frozen 14-predictor architecture depends on
# allowing predictors with up to 50% missingness in a system.
#
# Sensitivity rule:
#   Predictor must be usable in every primary system
#   AND maximum system-level missingness <= 40%.
#
# Expected:
#   14 primary predictors -> 12 sensitivity predictors
#
# Fits:
#   74 BEINGBULLIED models
#   75 x 4 subtype models
#   = 374 models total
#
# Uses same:
#   - systems
#   - outcomes
#   - weights
#   - folds
#   - XGBoost reference specification
#   - OOF TreeSHAP
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

MISS_DIR = Path(
    "../results/missingness40_sensitivity"
)

MISS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PRIMARY_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


MISS_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


MISS_BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


SYSTEM_UNIVERSE = sorted(
    set(PRIMARY_BEING_SYSTEMS)
    |
    set(PRIMARY_SUBTYPE_SYSTEMS)
)


# ============================================================
# 2. Reconstruct predictor missingness under the same
#    system-level availability logic
# ============================================================

availability_rows = []


for feature in PRIMARY_14:

    for country in SYSTEM_UNIVERSE:

        df = analysis_frames[
            country
        ]


        if feature not in df.columns:

            availability_rows.append(
                {
                    "Country": country,
                    "Feature": feature,
                    "Missing_pct": 100.0,
                    "N_nonmissing": 0,
                    "N_unique": 0,
                    "Usable": False,
                }
            )

            continue


        values = pd.to_numeric(
            df[feature],
            errors="coerce",
        )


        n_total = len(
            values
        )


        n_nonmissing = int(
            values.notna().sum()
        )


        n_unique = int(
            values.nunique(
                dropna=True
            )
        )


        missing_pct = float(
            100
            * values.isna().mean()
        )


        usable = (
            n_nonmissing > 0
            and
            n_unique > 1
        )


        availability_rows.append(
            {
                "Country": country,
                "Feature": feature,
                "Missing_pct": missing_pct,
                "N_nonmissing": n_nonmissing,
                "N_unique": n_unique,
                "Usable": usable,
            }
        )


availability40 = pd.DataFrame(
    availability_rows
)


predictor40_summary = (
    availability40
    .groupby(
        "Feature"
    )
    .agg(

        Max_missing_pct=(
            "Missing_pct",
            "max",
        ),

        Mean_missing_pct=(
            "Missing_pct",
            "mean",
        ),

        Usable_all_systems=(
            "Usable",
            "all",
        ),
    )
    .reset_index()
)


STRICT40_PREDICTORS = (
    predictor40_summary.loc[
        (
            predictor40_summary[
                "Usable_all_systems"
            ]
        )
        &
        (
            predictor40_summary[
                "Max_missing_pct"
            ]
            <= 40
        ),
        "Feature",
    ]
    .tolist()
)


STRICT40_PREDICTORS = [
    feature
    for feature in PRIMARY_14
    if feature in STRICT40_PREDICTORS
]


DROPPED40 = [
    feature
    for feature in PRIMARY_14
    if feature not in STRICT40_PREDICTORS
]


print(
    "Primary predictors:",
    len(PRIMARY_14)
)

print(
    "40% sensitivity predictors:",
    len(STRICT40_PREDICTORS)
)

print(
    "Retained:",
    STRICT40_PREDICTORS
)

print(
    "Dropped:",
    DROPPED40
)


print("\n")
print(
    predictor40_summary
    .sort_values(
        "Max_missing_pct",
        ascending=False,
    )
    .round(2)
    .to_string(
        index=False
    )
)


# Earlier audit indicated exactly 12.

if len(STRICT40_PREDICTORS) != 12:

    raise RuntimeError(
        "Expected 12 predictors under the 40% rule, "
        f"but found {len(STRICT40_PREDICTORS)}. "
        "Stop here so we can inspect why."
    )


# ============================================================
# 3. Stable sigmoid
# ============================================================

def sensitivity_sigmoid(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    result = np.empty_like(
        x
    )


    positive = (
        x >= 0
    )

    negative = ~positive


    result[
        positive
    ] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -x[
                    positive
                ]
            )
        )
    )


    exp_x = np.exp(
        x[
            negative
        ]
    )


    result[
        negative
    ] = (
        exp_x
        /
        (
            1.0
            +
            exp_x
        )
    )


    return result


# ============================================================
# 4. Preserve frozen primary predictor list
# ============================================================

ORIGINAL_ATTRIBUTION_PREDICTORS = (
    ATTRIBUTION_PREDICTORS.copy()
)


# ============================================================
# 5. Run 40% sensitivity models
# ============================================================

ranking_batches = []

performance_rows = []

qa_rows = []

error_rows = []


EXPECTED_MODELS = (
    len(PRIMARY_BEING_SYSTEMS)
    +
    4
    * len(PRIMARY_SUBTYPE_SYSTEMS)
)


model_counter = 0

start_time = time.time()


try:

    ATTRIBUTION_PREDICTORS[:] = (
        STRICT40_PREDICTORS
    )


    for outcome in MISS_OUTCOMES:

        if outcome == "BEINGBULLIED":

            systems = (
                PRIMARY_BEING_SYSTEMS
            )

        else:

            systems = (
                PRIMARY_SUBTYPE_SYSTEMS
            )


        for country in systems:

            model_counter += 1


            print(
                f"[{model_counter}/{EXPECTED_MODELS}] "
                f"{country} — {outcome}",
                end=" ... ",
            )


            try:

                (
                    ranking,
                    fold_importance,
                    qa,
                    row_shap,
                ) = run_country_oof_shap(
                    df=analysis_frames[
                        country
                    ],
                    country=country,
                    outcome=outcome,
                )


                # --------------------------------------------
                # Structural QA
                # --------------------------------------------

                if len(
                    ranking
                ) != len(
                    STRICT40_PREDICTORS
                ):

                    raise RuntimeError(
                        "Incorrect feature count."
                    )


                importance_sum = float(
                    ranking[
                        "ImportanceShare"
                    ]
                    .sum()
                )


                max_error = float(
                    qa[
                        "Max_SHAP_additivity_error"
                    ]
                    .max()
                )


                ranking_batches.append(
                    ranking.copy()
                )


                # --------------------------------------------
                # Recover exact OOF prediction from SHAP
                # --------------------------------------------

                shap_columns = [
                    f"SHAP__{feature}"
                    for feature
                    in STRICT40_PREDICTORS
                ]


                raw_prediction = (
                    row_shap[
                        "SHAP_Bias"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                    +
                    row_shap[
                        shap_columns
                    ]
                    .sum(
                        axis=1
                    )
                    .to_numpy(
                        dtype=float
                    )
                )


                observed = (
                    row_shap[
                        "Observed"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                weights = (
                    row_shap[
                        "W_FSTUWT"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                # --------------------------------------------
                # Performance
                # --------------------------------------------

                if outcome == "BEINGBULLIED":

                    metrics = (
                        regression_metrics(
                            observed,
                            raw_prediction,
                            weights,
                        )
                    )


                    perf = {
                        "Country": country,
                        "Outcome": outcome,

                        "Model_R2":
                            metrics[
                                "R2"
                            ],

                        "Model_RMSE":
                            metrics[
                                "RMSE"
                            ],

                        "Model_MAE":
                            metrics[
                                "MAE"
                            ],

                        "Model_ROC_AUC":
                            np.nan,

                        "Model_PR_AUC":
                            np.nan,
                    }


                else:

                    probability = (
                        sensitivity_sigmoid(
                            raw_prediction
                        )
                    )


                    metrics = (
                        classification_metrics(
                            observed,
                            probability,
                            weights,
                        )
                    )


                    perf = {
                        "Country": country,
                        "Outcome": outcome,

                        "Model_R2":
                            np.nan,

                        "Model_RMSE":
                            np.nan,

                        "Model_MAE":
                            np.nan,

                        "Model_ROC_AUC":
                            metrics[
                                "ROC_AUC"
                            ],

                        "Model_PR_AUC":
                            metrics[
                                "PR_AUC"
                            ],
                    }


                performance_rows.append(
                    perf
                )


                qa_rows.append(
                    {
                        "Country": country,
                        "Outcome": outcome,

                        "ImportanceSum":
                            importance_sum,

                        "MaxAdditivityError":
                            max_error,
                    }
                )


                top = (
                    ranking
                    .sort_values(
                        "Rank"
                    )
                    .iloc[0]
                )


                print(
                    f"OK | "
                    f"#1={top['Feature']}"
                )


                del fold_importance
                del qa
                del row_shap


                if (
                    model_counter
                    % 50
                    == 0
                ):

                    gc.collect()


            except Exception as e:

                error_rows.append(
                    {
                        "Country": country,
                        "Outcome": outcome,
                        "Error_type":
                            type(e).__name__,
                        "Error":
                            str(e),
                    }
                )


                print(
                    f"ERROR | "
                    f"{type(e).__name__}: "
                    f"{e}"
                )


finally:

    ATTRIBUTION_PREDICTORS[:] = (
        ORIGINAL_ATTRIBUTION_PREDICTORS
    )


# ============================================================
# 6. Combine and save
# ============================================================

strict_rankings = pd.concat(
    ranking_batches,
    ignore_index=True,
)


strict_performance = pd.DataFrame(
    performance_rows
)


strict_qa = pd.DataFrame(
    qa_rows
)


strict_errors = pd.DataFrame(
    error_rows
)


strict_rankings.to_csv(
    MISS_DIR
    / "oof_shap_rankings_40pct.csv",
    index=False,
)


strict_performance.to_csv(
    MISS_DIR
    / "overall_performance_40pct.csv",
    index=False,
)


strict_qa.to_csv(
    MISS_DIR
    / "shap_qa_40pct.csv",
    index=False,
)


strict_errors.to_csv(
    MISS_DIR
    / "errors_40pct.csv",
    index=False,
)


predictor40_summary.to_csv(
    MISS_DIR
    / "predictor_40pct_availability.csv",
    index=False,
)


# ============================================================
# 7. Completion QA
# ============================================================

elapsed = (
    time.time()
    -
    start_time
) / 60


print("\n")
print("=" * 110)
print("40% MISSINGNESS SENSITIVITY — COMPLETION")
print("=" * 110)


print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Successful models:",
    len(
        strict_performance
    )
)

print(
    "Errors:",
    len(
        strict_errors
    )
)

print(
    "Expected ranking rows:",
    EXPECTED_MODELS
    *
    len(
        STRICT40_PREDICTORS
    )
)

print(
    "Observed ranking rows:",
    len(
        strict_rankings
    )
)

print(
    f"Runtime: "
    f"{elapsed:.2f} minutes"
)

print(
    "Primary predictor list restored:",
    ATTRIBUTION_PREDICTORS
    ==
    ORIGINAL_ATTRIBUTION_PREDICTORS
)


print(
    "Importance sum range:",
    strict_qa[
        "ImportanceSum"
    ].min(),
    "to",
    strict_qa[
        "ImportanceSum"
    ].max(),
)


print(
    "Maximum SHAP additivity error:",
    strict_qa[
        "MaxAdditivityError"
    ].max(),
)


# ============================================================
# 8. Compare performance against frozen primary
# ============================================================

primary_perf = pd.read_csv(
    "../results/primary_models/"
    "overall_performance.csv"
)


perf_compare = (
    strict_performance
    .merge(
        primary_perf[
            [
                "Country",
                "Outcome",
                "Model_R2",
                "Model_ROC_AUC",
                "Model_PR_AUC",
            ]
        ],
        on=[
            "Country",
            "Outcome",
        ],
        suffixes=(
            "_Strict40",
            "_Primary",
        ),
        validate="one_to_one",
    )
)


perf_compare[
    "Delta_R2"
] = (
    perf_compare[
        "Model_R2_Strict40"
    ]
    -
    perf_compare[
        "Model_R2_Primary"
    ]
)


perf_compare[
    "Delta_ROC_AUC"
] = (
    perf_compare[
        "Model_ROC_AUC_Strict40"
    ]
    -
    perf_compare[
        "Model_ROC_AUC_Primary"
    ]
)


perf_compare[
    "Delta_PR_AUC"
] = (
    perf_compare[
        "Model_PR_AUC_Strict40"
    ]
    -
    perf_compare[
        "Model_PR_AUC_Primary"
    ]
)


# ============================================================
# 9. Performance summary
# ============================================================

perf_summary_rows = []


for outcome in MISS_OUTCOMES:

    temp = (
        perf_compare[
            perf_compare[
                "Outcome"
            ]
            == outcome
        ]
    )


    if outcome == "BEINGBULLIED":

        delta = (
            temp[
                "Delta_R2"
            ]
            .dropna()
        )


        perf_summary_rows.append(
            {
                "Outcome": outcome,
                "Metric": "R2",
                "N": len(delta),

                "Mean_delta":
                    delta.mean(),

                "Median_delta":
                    delta.median(),

                "Min_delta":
                    delta.min(),

                "Max_delta":
                    delta.max(),

                "Pct_strict40_better":
                    100
                    *
                    (
                        delta > 0
                    ).mean(),
            }
        )


    else:

        for metric, column in [
            (
                "ROC_AUC",
                "Delta_ROC_AUC",
            ),
            (
                "PR_AUC",
                "Delta_PR_AUC",
            ),
        ]:

            delta = (
                temp[
                    column
                ]
                .dropna()
            )


            perf_summary_rows.append(
                {
                    "Outcome": outcome,
                    "Metric": metric,
                    "N": len(delta),

                    "Mean_delta":
                        delta.mean(),

                    "Median_delta":
                        delta.median(),

                    "Min_delta":
                        delta.min(),

                    "Max_delta":
                        delta.max(),

                    "Pct_strict40_better":
                        100
                        *
                        (
                            delta > 0
                        ).mean(),
                }
            )


perf_summary = pd.DataFrame(
    perf_summary_rows
)


# ============================================================
# 10. Compare architecture fairly
#
# Primary model:
#   remove the two predictors excluded by 40% rule,
#   renormalize remaining 12.
#
# Strict model:
#   already contains those same 12.
# ============================================================

primary_rankings = pd.read_csv(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


architecture_rows = []


for (
    country,
    outcome
) in (
    strict_performance[
        [
            "Country",
            "Outcome",
        ]
    ]
    .itertuples(
        index=False,
        name=None,
    )
):

    # --------------------------------------------------------
    # Primary 12-feature projection
    # --------------------------------------------------------

    p = (
        primary_rankings[
            (
                primary_rankings[
                    "Country"
                ]
                == country
            )
            &
            (
                primary_rankings[
                    "Outcome"
                ]
                == outcome
            )
            &
            (
                primary_rankings[
                    "Feature"
                ]
                .isin(
                    STRICT40_PREDICTORS
                )
            )
        ]
        .copy()
    )


    p[
        "ProjectedShare"
    ] = (
        p[
            "ImportanceShare"
        ]
        /
        p[
            "ImportanceShare"
        ]
        .sum()
    )


    p[
        "ProjectedRank"
    ] = (
        p[
            "ProjectedShare"
        ]
        .rank(
            method="first",
            ascending=False,
        )
        .astype(int)
    )


    p = (
        p
        .set_index(
            "Feature"
        )
        .loc[
            STRICT40_PREDICTORS
        ]
    )


    # --------------------------------------------------------
    # Strict 40% model
    # --------------------------------------------------------

    s = (
        strict_rankings[
            (
                strict_rankings[
                    "Country"
                ]
                == country
            )
            &
            (
                strict_rankings[
                    "Outcome"
                ]
                == outcome
            )
        ]
        .set_index(
            "Feature"
        )
        .loc[
            STRICT40_PREDICTORS
        ]
    )


    p_share = (
        p[
            "ProjectedShare"
        ]
        .to_numpy(
            dtype=float
        )
    )


    s_share = (
        s[
            "ImportanceShare"
        ]
        .to_numpy(
            dtype=float
        )
    )


    p_rank = (
        p[
            "ProjectedRank"
        ]
        .to_numpy(
            dtype=float
        )
    )


    s_rank = (
        s[
            "Rank"
        ]
        .to_numpy(
            dtype=float
        )
    )


    cosine = float(
        np.dot(
            p_share,
            s_share,
        )
        /
        (
            np.linalg.norm(
                p_share
            )
            *
            np.linalg.norm(
                s_share
            )
        )
    )


    rank_corr = float(
        np.corrcoef(
            p_rank,
            s_rank,
        )[0, 1]
    )


    p_top1 = (
        p[
            "ProjectedRank"
        ]
        .idxmin()
    )


    s_top1 = (
        s[
            "Rank"
        ]
        .idxmin()
    )


    p_top5 = set(
        p.loc[
            p[
                "ProjectedRank"
            ]
            <= 5
        ]
        .index
    )


    s_top5 = set(
        s.loc[
            s[
                "Rank"
            ]
            <= 5
        ]
        .index
    )


    p_top7 = set(
        p.loc[
            p[
                "ProjectedRank"
            ]
            <= 7
        ]
        .index
    )


    s_top7 = set(
        s.loc[
            s[
                "Rank"
            ]
            <= 7
        ]
        .index
    )


    architecture_rows.append(
        {
            "Country": country,
            "Outcome": outcome,

            "CosineSimilarity":
                cosine,

            "RankCorrelation":
                rank_corr,

            "Top1Same":
                int(
                    p_top1
                    ==
                    s_top1
                ),

            "PrimaryProjectedTop1":
                p_top1,

            "Strict40Top1":
                s_top1,

            "Top5Jaccard":
                len(
                    p_top5
                    &
                    s_top5
                )
                /
                len(
                    p_top5
                    |
                    s_top5
                ),

            "Top7Jaccard":
                len(
                    p_top7
                    &
                    s_top7
                )
                /
                len(
                    p_top7
                    |
                    s_top7
                ),

            "MeanAbsoluteShareChange":
                float(
                    np.abs(
                        p_share
                        -
                        s_share
                    )
                    .mean()
                ),
        }
    )


architecture_compare = pd.DataFrame(
    architecture_rows
)


architecture_summary = (
    architecture_compare
    .groupby(
        "Outcome"
    )
    .agg(

        N_systems=(
            "Country",
            "count",
        ),

        Mean_cosine=(
            "CosineSimilarity",
            "mean",
        ),

        Median_cosine=(
            "CosineSimilarity",
            "median",
        ),

        Min_cosine=(
            "CosineSimilarity",
            "min",
        ),

        Mean_rank_corr=(
            "RankCorrelation",
            "mean",
        ),

        Median_rank_corr=(
            "RankCorrelation",
            "median",
        ),

        Min_rank_corr=(
            "RankCorrelation",
            "min",
        ),

        Top1_agreement_rate=(
            "Top1Same",
            "mean",
        ),

        Mean_top5_jaccard=(
            "Top5Jaccard",
            "mean",
        ),

        Mean_top7_jaccard=(
            "Top7Jaccard",
            "mean",
        ),

        Mean_abs_share_change=(
            "MeanAbsoluteShareChange",
            "mean",
        ),
    )
    .reset_index()
)


architecture_compare.to_csv(
    MISS_DIR
    / "primary_vs_40pct_architecture.csv",
    index=False,
)


architecture_summary.to_csv(
    MISS_DIR
    / "primary_vs_40pct_architecture_summary.csv",
    index=False,
)


perf_compare.to_csv(
    MISS_DIR
    / "performance_vs_primary.csv",
    index=False,
)


perf_summary.to_csv(
    MISS_DIR
    / "performance_vs_primary_summary.csv",
    index=False,
)


# ============================================================
# 11. How important were the predictors that got removed?
# ============================================================

dropped_primary = (
    primary_rankings[
        primary_rankings[
            "Feature"
        ]
        .isin(
            DROPPED40
        )
    ]
    .groupby(
        [
            "Outcome",
            "Feature",
        ]
    )
    .agg(

        MeanPrimaryShare=(
            "ImportanceShare",
            "mean",
        ),

        MedianPrimaryRank=(
            "Rank",
            "median",
        ),

        Top5Rate=(
            "Rank",
            lambda x: float(
                (
                    x <= 5
                ).mean()
            ),
        ),
    )
    .reset_index()
)


# ============================================================
# 12. Print final summaries
# ============================================================

print("\n")
print("=" * 110)
print("40% RULE — DROPPED PREDICTORS")
print("=" * 110)

print(
    dropped_primary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("40% RULE — PERFORMANCE CHANGE VS PRIMARY")
print("=" * 110)

print(
    perf_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("40% RULE — ARCHITECTURE ROBUSTNESS")
print("=" * 110)

print(
    architecture_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("LARGEST ARCHITECTURE CHANGES")
print("=" * 110)


for outcome in MISS_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        architecture_compare[
            architecture_compare[
                "Outcome"
            ]
            == outcome
        ]
        .sort_values(
            "CosineSimilarity"
        )
        .head(
            5
        )
    )


    print(
        temp
        .round(4)
        .to_string(
            index=False
        )
    )

Primary predictors: 14
40% sensitivity predictors: 12
Retained: ['ADAPTIVITY', 'AGE', 'ATTLNACT', 'DISCLIMA', 'EDUSHORT', 'ESCS', 'JOYREAD', 'METASPAM', 'METASUM', 'ST004D01T', 'STIMREAD', 'UNDREM']
Dropped: ['BELONG', 'MASTGOAL']


   Feature  Max_missing_pct  Mean_missing_pct  Usable_all_systems
    BELONG            47.11             10.28                True
  MASTGOAL            45.07             10.15                True
  EDUSHORT            37.19              4.47                True
  METASPAM            36.50             12.18                True
   METASUM            34.38             11.01                True
    UNDREM            33.24             11.05                True
  ATTLNACT            32.55              7.51                True
ADAPTIVITY            20.00              5.38                True
  STIMREAD            19.69              5.11                True
   JOYREAD            16.34              4.38                True
      ESCS            14.58              

In [41]:
# ============================================================
# FINAL MISSINGNESS SENSITIVITY
#
# Keep the frozen 14-predictor models unchanged.
#
# Instead of dropping predictors globally, restrict the
# international summary to education systems where EVERY
# frozen predictor has <=40% missingness.
#
# No model fitting.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

MISS_DIR = Path(
    "../results/missingness40_sensitivity"
)

MISS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PRIMARY_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


# ============================================================
# 2. Reconstruct whole-system predictor missingness
# ============================================================

all_systems = sorted(
    set(PRIMARY_BEING_SYSTEMS)
    |
    set(PRIMARY_SUBTYPE_SYSTEMS)
)


missing_rows = []


for country in all_systems:

    df = analysis_frames[
        country
    ]


    for feature in PRIMARY_14:

        values = pd.to_numeric(
            df[feature],
            errors="coerce",
        )


        missing_rows.append(
            {
                "Country": country,
                "Feature": feature,

                "Missing_pct": float(
                    100
                    *
                    values
                    .isna()
                    .mean()
                ),
            }
        )


system_predictor_missingness = pd.DataFrame(
    missing_rows
)


# ============================================================
# 3. Maximum frozen-predictor missingness per system
# ============================================================

system_missingness = (
    system_predictor_missingness
    .groupby(
        "Country"
    )
    .agg(

        Max_predictor_missing_pct=(
            "Missing_pct",
            "max",
        ),

        Mean_predictor_missing_pct=(
            "Missing_pct",
            "mean",
        ),
    )
    .reset_index()
)


LOWMISS40_SYSTEMS = set(
    system_missingness.loc[
        system_missingness[
            "Max_predictor_missing_pct"
        ]
        <= 40,
        "Country",
    ]
)


EXCLUDED40_SYSTEMS = sorted(
    set(all_systems)
    -
    LOWMISS40_SYSTEMS
)


print(
    "Total primary systems:",
    len(all_systems)
)

print(
    "Systems with all 14 predictors <=40% missing:",
    len(LOWMISS40_SYSTEMS)
)

print(
    "Excluded systems:",
    EXCLUDED40_SYSTEMS
)


# ============================================================
# 4. Show exactly why each system was excluded
# ============================================================

excluded_details = (
    system_predictor_missingness[
        system_predictor_missingness[
            "Country"
        ]
        .isin(
            EXCLUDED40_SYSTEMS
        )
    ]
    .sort_values(
        [
            "Country",
            "Missing_pct",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print("\n")
print("=" * 100)
print("SYSTEMS EXCLUDED BY 40% PREDICTOR-MISSINGNESS RULE")
print("=" * 100)


if excluded_details.empty:

    print("None.")

else:

    for country in EXCLUDED40_SYSTEMS:

        print("\n", country)

        temp = (
            excluded_details[
                excluded_details[
                    "Country"
                ]
                == country
            ]
            .head(
                5
            )
        )

        print(
            temp
            .round(2)
            .to_string(
                index=False
            )
        )


# ============================================================
# 5. Load frozen primary SHAP results
# ============================================================

primary_rankings = pd.read_csv(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


# ============================================================
# 6. Compare full vs low-missingness international profiles
# ============================================================

summary_rows = []

restricted_architecture_rows = []


for outcome in OUTCOMES:

    full = (
        primary_rankings[
            primary_rankings[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    restricted = (
        full[
            full[
                "Country"
            ]
            .isin(
                LOWMISS40_SYSTEMS
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Mean feature profile across systems
    # --------------------------------------------------------

    full_profile = (
        full
        .groupby(
            "Feature"
        )
        [
            "ImportanceShare"
        ]
        .mean()
        .reindex(
            PRIMARY_14
        )
    )


    restricted_profile = (
        restricted
        .groupby(
            "Feature"
        )
        [
            "ImportanceShare"
        ]
        .mean()
        .reindex(
            PRIMARY_14
        )
    )


    full_vector = (
        full_profile
        .to_numpy(
            dtype=float
        )
    )


    restricted_vector = (
        restricted_profile
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Cosine similarity
    # --------------------------------------------------------

    cosine = float(
        np.dot(
            full_vector,
            restricted_vector,
        )
        /
        (
            np.linalg.norm(
                full_vector
            )
            *
            np.linalg.norm(
                restricted_vector
            )
        )
    )


    # --------------------------------------------------------
    # Global ranking comparison
    # --------------------------------------------------------

    full_ranks = (
        full_profile
        .rank(
            method="first",
            ascending=False,
        )
    )


    restricted_ranks = (
        restricted_profile
        .rank(
            method="first",
            ascending=False,
        )
    )


    rank_corr = float(
        np.corrcoef(
            full_ranks,
            restricted_ranks,
        )[0, 1]
    )


    mean_abs_change = float(
        np.abs(
            full_vector
            -
            restricted_vector
        )
        .mean()
    )


    full_top1 = (
        full_profile
        .idxmax()
    )


    restricted_top1 = (
        restricted_profile
        .idxmax()
    )


    # --------------------------------------------------------
    # BELONG summaries
    # --------------------------------------------------------

    full_belong = float(
        full_profile[
            "BELONG"
        ]
    )


    restricted_belong = float(
        restricted_profile[
            "BELONG"
        ]
    )


    summary_rows.append(
        {
            "Outcome": outcome,

            "N_full_systems":
                full[
                    "Country"
                ]
                .nunique(),

            "N_lowmissing_systems":
                restricted[
                    "Country"
                ]
                .nunique(),

            "ProfileCosine":
                cosine,

            "ProfileRankCorrelation":
                rank_corr,

            "MeanAbsoluteShareChange":
                mean_abs_change,

            "FullTop1":
                full_top1,

            "RestrictedTop1":
                restricted_top1,

            "BELONG_full_share":
                full_belong,

            "BELONG_restricted_share":
                restricted_belong,

            "BELONG_share_delta":
                restricted_belong
                -
                full_belong,
        }
    )


    # --------------------------------------------------------
    # Save restricted international architecture
    # --------------------------------------------------------

    for feature in PRIMARY_14:

        restricted_architecture_rows.append(
            {
                "Outcome":
                    outcome,

                "Feature":
                    feature,

                "FullMeanShare":
                    float(
                        full_profile[
                            feature
                        ]
                    ),

                "RestrictedMeanShare":
                    float(
                        restricted_profile[
                            feature
                        ]
                    ),

                "ShareDelta":
                    float(
                        restricted_profile[
                            feature
                        ]
                        -
                        full_profile[
                            feature
                        ]
                    ),

                "FullRank":
                    int(
                        full_ranks[
                            feature
                        ]
                    ),

                "RestrictedRank":
                    int(
                        restricted_ranks[
                            feature
                        ]
                    ),
            }
        )


restriction_summary = pd.DataFrame(
    summary_rows
)


restricted_architecture = pd.DataFrame(
    restricted_architecture_rows
)


# ============================================================
# 7. Does predictor missingness correlate with SHAP share?
#
# Descriptive diagnostic only.
# ============================================================

importance_missingness = (
    primary_rankings[
        [
            "Country",
            "Outcome",
            "Feature",
            "ImportanceShare",
        ]
    ]
    .merge(
        system_predictor_missingness,
        on=[
            "Country",
            "Feature",
        ],
        how="left",
        validate="many_to_one",
    )
)


correlation_rows = []


for outcome in OUTCOMES:

    for feature in PRIMARY_14:

        temp = (
            importance_missingness[
                (
                    importance_missingness[
                        "Outcome"
                    ]
                    == outcome
                )
                &
                (
                    importance_missingness[
                        "Feature"
                    ]
                    == feature
                )
            ]
        )


        rho = (
            temp[
                "Missing_pct"
            ]
            .corr(
                temp[
                    "ImportanceShare"
                ],
                method="spearman",
            )
        )


        correlation_rows.append(
            {
                "Outcome":
                    outcome,

                "Feature":
                    feature,

                "Spearman_missingness_vs_share":
                    float(
                        rho
                    ),

                "N_systems":
                    len(
                        temp
                    ),
            }
        )


missingness_share_correlations = pd.DataFrame(
    correlation_rows
)


# ============================================================
# 8. Save
# ============================================================

system_predictor_missingness.to_csv(
    MISS_DIR
    / "system_predictor_missingness_14.csv",
    index=False,
)


system_missingness.to_csv(
    MISS_DIR
    / "system_max_predictor_missingness.csv",
    index=False,
)


restriction_summary.to_csv(
    MISS_DIR
    / "lowmissing_system_restriction_summary.csv",
    index=False,
)


restricted_architecture.to_csv(
    MISS_DIR
    / "lowmissing_international_architecture.csv",
    index=False,
)


missingness_share_correlations.to_csv(
    MISS_DIR
    / "missingness_vs_importance_correlations.csv",
    index=False,
)


# ============================================================
# 9. Print main sensitivity result
# ============================================================

print("\n")
print("=" * 110)
print("FULL vs <=40% PREDICTOR-MISSINGNESS SYSTEMS")
print("=" * 110)


print(
    restriction_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 10. BELONG / MASTGOAL missingness-attribution relationship
# ============================================================

print("\n")
print("=" * 110)
print("BELONG AND MASTGOAL: MISSINGNESS vs IMPORTANCE")
print("=" * 110)


print(
    missingness_share_correlations[
        missingness_share_correlations[
            "Feature"
        ]
        .isin(
            [
                "BELONG",
                "MASTGOAL",
            ]
        )
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Strongest missingness-share correlations
# ============================================================

print("\n")
print("=" * 110)
print("STRONGEST MISSINGNESS-IMPORTANCE CORRELATIONS")
print("=" * 110)


strongest = (
    missingness_share_correlations
    .assign(
        AbsCorrelation=lambda d:
            d[
                "Spearman_missingness_vs_share"
            ]
            .abs()
    )
    .sort_values(
        "AbsCorrelation",
        ascending=False,
    )
    .head(
        15
    )
)


print(
    strongest[
        [
            "Outcome",
            "Feature",
            "Spearman_missingness_vs_share",
            "N_systems",
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)

Total primary systems: 75
Systems with all 14 predictors <=40% missing: 73
Excluded systems: ['DOM', 'PAN']


SYSTEMS EXCLUDED BY 40% PREDICTOR-MISSINGNESS RULE

 DOM
Country  Feature  Missing_pct
    DOM   BELONG        47.11
    DOM MASTGOAL        45.07
    DOM METASPAM        36.50
    DOM  METASUM        34.38
    DOM   UNDREM        33.24

 PAN
Country  Feature  Missing_pct
    PAN   BELONG        43.84
    PAN MASTGOAL        40.65
    PAN METASPAM        33.17
    PAN  METASUM        30.61
    PAN ATTLNACT        28.80


FULL vs <=40% PREDICTOR-MISSINGNESS SYSTEMS
         Outcome  N_full_systems  N_lowmissing_systems  ProfileCosine  ProfileRankCorrelation  MeanAbsoluteShareChange FullTop1 RestrictedTop1  BELONG_full_share  BELONG_restricted_share  BELONG_share_delta
    BEINGBULLIED              74                    72            1.0                  1.0000                   0.0005   BELONG         BELONG             0.2952                   0.2969              0.0018
    BUL

c:\Users\adhiv\OneDrive\Desktop\EduMindsResearch\EduMinds-Research\.venv\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [42]:
# ============================================================
# CONVENTIONAL ADDITIVE-MODEL BENCHMARK
#
# Tests whether XGBoost materially outperforms:
#
#   BEINGBULLIED:
#       weighted multiple linear regression
#
#   Binary bullying outcomes:
#       weighted additive logistic regression
#
# SAME:
#   - 14 predictors
#   - education systems
#   - outcome definitions
#   - W_FSTUWT
#   - school-grouped 5-fold CV
#   - held-out students/schools
#
# PREPROCESSING:
#   - median imputation FIT INSIDE TRAINING FOLD ONLY
#   - standardization FIT INSIDE TRAINING FOLD ONLY
#
# No feature selection.
# No class balancing.
# No hyperparameter tuning.
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

from sklearn.exceptions import ConvergenceWarning


# ============================================================
# 1. Configuration
# ============================================================

BENCH_DIR = Path(
    "../results/conventional_benchmark"
)

BENCH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BENCHMARK_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


BENCHMARK_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_SPLITS = 5

RANDOM_STATE = 42


# ============================================================
# 2. Metric helpers
# ============================================================

def weighted_mean(
    y,
    w,
):

    y = np.asarray(
        y,
        dtype=float,
    )

    w = np.asarray(
        w,
        dtype=float,
    )

    return float(
        np.average(
            y,
            weights=w,
        )
    )


def weighted_rmse(
    y_true,
    y_pred,
    weights,
):

    return float(
        np.sqrt(
            np.average(
                (
                    np.asarray(y_true)
                    -
                    np.asarray(y_pred)
                ) ** 2,
                weights=weights,
            )
        )
    )


def weighted_mae(
    y_true,
    y_pred,
    weights,
):

    return float(
        np.average(
            np.abs(
                np.asarray(y_true)
                -
                np.asarray(y_pred)
            ),
            weights=weights,
        )
    )


def weighted_r2(
    y_true,
    y_pred,
    weights,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )


    y_bar = np.average(
        y_true,
        weights=weights,
    )


    ss_res = np.sum(
        weights
        *
        (
            y_true
            -
            y_pred
        ) ** 2
    )


    ss_tot = np.sum(
        weights
        *
        (
            y_true
            -
            y_bar
        ) ** 2
    )


    if ss_tot <= 0:

        return np.nan


    return float(
        1
        -
        ss_res
        /
        ss_tot
    )


# ============================================================
# 3. Prepare exactly the rows eligible for modeling
#
# Predictor missingness is RETAINED here.
# It will be handled by training-fold median imputation.
#
# Rows are excluded only if:
#   - outcome missing
#   - school ID missing
#   - final survey weight invalid/non-positive
# ============================================================

def prepare_benchmark_data(
    df,
    outcome,
):

    required = (
        BENCHMARK_PREDICTORS
        +
        [
            outcome,
            "CNTSCHID",
            "W_FSTUWT",
        ]
    )


    missing_columns = [
        col
        for col in required
        if col not in df.columns
    ]


    if missing_columns:

        raise RuntimeError(
            f"Missing columns: "
            f"{missing_columns}"
        )


    y = pd.to_numeric(
        df[outcome],
        errors="coerce",
    )


    weights = pd.to_numeric(
        df["W_FSTUWT"],
        errors="coerce",
    )


    groups = df[
        "CNTSCHID"
    ]


    valid = (
        y.notna()
        &
        groups.notna()
        &
        weights.notna()
        &
        np.isfinite(
            weights
        )
        &
        (
            weights > 0
        )
    )


    model_df = (
        df.loc[
            valid
        ]
        .copy()
    )


    X = (
        model_df[
            BENCHMARK_PREDICTORS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )


    y = (
        pd.to_numeric(
            model_df[
                outcome
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )


    groups = (
        model_df[
            "CNTSCHID"
        ]
        .to_numpy()
    )


    weights = (
        pd.to_numeric(
            model_df[
                "W_FSTUWT"
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )


    return (
        X,
        y,
        groups,
        weights,
        model_df.index.to_numpy(),
    )


# ============================================================
# 4. Training weight normalization
#
# Same principle as the XGBoost analysis.
# Constant scaling does not change the solution for these
# unregularized additive models, but keeps the convention
# consistent.
# ============================================================

def normalized_training_weights(
    weights,
):

    weights = np.asarray(
        weights,
        dtype=float,
    )


    return (
        weights
        /
        weights.mean()
    )


# ============================================================
# 5. Logistic regression helper
#
# We want an UNREGULARIZED additive logistic model so that the
# comparison is not dependent on an arbitrary regularization
# strength.
#
# sklearn versions differ slightly in whether they expect:
#   penalty=None
# or
#   penalty="none"
# ============================================================

def fit_unregularized_logistic(
    X_train,
    y_train,
    weights,
):

    last_error = None


    for penalty_value in [
        None,
        "none",
    ]:

        try:

            model = LogisticRegression(
                penalty=penalty_value,
                solver="lbfgs",
                max_iter=5000,
                tol=1e-8,
                random_state=RANDOM_STATE,
            )


            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore",
                    category=ConvergenceWarning,
                )


                model.fit(
                    X_train,
                    y_train,
                    sample_weight=weights,
                )


            return model


        except (
            ValueError,
            TypeError,
        ) as e:

            last_error = e


    raise RuntimeError(
        "Could not fit unregularized "
        f"logistic regression: {last_error}"
    )


# ============================================================
# 6. Run one regression benchmark
# ============================================================

def run_linear_cv(
    df,
    country,
):

    (
        X,
        y,
        groups,
        weights,
        original_indices,
    ) = prepare_benchmark_data(
        df,
        "BEINGBULLIED",
    )


    splitter = GroupKFold(
        n_splits=N_SPLITS
    )


    oof_prediction = np.full(
        len(y),
        np.nan,
        dtype=float,
    )


    fold_rows = []


    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        start=1,
    ):

        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]


        y_train = y[
            train_idx
        ]

        y_test = y[
            test_idx
        ]


        w_train = (
            normalized_training_weights(
                weights[
                    train_idx
                ]
            )
        )


        w_test = weights[
            test_idx
        ]


        # ----------------------------------------------------
        # Median imputation TRAINING FOLD ONLY
        # ----------------------------------------------------

        imputer = SimpleImputer(
            strategy="median"
        )


        X_train_imp = (
            imputer.fit_transform(
                X_train
            )
        )


        X_test_imp = (
            imputer.transform(
                X_test
            )
        )


        # ----------------------------------------------------
        # Scaling TRAINING FOLD ONLY
        # ----------------------------------------------------

        scaler = StandardScaler()


        X_train_scaled = (
            scaler.fit_transform(
                X_train_imp
            )
        )


        X_test_scaled = (
            scaler.transform(
                X_test_imp
            )
        )


        # ----------------------------------------------------
        # Weighted linear regression
        # ----------------------------------------------------

        model = LinearRegression()


        model.fit(
            X_train_scaled,
            y_train,
            sample_weight=w_train,
        )


        prediction = model.predict(
            X_test_scaled
        )


        oof_prediction[
            test_idx
        ] = prediction


        fold_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    "BEINGBULLIED",

                "Fold":
                    fold,

                "N_train":
                    len(train_idx),

                "N_test":
                    len(test_idx),

                "N_train_schools":
                    len(
                        np.unique(
                            groups[
                                train_idx
                            ]
                        )
                    ),

                "N_test_schools":
                    len(
                        np.unique(
                            groups[
                                test_idx
                            ]
                        )
                    ),

                "Linear_R2":
                    weighted_r2(
                        y_test,
                        prediction,
                        w_test,
                    ),

                "Linear_RMSE":
                    weighted_rmse(
                        y_test,
                        prediction,
                        w_test,
                    ),

                "Linear_MAE":
                    weighted_mae(
                        y_test,
                        prediction,
                        w_test,
                    ),
            }
        )


    if np.isnan(
        oof_prediction
    ).any():

        raise RuntimeError(
            f"{country}: missing OOF predictions."
        )


    overall = {
        "Country":
            country,

        "Outcome":
            "BEINGBULLIED",

        "ModelType":
            "Weighted linear regression",

        "N":
            len(y),

        "N_schools":
            len(
                np.unique(
                    groups
                )
            ),

        "Linear_R2":
            weighted_r2(
                y,
                oof_prediction,
                weights,
            ),

        "Linear_RMSE":
            weighted_rmse(
                y,
                oof_prediction,
                weights,
            ),

        "Linear_MAE":
            weighted_mae(
                y,
                oof_prediction,
                weights,
            ),

        "Logistic_ROC_AUC":
            np.nan,

        "Logistic_PR_AUC":
            np.nan,
    }


    return (
        overall,
        fold_rows,
    )


# ============================================================
# 7. Run one classification benchmark
# ============================================================

def run_logistic_cv(
    df,
    country,
    outcome,
):

    (
        X,
        y,
        groups,
        weights,
        original_indices,
    ) = prepare_benchmark_data(
        df,
        outcome,
    )


    y = y.astype(
        int
    )


    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


    oof_probability = np.full(
        len(y),
        np.nan,
        dtype=float,
    )


    fold_rows = []


    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        start=1,
    ):

        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]


        y_train = y[
            train_idx
        ]

        y_test = y[
            test_idx
        ]


        w_train = (
            normalized_training_weights(
                weights[
                    train_idx
                ]
            )
        )


        w_test = weights[
            test_idx
        ]


        # ----------------------------------------------------
        # Median imputation inside training fold
        # ----------------------------------------------------

        imputer = SimpleImputer(
            strategy="median"
        )


        X_train_imp = (
            imputer.fit_transform(
                X_train
            )
        )


        X_test_imp = (
            imputer.transform(
                X_test
            )
        )


        # ----------------------------------------------------
        # Scaling inside training fold
        # ----------------------------------------------------

        scaler = StandardScaler()


        X_train_scaled = (
            scaler.fit_transform(
                X_train_imp
            )
        )


        X_test_scaled = (
            scaler.transform(
                X_test_imp
            )
        )


        # ----------------------------------------------------
        # Unregularized weighted logistic regression
        # ----------------------------------------------------

        model = (
            fit_unregularized_logistic(
                X_train_scaled,
                y_train,
                w_train,
            )
        )


        probability = (
            model.predict_proba(
                X_test_scaled
            )[:, 1]
        )


        oof_probability[
            test_idx
        ] = probability


        fold_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    outcome,

                "Fold":
                    fold,

                "N_train":
                    len(train_idx),

                "N_test":
                    len(test_idx),

                "N_train_positive":
                    int(
                        y_train.sum()
                    ),

                "N_test_positive":
                    int(
                        y_test.sum()
                    ),

                "N_train_schools":
                    len(
                        np.unique(
                            groups[
                                train_idx
                            ]
                        )
                    ),

                "N_test_schools":
                    len(
                        np.unique(
                            groups[
                                test_idx
                            ]
                        )
                    ),

                "Logistic_ROC_AUC":
                    float(
                        roc_auc_score(
                            y_test,
                            probability,
                            sample_weight=w_test,
                        )
                    ),

                "Logistic_PR_AUC":
                    float(
                        average_precision_score(
                            y_test,
                            probability,
                            sample_weight=w_test,
                        )
                    ),
            }
        )


    if np.isnan(
        oof_probability
    ).any():

        raise RuntimeError(
            f"{country} — {outcome}: "
            "missing OOF predictions."
        )


    overall = {
        "Country":
            country,

        "Outcome":
            outcome,

        "ModelType":
            "Weighted additive logistic regression",

        "N":
            len(y),

        "N_schools":
            len(
                np.unique(
                    groups
                )
            ),

        "Linear_R2":
            np.nan,

        "Linear_RMSE":
            np.nan,

        "Linear_MAE":
            np.nan,

        "Logistic_ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    oof_probability,
                    sample_weight=weights,
                )
            ),

        "Logistic_PR_AUC":
            float(
                average_precision_score(
                    y,
                    oof_probability,
                    sample_weight=weights,
                )
            ),
    }


    return (
        overall,
        fold_rows,
    )


# ============================================================
# 8. Run all 374 conventional benchmark models
# ============================================================

overall_rows = []

fold_rows = []

error_rows = []


EXPECTED_MODELS = (
    len(
        PRIMARY_BEING_SYSTEMS
    )
    +
    (
        4
        *
        len(
            PRIMARY_SUBTYPE_SYSTEMS
        )
    )
)


counter = 0

start_time = time.time()


# ------------------------------------------------------------
# BEINGBULLIED
# ------------------------------------------------------------

for country in PRIMARY_BEING_SYSTEMS:

    counter += 1


    print(
        f"[{counter}/{EXPECTED_MODELS}] "
        f"{country} — BEINGBULLIED",
        end=" ... ",
    )


    try:

        overall, folds = (
            run_linear_cv(
                analysis_frames[
                    country
                ],
                country,
            )
        )


        overall_rows.append(
            overall
        )


        fold_rows.extend(
            folds
        )


        print(
            f"OK | "
            f"R2={overall['Linear_R2']:.4f}"
        )


    except Exception as e:

        error_rows.append(
            {
                "Country":
                    country,

                "Outcome":
                    "BEINGBULLIED",

                "ErrorType":
                    type(e).__name__,

                "Error":
                    str(e),
            }
        )


        print(
            f"ERROR | {e}"
        )


# ------------------------------------------------------------
# Binary subtypes
# ------------------------------------------------------------

for country in PRIMARY_SUBTYPE_SYSTEMS:

    for outcome in BINARY_OUTCOMES:

        counter += 1


        print(
            f"[{counter}/{EXPECTED_MODELS}] "
            f"{country} — {outcome}",
            end=" ... ",
        )


        try:

            overall, folds = (
                run_logistic_cv(
                    analysis_frames[
                        country
                    ],
                    country,
                    outcome,
                )
            )


            overall_rows.append(
                overall
            )


            fold_rows.extend(
                folds
            )


            print(
                f"OK | "
                f"ROC={overall['Logistic_ROC_AUC']:.4f} | "
                f"PR={overall['Logistic_PR_AUC']:.4f}"
            )


        except Exception as e:

            error_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "ErrorType":
                        type(e).__name__,

                    "Error":
                        str(e),
                }
            )


            print(
                f"ERROR | {e}"
            )


# ============================================================
# 9. Save conventional benchmark
# ============================================================

benchmark_overall = pd.DataFrame(
    overall_rows
)


benchmark_folds = pd.DataFrame(
    fold_rows
)


benchmark_errors = pd.DataFrame(
    error_rows
)


benchmark_overall.to_csv(
    BENCH_DIR
    / "benchmark_overall_performance.csv",
    index=False,
)


benchmark_folds.to_csv(
    BENCH_DIR
    / "benchmark_fold_performance.csv",
    index=False,
)


benchmark_errors.to_csv(
    BENCH_DIR
    / "benchmark_errors.csv",
    index=False,
)


# ============================================================
# 10. Completion QA
# ============================================================

runtime_minutes = (
    time.time()
    -
    start_time
) / 60


print("\n")
print("=" * 110)
print("CONVENTIONAL BENCHMARK — COMPLETION")
print("=" * 110)


print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Successful models:",
    len(
        benchmark_overall
    )
)

print(
    "Errors:",
    len(
        benchmark_errors
    )
)

print(
    "Expected folds:",
    EXPECTED_MODELS
    *
    N_SPLITS
)

print(
    "Observed folds:",
    len(
        benchmark_folds
    )
)

print(
    f"Runtime: "
    f"{runtime_minutes:.2f} minutes"
)


# ============================================================
# 11. Load frozen XGBoost performance
# ============================================================

xgb_perf = pd.read_csv(
    "../results/primary_models/"
    "overall_performance.csv"
)


# ============================================================
# 12. Merge XGBoost vs additive benchmark
# ============================================================

comparison = (
    benchmark_overall
    .merge(
        xgb_perf[
            [
                "Country",
                "Outcome",
                "Model_R2",
                "Model_RMSE",
                "Model_MAE",
                "Model_ROC_AUC",
                "Model_PR_AUC",
            ]
        ],
        on=[
            "Country",
            "Outcome",
        ],
        how="inner",
        validate="one_to_one",
    )
)


# Regression

comparison[
    "Delta_XGB_minus_Linear_R2"
] = (
    comparison[
        "Model_R2"
    ]
    -
    comparison[
        "Linear_R2"
    ]
)


comparison[
    "Delta_XGB_minus_Linear_RMSE"
] = (
    comparison[
        "Model_RMSE"
    ]
    -
    comparison[
        "Linear_RMSE"
    ]
)


# Classification

comparison[
    "Delta_XGB_minus_Logistic_ROC"
] = (
    comparison[
        "Model_ROC_AUC"
    ]
    -
    comparison[
        "Logistic_ROC_AUC"
    ]
)


comparison[
    "Delta_XGB_minus_Logistic_PR"
] = (
    comparison[
        "Model_PR_AUC"
    ]
    -
    comparison[
        "Logistic_PR_AUC"
    ]
)


comparison.to_csv(
    BENCH_DIR
    / "xgboost_vs_conventional.csv",
    index=False,
)


# ============================================================
# 13. Summary
# ============================================================

summary_rows = []


# ------------------------------------------------------------
# Regression
# ------------------------------------------------------------

reg = (
    comparison[
        comparison[
            "Outcome"
        ]
        == "BEINGBULLIED"
    ]
)


delta = (
    reg[
        "Delta_XGB_minus_Linear_R2"
    ]
    .dropna()
)


summary_rows.append(
    {
        "Outcome":
            "BEINGBULLIED",

        "Metric":
            "R2",

        "N_systems":
            len(delta),

        "Conventional_mean":
            reg[
                "Linear_R2"
            ].mean(),

        "XGBoost_mean":
            reg[
                "Model_R2"
            ].mean(),

        "Mean_XGB_minus_conventional":
            delta.mean(),

        "Median_XGB_minus_conventional":
            delta.median(),

        "Min_difference":
            delta.min(),

        "Max_difference":
            delta.max(),

        "Pct_XGBoost_better":
            100
            *
            (
                delta > 0
            ).mean(),
    }
)


# ------------------------------------------------------------
# Classification
# ------------------------------------------------------------

for outcome in BINARY_OUTCOMES:

    temp = (
        comparison[
            comparison[
                "Outcome"
            ]
            == outcome
        ]
    )


    for (
        metric_name,
        conventional_column,
        xgb_column,
        delta_column,
    ) in [
        (
            "ROC_AUC",
            "Logistic_ROC_AUC",
            "Model_ROC_AUC",
            "Delta_XGB_minus_Logistic_ROC",
        ),
        (
            "PR_AUC",
            "Logistic_PR_AUC",
            "Model_PR_AUC",
            "Delta_XGB_minus_Logistic_PR",
        ),
    ]:

        delta = (
            temp[
                delta_column
            ]
            .dropna()
        )


        summary_rows.append(
            {
                "Outcome":
                    outcome,

                "Metric":
                    metric_name,

                "N_systems":
                    len(delta),

                "Conventional_mean":
                    temp[
                        conventional_column
                    ].mean(),

                "XGBoost_mean":
                    temp[
                        xgb_column
                    ].mean(),

                "Mean_XGB_minus_conventional":
                    delta.mean(),

                "Median_XGB_minus_conventional":
                    delta.median(),

                "Min_difference":
                    delta.min(),

                "Max_difference":
                    delta.max(),

                "Pct_XGBoost_better":
                    100
                    *
                    (
                        delta > 0
                    ).mean(),
            }
        )


benchmark_summary = pd.DataFrame(
    summary_rows
)


benchmark_summary.to_csv(
    BENCH_DIR
    / "xgboost_vs_conventional_summary.csv",
    index=False,
)


# ============================================================
# 14. Systems where conventional model wins most
# ============================================================

exceptions = []


for outcome in BENCHMARK_OUTCOMES:

    temp = (
        comparison[
            comparison[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    if outcome == "BEINGBULLIED":

        temp[
            "Difference"
        ] = (
            temp[
                "Delta_XGB_minus_Linear_R2"
            ]
        )


    else:

        temp[
            "Difference"
        ] = (
            temp[
                "Delta_XGB_minus_Logistic_ROC"
            ]
        )


    temp = (
        temp
        .sort_values(
            "Difference"
        )
        .head(
            5
        )
    )


    for _, row in temp.iterrows():

        exceptions.append(
            {
                "Outcome":
                    outcome,

                "Country":
                    row[
                        "Country"
                    ],

                "Difference":
                    row[
                        "Difference"
                    ],
            }
        )


benchmark_exceptions = pd.DataFrame(
    exceptions
)


# ============================================================
# 15. Print main results
# ============================================================

print("\n")
print("=" * 110)
print("XGBOOST vs CONVENTIONAL ADDITIVE MODELS")
print("=" * 110)


print(
    benchmark_summary
    .round(4)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 110)
print("SYSTEMS WITH SMALLEST XGBOOST ADVANTAGE")
print("=" * 110)


for outcome in BENCHMARK_OUTCOMES:

    print("\n")
    print("-" * 90)
    print(outcome)
    print("-" * 90)


    temp = (
        benchmark_exceptions[
            benchmark_exceptions[
                "Outcome"
            ]
            == outcome
        ]
    )


    print(
        temp
        .round(4)
        .to_string(
            index=False
        )
    )

[1/374] ALB — BEINGBULLIED ... OK | R2=0.1125
[2/374] ARE — BEINGBULLIED ... OK | R2=0.1757
[3/374] ARG — BEINGBULLIED ... OK | R2=0.1253
[4/374] AUS — BEINGBULLIED ... OK | R2=0.1686
[5/374] AUT — BEINGBULLIED ... OK | R2=0.1084
[6/374] BEL — BEINGBULLIED ... OK | R2=0.1268
[7/374] BGR — BEINGBULLIED ... OK | R2=0.0952
[8/374] BIH — BEINGBULLIED ... OK | R2=0.0884
[9/374] BLR — BEINGBULLIED ... OK | R2=0.1234
[10/374] BRA — BEINGBULLIED ... OK | R2=0.0946
[11/374] BRN — BEINGBULLIED ... OK | R2=0.1543
[12/374] CAN — BEINGBULLIED ... OK | R2=0.1220
[13/374] CHE — BEINGBULLIED ... OK | R2=0.1133
[14/374] CHL — BEINGBULLIED ... OK | R2=0.0883
[15/374] COL — BEINGBULLIED ... OK | R2=0.0915
[16/374] CRI — BEINGBULLIED ... OK | R2=0.1015
[17/374] CZE — BEINGBULLIED ... OK | R2=0.1196
[18/374] DEU — BEINGBULLIED ... OK | R2=0.1479
[19/374] DNK — BEINGBULLIED ... OK | R2=0.1237
[20/374] DOM — BEINGBULLIED ... OK | R2=0.1175
[21/374] ESP — BEINGBULLIED ... OK | R2=0.1088
[22/374] EST — BEINGBU

In [43]:
# ============================================================
# PERMUTATION-NULL TEST OF INTERNATIONAL ARCHITECTURE
#
# Central question:
#
# Are education systems more aligned on WHICH predictors
# matter than would be expected simply from having similarly
# concentrated non-negative SHAP profiles?
#
# Null model:
#   For each system independently:
#       preserve its exact set of SHAP importance shares
#       but randomly permute those shares across predictor labels.
#
# Therefore the null preserves:
#   - number of predictors
#   - each system's exact importance distribution
#   - profile concentration / sparsity
#   - each system's Euclidean norm
#
# But destroys:
#   - agreement across systems about predictor identity
#
# Tests:
#   1. Full 14-feature architecture
#   2. BELONG removed, remaining 13 features renormalized
#
# No model fitting.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Configuration
# ============================================================

NULL_DIR = Path(
    "../results/architecture_null"
)

NULL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


FEATURES_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FEATURES_13 = [
    feature
    for feature in FEATURES_14
    if feature != "BELONG"
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_PERMUTATIONS = 10_000

RANDOM_SEED = 42

BATCH_SIZE = 500


rng = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 2. Load frozen attribution results
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


# ============================================================
# 3. Utility:
# mean pairwise cosine WITHOUT constructing every pair
#
# If rows u_i are unit vectors:
#
# sum_{i != j} u_i dot u_j
# =
# ||sum_i u_i||^2 - n
#
# Therefore:
#
# mean over unordered pairs
# =
# (||sum_i u_i||^2 - n)
# /
# [n(n-1)]
#
# This is exactly equal to averaging all pairwise cosines.
# ============================================================

def mean_pairwise_cosine(
    matrix,
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )


    if (
        norms <= 0
    ).any():

        raise RuntimeError(
            "Zero-length attribution profile found."
        )


    unit = (
        matrix
        /
        norms
    )


    n = unit.shape[0]


    sum_vector = (
        unit.sum(
            axis=0
        )
    )


    mean_cosine = (
        (
            np.dot(
                sum_vector,
                sum_vector,
            )
            -
            n
        )
        /
        (
            n
            *
            (
                n - 1
            )
        )
    )


    return float(
        mean_cosine
    )


# ============================================================
# 4. Build one outcome matrix
# ============================================================

def build_profile_matrix(
    outcome,
    features,
    remove_belong=False,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            == outcome
        ]
        .copy()
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    rows = []


    for country in systems:

        system_data = (
            temp[
                temp[
                    "Country"
                ]
                == country
            ]
            .set_index(
                "Feature"
            )
        )


        vector = (
            system_data
            .loc[
                features,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        if remove_belong:

            total = (
                vector.sum()
            )


            if total <= 0:

                raise RuntimeError(
                    f"{country} — {outcome}: "
                    "non-positive remaining attribution."
                )


            vector = (
                vector
                /
                total
            )


        rows.append(
            vector
        )


    matrix = np.vstack(
        rows
    )


    return (
        systems,
        matrix,
    )


# ============================================================
# 5. Permutation generator
#
# Efficient batched implementation.
#
# For every permutation:
#   independently scramble feature positions within each system.
#
# Because permutations preserve each row's Euclidean norm,
# we can unit-normalize ONCE before permutation.
# ============================================================

def generate_null_distribution(
    matrix,
    n_permutations=N_PERMUTATIONS,
    batch_size=BATCH_SIZE,
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    n_systems, n_features = (
        matrix.shape
    )


    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )


    unit = (
        matrix
        /
        norms
    )


    null_values = np.empty(
        n_permutations,
        dtype=float,
    )


    cursor = 0


    while cursor < n_permutations:

        current_batch = min(
            batch_size,
            n_permutations
            -
            cursor,
        )


        # ----------------------------------------------------
        # Generate independent random permutations for
        # every [permutation x system] row.
        #
        # argsort of random uniforms gives permutation indices.
        # ----------------------------------------------------

        random_scores = rng.random(
            (
                current_batch,
                n_systems,
                n_features,
            )
        )


        permutation_indices = (
            np.argsort(
                random_scores,
                axis=2,
            )
        )


        # ----------------------------------------------------
        # Apply those permutations to the already
        # unit-normalized system profiles.
        # ----------------------------------------------------

        permuted = np.take_along_axis(
            np.broadcast_to(
                unit,
                (
                    current_batch,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_indices,
            axis=2,
        )


        # ----------------------------------------------------
        # Sum system vectors for each permutation.
        # ----------------------------------------------------

        summed = (
            permuted.sum(
                axis=1
            )
        )


        squared_norm = (
            np.square(
                summed
            )
            .sum(
                axis=1
            )
        )


        # ----------------------------------------------------
        # Exact mean pairwise cosine
        # ----------------------------------------------------

        batch_cosines = (
            (
                squared_norm
                -
                n_systems
            )
            /
            (
                n_systems
                *
                (
                    n_systems
                    -
                    1
                )
            )
        )


        null_values[
            cursor:
            cursor
            +
            current_batch
        ] = batch_cosines


        cursor += (
            current_batch
        )


    return null_values


# ============================================================
# 6. Run tests
# ============================================================

summary_rows = []

null_rows = []


for outcome in OUTCOMES:

    print("\n")
    print(
        "=" * 100
    )

    print(
        outcome
    )

    print(
        "=" * 100
    )


    for analysis_name, features, remove_belong in [

        (
            "Full14",
            FEATURES_14,
            False,
        ),

        (
            "NoBELONG13",
            FEATURES_13,
            True,
        ),

    ]:

        (
            systems,
            matrix,
        ) = build_profile_matrix(
            outcome=outcome,
            features=features,
            remove_belong=remove_belong,
        )


        observed = (
            mean_pairwise_cosine(
                matrix
            )
        )


        null_distribution = (
            generate_null_distribution(
                matrix
            )
        )


        null_mean = float(
            null_distribution.mean()
        )


        null_sd = float(
            null_distribution.std(
                ddof=1
            )
        )


        null_q025 = float(
            np.quantile(
                null_distribution,
                0.025,
            )
        )


        null_q975 = float(
            np.quantile(
                null_distribution,
                0.975,
            )
        )


        null_q999 = float(
            np.quantile(
                null_distribution,
                0.999,
            )
        )


        # ----------------------------------------------------
        # One-sided permutation p-value
        #
        # +1 correction prevents reporting p=0.
        # ----------------------------------------------------

        p_value = float(
            (
                1
                +
                (
                    null_distribution
                    >=
                    observed
                ).sum()
            )
            /
            (
                N_PERMUTATIONS
                +
                1
            )
        )


        if null_sd > 0:

            standardized_excess = float(
                (
                    observed
                    -
                    null_mean
                )
                /
                null_sd
            )

        else:

            standardized_excess = (
                np.nan
            )


        absolute_excess = float(
            observed
            -
            null_mean
        )


        relative_excess = float(
            observed
            /
            null_mean
        )


        summary_rows.append(
            {
                "Outcome":
                    outcome,

                "Analysis":
                    analysis_name,

                "N_systems":
                    len(
                        systems
                    ),

                "N_features":
                    len(
                        features
                    ),

                "ObservedMeanCosine":
                    observed,

                "NullMeanCosine":
                    null_mean,

                "NullSD":
                    null_sd,

                "NullQ025":
                    null_q025,

                "NullQ975":
                    null_q975,

                "NullQ999":
                    null_q999,

                "AbsoluteExcess":
                    absolute_excess,

                "ObservedDividedByNull":
                    relative_excess,

                "StandardizedExcess":
                    standardized_excess,

                "PermutationP":
                    p_value,
            }
        )


        for permutation_id, value in enumerate(
            null_distribution,
            start=1,
        ):

            null_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Analysis":
                        analysis_name,

                    "Permutation":
                        permutation_id,

                    "MeanCosine":
                        value,
                }
            )


        print(
            f"{analysis_name}: "
            f"observed={observed:.4f} | "
            f"null={null_mean:.4f} "
            f"(95% "
            f"{null_q025:.4f}–"
            f"{null_q975:.4f}) | "
            f"excess={absolute_excess:.4f} | "
            f"z-like={standardized_excess:.1f} | "
            f"p={p_value:.6f}"
        )


# ============================================================
# 7. Combine and save
# ============================================================

null_summary = pd.DataFrame(
    summary_rows
)


null_distribution_long = pd.DataFrame(
    null_rows
)


null_summary.to_csv(
    NULL_DIR
    / "architecture_permutation_null_summary.csv",
    index=False,
)


null_distribution_long.to_csv(
    NULL_DIR
    / "architecture_permutation_null_distribution.csv",
    index=False,
)


# ============================================================
# 8. QA against previously observed cosine results
# ============================================================

print("\n")
print("=" * 110)
print("PERMUTATION NULL — MAIN RESULTS")
print("=" * 110)


print(
    null_summary[
        [
            "Outcome",
            "Analysis",
            "N_systems",
            "N_features",
            "ObservedMeanCosine",
            "NullMeanCosine",
            "NullQ025",
            "NullQ975",
            "NullQ999",
            "AbsoluteExcess",
            "StandardizedExcess",
            "PermutationP",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 9. Compact comparison by outcome
# ============================================================

compact = (
    null_summary
    .pivot(
        index="Outcome",
        columns="Analysis",
        values=[
            "ObservedMeanCosine",
            "NullMeanCosine",
            "AbsoluteExcess",
            "PermutationP",
        ],
    )
)


compact.columns = [
    f"{metric}_{analysis}"
    for metric, analysis
    in compact.columns
]


compact = (
    compact
    .reset_index()
)


compact.to_csv(
    NULL_DIR
    / "architecture_null_compact.csv",
    index=False,
)


print("\n")
print("=" * 110)
print("FULL vs BELONG-REMOVED NULL COMPARISON")
print("=" * 110)


print(
    compact
    .round(
        6
    )
    .to_string(
        index=False
    )
)



BEINGBULLIED
Full14: observed=0.9546 | null=0.5117 (95% 0.5077–0.5176) | excess=0.4429 | z-like=172.6 | p=0.000100
NoBELONG13: observed=0.9286 | null=0.7377 (95% 0.7355–0.7411) | excess=0.1909 | z-like=131.8 | p=0.000100


BULLY_VERBAL
Full14: observed=0.9401 | null=0.6207 (95% 0.6176–0.6253) | excess=0.3194 | z-like=160.6 | p=0.000100
NoBELONG13: observed=0.9189 | null=0.8086 (95% 0.8070–0.8110) | excess=0.1103 | z-like=107.0 | p=0.000100


BULLY_THREAT
Full14: observed=0.9357 | null=0.7066 (95% 0.7042–0.7102) | excess=0.2291 | z-like=149.9 | p=0.000100
NoBELONG13: observed=0.9164 | null=0.8450 (95% 0.8437–0.8469) | excess=0.0714 | z-like=84.5 | p=0.000100


BULLY_PHYSICAL
Full14: observed=0.9437 | null=0.7060 (95% 0.7036–0.7096) | excess=0.2377 | z-like=153.8 | p=0.000100
NoBELONG13: observed=0.9275 | null=0.8036 (95% 0.8019–0.8060) | excess=0.1240 | z-like=116.9 | p=0.000100


BULLY_RELATIONAL
Full14: observed=0.9473 | null=0.6358 (95% 0.6328–0.6402) | excess=0.3115 | z-like=162.0

In [44]:
# ============================================================
# FOCUSED DIRECTIONAL OOF SHAP ANALYSIS
#
# Scientific question:
#
# For the dominant predictors, is the DIRECTION of their
# predictive association consistent across education systems?
#
# Focal predictors:
#   BELONG
#   DISCLIMA
#   ST004D01T
#
# Continuous predictors:
#   Compare weighted mean signed SHAP in the highest vs lowest
#   weighted quartile of observed predictor values.
#
# Gender:
#   ST004D01T = 0 female
#   ST004D01T = 1 male
#   Compare weighted mean signed SHAP:
#       male - female
#
# Interpretation:
#
#   Positive SHAP:
#       pushes the fitted prediction toward MORE bullying
#
#   Negative SHAP:
#       pushes the fitted prediction toward LESS bullying
#
# For binary XGBoost models, SHAP is on the raw-margin
# (log-odds) scale.
#
# For BEINGBULLIED, SHAP is on the continuous outcome scale.
#
# IMPORTANT:
#   This is predictive/model-based association.
#   It is NOT causal.
#
# Same:
#   - 14 predictors
#   - systems
#   - outcomes
#   - 5 school-grouped folds
#   - W_FSTUWT
#   - reference XGBoost specification
#   - exact held-out TreeSHAP
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)


# ============================================================
# 1. Configuration
# ============================================================

DIR_DIR = Path(
    "../results/directional_analysis"
)

DIR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


DIRECTION_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FOCAL_CONTINUOUS = [
    "BELONG",
    "DISCLIMA",
]


FOCAL_BINARY = [
    "ST004D01T",
]


FOCAL_FEATURES = (
    FOCAL_CONTINUOUS
    +
    FOCAL_BINARY
)


DIRECTION_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


DIRECTION_BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_SPLITS = 5

RANDOM_STATE = 42


# ============================================================
# 2. Weighted helpers
# ============================================================

def direction_weighted_mean(
    values,
    weights,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )


    mask = (
        np.isfinite(values)
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )


    if mask.sum() == 0:

        return np.nan


    return float(
        np.average(
            values[mask],
            weights=weights[mask],
        )
    )


def direction_weighted_quantile(
    values,
    weights,
    quantile,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )


    mask = (
        np.isfinite(values)
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )


    values = values[
        mask
    ]

    weights = weights[
        mask
    ]


    if len(values) == 0:

        return np.nan


    order = np.argsort(
        values
    )


    values = values[
        order
    ]

    weights = weights[
        order
    ]


    cumulative = (
        np.cumsum(
            weights
        )
        -
        0.5
        *
        weights
    )


    cumulative = (
        cumulative
        /
        weights.sum()
    )


    return float(
        np.interp(
            quantile,
            cumulative,
            values,
        )
    )


def normalize_direction_weights(
    weights,
):

    weights = np.asarray(
        weights,
        dtype=float,
    )


    return (
        weights
        /
        weights.mean()
    )


# ============================================================
# 3. Prepare modeling rows
# ============================================================

def prepare_direction_data(
    df,
    outcome,
):

    required = (
        DIRECTION_PREDICTORS
        +
        [
            outcome,
            "CNTSCHID",
            "W_FSTUWT",
        ]
    )


    missing = [
        column
        for column in required
        if column not in df.columns
    ]


    if missing:

        raise RuntimeError(
            f"Missing required columns: "
            f"{missing}"
        )


    y = pd.to_numeric(
        df[outcome],
        errors="coerce",
    )


    weights = pd.to_numeric(
        df["W_FSTUWT"],
        errors="coerce",
    )


    group = df[
        "CNTSCHID"
    ]


    valid = (
        y.notna()
        &
        group.notna()
        &
        weights.notna()
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )


    model_df = (
        df.loc[
            valid
        ]
        .copy()
    )


    X = (
        model_df[
            DIRECTION_PREDICTORS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )


    y = (
        pd.to_numeric(
            model_df[outcome],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )


    groups = (
        model_df[
            "CNTSCHID"
        ]
        .to_numpy()
    )


    weights = (
        pd.to_numeric(
            model_df["W_FSTUWT"],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )


    return (
        X,
        y,
        groups,
        weights,
    )


# ============================================================
# 4. Fit one country/outcome and return held-out SHAP only
# ============================================================

def run_direction_oof_shap(
    df,
    country,
    outcome,
):

    (
        X,
        y,
        groups,
        weights,
    ) = prepare_direction_data(
        df,
        outcome,
    )


    n = len(
        y
    )


    # --------------------------------------------------------
    # Exact same fold logic as primary analysis
    # --------------------------------------------------------

    if outcome == "BEINGBULLIED":

        splitter = GroupKFold(
            n_splits=N_SPLITS
        )


        split_iterator = (
            splitter.split(
                X,
                y,
                groups,
            )
        )


    else:

        y_for_split = (
            y.astype(int)
        )


        splitter = StratifiedGroupKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE,
        )


        split_iterator = (
            splitter.split(
                X,
                y_for_split,
                groups,
            )
        )


    # --------------------------------------------------------
    # OOF storage
    # --------------------------------------------------------

    shap_oof = {
        feature:
            np.full(
                n,
                np.nan,
                dtype=float,
            )
        for feature in FOCAL_FEATURES
    }


    fold_oof = np.full(
        n,
        -1,
        dtype=int,
    )


    max_additivity_error = 0.0


    feature_index = {
        feature:
            DIRECTION_PREDICTORS.index(
                feature
            )
        for feature in FOCAL_FEATURES
    }


    # --------------------------------------------------------
    # Five held-out-school folds
    # --------------------------------------------------------

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        split_iterator,
        start=1,
    ):

        X_train = X.iloc[
            train_idx
        ]


        X_test = X.iloc[
            test_idx
        ]


        y_train = y[
            train_idx
        ]


        w_train = (
            normalize_direction_weights(
                weights[
                    train_idx
                ]
            )
        )


        # ----------------------------------------------------
        # Exact reference XGBoost constructors already frozen
        # ----------------------------------------------------

        if outcome == "BEINGBULLIED":

            model = (
                make_xgb_regressor()
            )


        else:

            model = (
                make_xgb_classifier()
            )


        model.fit(
            X_train,
            y_train,
            sample_weight=w_train,
        )


        # ----------------------------------------------------
        # Exact TreeSHAP on HELD-OUT students only
        # ----------------------------------------------------

        dtest = xgb.DMatrix(
            X_test,
            feature_names=DIRECTION_PREDICTORS,
        )


        booster = (
            model.get_booster()
        )


        contributions = (
            booster.predict(
                dtest,
                pred_contribs=True,
                approx_contribs=False,
            )
        )


        raw_margin = (
            booster.predict(
                dtest,
                output_margin=True,
            )
        )


        reconstructed_margin = (
            contributions[
                :,
                :-1
            ]
            .sum(
                axis=1
            )
            +
            contributions[
                :,
                -1
            ]
        )


        fold_error = float(
            np.max(
                np.abs(
                    reconstructed_margin
                    -
                    raw_margin
                )
            )
        )


        max_additivity_error = max(
            max_additivity_error,
            fold_error,
        )


        # ----------------------------------------------------
        # Store only focal predictor SHAP values
        # ----------------------------------------------------

        for feature in FOCAL_FEATURES:

            shap_oof[
                feature
            ][
                test_idx
            ] = (
                contributions[
                    :,
                    feature_index[
                        feature
                    ]
                ]
            )


        fold_oof[
            test_idx
        ] = fold


    # --------------------------------------------------------
    # QA
    # --------------------------------------------------------

    if (
        fold_oof
        <
        1
    ).any():

        raise RuntimeError(
            f"{country} — {outcome}: "
            "some observations never entered a test fold."
        )


    for feature in FOCAL_FEATURES:

        if np.isnan(
            shap_oof[
                feature
            ]
        ).any():

            raise RuntimeError(
                f"{country} — {outcome}: "
                f"missing OOF SHAP for {feature}."
            )


    return {
        "X":
            X,

        "y":
            y,

        "weights":
            weights,

        "fold":
            fold_oof,

        "shap":
            shap_oof,

        "max_additivity_error":
            max_additivity_error,
    }


# ============================================================
# 5. Summarize one continuous predictor
#
# Directional contrast:
#
# weighted mean SHAP among highest quartile
# MINUS
# weighted mean SHAP among lowest quartile
#
# Negative:
# higher values push predictions toward LOWER bullying
#
# Positive:
# higher values push predictions toward HIGHER bullying
# ============================================================

def summarize_continuous_direction(
    result,
    country,
    outcome,
    feature,
):

    x = (
        result[
            "X"
        ][
            feature
        ]
        .to_numpy(
            dtype=float
        )
    )


    shap = (
        result[
            "shap"
        ][
            feature
        ]
    )


    weights = (
        result[
            "weights"
        ]
    )


    observed = (
        np.isfinite(x)
        &
        np.isfinite(shap)
        &
        np.isfinite(weights)
        &
        (weights > 0)
    )


    x_obs = x[
        observed
    ]

    shap_obs = shap[
        observed
    ]

    w_obs = weights[
        observed
    ]


    q25 = (
        direction_weighted_quantile(
            x_obs,
            w_obs,
            0.25,
        )
    )


    q75 = (
        direction_weighted_quantile(
            x_obs,
            w_obs,
            0.75,
        )
    )


    low = (
        x_obs
        <=
        q25
    )


    high = (
        x_obs
        >=
        q75
    )


    low_mean_shap = (
        direction_weighted_mean(
            shap_obs[
                low
            ],
            w_obs[
                low
            ],
        )
    )


    high_mean_shap = (
        direction_weighted_mean(
            shap_obs[
                high
            ],
            w_obs[
                high
            ],
        )
    )


    contrast = float(
        high_mean_shap
        -
        low_mean_shap
    )


    mean_abs_shap = (
        direction_weighted_mean(
            np.abs(
                shap_obs
            ),
            w_obs,
        )
    )


    if (
        np.isfinite(
            mean_abs_shap
        )
        and
        mean_abs_shap > 0
    ):

        normalized_contrast = float(
            contrast
            /
            mean_abs_shap
        )


    else:

        normalized_contrast = (
            np.nan
        )


    return {
        "Country":
            country,

        "Outcome":
            outcome,

        "Feature":
            feature,

        "ContrastDefinition":
            "Upper weighted quartile minus lower weighted quartile",

        "N_observed":
            int(
                observed.sum()
            ),

        "Q25":
            q25,

        "Q75":
            q75,

        "MeanSHAP_Low":
            low_mean_shap,

        "MeanSHAP_High":
            high_mean_shap,

        "DirectionalContrast":
            contrast,

        "MeanAbsSHAP":
            mean_abs_shap,

        "NormalizedContrast":
            normalized_contrast,

        "Direction":
            (
                "Higher -> higher predicted bullying"
                if contrast > 0
                else
                "Higher -> lower predicted bullying"
                if contrast < 0
                else
                "No contrast"
            ),
    }


# ============================================================
# 6. Summarize gender
#
# ST004D01T:
#   0 = female
#   1 = male
#
# DirectionalContrast =
#   weighted mean male SHAP
#   -
#   weighted mean female SHAP
#
# Positive:
# male value contributes more toward predicted bullying
# ============================================================

def summarize_gender_direction(
    result,
    country,
    outcome,
):

    feature = (
        "ST004D01T"
    )


    x = (
        result[
            "X"
        ][
            feature
        ]
        .to_numpy(
            dtype=float
        )
    )


    shap = (
        result[
            "shap"
        ][
            feature
        ]
    )


    weights = (
        result[
            "weights"
        ]
    )


    valid = (
        np.isfinite(x)
        &
        np.isfinite(shap)
        &
        np.isfinite(weights)
        &
        (weights > 0)
        &
        np.isin(
            x,
            [
                0,
                1,
            ]
        )
    )


    female = (
        valid
        &
        (x == 0)
    )


    male = (
        valid
        &
        (x == 1)
    )


    female_mean = (
        direction_weighted_mean(
            shap[
                female
            ],
            weights[
                female
            ],
        )
    )


    male_mean = (
        direction_weighted_mean(
            shap[
                male
            ],
            weights[
                male
            ],
        )
    )


    contrast = float(
        male_mean
        -
        female_mean
    )


    mean_abs_shap = (
        direction_weighted_mean(
            np.abs(
                shap[
                    valid
                ]
            ),
            weights[
                valid
            ],
        )
    )


    if (
        np.isfinite(
            mean_abs_shap
        )
        and
        mean_abs_shap > 0
    ):

        normalized_contrast = float(
            contrast
            /
            mean_abs_shap
        )


    else:

        normalized_contrast = (
            np.nan
        )


    return {
        "Country":
            country,

        "Outcome":
            outcome,

        "Feature":
            feature,

        "ContrastDefinition":
            "Male minus female mean SHAP",

        "N_observed":
            int(
                valid.sum()
            ),

        "Q25":
            np.nan,

        "Q75":
            np.nan,

        "MeanSHAP_Low":
            female_mean,

        "MeanSHAP_High":
            male_mean,

        "DirectionalContrast":
            contrast,

        "MeanAbsSHAP":
            mean_abs_shap,

        "NormalizedContrast":
            normalized_contrast,

        "Direction":
            (
                "Male -> higher predicted bullying contribution"
                if contrast > 0
                else
                "Male -> lower predicted bullying contribution"
                if contrast < 0
                else
                "No contrast"
            ),
    }


# ============================================================
# 7. Run all 374 primary models
# ============================================================

direction_rows = []

qa_rows = []

error_rows = []


EXPECTED_MODELS = (
    len(
        PRIMARY_BEING_SYSTEMS
    )
    +
    4
    *
    len(
        PRIMARY_SUBTYPE_SYSTEMS
    )
)


counter = 0

start_time = time.time()


for outcome in DIRECTION_OUTCOMES:

    if outcome == "BEINGBULLIED":

        systems = (
            PRIMARY_BEING_SYSTEMS
        )


    else:

        systems = (
            PRIMARY_SUBTYPE_SYSTEMS
        )


    for country in systems:

        counter += 1


        print(
            f"[{counter}/{EXPECTED_MODELS}] "
            f"{country} — {outcome}",
            end=" ... ",
        )


        try:

            result = (
                run_direction_oof_shap(
                    df=analysis_frames[
                        country
                    ],
                    country=country,
                    outcome=outcome,
                )
            )


            # ------------------------------------------------
            # Continuous focal predictors
            # ------------------------------------------------

            for feature in (
                FOCAL_CONTINUOUS
            ):

                row = (
                    summarize_continuous_direction(
                        result=result,
                        country=country,
                        outcome=outcome,
                        feature=feature,
                    )
                )


                direction_rows.append(
                    row
                )


            # ------------------------------------------------
            # Gender
            # ------------------------------------------------

            direction_rows.append(
                summarize_gender_direction(
                    result=result,
                    country=country,
                    outcome=outcome,
                )
            )


            qa_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "MaxAdditivityError":
                        result[
                            "max_additivity_error"
                        ],
                }
            )


            print(
                "OK | "
                f"max error="
                f"{result['max_additivity_error']:.2e}"
            )


            del result


            if (
                counter
                %
                25
                ==
                0
            ):

                gc.collect()


        except Exception as e:

            error_rows.append(
                {
                    "Country":
                        country,

                    "Outcome":
                        outcome,

                    "ErrorType":
                        type(e).__name__,

                    "Error":
                        str(e),
                }
            )


            print(
                f"ERROR | "
                f"{type(e).__name__}: "
                f"{e}"
            )


# ============================================================
# 8. Combine and save
# ============================================================

direction_results = pd.DataFrame(
    direction_rows
)


direction_qa = pd.DataFrame(
    qa_rows
)


direction_errors = pd.DataFrame(
    error_rows
)


direction_results.to_csv(
    DIR_DIR
    / "system_directional_shap.csv",
    index=False,
)


direction_qa.to_csv(
    DIR_DIR
    / "directional_shap_qa.csv",
    index=False,
)


direction_errors.to_csv(
    DIR_DIR
    / "directional_errors.csv",
    index=False,
)


# ============================================================
# 9. Completion QA
# ============================================================

runtime_minutes = (
    time.time()
    -
    start_time
) / 60


expected_direction_rows = (
    EXPECTED_MODELS
    *
    len(
        FOCAL_FEATURES
    )
)


print("\n")
print("=" * 110)
print("DIRECTIONAL SHAP — COMPLETION")
print("=" * 110)


print(
    "Expected models:",
    EXPECTED_MODELS
)

print(
    "Successful models:",
    len(
        direction_qa
    )
)

print(
    "Errors:",
    len(
        direction_errors
    )
)

print(
    "Expected directional rows:",
    expected_direction_rows
)

print(
    "Observed directional rows:",
    len(
        direction_results
    )
)

print(
    "Maximum additivity error:",
    direction_qa[
        "MaxAdditivityError"
    ].max()
)

print(
    f"Runtime: "
    f"{runtime_minutes:.2f} minutes"
)


# ============================================================
# 10. Cross-system directional consistency
# ============================================================

summary_rows = []


for outcome in DIRECTION_OUTCOMES:

    for feature in FOCAL_FEATURES:

        temp = (
            direction_results[
                (
                    direction_results[
                        "Outcome"
                    ]
                    == outcome
                )
                &
                (
                    direction_results[
                        "Feature"
                    ]
                    == feature
                )
            ]
            .copy()
        )


        contrast = (
            temp[
                "DirectionalContrast"
            ]
            .dropna()
        )


        normalized = (
            temp[
                "NormalizedContrast"
            ]
            .dropna()
        )


        n_positive = int(
            (
                contrast > 0
            ).sum()
        )


        n_negative = int(
            (
                contrast < 0
            ).sum()
        )


        n_total = len(
            contrast
        )


        if n_total > 0:

            positive_rate = (
                n_positive
                /
                n_total
            )


            negative_rate = (
                n_negative
                /
                n_total
            )


            dominant_rate = max(
                positive_rate,
                negative_rate,
            )


        else:

            positive_rate = (
                np.nan
            )

            negative_rate = (
                np.nan
            )

            dominant_rate = (
                np.nan
            )


        dominant_direction = (
            "Positive"
            if positive_rate > negative_rate
            else
            "Negative"
            if negative_rate > positive_rate
            else
            "Tie"
        )


        summary_rows.append(
            {
                "Outcome":
                    outcome,

                "Feature":
                    feature,

                "N_systems":
                    n_total,

                "N_positive":
                    n_positive,

                "N_negative":
                    n_negative,

                "Pct_positive":
                    100
                    *
                    positive_rate,

                "Pct_negative":
                    100
                    *
                    negative_rate,

                "DominantDirection":
                    dominant_direction,

                "DirectionConsistency":
                    dominant_rate,

                "MedianContrast":
                    contrast.median(),

                "MeanContrast":
                    contrast.mean(),

                "MedianNormalizedContrast":
                    normalized.median(),
            }
        )


direction_summary = pd.DataFrame(
    summary_rows
)


direction_summary.to_csv(
    DIR_DIR
    / "international_directional_summary.csv",
    index=False,
)


# ============================================================
# 11. Print main result
# ============================================================

print("\n")
print("=" * 110)
print("INTERNATIONAL DIRECTIONAL CONSISTENCY")
print("=" * 110)


print(
    direction_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 12. Show exceptions to dominant direction
# ============================================================

print("\n")
print("=" * 110)
print("DIRECTIONAL EXCEPTIONS")
print("=" * 110)


for outcome in DIRECTION_OUTCOMES:

    print("\n")
    print("-" * 100)
    print(outcome)
    print("-" * 100)


    for feature in FOCAL_FEATURES:

        row = (
            direction_summary[
                (
                    direction_summary[
                        "Outcome"
                    ]
                    == outcome
                )
                &
                (
                    direction_summary[
                        "Feature"
                    ]
                    == feature
                )
            ]
            .iloc[0]
        )


        dominant = (
            row[
                "DominantDirection"
            ]
        )


        temp = (
            direction_results[
                (
                    direction_results[
                        "Outcome"
                    ]
                    == outcome
                )
                &
                (
                    direction_results[
                        "Feature"
                    ]
                    == feature
                )
            ]
            .copy()
        )


        if dominant == "Negative":

            exceptions = (
                temp[
                    temp[
                        "DirectionalContrast"
                    ]
                    >= 0
                ]
            )


        elif dominant == "Positive":

            exceptions = (
                temp[
                    temp[
                        "DirectionalContrast"
                    ]
                    <= 0
                ]
            )


        else:

            exceptions = (
                temp.iloc[
                    0:0
                ]
            )


        print(
            f"\n{feature}: "
            f"dominant={dominant}, "
            f"exceptions="
            f"{len(exceptions)}"
        )


        if len(
            exceptions
        ) > 0:

            print(
                exceptions[
                    [
                        "Country",
                        "DirectionalContrast",
                        "NormalizedContrast",
                    ]
                ]
                .sort_values(
                    "DirectionalContrast"
                )
                .round(
                    4
                )
                .to_string(
                    index=False
                )
            )

[1/374] ALB — BEINGBULLIED ... OK | max error=1.67e-06
[2/374] ARE — BEINGBULLIED ... OK | max error=2.86e-06
[3/374] ARG — BEINGBULLIED ... OK | max error=2.62e-06
[4/374] AUS — BEINGBULLIED ... OK | max error=2.62e-06
[5/374] AUT — BEINGBULLIED ... OK | max error=1.79e-06
[6/374] BEL — BEINGBULLIED ... OK | max error=2.15e-06
[7/374] BGR — BEINGBULLIED ... OK | max error=2.38e-06
[8/374] BIH — BEINGBULLIED ... OK | max error=1.91e-06
[9/374] BLR — BEINGBULLIED ... OK | max error=1.55e-06
[10/374] BRA — BEINGBULLIED ... OK | max error=1.79e-06
[11/374] BRN — BEINGBULLIED ... OK | max error=2.38e-06
[12/374] CAN — BEINGBULLIED ... OK | max error=2.26e-06
[13/374] CHE — BEINGBULLIED ... OK | max error=2.15e-06
[14/374] CHL — BEINGBULLIED ... OK | max error=1.31e-06
[15/374] COL — BEINGBULLIED ... OK | max error=1.43e-06
[16/374] CRI — BEINGBULLIED ... OK | max error=1.55e-06
[17/374] CZE — BEINGBULLIED ... OK | max error=1.67e-06
[18/374] DEU — BEINGBULLIED ... OK | max error=1.67e-06
[

In [ ]:
# ============================================================
# FINAL REPRODUCIBILITY + VERSION MANIFEST
#
# PURPOSE
# -------
# Freeze the computational and methodological state of the
# completed PISA 2018 bullying analysis.
#
# CREATES
# -------
# ../results/reproducibility/
#
#   reproducibility_manifest.json
#   package_versions.csv
#   pip_freeze.txt
#   git_status.txt
#   git_diff.patch
#   result_file_hashes.csv
#   source_file_hashes.csv
#   expected_outputs_qa.csv
#
# NO MODEL FITTING.
# NO RESULT RECOMPUTATION.
# ============================================================


from pathlib import Path
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

RESULT_ROOT = Path(
    "../results"
)

REPRO_DIR = (
    RESULT_ROOT
    / "reproducibility"
)

REPRO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Optional repository folders.
# Only existing folders will be hashed.

SOURCE_DIR_CANDIDATES = [
    Path("../src"),
    Path("../scripts"),
    Path("../config"),
    Path("../notebooks"),
]


# ============================================================
# 2. Frozen study constants
# ============================================================

PRIMARY_PREDICTORS = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


PRIMARY_OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


BINARY_OUTCOMES = [
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


REFERENCE_XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 5,
    "reg_lambda": 4.0,
    "reg_alpha": 0.6,
    "gamma": 0.0,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}


GLOBAL_RANDOM_SEED = 42

N_CV_FOLDS = 5

N_PV_RUNS = 10

N_REPLICATE_WEIGHTS = 80

FAY_FACTOR = 0.5

N_ARCHITECTURE_PERMUTATIONS = 10_000


# ============================================================
# 3. Package versions
# ============================================================

PACKAGES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "xgboost",
    "pyreadstat",
    "matplotlib",
    "joblib",
    "threadpoolctl",
]


package_rows = []


for package in PACKAGES:

    try:

        version = (
            importlib.metadata.version(
                package
            )
        )

    except (
        importlib.metadata.PackageNotFoundError
    ):

        version = None


    package_rows.append(
        {
            "Package":
                package,

            "Version":
                version,
        }
    )


package_versions = pd.DataFrame(
    package_rows
)


package_versions.to_csv(
    REPRO_DIR
    / "package_versions.csv",
    index=False,
)


# ============================================================
# 4. Full pip environment
# ============================================================

def safe_subprocess(
    command,
):

    try:

        return subprocess.check_output(
            command,
            text=True,
            stderr=subprocess.STDOUT,
        ).strip()

    except Exception as e:

        return (
            f"COMMAND FAILED: "
            f"{type(e).__name__}: {e}"
        )


pip_freeze = safe_subprocess(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ]
)


(
    REPRO_DIR
    / "pip_freeze.txt"
).write_text(
    pip_freeze
    +
    "\n",
    encoding="utf-8",
)


# ============================================================
# 5. Git metadata
# ============================================================

def git_command(
    args,
):

    try:

        return subprocess.check_output(
            [
                "git",
                *args,
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()

    except Exception:

        return None


git_root = git_command(
    [
        "rev-parse",
        "--show-toplevel",
    ]
)


git_commit = git_command(
    [
        "rev-parse",
        "HEAD",
    ]
)


git_branch = git_command(
    [
        "rev-parse",
        "--abbrev-ref",
        "HEAD",
    ]
)


git_status = git_command(
    [
        "status",
        "--porcelain",
    ]
)


git_clean = (
    git_status == ""
    if git_status is not None
    else None
)


git_diff = git_command(
    [
        "diff",
        "--no-ext-diff",
    ]
)


(
    REPRO_DIR
    / "git_status.txt"
).write_text(
    (
        git_status
        if git_status is not None
        else "Not inside a Git repository."
    )
    +
    "\n",
    encoding="utf-8",
)


(
    REPRO_DIR
    / "git_diff.patch"
).write_text(
    (
        git_diff
        if git_diff is not None
        else "No Git diff available."
    )
    +
    "\n",
    encoding="utf-8",
)


# ============================================================
# 6. Hash helper
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ============================================================
# 7. Hash analytical result files
#
# Exclude reproducibility directory itself so repeated runs
# do not create recursive/self-referential hashes.
# ============================================================

RESULT_EXTENSIONS = {
    ".csv",
    ".json",
    ".txt",
    ".pkl",
    ".pickle",
    ".parquet",
}


result_hash_rows = []


if RESULT_ROOT.exists():

    for path in sorted(
        RESULT_ROOT.rglob("*")
    ):

        if not path.is_file():

            continue


        if REPRO_DIR in path.parents:

            continue


        if (
            path.suffix.lower()
            not in RESULT_EXTENSIONS
        ):

            continue


        try:

            result_hash_rows.append(
                {
                    "File":
                        str(path),

                    "RelativeFile":
                        str(
                            path.relative_to(
                                RESULT_ROOT
                            )
                        ),

                    "Size_bytes":
                        path.stat().st_size,

                    "SHA256":
                        sha256_file(
                            path
                        ),
                }
            )


        except Exception as e:

            result_hash_rows.append(
                {
                    "File":
                        str(path),

                    "RelativeFile":
                        str(path),

                    "Size_bytes":
                        np.nan,

                    "SHA256":
                        (
                            f"ERROR: "
                            f"{type(e).__name__}: {e}"
                        ),
                }
            )


result_hashes = pd.DataFrame(
    result_hash_rows
)


result_hashes.to_csv(
    REPRO_DIR
    / "result_file_hashes.csv",
    index=False,
)


# ============================================================
# 8. Hash source / script / config files
# ============================================================

SOURCE_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".ini",
    ".cfg",
    ".md",
    ".txt",
}


source_hash_rows = []


for source_root in SOURCE_DIR_CANDIDATES:

    if not source_root.exists():

        continue


    for path in sorted(
        source_root.rglob("*")
    ):

        if not path.is_file():

            continue


        if (
            path.suffix.lower()
            not in SOURCE_EXTENSIONS
        ):

            continue


        try:

            source_hash_rows.append(
                {
                    "Root":
                        str(source_root),

                    "File":
                        str(path),

                    "Size_bytes":
                        path.stat().st_size,

                    "SHA256":
                        sha256_file(
                            path
                        ),
                }
            )


        except Exception as e:

            source_hash_rows.append(
                {
                    "Root":
                        str(source_root),

                    "File":
                        str(path),

                    "Size_bytes":
                        np.nan,

                    "SHA256":
                        (
                            f"ERROR: "
                            f"{type(e).__name__}: {e}"
                        ),
                }
            )


source_hashes = pd.DataFrame(
    source_hash_rows,
    columns=[
        "Root",
        "File",
        "Size_bytes",
        "SHA256",
    ],
)


source_hashes.to_csv(
    REPRO_DIR
    / "source_file_hashes.csv",
    index=False,
)


# ============================================================
# 9. Expected frozen output files
#
# This catches accidental missing stages before we declare
# the project reproducible.
# ============================================================

EXPECTED_OUTPUTS = [

    # --------------------------------------------------------
    # Primary models
    # --------------------------------------------------------

    RESULT_ROOT
    / "primary_models"
    / "overall_performance.csv",


    # --------------------------------------------------------
    # Primary attribution
    # --------------------------------------------------------

    RESULT_ROOT
    / "primary_attribution"
    / "oof_shap_rankings.csv",


    # --------------------------------------------------------
    # Achievement plausible values
    # --------------------------------------------------------

    RESULT_ROOT
    / "achievement_pv"
    / "pv_model_rankings.csv",

    RESULT_ROOT
    / "achievement_pv"
    / "pv_model_performance.csv",

    RESULT_ROOT
    / "achievement_pv"
    / "pv_model_qa.csv",

    RESULT_ROOT
    / "achievement_pv"
    / "system_conceptual_predictor_summary.csv",

    RESULT_ROOT
    / "achievement_pv"
    / "primary_vs_augmented_architecture_summary.csv",


    # --------------------------------------------------------
    # Missingness sensitivity
    # --------------------------------------------------------

    RESULT_ROOT
    / "missingness40_sensitivity"
    / "primary_vs_40pct_architecture_summary.csv",

    RESULT_ROOT
    / "missingness40_sensitivity"
    / "lowmissing_system_restriction_summary.csv",

    RESULT_ROOT
    / "missingness40_sensitivity"
    / "missingness_vs_importance_correlations.csv",


    # --------------------------------------------------------
    # Survey-design uncertainty
    # --------------------------------------------------------

    RESULT_ROOT
    / "design_uncertainty"
    / "outcome_replicate_weight_uncertainty.csv",

    RESULT_ROOT
    / "design_uncertainty"
    / "outcome_uncertainty_summary.csv",


    # --------------------------------------------------------
    # Conventional benchmark
    # --------------------------------------------------------

    RESULT_ROOT
    / "conventional_benchmark"
    / "benchmark_overall_performance.csv",

    RESULT_ROOT
    / "conventional_benchmark"
    / "xgboost_vs_conventional.csv",

    RESULT_ROOT
    / "conventional_benchmark"
    / "xgboost_vs_conventional_summary.csv",


    # --------------------------------------------------------
    # Architecture permutation null
    # --------------------------------------------------------

    RESULT_ROOT
    / "architecture_null"
    / "architecture_permutation_null_summary.csv",

    RESULT_ROOT
    / "architecture_null"
    / "architecture_null_compact.csv",


    # --------------------------------------------------------
    # Directional analysis
    # --------------------------------------------------------

    RESULT_ROOT
    / "directional_analysis"
    / "system_directional_shap.csv",

    RESULT_ROOT
    / "directional_analysis"
    / "directional_shap_qa.csv",

    RESULT_ROOT
    / "directional_analysis"
    / "international_directional_summary.csv",
]


output_qa_rows = []


for path in EXPECTED_OUTPUTS:

    output_qa_rows.append(
        {
            "File":
                str(path),

            "Exists":
                path.exists(),

            "Size_bytes":
                (
                    path.stat().st_size
                    if path.exists()
                    else np.nan
                ),
        }
    )


expected_output_qa = pd.DataFrame(
    output_qa_rows
)


expected_output_qa.to_csv(
    REPRO_DIR
    / "expected_outputs_qa.csv",
    index=False,
)


missing_expected_outputs = (
    expected_output_qa[
        ~expected_output_qa[
            "Exists"
        ]
    ]
)


# ============================================================
# 10. Frozen methodology manifest
# ============================================================

manifest = {

    # ========================================================
    # Manifest
    # ========================================================

    "manifest": {

        "generated_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "status":
            "ANALYTICAL METHODOLOGY LOCKED",

        "purpose":
            (
                "Reproducibility record for the "
                "PISA 2018 cross-system bullying "
                "predictor analysis."
            ),
    },


    # ========================================================
    # Environment
    # ========================================================

    "environment": {

        "python_version":
            platform.python_version(),

        "python_full_version":
            sys.version,

        "python_executable":
            sys.executable,

        "platform":
            platform.platform(),

        "operating_system":
            platform.system(),

        "os_release":
            platform.release(),

        "machine":
            platform.machine(),

        "processor":
            platform.processor(),

        "cpu_count":
            os.cpu_count(),
    },


    # ========================================================
    # Git
    # ========================================================

    "git": {

        "repository_root":
            git_root,

        "branch":
            git_branch,

        "commit":
            git_commit,

        "working_tree_clean":
            git_clean,

        "status_saved_to":
            "git_status.txt",

        "diff_saved_to":
            "git_diff.patch",
    },


    # ========================================================
    # Dataset
    # ========================================================

    "dataset": {

        "dataset":
            (
                "Programme for International "
                "Student Assessment (PISA) 2018"
            ),

        "student_file_raw_rows":
            612004,

        "student_file_raw_columns":
            1119,

        "school_file_raw_rows":
            21903,

        "school_file_raw_columns":
            197,

        "raw_CNT_codes":
            80,

        "official_PISA_2018_participants":
            79,

        "main_analysis_system_universe":
            78,

        "excluded_subregional_codes": [
            "QMR",
            "QRT",
        ],

        "notes": [

            (
                "QMR and QRT were excluded because "
                "the national Russian Federation "
                "entry was retained."
            ),

            (
                "QAZ was retained as the "
                "Baku/Azerbaijan PISA entity "
                "represented in the supplied data."
            ),

            (
                "QCI was retained as the "
                "B-S-J-Z (China) PISA entity "
                "represented in the supplied data."
            ),

            (
                "CYP was not present in the "
                "supplied student dataset and "
                "was not artificially restored."
            ),

            (
                "VNM contained the achievement "
                "plausible-value columns but no "
                "usable achievement PV observations; "
                "it was therefore retained in "
                "non-achievement analyses and "
                "excluded only from achievement "
                "sensitivity models."
            ),
        ],
    },


    # ========================================================
    # Merge
    # ========================================================

    "student_school_merge": {

        "keys": [
            "CNT",
            "CNTSCHID",
        ],

        "relationship":
            "many-to-one student-to-school",

        "join":
            "left",

        "duplicate_school_key_check":
            True,

        "pandas_merge_validation":
            "many_to_one",

        "student_row_preservation_asserted":
            True,

        "school_match_indicator_preserved":
            True,
    },


    # ========================================================
    # Outcomes
    # ========================================================

    "outcomes": {

        "BEINGBULLIED": {

            "type":
                "continuous",

            "definition":
                (
                    "Official supplied PISA "
                    "BEINGBULLIED bullying index; "
                    "retained unchanged."
                ),

            "primary_systems":
                74,

            "special_missing_codes": [
                95,
                97,
                98,
                99,
            ],
        },


        "BULLY_VERBAL": {

            "type":
                "binary",

            "source":
                "ST038Q04NA",

            "behavior":
                "Made fun of.",

            "primary_systems":
                75,
        },


        "BULLY_THREAT": {

            "type":
                "binary",

            "source":
                "ST038Q05NA",

            "behavior":
                "Threatened.",

            "note":
                (
                    "Retained as separate "
                    "behavior-specific subtype; "
                    "physical modality not assumed."
                ),

            "primary_systems":
                75,
        },


        "BULLY_PHYSICAL": {

            "type":
                "binary",

            "sources": [
                "ST038Q06NA",
                "ST038Q07NA",
            ],

            "behavior":
                (
                    "Belongings taken/destroyed "
                    "OR hit/pushed."
                ),

            "combination_rule":
                (
                    "Any observed positive -> 1; "
                    "both observed and both negative -> 0; "
                    "otherwise missing."
                ),

            "primary_systems":
                75,
        },


        "BULLY_RELATIONAL": {

            "type":
                "binary",

            "source":
                "ST038Q08NA",

            "behavior":
                "Nasty rumours.",

            "primary_systems":
                75,
        },


        "ST038_binary_recoding": {

            "codes_1_2":
                0,

            "codes_3_4":
                1,

            "special_missing_codes_5_to_9":
                "NaN",
        },


        "ST038Q03NA_used_in_behavior_specific_outcomes":
            False,
    },


    # ========================================================
    # Predictors
    # ========================================================

    "predictors": {

        "primary_predictors":
            PRIMARY_PREDICTORS,

        "n_primary_predictors":
            14,

        "selection_rule":
            (
                "Common predictor set usable in "
                "every included primary system "
                "with maximum system-level "
                "missingness <=50%."
            ),

        "primary_missingness_threshold_pct":
            50,

        "scaling_for_xgboost":
            False,

        "imputation_for_xgboost":
            False,

        "xgboost_native_missing_value_handling":
            True,

        "gender_variable":
            "ST004D01T",

        "gender_recode": {

            "female":
                0,

            "male":
                1,

            "original_code_1":
                0,

            "original_code_2":
                1,
        },

        "school_level_predictor_in_primary_set":
            "EDUSHORT",
    },


    # ========================================================
    # Survey design
    # ========================================================

    "survey_design": {

        "final_student_weight":
            "W_FSTUWT",

        "replicate_weights":
            "W_FSTURWT1 ... W_FSTURWT80",

        "n_replicate_weights":
            80,

        "fay_factor":
            0.5,

        "fay_variance_multiplier":
            0.05,

        "training_weight_normalization":
            (
                "Within each training fold, "
                "positive finite W_FSTUWT values "
                "were divided by their mean."
            ),

        "model_training_weighted":
            True,

        "performance_metrics_weighted":
            True,

        "shap_aggregation_weighted":
            True,

        "replicate_weight_use":
            (
                "Design-based standard errors and "
                "95% confidence intervals for "
                "descriptive outcome estimates."
            ),

        "replicate_weight_model_refitting":
            False,

        "model_uncertainty_policy":
            (
                "Cross-validated model performance "
                "and SHAP attribution are not "
                "presented with design-based "
                "replicate-weight confidence intervals, "
                "because valid design-based uncertainty "
                "for learned models would require "
                "refitting the learning procedure "
                "under replicate weights."
            ),
    },


    # ========================================================
    # Cross-validation
    # ========================================================

    "cross_validation": {

        "folds":
            5,

        "group_variable":
            "CNTSCHID",

        "regression_splitter":
            "GroupKFold",

        "classification_splitter":
            "StratifiedGroupKFold",

        "classification_shuffle":
            True,

        "classification_random_state":
            42,

        "held_out_unit":
            "schools",

        "performance_evaluated_out_of_fold":
            True,

        "shap_evaluated_out_of_fold":
            True,
    },


    # ========================================================
    # XGBoost
    # ========================================================

    "xgboost": {

        "algorithm":
            "XGBoost",

        "reference_parameters":
            REFERENCE_XGB_PARAMS,

        "regression_objective":
            "reg:squarederror",

        "classification_objective":
            "binary:logistic",

        "classification_eval_metric":
            "logloss",

        "class_balancing":
            False,

        "scale_pos_weight":
            False,

        "reference_model_policy":
            (
                "A common reference specification "
                "was used across education systems "
                "and outcomes; no per-system "
                "hyperparameter optimization."
            ),

        "parameter_interpretation":
            (
                "Reference specification rather "
                "than retrospectively selected "
                "best-performing specification."
            ),
    },


    # ========================================================
    # Performance
    # ========================================================

    "performance": {

        "BEINGBULLIED_metrics": [
            "survey-weighted RMSE",
            "survey-weighted MAE",
            "survey-weighted R2",
        ],

        "binary_metrics": [
            "survey-weighted ROC-AUC",
            "survey-weighted PR-AUC",
        ],

        "regression_baseline":
            (
                "Weighted training-fold "
                "outcome mean."
            ),

        "classification_ROC_baseline":
            0.5,

        "classification_PR_baseline":
            (
                "Survey-weighted outcome prevalence."
            ),

        "fold_stability_evaluated":
            True,
    },


    # ========================================================
    # SHAP
    # ========================================================

    "shap_attribution": {

        "method":
            "exact native XGBoost TreeSHAP",

        "implementation":
            (
                "Booster.predict("
                "DMatrix, "
                "pred_contribs=True, "
                "approx_contribs=False)"
            ),

        "data_used":
            "held-out OOF observations only",

        "system_level_importance":
            (
                "W_FSTUWT-weighted mean "
                "absolute SHAP."
            ),

        "normalization":
            (
                "Feature importance shares "
                "normalized to sum to 1 within "
                "each education-system/outcome model."
            ),

        "fold_rank_stability_evaluated":
            True,

        "signed_SHAP_primary_ranking":
            False,

        "causal_interpretation":
            False,
    },


    # ========================================================
    # Cross-system architecture
    # ========================================================

    "cross_system_architecture": {

        "profile":
            (
                "Complete normalized SHAP "
                "importance vector."
            ),

        "primary_similarity":
            "cosine similarity",

        "rank_similarity":
            "Spearman rank correlation",

        "top_k_set_similarity": [
            "Top-5 Jaccard",
            "Top-7 Jaccard",
        ],

        "international_consensus_profile":
            (
                "Equal-weight mean of normalized "
                "system-level SHAP importance shares."
            ),

        "system_heterogeneity":
            (
                "Continuous deviation from "
                "international consensus."
            ),

        "hard_clustering":
            False,

        "reason_clustering_omitted":
            (
                "Results were better characterized "
                "by a conserved international "
                "architecture with continuous "
                "system-specific reweighting than "
                "by clearly separated discrete groups."
            ),
    },


    # ========================================================
    # Permutation null
    # ========================================================

    "architecture_permutation_null": {

        "performed":
            True,

        "n_permutations":
            10000,

        "random_seed":
            42,

        "null_unit":
            "education-system attribution vector",

        "null_operation":
            (
                "Independently permute predictor "
                "labels within each system while "
                "preserving that system's exact "
                "set of normalized SHAP shares."
            ),

        "preserves": [
            "number of predictors",
            "system-specific attribution concentration",
            "system-specific attribution magnitudes",
            "system-vector norm",
        ],

        "destroys":
            (
                "cross-system agreement about "
                "predictor identity"
            ),

        "tested_profiles": [
            "Full 14-predictor architecture",
            "13-predictor architecture after BELONG removal",
        ],

        "p_value_correction":
            (
                "(1 + number of null statistics "
                ">= observed) / "
                "(number of permutations + 1)"
            ),
    },


    # ========================================================
    # Achievement PV analysis
    # ========================================================

    "achievement_plausible_values": {

        "role":
            "supplementary sensitivity analysis",

        "n_pv_runs":
            10,

        "per_run_achievement_predictors":
            (
                "PViREAD, PViMATH, PViSCIE "
                "for i=1,...,10"
            ),

        "student_level_PV_averaging":
            False,

        "matched_PV_rule":
            (
                "PV1READ/PV1MATH/PV1SCIE together, "
                "... through "
                "PV10READ/PV10MATH/PV10SCIE together."
            ),

        "conceptual_domains": [
            "READ_ACHIEVEMENT",
            "MATH_ACHIEVEMENT",
            "SCIENCE_ACHIEVEMENT",
        ],

        "predictors_per_augmented_model":
            17,

        "BEINGBULLIED_systems":
            73,

        "subtype_systems":
            74,

        "excluded_for_achievement":
            [
                "VNM",
            ],

        "expected_models":
            3690,

        "completed_models":
            3690,

        "PV_variability_interpretation":
            (
                "Across-PV variability was used as "
                "plausible-value sensitivity, "
                "not as a sampling standard error."
            ),
    },


    # ========================================================
    # Missingness analyses
    # ========================================================

    "missingness_robustness": {

        "primary_threshold_pct":
            50,

        "strict_predictor_threshold_pct":
            40,

        "strict_40pct_predictors_retained":
            12,

        "strict_40pct_predictors_removed": [
            "BELONG",
            "MASTGOAL",
        ],

        "strict_40pct_role":
            (
                "Extreme predictor-universe "
                "sensitivity only; not used to "
                "redefine the primary analysis."
            ),

        "preferred_low_missingness_sensitivity":
            (
                "Keep all 14 frozen predictors "
                "and exclude systems where any "
                "primary predictor exceeds 40% "
                "missingness."
            ),

        "systems_excluded_under_40pct_system_rule": [
            "DOM",
            "PAN",
        ],

        "low_missingness_architecture_recomputed":
            False,

        "reason":
            (
                "The sensitivity summarized "
                "the already-frozen 14-predictor "
                "models after restricting systems; "
                "no refitting was required."
            ),

        "missingness_vs_importance_diagnostic":
            True,
    },


    # ========================================================
    # Outcome coverage
    # ========================================================

    "outcome_coverage_robustness": {

        "primary_policy":
            (
                "Systems with any valid outcome data "
                "were retained in the primary analysis."
            ),

        "sensitivity_threshold_pct":
            50,

        "performed":
            True,

        "notable_low_coverage_systems": [
            "DOM",
            "PAN",
            "MAR",
            "PER",
            "DEU",
        ],
    },


    # ========================================================
    # BELONG sensitivity
    # ========================================================

    "BELONG_removed_sensitivity": {

        "performed":
            True,

        "purpose":
            (
                "Determine whether international "
                "architectural similarity was "
                "solely caused by BELONG dominance."
            ),

        "remaining_predictors":
            13,

        "renormalized_after_removal":
            True,
    },


    # ========================================================
    # Hyperparameter robustness
    # ========================================================

    "hyperparameter_robustness": {

        "performed":
            True,

        "alternative_specs": [
            "shallower",
            "deeper",
            "more_regularized",
        ],

        "reference_model_replaced_after_robustness":
            False,
    },


    # ========================================================
    # Conventional benchmark
    # ========================================================

    "conventional_benchmark": {

        "performed":
            True,

        "same_primary_predictors":
            True,

        "same_students_and_outcomes":
            True,

        "same_school_grouped_CV":
            True,

        "same_final_student_weight":
            "W_FSTUWT",

        "BEINGBULLIED_model":
            "weighted multiple linear regression",

        "binary_model":
            (
                "weighted unregularized "
                "additive logistic regression"
            ),

        "median_imputation":
            (
                "fit within each training fold only"
            ),

        "standardization":
            (
                "fit within each training fold only"
            ),

        "primary_interpretation":
            (
                "XGBoost was not assumed or shown "
                "to universally outperform "
                "conventional additive models. "
                "Its role is a flexible framework "
                "for nonlinear predictor-attribution "
                "profiling across systems."
            ),
    },


    # ========================================================
    # Directional SHAP
    # ========================================================

    "directional_SHAP": {

        "performed":
            True,

        "focal_features": [
            "BELONG",
            "DISCLIMA",
            "ST004D01T",
        ],

        "continuous_feature_contrast":
            (
                "W_FSTUWT-weighted mean signed SHAP "
                "among observations in the upper "
                "weighted quartile minus the lower "
                "weighted quartile of the predictor."
            ),

        "gender_contrast":
            (
                "W_FSTUWT-weighted mean signed SHAP "
                "for male minus female."
            ),

        "gender_coding": {
            "female": 0,
            "male": 1,
        },

        "binary_SHAP_scale":
            "raw XGBoost margin / log-odds",

        "BEINGBULLIED_SHAP_scale":
            "continuous model-output scale",

        "directional_interpretation":
            (
                "model-based predictive association only"
            ),

        "causal_interpretation":
            False,

        "expected_models":
            374,

        "completed_models":
            374,

        "expected_directional_rows":
            1122,

        "completed_directional_rows":
            1122,
    },


    # ========================================================
    # Reproducibility
    # ========================================================

    "reproducibility": {

        "global_random_seed":
            GLOBAL_RANDOM_SEED,

        "package_versions_saved":
            True,

        "pip_freeze_saved":
            True,

        "git_commit_saved":
            True,

        "git_status_saved":
            True,

        "git_diff_saved":
            True,

        "analytical_result_hashes_saved":
            True,

        "source_file_hashes_saved":
            True,

        "expected_output_QA_saved":
            True,
    },
}


# ============================================================
# 11. Save manifest
# ============================================================

MANIFEST_PATH = (
    REPRO_DIR
    / "reproducibility_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 12. Internal manifest QA
# ============================================================

assert (
    len(
        PRIMARY_PREDICTORS
    )
    ==
    14
)


assert (
    len(
        PRIMARY_OUTCOMES
    )
    ==
    5
)


assert (
    manifest[
        "achievement_plausible_values"
    ][
        "expected_models"
    ]
    ==
    3690
)


assert (
    manifest[
        "directional_SHAP"
    ][
        "completed_models"
    ]
    ==
    374
)


assert np.isclose(
    manifest[
        "survey_design"
    ][
        "fay_variance_multiplier"
    ],
    0.05,
)


# ============================================================
# 13. Final report
# ============================================================

print("\n")
print("=" * 110)
print("REPRODUCIBILITY MANIFEST COMPLETE")
print("=" * 110)


print(
    "Manifest:",
    MANIFEST_PATH
)


print(
    "Packages recorded:",
    len(
        package_versions
    )
)


print(
    "Analytical result files hashed:",
    len(
        result_hashes
    )
)


print(
    "Source/config files hashed:",
    len(
        source_hashes
    )
)


print(
    "Expected frozen outputs:",
    len(
        expected_output_qa
    )
)


print(
    "Missing expected outputs:",
    len(
        missing_expected_outputs
    )
)


if len(
    missing_expected_outputs
) > 0:

    print("\nMISSING EXPECTED FILES")

    print(
        missing_expected_outputs[
            [
                "File",
                "Exists",
            ]
        ]
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 110)
print("GIT STATE")
print("=" * 110)


print(
    "Repository root:",
    git_root
)

print(
    "Branch:",
    git_branch
)

print(
    "Commit:",
    git_commit
)

print(
    "Working tree clean:",
    git_clean
)


print("\n")
print("=" * 110)
print("PACKAGE VERSIONS")
print("=" * 110)


print(
    package_versions
    .to_string(
        index=False
    )
)


print("\nPython:")
print(
    sys.version
)


print("\nPlatform:")
print(
    platform.platform()
)


print("\n")
print("=" * 110)

if len(
    missing_expected_outputs
) == 0:

    print(
        "METHODOLOGY STATUS: LOCKED"
    )

else:

    print(
        "METHODOLOGY STATUS: ANALYSIS COMPLETE, "
        "BUT EXPECTED OUTPUT FILE QA NEEDS REVIEW"
    )

print("=" * 110)



REPRODUCIBILITY MANIFEST COMPLETE
Manifest: ..\results\reproducibility\reproducibility_manifest.json
Packages recorded: 9
Analytical result files hashed: 687
Source/config files hashed: 10
Expected frozen outputs: 20
Missing expected outputs: 0


GIT STATE
Repository root: C:/Users/adhiv/OneDrive/Desktop/EduMindsResearch/EduMinds-Research
Branch: main
Commit: c9563a288e91c47ef0eb03976db511007a436218
Working tree clean: False


PACKAGE VERSIONS
      Package Version
        numpy   2.2.6
       pandas   2.2.3
        scipy  1.15.3
 scikit-learn   1.6.1
      xgboost   3.0.2
   pyreadstat   1.2.9
   matplotlib  3.10.3
       joblib   1.5.1
threadpoolctl   3.6.0

Python:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

Platform:
Windows-10-10.0.26200-SP0


METHODOLOGY STATUS: LOCKED


: 